# dropout_bench

Reordered working notebook derived from `main_markdown_revised.ipynb`.

This version only **reorders existing cells** and inserts a few **placeholder cells** for future refactoring. Existing P-cells keep their original content; new planning cells use section letters **A–E**.

# A. Foundation

This section establishes the benchmark foundation.

It covers:
- imports, configuration, and reproducibility
- dataset download and availability checks
- output reset and directory initialization
- DuckDB setup and source registration
- raw structural audit
- enrollment backbone construction
- survival-ready event and time construction

These steps create the canonical data and storage foundation used by all later sections.


## A0 - Dependency Check and Auto-Install

Run this step before A1 to ensure required Python libraries are available in the active environment.

In [ ]:
# ==============================================================
# A0 - Dependency Check and Auto-Install
# --------------------------------------------------------------
# Purpose:
#   Validate required third-party libraries and install missing
#   packages in the current Python environment before A1 imports.
# ==============================================================

import importlib.util
import subprocess
import sys

print("\n" + "=" * 70)
print("A0 - Dependency Check and Auto-Install")
print("=" * 70)

# ------------------------------
# 1) Required and optional packages
# ------------------------------
REQUIRED_PACKAGES = {
    "duckdb": "duckdb",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "lifelines": "lifelines",
    "torch": "torch",
}

OPTIONAL_PACKAGES = {
    "torchtuples": "torchtuples",
    "pycox": "pycox",
}


### Funcao: is_available

Definicao isolada de is_available para reutilizacao nas etapas seguintes.


In [ ]:


def is_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


### Funcao: install_package

Definicao isolada de install_package para reutilizacao nas etapas seguintes.


In [ ]:


def install_package(package_name: str) -> tuple[bool, str]:
    cmd = [sys.executable, "-m", "pip", "install", package_name]
    result = subprocess.run(cmd, capture_output=True, text=True)
    ok = result.returncode == 0
    details = result.stdout.strip() if ok else result.stderr.strip()
    return ok, details


In [ ]:


# ------------------------------
# 2) Install missing required packages
# ------------------------------
required_missing = [m for m in REQUIRED_PACKAGES if not is_available(m)]
optional_missing = [m for m in OPTIONAL_PACKAGES if not is_available(m)]

if required_missing:
    print("Missing required modules:", ", ".join(required_missing))
else:
    print("All required modules are already installed.")

install_failures = []

for module_name in required_missing:
    package_name = REQUIRED_PACKAGES[module_name]
    print(f"Installing required package: {package_name}")
    ok, details = install_package(package_name)
    if not ok:
        install_failures.append((module_name, package_name, details))

# ------------------------------
# 3) Try to install optional packages
# ------------------------------
if optional_missing:
    print("Missing optional modules:", ", ".join(optional_missing))

for module_name in optional_missing:
    package_name = OPTIONAL_PACKAGES[module_name]
    print(f"Installing optional package: {package_name}")
    ok, details = install_package(package_name)
    if not ok:
        print(f"Optional package failed to install: {package_name}")
        print(details.splitlines()[-1] if details else "No error details available.")

# ------------------------------
# 4) Final validation
# ------------------------------
still_missing_required = [m for m in REQUIRED_PACKAGES if not is_available(m)]

if still_missing_required:
    print("\nFailed required modules:", ", ".join(still_missing_required))
    if install_failures:
        print("\nInstall errors (required):")
        for module_name, package_name, details in install_failures:
            print(f"- {module_name} ({package_name})")
            print(details.splitlines()[-1] if details else "No error details available.")
    raise RuntimeError(
        "A0 could not install all required dependencies. "
        "Fix the errors above and rerun A0 before A1."
    )

print("\nDependency check completed. You can now run A1 safely.")


## A1 — Imports, Configuration, and Reproducibility


In [ ]:
# ==============================================================
# A1 — Imports, Configuration, and Reproducibility
# --------------------------------------------------------------
# Purpose:
#   Centralize imports, global configuration, random seeds,
#   project paths, output directories, and lightweight helpers
#   for reproducible execution.
#
# Outputs:
#   - Global configuration variables
#   - Reproducibility seeds
#   - Standard output directory tree
#   - Environment/version summary
# ==============================================================

# ------------------------------
# 1) Standard library
# ------------------------------
from pathlib import Path
from datetime import datetime
import os
import json
import random
import warnings
import platform
import requests
import zipfile
import duckdb
import shutil

# ------------------------------
# 2) Core scientific stack
# ------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------
# 3) Machine learning / survival
# ------------------------------
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from lifelines import CoxPHFitter, CoxTimeVaryingFitter
import lifelines

# ------------------------------
# 4) Deep learning
# ------------------------------
import torch
import torch.nn as nn

# Optional: enable only if installed / needed later
try:
    import torchtuples as tt
    TORCHTUPLES_AVAILABLE = True
except ImportError:
    TORCHTUPLES_AVAILABLE = False

try:
    from pycox.models import LogisticHazard, CoxPH
    PYCOX_AVAILABLE = True
    PYCOX_IMPORT_ERROR = None
except Exception as e:
    PYCOX_AVAILABLE = False
    PYCOX_IMPORT_ERROR = str(e)

# ------------------------------
# 5) Global display / warning settings
# ------------------------------
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)

# ------------------------------
# 6) Reproducibility
# ------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# More deterministic behavior (may reduce speed a bit)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ------------------------------
# 7) Project paths
# ------------------------------
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "content"          # same convention you already use
OUTPUT_DIR = PROJECT_ROOT / "outputs_benchmark_survival"

TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
METADATA_DIR = OUTPUT_DIR / "metadata"

for d in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, METADATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 8) Protocol configuration
# ------------------------------
CONFIG = {
    "seed": SEED,
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "tables_dir": str(TABLES_DIR),
    "figures_dir": str(FIGURES_DIR),
    "models_dir": str(MODELS_DIR),
    "metadata_dir": str(METADATA_DIR),
    "unit_of_analysis": "enrollment",
    "enrollment_key": ["id_student", "code_module", "code_presentation"],
    "time_granularity": "weekly",
    "event_definition": "Withdrawn with valid date_unregistration",
    "test_size": 0.30,
    "temporal_buckets_q": 4,
}


### Funcao: print_section

Definicao isolada de print_section para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 9) Lightweight utility helpers
# ------------------------------
def print_section(title: str) -> None:
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


### Funcao: save_json

Definicao isolada de save_json para reutilizacao nas etapas seguintes.


In [ ]:

def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


### Funcao: get_environment_summary

Definicao isolada de get_environment_summary para reutilizacao nas etapas seguintes.


In [ ]:

def get_environment_summary() -> dict:
    return {
        "timestamp": datetime.now().isoformat(),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "sklearn_version": sklearn_version,
        "lifelines_version": lifelines.__version__,
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "torchtuples_available": TORCHTUPLES_AVAILABLE,
        "pycox_available": PYCOX_AVAILABLE,
        "seed": SEED,
    }


In [ ]:

# ------------------------------
# 10) Persist metadata
# ------------------------------
env_summary = get_environment_summary()
save_json(CONFIG, METADATA_DIR / "config.json")
save_json(env_summary, METADATA_DIR / "environment_summary.json")

# ------------------------------
# 11) Execution summary
# ------------------------------
print_section("A1 — Imports, Configuration, and Reproducibility")
print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("SEED         :", SEED)
print("CUDA         :", torch.cuda.is_available())
print("PYCOX        :", PYCOX_AVAILABLE)
print("TORCHTUPLES  :", TORCHTUPLES_AVAILABLE)

if PYCOX_IMPORT_ERROR is not None:
    print("PYCOX ERROR  :", PYCOX_IMPORT_ERROR)

print("\nSaved metadata files:")
print("-", METADATA_DIR / "config.json")
print("-", METADATA_DIR / "environment_summary.json")


### A1.1 — Download and Extract OULAD Dataset

In [ ]:
# ==============================================================
# A1.1 — Download and Extract OULAD Dataset
# --------------------------------------------------------------
# Purpose:
#   Ensure the OULAD raw files are available under DATA_DIR.
#   If needed, download the official zip file and extract it
#   into the project content directory.
#
# Notes:
#   - Reuses the project paths defined in A1
#   - Avoids re-downloading if the zip already exists
#   - Avoids re-extracting if the required CSV files already exist
# ==============================================================

import requests
import zipfile

print_section("A1.1 — Download and Extract OULAD Dataset")

# ------------------------------
# 1) Dataset source and paths
# ------------------------------
BASE_PATH = Path.cwd()
DATASET_URL = "https://analyse.kmi.open.ac.uk/open-dataset/download"
ZIP_FILENAME = "anonymisedData.zip"
ZIP_PATH = PROJECT_ROOT / ZIP_FILENAME

EXPECTED_CSVS = [
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentVle.csv",
    "courses.csv",
    "vle.csv",
    "studentAssessment.csv",
    "assessments.csv",
]


### Funcao: list_missing_csvs

Definicao isolada de list_missing_csvs para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 2) Helper checks
# ------------------------------
def list_missing_csvs(data_dir: Path, expected_files: list[str]) -> list[str]:
    return [fname for fname in expected_files if not (data_dir / fname).exists()]


### Funcao: dataset_is_ready

Definicao isolada de dataset_is_ready para reutilizacao nas etapas seguintes.


In [ ]:

def dataset_is_ready(data_dir: Path, expected_files: list[str]) -> bool:
    return len(list_missing_csvs(data_dir, expected_files)) == 0


### Funcao: download_oulad_zip

Definicao isolada de download_oulad_zip para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 3) Download function
# ------------------------------
def download_oulad_zip(url: str, zip_path: Path) -> None:
    if zip_path.exists():
        print(f"Zip file already exists: {zip_path}")
        return

    print(f"Downloading OULAD zip from: {url}")
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0.0.0 Safari/537.36"
        )
    }

    try:
        with requests.get(url, headers=headers, stream=True, timeout=60) as response:
            response.raise_for_status()
            with open(zip_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        print(f"Download completed: {zip_path}")
    except Exception as e:
        if zip_path.exists():
            zip_path.unlink()
        raise RuntimeError(f"Download failed: {e}")


### Funcao: extract_oulad_zip

Definicao isolada de extract_oulad_zip para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 4) Extraction function
# ------------------------------
def extract_oulad_zip(zip_path: Path, extract_dir: Path) -> None:
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")

    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting zip into: {extract_dir}")
    try:
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_dir)
        print("Extraction completed successfully.")
    except zipfile.BadZipFile:
        raise RuntimeError(f"Bad zip file: {zip_path}")
    except Exception as e:
        raise RuntimeError(f"Extraction failed: {e}")


In [ ]:

# ------------------------------
# 5) Main execution logic
# ------------------------------
DATA_DIR.mkdir(parents=True, exist_ok=True)

missing_before = list_missing_csvs(DATA_DIR, EXPECTED_CSVS)

if dataset_is_ready(DATA_DIR, EXPECTED_CSVS):
    print("OULAD dataset is already available in DATA_DIR. Skipping download and extraction.")
else:
    print("Missing files before setup:")
    for fname in missing_before:
        print(f" - {fname}")

    download_oulad_zip(DATASET_URL, ZIP_PATH)
    extract_oulad_zip(ZIP_PATH, DATA_DIR)

# ------------------------------
# 6) Final validation
# ------------------------------
missing_after = list_missing_csvs(DATA_DIR, EXPECTED_CSVS)

print("\nFinal validation:")
print(f"DATA_DIR: {DATA_DIR}")

if missing_after:
    print("Some expected CSV files are still missing after extraction:")
    for fname in missing_after:
        print(f" - {fname}")
    raise FileNotFoundError("Dataset extraction finished, but required OULAD files are missing.")
else:
    print("All expected OULAD CSV files are available.")
    for fname in EXPECTED_CSVS:
        print(f" - {(DATA_DIR / fname).resolve()}")

# ------------------------------
# 7) Save dataset metadata
# ------------------------------
dataset_metadata = {
    "dataset_url": DATASET_URL,
    "zip_filename": ZIP_FILENAME,
    "zip_path": str(ZIP_PATH),
    "data_dir": str(DATA_DIR),
    "expected_csvs": EXPECTED_CSVS,
    "missing_before": missing_before,
    "missing_after": missing_after,
}

save_json(dataset_metadata, METADATA_DIR / "dataset_setup.json")
print("\nSaved:")
print("-", (METADATA_DIR / "dataset_setup.json").resolve())


### A1.2 — Reset Output Directory

In [ ]:
# ==============================================================
# A1.2 — Reset Output Directory
# --------------------------------------------------------------
# Purpose:
#   Remove the previous benchmark output directory so the notebook
#   can recreate a clean output structure for the current run.
# ==============================================================

print("\n" + "=" * 70)
print("A1.2 — Reset Output Directory")
print("=" * 70)

from pathlib import Path
import shutil

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

target_path = Path(OUTPUT_DIR)

# ------------------------------
# 2) Reset output directory
# ------------------------------
if target_path.exists():
    shutil.rmtree(target_path)
    print(f"Directory '{target_path.name}' successfully deleted at: {target_path}")
else:
    print(f"Directory '{target_path.name}' does not exist. Nothing to delete.")

### A1.3 — Create Required Output Directory Tree

In [ ]:
# ==============================================================
# A1.3 — Create Required Output Directory Tree
# --------------------------------------------------------------
# Purpose:
#   Recreate the canonical output directory tree required by the
#   benchmark after the output reset step.
# ==============================================================
from pathlib import Path

# ------------------------------
# 1) Basic checks (reuse upstream variable)
# ------------------------------
required_names = ["OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run A1 first."
    )

OUTPUT_DIR = Path(OUTPUT_DIR)

print("\n" + "=" * 70)
print("A1.3 — Create Required Output Directory Tree")
print("=" * 70)

REQUIRED_DIRS = [
    OUTPUT_DIR,  # outputs_benchmark_survival
    OUTPUT_DIR / "data",
    OUTPUT_DIR / "tables",
    OUTPUT_DIR / "tables" / "paper_main",
    OUTPUT_DIR / "tables" / "paper_appendix",
    OUTPUT_DIR / "figures",
    OUTPUT_DIR / "models",
    OUTPUT_DIR / "metadata",
]

for d in REQUIRED_DIRS:
    d.mkdir(parents=True, exist_ok=True)

print("Directory tree ensured:")
for d in REQUIRED_DIRS:
    print(f" - {d}")

## A2 — DuckDB Setup and OULAD Registration


In [ ]:
# ==============================================================
# A2 — DuckDB Setup and OULAD Registration
# --------------------------------------------------------------
# Purpose:
#   Initialize a persistent DuckDB database for the project,
#   register the extracted OULAD CSV files as SQL views,
#   validate access to each view, and save an initial registry
#   audit for downstream data preparation steps.
#
# Inputs expected from previous cells:
#   - PROJECT_ROOT
#   - DATA_DIR
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - print_section
#
# Outputs:
#   - benchmark_survival.duckdb
#   - table_duckdb_registered_views.csv
#   - duckdb_setup.json
# ==============================================================

# ------------------------------
# 1) Imports
# ------------------------------
print_section("A2 — DuckDB Setup and OULAD Registration")

# ------------------------------
# 2) Paths and directories
# ------------------------------
DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DUCKDB_PATH = OUTPUT_DIR / "benchmark_survival.duckdb"

EXPECTED_FILES = {
    "studentInfo": DATA_DIR / "studentInfo.csv",
    "studentRegistration": DATA_DIR / "studentRegistration.csv",
    "studentVle": DATA_DIR / "studentVle.csv",
    "courses": DATA_DIR / "courses.csv",
    "vle": DATA_DIR / "vle.csv",
    "studentAssessment": DATA_DIR / "studentAssessment.csv",
    "assessments": DATA_DIR / "assessments.csv",
}

# ------------------------------
# 3) Validate expected CSV files
# ------------------------------
missing_files = [name for name, path in EXPECTED_FILES.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing expected OULAD file(s) in DATA_DIR={DATA_DIR}:\n- " + "\n- ".join(missing_files)
    )

print(f"DuckDB file will be created/used at: {DUCKDB_PATH}")
print("All expected OULAD CSV files are available.")

# ------------------------------
# 4) Open persistent DuckDB connection
# ------------------------------
con = duckdb.connect(str(DUCKDB_PATH))

# ------------------------------
# 5) Register CSVs as views
# ------------------------------
registered_views = []

for view_name, csv_path in EXPECTED_FILES.items():
    sql = f"""
    CREATE OR REPLACE VIEW {view_name} AS
    SELECT *
    FROM read_csv_auto('{csv_path.as_posix()}', HEADER=TRUE);
    """
    con.execute(sql)
    registered_views.append(view_name)

print("\nRegistered DuckDB views:")
for v in registered_views:
    print(f" - {v}")

# ------------------------------
# 6) Validate row counts and columns
# ------------------------------
audit_rows = []

for view_name in registered_views:
    row_count = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]

    columns_info = con.execute(f"DESCRIBE {view_name}").fetchdf()
    n_cols = columns_info.shape[0]
    col_names = columns_info["column_name"].tolist()

    audit_rows.append({
        "view_name": view_name,
        "n_rows": int(row_count),
        "n_cols": int(n_cols),
        "columns": ", ".join(col_names),
        "source_csv": str(EXPECTED_FILES[view_name])
    })

audit_duckdb_views = pd.DataFrame(audit_rows).sort_values("view_name").reset_index(drop=True)

# ------------------------------
# 7) Save audit outputs
# ------------------------------
audit_csv_path = TABLES_DIR / "table_duckdb_registered_views.csv"
audit_duckdb_views.to_csv(audit_csv_path, index=False)

duckdb_setup_metadata = {
    "duckdb_path": str(DUCKDB_PATH),
    "data_output_dir": str(DATA_OUTPUT_DIR),
    "registered_views": registered_views,
    "source_files": {k: str(v) for k, v in EXPECTED_FILES.items()},
    "audit_csv_path": str(audit_csv_path),
}

save_json(duckdb_setup_metadata, METADATA_DIR / "duckdb_setup.json")

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nDuckDB view registry summary:")
display(audit_duckdb_views[["view_name", "n_rows", "n_cols"]])

print("\nSaved:")
print("-", audit_csv_path.resolve())
print("-", (METADATA_DIR / "duckdb_setup.json").resolve())

print("\nNotes:")
print("- DuckDB connection object available as: con")
print("- Materialized tables will be stored later in DuckDB and/or outputs_benchmark_survival/data/")


DuckDB view registry summary:


,view_name,n_rows,n_cols
0,assessments,206,6
1,courses,22,3
2,studentAssessment,173912,5
3,studentInfo,32593,12
4,studentRegistration,32593,5
5,studentVle,10655280,6
6,vle,6364,6



Saved:
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_duckdb_registered_views.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/metadata/duckdb_setup.json

Notes:
- DuckDB connection object available as: con
- Materialized tables will be stored later in DuckDB and/or outputs_benchmark_survival/data/


## A3 — Backbone-Oriented Raw Audit with DuckDB


### Funcao: get_duckdb_columns

Definicao isolada de get_duckdb_columns para reutilizacao nas etapas seguintes.


In [8]:
# ==============================================================
# A3 — Backbone-Oriented Raw Audit with DuckDB
# --------------------------------------------------------------
# Purpose:
#   Audit the raw OULAD tables with a backbone-oriented focus,
#   validating whether the minimal enrollment backbone can be
#   built cleanly from studentInfo + studentRegistration, and
#   whether the transactional tables are structurally ready for
#   later temporal expansion.
#
# Inputs expected from previous cells:
#   - con (DuckDB connection)
#   - TABLES_DIR
#   - OUTPUT_DIR
#
# Outputs:
#   - table_oulad_backbone_oriented_audit.csv
#   - table_oulad_critical_missing_audit.csv
#   - table_oulad_backbone_key_uniqueness.csv
# ==============================================================

print("\n" + "=" * 70)
print("A3 — Backbone-Oriented Raw Audit with DuckDB")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "TABLES_DIR", "OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

KEYS = ["id_student", "code_module", "code_presentation"]

# ------------------------------
# 2) Target tables and critical columns
# ------------------------------
audit_tables = {
    "studentInfo": {
        "role": "backbone_core",
        "critical_columns": ["id_student", "code_module", "code_presentation", "final_result"]
    },
    "studentRegistration": {
        "role": "backbone_core",
        "critical_columns": ["id_student", "code_module", "code_presentation", "date_registration", "date_unregistration"]
    },
    "studentVle": {
        "role": "transactional_minimal",
        "critical_columns": ["id_student", "code_module", "code_presentation", "id_site", "date", "sum_click"]
    },
    "studentAssessment": {
        "role": "transactional_optional",
        "critical_columns": ["id_student", "code_module", "code_presentation", "id_assessment", "date_submitted"]
    },
    "assessments": {
        "role": "transactional_optional",
        "critical_columns": ["id_assessment", "code_module", "code_presentation"]
    },
    "courses": {
        "role": "auxiliary",
        "critical_columns": ["code_module", "code_presentation"]
    },
    "vle": {
        "role": "auxiliary",
        "critical_columns": ["id_site", "code_module", "code_presentation"]
    },
}

# ------------------------------
# 3) Helper: get column names from DuckDB
# ------------------------------
def get_duckdb_columns(view_name: str) -> list:
    desc_df = con.execute(f"DESCRIBE {view_name}").fetchdf()
    return desc_df["column_name"].tolist()

# ------------------------------
# 4) Structural audit
# ------------------------------
structural_rows = []

for table_name, meta in audit_tables.items():
    columns = get_duckdb_columns(table_name)
    n_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    n_cols = len(columns)

    critical_cols = meta["critical_columns"]
    present_critical = [c for c in critical_cols if c in columns]
    missing_critical = [c for c in critical_cols if c not in columns]

    structural_rows.append({
        "table_name": table_name,
        "role": meta["role"],
        "n_rows": int(n_rows),
        "n_cols": int(n_cols),
        "critical_columns_expected": ", ".join(critical_cols),
        "critical_columns_present": ", ".join(present_critical),
        "critical_columns_missing": ", ".join(missing_critical),
        "all_columns": ", ".join(columns)
    })

audit_structural = pd.DataFrame(structural_rows).sort_values(["role", "table_name"]).reset_index(drop=True)

# ------------------------------
# 5) Critical missing audit
# ------------------------------
missing_rows = []

for table_name, meta in audit_tables.items():
    columns = get_duckdb_columns(table_name)

    for col in meta["critical_columns"]:
        if col in columns:
            q = f"""
            SELECT
                COUNT(*) AS n_total,
                SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS n_missing
            FROM {table_name}
            """
            result = con.execute(q).fetchdf().iloc[0]

            n_total = int(result["n_total"])
            n_missing = int(result["n_missing"]) if pd.notna(result["n_missing"]) else 0
            pct_missing = (n_missing / n_total * 100.0) if n_total > 0 else np.nan
        else:
            n_total = None
            n_missing = None
            pct_missing = None

        missing_rows.append({
            "table_name": table_name,
            "column_name": col,
            "exists_in_table": col in columns,
            "n_total": n_total,
            "n_missing": n_missing,
            "pct_missing": pct_missing
        })

audit_missing = pd.DataFrame(missing_rows).sort_values(["table_name", "column_name"]).reset_index(drop=True)

# ------------------------------
# 6) Backbone key uniqueness audit
# ------------------------------
uniqueness_rows = []

for table_name in ["studentInfo", "studentRegistration"]:
    total_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

    distinct_rows = con.execute(f"""
        SELECT COUNT(*) 
        FROM (
            SELECT DISTINCT id_student, code_module, code_presentation
            FROM {table_name}
        ) t
    """).fetchone()[0]

    duplicate_rows = int(total_rows - distinct_rows)

    uniqueness_rows.append({
        "table_name": table_name,
        "keys_checked": ", ".join(KEYS),
        "n_rows_total": int(total_rows),
        "n_distinct_keys": int(distinct_rows),
        "n_excess_duplicate_rows": duplicate_rows,
        "is_unique_by_keys": duplicate_rows == 0
    })

audit_uniqueness = pd.DataFrame(uniqueness_rows)

# ------------------------------
# 7) Save outputs
# ------------------------------
structural_path = TABLES_DIR / "table_oulad_backbone_oriented_audit.csv"
missing_path = TABLES_DIR / "table_oulad_critical_missing_audit.csv"
uniqueness_path = TABLES_DIR / "table_oulad_backbone_key_uniqueness.csv"

audit_structural.to_csv(structural_path, index=False)
audit_missing.to_csv(missing_path, index=False)
audit_uniqueness.to_csv(uniqueness_path, index=False)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nBackbone-oriented structural summary:")
display(audit_structural[[
    "table_name", "role", "n_rows", "n_cols", "critical_columns_missing"
]])

print("\nCritical missing summary (only rows with missing > 0 or missing columns):")
audit_missing_filtered = audit_missing[
    (~audit_missing["exists_in_table"]) | (audit_missing["n_missing"].fillna(0) > 0)
].copy()
display(audit_missing_filtered)

print("\nBackbone key uniqueness:")
display(audit_uniqueness)

print("\nSaved:")
print("-", structural_path.resolve())
print("-", missing_path.resolve())
print("-", uniqueness_path.resolve())


A3 — Backbone-Oriented Raw Audit with DuckDB

Backbone-oriented structural summary:


,table_name,role,n_rows,n_cols,critical_columns_missing
0,courses,auxiliary,22,3,
1,vle,auxiliary,6364,6,
2,studentInfo,backbone_core,32593,12,
3,studentRegistration,backbone_core,32593,5,
4,studentVle,transactional_minimal,10655280,6,
5,assessments,transactional_optional,206,6,
6,studentAssessment,transactional_optional,173912,5,"code_module, code_presentation"



Critical missing summary (only rows with missing > 0 or missing columns):


,table_name,column_name,exists_in_table,n_total,n_missing,pct_missing
5,studentAssessment,code_module,False,NaN,NaN,NaN
6,studentAssessment,code_presentation,False,NaN,NaN,NaN
16,studentRegistration,date_registration,True,32593.0,45.0,0.138066
17,studentRegistration,date_unregistration,True,32593.0,22521.0,69.097659



Backbone key uniqueness:


,table_name,keys_checked,n_rows_total,n_distinct_keys,n_excess_duplicate_rows,is_unique_by_keys
0,studentInfo,"id_student, code_module, code_presentation",32593,32593,0,True
1,studentRegistration,"id_student, code_module, code_presentation",32593,32593,0,True



Saved:
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_oulad_backbone_oriented_audit.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_oulad_critical_missing_audit.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_oulad_backbone_key_uniqueness.csv


## A4 — Build Minimal Enrollment Backbone


In [9]:
# ==============================================================
# A4 — Build Minimal Enrollment Backbone
# --------------------------------------------------------------
# Purpose:
#   Materialize the minimal enrollment-level backbone table
#   by joining studentInfo and studentRegistration on the
#   enrollment key, preserving one row per enrollment and
#   exporting the result for downstream survival preparation.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - DuckDB table: enrollment_backbone
#   - enrollment_backbone.parquet
#   - enrollment_backbone.csv
#   - table_enrollment_backbone_audit.csv
# ==============================================================

print("\n" + "=" * 70)
print("A4 — Build Minimal Enrollment Backbone")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KEYS = ["id_student", "code_module", "code_presentation"]

# ------------------------------
# 2) Create minimal enrollment backbone
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_backbone")

create_sql = """
CREATE TABLE enrollment_backbone AS
SELECT
    si.id_student,
    si.code_module,
    si.code_presentation,
    si.gender,
    si.region,
    si.highest_education,
    si.imd_band,
    si.age_band,
    si.num_of_prev_attempts,
    si.studied_credits,
    si.disability,
    si.final_result,
    sr.date_registration,
    sr.date_unregistration
FROM studentInfo AS si
LEFT JOIN studentRegistration AS sr
    ON si.id_student = sr.id_student
   AND si.code_module = sr.code_module
   AND si.code_presentation = sr.code_presentation
"""

con.execute(create_sql)

# ------------------------------
# 3) Validate uniqueness after join
# ------------------------------
n_rows_total = con.execute("SELECT COUNT(*) FROM enrollment_backbone").fetchone()[0]

n_distinct_keys = con.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT id_student, code_module, code_presentation
        FROM enrollment_backbone
    ) t
""").fetchone()[0]

is_unique = (n_rows_total == n_distinct_keys)

# ------------------------------
# 4) Final result distribution
# ------------------------------
final_result_dist = con.execute("""
    SELECT
        final_result,
        COUNT(*) AS n
    FROM enrollment_backbone
    GROUP BY final_result
    ORDER BY n DESC
""").fetchdf()

final_result_dist["pct"] = final_result_dist["n"] / final_result_dist["n"].sum() * 100.0

# ------------------------------
# 5) Missing audit for key date fields
# ------------------------------
date_missing_audit = con.execute("""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN date_registration IS NULL THEN 1 ELSE 0 END) AS n_missing_date_registration,
        SUM(CASE WHEN date_unregistration IS NULL THEN 1 ELSE 0 END) AS n_missing_date_unregistration
    FROM enrollment_backbone
""").fetchdf()

date_missing_audit["pct_missing_date_registration"] = (
    date_missing_audit["n_missing_date_registration"] / date_missing_audit["n_total"] * 100.0
)
date_missing_audit["pct_missing_date_unregistration"] = (
    date_missing_audit["n_missing_date_unregistration"] / date_missing_audit["n_total"] * 100.0
)

# ------------------------------
# 6) Build audit table
# ------------------------------
audit_rows = [{
    "table_name": "enrollment_backbone",
    "n_rows_total": int(n_rows_total),
    "n_distinct_enrollments": int(n_distinct_keys),
    "is_unique_by_keys": bool(is_unique),
    "n_missing_date_registration": int(date_missing_audit.loc[0, "n_missing_date_registration"]),
    "pct_missing_date_registration": float(date_missing_audit.loc[0, "pct_missing_date_registration"]),
    "n_missing_date_unregistration": int(date_missing_audit.loc[0, "n_missing_date_unregistration"]),
    "pct_missing_date_unregistration": float(date_missing_audit.loc[0, "pct_missing_date_unregistration"]),
}]

audit_backbone = pd.DataFrame(audit_rows)

# ------------------------------
# 7) Export backbone
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "enrollment_backbone.parquet"
csv_path = DATA_OUTPUT_DIR / "enrollment_backbone.csv"
audit_path = TABLES_DIR / "table_enrollment_backbone_audit.csv"

con.execute(f"COPY enrollment_backbone TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY enrollment_backbone TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")
audit_backbone.to_csv(audit_path, index=False)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nBackbone table summary:")
display(audit_backbone)

print("\nFinal result distribution:")
display(final_result_dist)

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())


A4 — Build Minimal Enrollment Backbone

Backbone table summary:


,table_name,n_rows_total,n_distinct_enrollments,is_unique_by_keys,n_missing_date_registration,pct_missing_date_registration,n_missing_date_unregistration,pct_missing_date_unregistration
0,enrollment_backbone,32593,32593,True,45,0.138066,22521,69.097659



Final result distribution:


,final_result,n,pct
0,Pass,12361,37.925321
1,Withdrawn,10156,31.160065
2,Fail,7052,21.636548
3,Distinction,3024,9.278066



Saved:
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/data/enrollment_backbone.parquet
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/data/enrollment_backbone.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_enrollment_backbone_audit.csv


## A5 — Final Event, Time, and Censoring Logic


In [10]:
# ==============================================================
# A5 — Final event/time construction for enrollment_survival_ready
# --------------------------------------------------------------
# Purpose:
#   Build the canonical enrollment-level survival-ready table
#   from the minimal enrollment backbone plus enrollment-level
#   VLE activity aggregates, then derive the final event/time
#   variables for temporal survival modeling.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - enrollment_backbone (DuckDB table from A4)
#   - studentVle (DuckDB table/view loaded upstream)
#
# Outputs:
#   - DuckDB table: enrollment_survival_ready
#   - enrollment_survival_ready.parquet
#   - enrollment_survival_ready.csv
#   - table_enrollment_survival_ready_audit.csv
#   - table_enrollment_survival_ready_final_result_by_event.csv
# ==============================================================

print("\n" + "=" * 70)
print("A5 — Final event/time construction for enrollment_survival_ready")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

required_tables = ["enrollment_backbone", "studentVle"]
missing_tables = [t for t in required_tables if t not in available_tables]
if missing_tables:
    raise RuntimeError(
        "Missing required DuckDB table(s)/view(s): "
        + ", ".join(missing_tables)
        + ". Please run the upstream preparation cells first."
    )

# ------------------------------
# 2) Aggregate VLE activity to enrollment level
# ------------------------------
con.execute("DROP TABLE IF EXISTS tmp_p5_vle_enrollment_agg")

con.execute("""
CREATE TABLE tmp_p5_vle_enrollment_agg AS
SELECT
    id_student,
    code_module,
    code_presentation,
    MAX(date) AS max_vle_day,
    COUNT(*) AS n_vle_rows,
    COALESCE(SUM(sum_click), 0) AS total_clicks_all_time,
    CASE WHEN COUNT(*) > 0 THEN 1 ELSE 0 END AS has_any_vle_activity
FROM studentVle
GROUP BY
    id_student,
    code_module,
    code_presentation
""")

# ------------------------------
# 3) Build raw survival-ready base
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_survival_ready_raw")

con.execute("""
CREATE TABLE enrollment_survival_ready_raw AS
SELECT
    eb.*,

    COALESCE(v.has_any_vle_activity, 0) AS has_any_vle_activity,
    v.max_vle_day,
    COALESCE(v.n_vle_rows, 0) AS n_vle_rows,
    COALESCE(v.total_clicks_all_time, 0) AS total_clicks_all_time

FROM enrollment_backbone AS eb
LEFT JOIN tmp_p5_vle_enrollment_agg AS v
    ON eb.id_student = v.id_student
   AND eb.code_module = v.code_module
   AND eb.code_presentation = v.code_presentation
""")

base = con.execute("""
SELECT *
FROM enrollment_survival_ready_raw
""").fetchdf()

# ------------------------------
# 4) Final event/time logic in pandas
# ------------------------------
import numpy as np
import pandas as pd

df = base.copy()

# Canonical enrollment identifier
df["enrollment_id"] = (
    df["id_student"].astype(str)
    + "||" + df["code_module"].astype(str)
    + "||" + df["code_presentation"].astype(str)
)

# Robust coercions
date_registration_num = pd.to_numeric(df["date_registration"], errors="coerce")
date_unregistration_num = pd.to_numeric(df["date_unregistration"], errors="coerce")
max_vle_day_num = pd.to_numeric(df["max_vle_day"], errors="coerce")
n_vle_rows_num = pd.to_numeric(df["n_vle_rows"], errors="coerce").fillna(0)
total_clicks_all_time_num = pd.to_numeric(df["total_clicks_all_time"], errors="coerce").fillna(0)
has_any_vle_activity_num = pd.to_numeric(df["has_any_vle_activity"], errors="coerce").fillna(0).astype(int)

# Withdrawn and event observability
final_result_clean = df["final_result"].astype(str).str.strip().str.upper()

df["is_withdrawn"] = (final_result_clean == "WITHDRAWN").astype(int)
df["has_valid_unregistration_date"] = (
    date_unregistration_num.notna() & (date_unregistration_num >= 0)
).astype(int)

# Event observed = withdrawn with valid observed unregistration date
df["event_observed"] = (
    (df["is_withdrawn"] == 1) &
    (df["has_valid_unregistration_date"] == 1)
).astype(int)

# Withdrawn without usable event date
df["is_withdrawn_without_valid_unregistration"] = (
    (df["is_withdrawn"] == 1) &
    (df["event_observed"] == 0)
).astype(int)

# Event week
df["t_event_week"] = np.where(
    df["event_observed"] == 1,
    np.floor(date_unregistration_num / 7.0),
    np.nan
)

# Raw censoring week based on latest observed VLE activity
df["t_last_obs_week_raw"] = np.where(
    max_vle_day_num.notna(),
    np.floor(max_vle_day_num / 7.0),
    np.nan
)

# Flag cases with only pre-start activity
df["has_pre_start_only_activity"] = (
    (has_any_vle_activity_num == 1) &
    max_vle_day_num.notna() &
    (max_vle_day_num < 0)
).astype(int)

# Final censoring week:
# - if no activity at all -> fallback to 0
# - if activity only before official start -> clamp to 0
# - otherwise -> floor(max_vle_day / 7), clamped at 0
df["t_last_obs_week"] = np.where(
    has_any_vle_activity_num == 0,
    0,
    np.where(
        max_vle_day_num.notna(),
        np.maximum(np.floor(max_vle_day_num / 7.0), 0),
        0
    )
)

# Final analysis time:
# - event week if event observed
# - censoring week otherwise
df["t_final_week"] = np.where(
    df["event_observed"] == 1,
    np.floor(date_unregistration_num / 7.0),
    df["t_last_obs_week"]
)

# Clamp final times to valid non-negative range
df["t_event_week"] = np.where(
    pd.notna(df["t_event_week"]),
    np.maximum(df["t_event_week"], 0),
    np.nan
)
df["t_last_obs_week"] = np.maximum(df["t_last_obs_week"], 0)
df["t_final_week"] = np.maximum(df["t_final_week"], 0)

# Integer typing where appropriate
df["t_last_obs_week"] = pd.to_numeric(df["t_last_obs_week"], errors="coerce").fillna(0).astype(int)
df["t_final_week"] = pd.to_numeric(df["t_final_week"], errors="coerce").fillna(0).astype(int)

# Audit flags
df["used_zero_week_fallback_for_censoring"] = (
    (df["event_observed"] == 0) &
    (
        (has_any_vle_activity_num == 0) |
        (df["has_pre_start_only_activity"] == 1)
    )
).astype(int)

df["t_last_obs_week_was_clamped"] = (
    pd.notna(df["t_last_obs_week_raw"]) &
    (pd.to_numeric(df["t_last_obs_week_raw"], errors="coerce") < 0)
).astype(int)

df["t_final_week_was_clamped"] = (
    (df["event_observed"] == 0) &
    (df["t_last_obs_week_was_clamped"] == 1)
).astype(int)

# Reorder some important columns near the front
priority_cols = [
    "enrollment_id",
    "id_student", "code_module", "code_presentation",
    "final_result",
    "date_registration", "date_unregistration",
    "is_withdrawn", "has_valid_unregistration_date",
    "event_observed", "is_withdrawn_without_valid_unregistration",
    "has_any_vle_activity", "max_vle_day", "n_vle_rows", "total_clicks_all_time",
    "t_event_week", "t_last_obs_week_raw", "t_last_obs_week", "t_final_week",
    "has_pre_start_only_activity",
    "used_zero_week_fallback_for_censoring",
    "t_last_obs_week_was_clamped",
    "t_final_week_was_clamped"
]
remaining_cols = [c for c in df.columns if c not in priority_cols]
df = df[priority_cols + remaining_cols]

# ------------------------------
# 5) Materialize final DuckDB table
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_survival_ready")
con.register("enrollment_survival_ready_df", df)
con.execute("""
CREATE TABLE enrollment_survival_ready AS
SELECT *
FROM enrollment_survival_ready_df
""")
con.unregister("enrollment_survival_ready_df")

# ------------------------------
# 6) Audits
# ------------------------------
n_total = len(df)
n_unique_enrollments = df["enrollment_id"].nunique()
is_unique_by_enrollment = (n_total == n_unique_enrollments)

n_withdrawn = int(df["is_withdrawn"].sum())
n_event_observed = int(df["event_observed"].sum())
n_withdrawn_without_valid_unregistration = int(df["is_withdrawn_without_valid_unregistration"].sum())
n_no_vle_activity = int((df["has_any_vle_activity"] == 0).sum())
n_pre_start_only = int(df["has_pre_start_only_activity"].sum())
n_zero_week_fallback = int(df["used_zero_week_fallback_for_censoring"].sum())

max_t_event_week = (
    float(pd.to_numeric(df["t_event_week"], errors="coerce").dropna().max())
    if pd.to_numeric(df["t_event_week"], errors="coerce").notna().any()
    else np.nan
)
max_t_last_obs_week = float(pd.to_numeric(df["t_last_obs_week"], errors="coerce").max())
max_t_final_week = float(pd.to_numeric(df["t_final_week"], errors="coerce").max())

audit_rows = [{
    "table_name": "enrollment_survival_ready",
    "n_rows_total": int(n_total),
    "n_unique_enrollments": int(n_unique_enrollments),
    "is_unique_by_enrollment": bool(is_unique_by_enrollment),
    "n_withdrawn": n_withdrawn,
    "n_event_observed": n_event_observed,
    "event_rate_over_all_enrollments": float(n_event_observed / n_total) if n_total > 0 else np.nan,
    "n_withdrawn_without_valid_unregistration": n_withdrawn_without_valid_unregistration,
    "n_no_vle_activity": n_no_vle_activity,
    "n_pre_start_only_activity": n_pre_start_only,
    "n_zero_week_fallback_for_censoring": n_zero_week_fallback,
    "max_t_event_week": max_t_event_week,
    "max_t_last_obs_week": max_t_last_obs_week,
    "max_t_final_week": max_t_final_week
}]
audit_df = pd.DataFrame(audit_rows)

final_result_by_event = (
    df.groupby(["final_result", "event_observed"], dropna=False)
      .size()
      .reset_index(name="n")
      .sort_values(["final_result", "event_observed"], ascending=[True, True])
)

final_result_by_event["pct_within_final_result"] = (
    final_result_by_event["n"] /
    final_result_by_event.groupby("final_result")["n"].transform("sum") * 100.0
)

# ------------------------------
# 7) Exports
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "enrollment_survival_ready.parquet"
csv_path = DATA_OUTPUT_DIR / "enrollment_survival_ready.csv"
audit_path = TABLES_DIR / "table_enrollment_survival_ready_audit.csv"
result_event_path = TABLES_DIR / "table_enrollment_survival_ready_final_result_by_event.csv"

con.execute(f"COPY enrollment_survival_ready TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY enrollment_survival_ready TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_df.to_csv(audit_path, index=False)
final_result_by_event.to_csv(result_event_path, index=False)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nEnrollment survival-ready audit:")
display(audit_df)

print("\nFinal result × event observed:")
display(final_result_by_event)

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", result_event_path.resolve())


A5 — Final event/time construction for enrollment_survival_ready

Enrollment survival-ready audit:


,table_name,n_rows_total,n_unique_enrollments,is_unique_by_enrollment,n_withdrawn,n_event_observed,event_rate_over_all_enrollments,n_withdrawn_without_valid_unregistration,n_no_vle_activity,n_pre_start_only_activity,n_zero_week_fallback_for_censoring,max_t_event_week,max_t_last_obs_week,max_t_final_week
0,enrollment_survival_ready,32593,32593,True,10156,7387,0.226644,2769,3365,728,3078,63.0,38.0,63.0



Final result × event observed:


,final_result,event_observed,n,pct_within_final_result
0,Distinction,0,3024,100.000000
1,Fail,0,7052,100.000000
2,Pass,0,12361,100.000000
3,Withdrawn,0,2769,27.264671
4,Withdrawn,1,7387,72.735329



Saved:
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/data/enrollment_survival_ready.parquet
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/data/enrollment_survival_ready.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_enrollment_survival_ready_audit.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_enrollment_survival_ready_final_result_by_event.csv


# B - Feature engineering and benchmark input construction

This section transforms the canonical survival backbone into the model-ready benchmark inputs.

It covers:
- person-period expansion
- weekly behavioral feature aggregation
- temporal feature engineering
- enrollment-level model tables
- canonical benchmark configuration
- model-specific input views
- window-based comparable features

The outputs of this section feed the split, preprocessing, and modeling steps in the following sections.

## B1 — Build Minimal Person-Period Skeleton


In [11]:
# ==============================================================
# B1 — Build Minimal Person-Period Skeleton
# --------------------------------------------------------------
# Purpose:
#   Expand the enrollment-level survival-ready table into a
#   weekly person-period skeleton, preserving one row per
#   enrollment-week and creating the discrete-time event target.
#
#   This revised version explicitly clamps negative t_final_week
#   values to zero before expansion so that no enrollment is lost.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - DuckDB table: person_period_min
#   - person_period_min.parquet
#   - person_period_min.csv
#   - table_person_period_min_audit.csv
#   - table_person_period_event_check.csv
#   - table_person_period_week_distribution.csv
# ==============================================================

print("\n" + "=" * 70)
print("B1 — Build Minimal Person-Period Skeleton")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Rebuild person-period skeleton
# ------------------------------
con.execute("DROP TABLE IF EXISTS person_period_min")

con.execute("""
CREATE TABLE person_period_min AS
WITH base AS (
    SELECT
        esr.*,
        GREATEST(esr.t_final_week, 0) AS t_final_week_clamped
    FROM enrollment_survival_ready AS esr
),
expanded AS (
    SELECT
        b.id_student,
        b.code_module,
        b.code_presentation,
        b.gender,
        b.region,
        b.highest_education,
        b.imd_band,
        b.age_band,
        b.num_of_prev_attempts,
        b.studied_credits,
        b.disability,
        b.final_result,
        b.date_registration,
        b.date_unregistration,
        b.is_withdrawn,
        b.has_valid_unregistration_date,
        b.event_observed,
        b.is_withdrawn_without_valid_unregistration,
        b.has_any_vle_activity,
        b.max_vle_day,
        b.n_vle_rows,
        b.total_clicks_all_time,
        b.t_event_week,
        b.t_last_obs_week,
        b.t_final_week,
        b.t_final_week_clamped,
        b.used_zero_week_fallback_for_censoring,
        gs.week
    FROM base AS b
    CROSS JOIN LATERAL generate_series(0, b.t_final_week_clamped) AS gs(week)
)
SELECT
    *,
    CAST(id_student AS VARCHAR) || '|' || code_module || '|' || code_presentation AS enrollment_id,
    CASE
        WHEN event_observed = 1 AND week = t_event_week THEN 1
        ELSE 0
    END AS event_t
FROM expanded
""")

# ------------------------------
# 3) Main audit summary
# ------------------------------
audit_summary = con.execute("""
SELECT
    COUNT(*) AS n_person_period_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments,
    MAX(week) AS max_week,
    MAX(t_final_week) AS max_t_final_week,
    MAX(t_final_week_clamped) AS max_t_final_week_clamped,
    MIN(t_final_week) AS min_t_final_week,
    MIN(t_final_week_clamped) AS min_t_final_week_clamped,
    SUM(event_t) AS n_total_event_rows,
    SUM(CASE WHEN used_zero_week_fallback_for_censoring = 1 THEN 1 ELSE 0 END) AS n_rows_from_zero_week_fallback,
    COUNT(DISTINCT CASE WHEN used_zero_week_fallback_for_censoring = 1 THEN enrollment_id END) AS n_enrollments_from_zero_week_fallback,
    COUNT(DISTINCT CASE WHEN t_final_week < 0 THEN enrollment_id END) AS n_enrollments_with_negative_t_final_week_raw
FROM person_period_min
""").fetchdf()

# ------------------------------
# 4) Event integrity check
# ------------------------------
event_check = con.execute("""
WITH per_enrollment AS (
    SELECT
        enrollment_id,
        MAX(event_observed) AS event_observed,
        SUM(event_t) AS n_event_rows
    FROM person_period_min
    GROUP BY 1
)
SELECT
    event_observed,
    n_event_rows,
    COUNT(*) AS n_enrollments
FROM per_enrollment
GROUP BY event_observed, n_event_rows
ORDER BY event_observed, n_event_rows
""").fetchdf()

# ------------------------------
# 5) Week distribution
# ------------------------------
week_distribution = con.execute("""
SELECT
    week,
    COUNT(*) AS n_rows,
    SUM(event_t) AS n_events
FROM person_period_min
GROUP BY week
ORDER BY week
""").fetchdf()

# ------------------------------
# 6) Enrollment coverage check
# ------------------------------
coverage_check = con.execute("""
SELECT
    (SELECT COUNT(*) FROM enrollment_survival_ready) AS n_enrollments_survival_ready,
    (SELECT COUNT(DISTINCT enrollment_id) FROM person_period_min) AS n_enrollments_person_period
""").fetchdf()

coverage_check["n_missing_enrollments_after_expansion"] = (
    coverage_check["n_enrollments_survival_ready"] - coverage_check["n_enrollments_person_period"]
)

# ------------------------------
# 7) Save outputs
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "person_period_min.parquet"
csv_path = DATA_OUTPUT_DIR / "person_period_min.csv"
audit_path = TABLES_DIR / "table_person_period_min_audit.csv"
event_check_path = TABLES_DIR / "table_person_period_event_check.csv"
week_dist_path = TABLES_DIR / "table_person_period_week_distribution.csv"
coverage_check_path = TABLES_DIR / "table_person_period_coverage_check.csv"

con.execute(f"COPY person_period_min TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY person_period_min TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_summary.to_csv(audit_path, index=False)
event_check.to_csv(event_check_path, index=False)
week_distribution.to_csv(week_dist_path, index=False)
coverage_check.to_csv(coverage_check_path, index=False)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nPerson-period summary:")
display(audit_summary)

print("\nEnrollment coverage check:")
display(coverage_check)

print("\nEvent integrity check:")
display(event_check)

print("\nWeek distribution (first rows):")
display(week_distribution.head(15))

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", event_check_path.resolve())
print("-", week_dist_path.resolve())
print("-", coverage_check_path.resolve())


B1 — Build Minimal Person-Period Skeleton

Person-period summary:


,n_person_period_rows,n_distinct_enrollments,max_week,max_t_final_week,max_t_final_week_clamped,min_t_final_week,min_t_final_week_clamped,n_total_event_rows,n_rows_from_zero_week_fallback,n_enrollments_from_zero_week_fallback,n_enrollments_with_negative_t_final_week_raw
0,775295,32593,63,63,63,0,0,7387.0,3078.0,3078,0



Enrollment coverage check:


,n_enrollments_survival_ready,n_enrollments_person_period,n_missing_enrollments_after_expansion
0,32593,32593,0



Event integrity check:


,event_observed,n_event_rows,n_enrollments
0,0,0.0,25206
1,1,1.0,7387



Week distribution (first rows):


,week,n_rows,n_events
0,0,32593,713.0
1,1,28633,1068.0
2,2,27396,215.0
3,3,26956,360.0
4,4,26363,303.0
5,5,25843,216.0
6,6,25459,246.0
7,7,25010,256.0
8,8,24554,230.0
9,9,24160,220.0



Saved:
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/data/person_period_min.parquet
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/data/person_period_min.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_person_period_min_audit.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_person_period_event_check.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_person_period_week_distribution.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_person_period_coverage_check.csv


## B2 — Build Weekly VLE Feature Layer


In [12]:
# ==============================================================
# B2 — Build Weekly VLE Feature Layer
# --------------------------------------------------------------
# Purpose:
#   Aggregate studentVle into a weekly LMS activity layer at the
#   enrollment-week level, creating the minimal weekly behavioral
#   signals needed for downstream temporal feature engineering.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - DuckDB table: vle_weekly_features
#   - vle_weekly_features.parquet
#   - vle_weekly_features.csv
#   - table_vle_weekly_features_audit.csv
#   - table_vle_weekly_features_week_distribution.csv
# ==============================================================

print("\n" + "=" * 70)
print("B2 — Build Weekly VLE Feature Layer")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Build weekly VLE features
# ------------------------------
con.execute("DROP TABLE IF EXISTS vle_weekly_features")

con.execute("""
CREATE TABLE vle_weekly_features AS
SELECT
    id_student,
    code_module,
    code_presentation,
    CAST(FLOOR(date / 7.0) AS INTEGER) AS week,
    SUM(sum_click) AS total_clicks_week,
    COUNT(*) AS n_vle_rows_week,
    1 AS active_this_week,
    COUNT(DISTINCT id_site) AS n_distinct_sites_week
FROM studentVle
GROUP BY
    id_student,
    code_module,
    code_presentation,
    CAST(FLOOR(date / 7.0) AS INTEGER)
""")

# ------------------------------
# 3) Main audit summary
# ------------------------------
audit_summary = con.execute("""
SELECT
    COUNT(*) AS n_enrollment_week_rows,
    COUNT(DISTINCT CAST(id_student AS VARCHAR) || '|' || code_module || '|' || code_presentation) AS n_distinct_enrollments,
    MIN(week) AS min_week,
    MAX(week) AS max_week,
    SUM(CASE WHEN week < 0 THEN 1 ELSE 0 END) AS n_rows_negative_weeks,
    SUM(CASE WHEN total_clicks_week = 0 THEN 1 ELSE 0 END) AS n_rows_zero_clicks,
    MIN(total_clicks_week) AS min_total_clicks_week,
    MAX(total_clicks_week) AS max_total_clicks_week
FROM vle_weekly_features
""").fetchdf()

# ------------------------------
# 4) Week distribution
# ------------------------------
week_distribution = con.execute("""
SELECT
    week,
    COUNT(*) AS n_rows,
    SUM(total_clicks_week) AS total_clicks_sum,
    AVG(total_clicks_week) AS avg_total_clicks_week
FROM vle_weekly_features
GROUP BY week
ORDER BY week
""").fetchdf()

# ------------------------------
# 5) Coverage relative to survival-ready enrollments
# ------------------------------
coverage_check = con.execute("""
SELECT
    (SELECT COUNT(*) FROM enrollment_survival_ready) AS n_enrollments_survival_ready,
    (SELECT COUNT(DISTINCT CAST(id_student AS VARCHAR) || '|' || code_module || '|' || code_presentation)
     FROM vle_weekly_features) AS n_enrollments_with_weekly_vle
""").fetchdf()

coverage_check["n_enrollments_without_weekly_vle"] = (
    coverage_check["n_enrollments_survival_ready"] - coverage_check["n_enrollments_with_weekly_vle"]
)

# ------------------------------
# 6) Save outputs
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "vle_weekly_features.parquet"
csv_path = DATA_OUTPUT_DIR / "vle_weekly_features.csv"
audit_path = TABLES_DIR / "table_vle_weekly_features_audit.csv"
week_dist_path = TABLES_DIR / "table_vle_weekly_features_week_distribution.csv"
coverage_path = TABLES_DIR / "table_vle_weekly_features_coverage_check.csv"

con.execute(f"COPY vle_weekly_features TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY vle_weekly_features TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_summary.to_csv(audit_path, index=False)
week_distribution.to_csv(week_dist_path, index=False)
coverage_check.to_csv(coverage_path, index=False)

# ------------------------------
# 7) Output for feedback
# ------------------------------
print("\nWeekly VLE summary:")
display(audit_summary)

print("\nCoverage relative to enrollment_survival_ready:")
display(coverage_check)

print("\nWeek distribution (first rows):")
display(week_distribution.head(20))

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", week_dist_path.resolve())
print("-", coverage_path.resolve())


B2 — Build Weekly VLE Feature Layer

Weekly VLE summary:


,n_enrollment_week_rows,n_distinct_enrollments,min_week,max_week,n_rows_negative_weeks,n_rows_zero_clicks,min_total_clicks_week,max_total_clicks_week
0,627031,29228,-4,38,47593.0,0.0,1.0,6999.0



Coverage relative to enrollment_survival_ready:


,n_enrollments_survival_ready,n_enrollments_with_weekly_vle,n_enrollments_without_weekly_vle
0,32593,29228,3365



Week distribution (first rows):


,week,n_rows,total_clicks_sum,avg_total_clicks_week
0,-4,1076,43298.0,40.239777
1,-3,10532,433407.0,41.151443
2,-2,16176,646001.0,39.935769
3,-1,19809,1025241.0,51.756323
4,0,23658,2013077.0,85.090752
5,1,23168,1797648.0,77.591851
6,2,23822,2307080.0,96.846612
7,3,22135,1663598.0,75.156901
8,4,21661,1505334.0,69.495129
9,5,20361,1228901.0,60.355631



Saved:
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/data/vle_weekly_features.parquet
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/data/vle_weekly_features.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_vle_weekly_features_audit.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_vle_weekly_features_week_distribution.csv
- /workspaces/student-dropout-survival-benchmark/outputs_benchmark_survival/tables/table_vle_weekly_features_coverage_check.csv


## B3 — Enrich Person-Period with Temporal Features


In [ ]:
# ==============================================================
# B3 — Enrich Person-Period with Temporal Features
# --------------------------------------------------------------
# Purpose:
#   Enrich the weekly person-period skeleton with minimal LMS
#   behavioral features and derive temporal covariates required
#   for discrete-time survival modeling.
#
# Methodological note:
#   The survival modeling grid in this project starts at week = 0.
#   Although raw LMS activity includes negative weeks (audited in B2),
#   only week >= 0 is used when enriching the modelable person-period
#   table. This keeps the behavioral covariates aligned with the
#   temporal frame used by the discrete-time survival models and avoids
#   mixing pre-period activity with the formal modeling horizon.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - DuckDB table: person_period_enriched
#   - person_period_enriched.parquet
#   - person_period_enriched.csv
#   - table_person_period_enriched_audit.csv
#   - table_person_period_enriched_missing_check.csv
# ==============================================================

print("\n" + "=" * 70)
print("B3 — Enrich Person-Period with Temporal Features")
print("=" * 70)
print("Methodological note: only VLE weeks >= 0 were joined into person_period_enriched.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Create non-negative weekly VLE layer
# ------------------------------
con.execute("DROP VIEW IF EXISTS vle_weekly_features_nonnegative")

con.execute("""
CREATE VIEW vle_weekly_features_nonnegative AS
SELECT *
FROM vle_weekly_features
WHERE week >= 0
""")

# ------------------------------
# 3) Join person-period skeleton with weekly VLE features
# ------------------------------
con.execute("DROP TABLE IF EXISTS person_period_enriched_stage1")

con.execute("""
CREATE TABLE person_period_enriched_stage1 AS
SELECT
    ppm.*,

    COALESCE(vwf.total_clicks_week, 0) AS total_clicks_week,
    COALESCE(vwf.n_vle_rows_week, 0) AS n_vle_rows_week,
    COALESCE(vwf.active_this_week, 0) AS active_this_week,
    COALESCE(vwf.n_distinct_sites_week, 0) AS n_distinct_sites_week

FROM person_period_min AS ppm
LEFT JOIN vle_weekly_features_nonnegative AS vwf
    ON ppm.id_student = vwf.id_student
   AND ppm.code_module = vwf.code_module
   AND ppm.code_presentation = vwf.code_presentation
   AND ppm.week = vwf.week
""")

# ------------------------------
# 4) Build enriched person-period table with temporal covariates
# ------------------------------
con.execute("DROP TABLE IF EXISTS person_period_enriched")

con.execute("""
CREATE TABLE person_period_enriched AS
WITH base AS (
    SELECT
        *,

        SUM(total_clicks_week) OVER (
            PARTITION BY enrollment_id
            ORDER BY week
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_clicks_until_t,

        SUM(active_this_week) OVER (
            PARTITION BY enrollment_id
            ORDER BY week
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_active_weeks_until_t,

        MAX(CASE WHEN active_this_week = 1 THEN week ELSE NULL END) OVER (
            PARTITION BY enrollment_id
            ORDER BY week
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS last_active_week_so_far
    FROM person_period_enriched_stage1
),
with_recency AS (
    SELECT
        *,

        CASE
            WHEN active_this_week = 1 THEN 0
            WHEN last_active_week_so_far IS NOT NULL THEN week - last_active_week_so_far
            ELSE week + 1
        END AS recency
    FROM base
),
with_groups AS (
    SELECT
        *,
        SUM(CASE WHEN active_this_week = 0 THEN 1 ELSE 0 END) OVER (
            PARTITION BY enrollment_id
            ORDER BY week
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS inactivity_group
    FROM with_recency
)
SELECT
    *,

    CASE
        WHEN active_this_week = 1 THEN
            ROW_NUMBER() OVER (
                PARTITION BY enrollment_id, inactivity_group
                ORDER BY week
            )
        ELSE 0
    END AS streak
FROM with_groups
""")

# ------------------------------
# 5) Main audit summary
# ------------------------------
audit_summary = con.execute("""
SELECT
    COUNT(*) AS n_person_period_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments,
    MAX(week) AS max_week,
    AVG(total_clicks_week) AS avg_total_clicks_week,
    AVG(cum_clicks_until_t) AS avg_cum_clicks_until_t,
    AVG(active_this_week) AS avg_active_this_week,
    MAX(recency) AS max_recency,
    MAX(streak) AS max_streak
FROM person_period_enriched
""").fetchdf()

# ------------------------------
# 6) Missing check
# ------------------------------
missing_check = con.execute("""
SELECT
    SUM(CASE WHEN total_clicks_week IS NULL THEN 1 ELSE 0 END) AS n_missing_total_clicks_week,
    SUM(CASE WHEN n_vle_rows_week IS NULL THEN 1 ELSE 0 END) AS n_missing_n_vle_rows_week,
    SUM(CASE WHEN active_this_week IS NULL THEN 1 ELSE 0 END) AS n_missing_active_this_week,
    SUM(CASE WHEN n_distinct_sites_week IS NULL THEN 1 ELSE 0 END) AS n_missing_n_distinct_sites_week,
    SUM(CASE WHEN cum_clicks_until_t IS NULL THEN 1 ELSE 0 END) AS n_missing_cum_clicks_until_t,
    SUM(CASE WHEN recency IS NULL THEN 1 ELSE 0 END) AS n_missing_recency,
    SUM(CASE WHEN streak IS NULL THEN 1 ELSE 0 END) AS n_missing_streak
FROM person_period_enriched
""").fetchdf()

# ------------------------------
# 7) Small ordered sample for inspection
# ------------------------------
sample_rows = con.execute("""
SELECT
    enrollment_id,
    week,
    event_t,
    total_clicks_week,
    active_this_week,
    cum_clicks_until_t,
    recency,
    streak
FROM person_period_enriched
ORDER BY enrollment_id, week
LIMIT 30
""").fetchdf()

# ------------------------------
# 8) Save outputs
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "person_period_enriched.parquet"
csv_path = DATA_OUTPUT_DIR / "person_period_enriched.csv"
audit_path = TABLES_DIR / "table_person_period_enriched_audit.csv"
missing_path = TABLES_DIR / "table_person_period_enriched_missing_check.csv"
sample_path = TABLES_DIR / "table_person_period_enriched_sample.csv"

con.execute(f"COPY person_period_enriched TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY person_period_enriched TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_summary.to_csv(audit_path, index=False)
missing_check.to_csv(missing_path, index=False)
sample_rows.to_csv(sample_path, index=False)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nPerson-period enriched summary:")
display(audit_summary)

print("\nMissing check for new temporal features:")
display(missing_check)

print("\nOrdered sample (first rows):")
display(sample_rows)

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", missing_path.resolve())
print("-", sample_path.resolve())


B3 — Enrich Person-Period with Temporal Features
Methodological note: only VLE weeks >= 0 were joined into person_period_enriched.


## B4 — Build Enrollment-Level Model Table (Revised v2)


In [ ]:
# ==============================================================
# B4 — Build Enrollment-Level Model Table (Revised v2)
# --------------------------------------------------------------
# Purpose:
#   Build the canonical enrollment-level model table for the
#   benchmark, containing only structural enrollment-level fields,
#   survival targets, and static covariates.
#
# Methodological note:
#   This revised version intentionally avoids embedding
#   retrospective whole-course temporal summary features (e.g.,
#   all-time activity summaries) into the canonical enrollment-level
#   benchmark table.
#
#   The goal is to preserve a clean structural base that can later
#   be combined with early-window features in a controlled and
#   comparable way, especially for enrollment-level models such as
#   Cox.
#
# Inputs expected from previous cells:
#   - con
#   - DATA_OUTPUT_DIR
#   - TABLES_DIR
#
# Outputs:
#   - enrollment_model_table.parquet
#   - enrollment_model_table.csv
#   - table_enrollment_model_table_audit.csv
#   - table_enrollment_model_table_missing_check.csv
#   - table_enrollment_model_table_event_distribution.csv
#   - table_enrollment_model_table_sample.csv
# ==============================================================

print("\n" + "=" * 70)
print("B4 — Build Enrollment-Level Model Table (Revised v2)")
print("=" * 70)
print("Methodological note: this revised table is a canonical structural")
print("enrollment-level base and intentionally excludes all-time temporal")
print("summary features from the benchmark core.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "DATA_OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd
import numpy as np

# ------------------------------
# 2) Build revised canonical table
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_model_table")

con.execute("""
CREATE TABLE enrollment_model_table AS
SELECT
    CAST(id_student AS VARCHAR) || '|' || code_module || '|' || code_presentation AS enrollment_id,
    id_student,
    code_module,
    code_presentation,

    -- survival targets
    event_observed AS event,
    t_last_obs_week AS duration_raw,
    CASE
        WHEN t_last_obs_week < 0 THEN 0
        ELSE t_last_obs_week
    END AS duration,

    -- structural note
    used_zero_week_fallback_for_censoring,

    -- static covariates
    gender,
    region,
    highest_education,
    imd_band,
    age_band,
    num_of_prev_attempts,
    studied_credits,
    disability

FROM enrollment_survival_ready
""")

# ------------------------------
# 3) Load for audit/export
# ------------------------------
enrollment_model_table = con.execute("""
SELECT *
FROM enrollment_model_table
ORDER BY enrollment_id
""").fetchdf()

# ------------------------------
# 4) Audit summaries
# ------------------------------
summary_df = con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments,
    MIN(duration_raw) AS min_duration_raw,
    MIN(duration) AS min_duration,
    AVG(duration) AS avg_duration,
    MAX(duration) AS max_duration,
    SUM(event) AS n_events,
    100.0 * AVG(event) AS pct_events,
    SUM(CASE WHEN duration_raw < 0 THEN 1 ELSE 0 END) AS n_negative_duration_raw,
    CASE
        WHEN COUNT(*) = COUNT(DISTINCT enrollment_id) THEN TRUE
        ELSE FALSE
    END AS is_unique_by_enrollment_id
FROM enrollment_model_table
""").fetchdf()

event_distribution_df = con.execute("""
SELECT
    event,
    COUNT(*) AS n
FROM enrollment_model_table
GROUP BY event
ORDER BY event
""").fetchdf()

missing_check_df = con.execute("""
SELECT
    SUM(CASE WHEN duration_raw IS NULL THEN 1 ELSE 0 END) AS n_missing_duration_raw,
    SUM(CASE WHEN duration IS NULL THEN 1 ELSE 0 END) AS n_missing_duration,
    SUM(CASE WHEN event IS NULL THEN 1 ELSE 0 END) AS n_missing_event,
    SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END) AS n_missing_gender,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS n_missing_region,
    SUM(CASE WHEN highest_education IS NULL THEN 1 ELSE 0 END) AS n_missing_highest_education,
    SUM(CASE WHEN imd_band IS NULL THEN 1 ELSE 0 END) AS n_missing_imd_band,
    SUM(CASE WHEN age_band IS NULL THEN 1 ELSE 0 END) AS n_missing_age_band,
    SUM(CASE WHEN num_of_prev_attempts IS NULL THEN 1 ELSE 0 END) AS n_missing_num_of_prev_attempts,
    SUM(CASE WHEN studied_credits IS NULL THEN 1 ELSE 0 END) AS n_missing_studied_credits,
    SUM(CASE WHEN disability IS NULL THEN 1 ELSE 0 END) AS n_missing_disability,
    SUM(CASE WHEN used_zero_week_fallback_for_censoring IS NULL THEN 1 ELSE 0 END) AS n_missing_zero_week_fallback_flag
FROM enrollment_model_table
""").fetchdf()

sample_df = con.execute("""
SELECT *
FROM enrollment_model_table
ORDER BY enrollment_id
LIMIT 30
""").fetchdf()

# ------------------------------
# 5) Save outputs
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "enrollment_model_table.parquet"
csv_path = DATA_OUTPUT_DIR / "enrollment_model_table.csv"
audit_path = TABLES_DIR / "table_enrollment_model_table_audit.csv"
missing_path = TABLES_DIR / "table_enrollment_model_table_missing_check.csv"
event_dist_path = TABLES_DIR / "table_enrollment_model_table_event_distribution.csv"
sample_path = TABLES_DIR / "table_enrollment_model_table_sample.csv"

enrollment_model_table.to_parquet(parquet_path, index=False)
enrollment_model_table.to_csv(csv_path, index=False)

summary_df.to_csv(audit_path, index=False)
missing_check_df.to_csv(missing_path, index=False)
event_distribution_df.to_csv(event_dist_path, index=False)
sample_df.to_csv(sample_path, index=False)

# ------------------------------
# 6) Output for feedback
# ------------------------------
print("\nEnrollment model table summary:")
display(summary_df)

print("\nEvent distribution:")
display(event_distribution_df)

print("\nMissing check:")
display(missing_check_df)

print("\nSample rows:")
display(sample_df)

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", missing_path.resolve())
print("-", event_dist_path.resolve())
print("-", sample_path.resolve())

## B5 — Define Canonical Modeling Configuration Layer


In [ ]:
# ==============================================================
# B5 — Define Canonical Modeling Configuration Layer
# 
# ==============================================================
# Purpose:
#   Centralize the benchmark configuration under the corrected
#   benchmark architecture.
#
# This step:
#   - preserves current operational configuration objects
#   - adds explicit benchmark-arm structure
#   - officially incorporates the future
#     discrete_time_comparable_minimal arm
#   - keeps downstream compatibility as much as possible
#
# Main outputs:
#   - benchmark configuration objects in memory
#   - table_p10_canonical_modeling_configuration.csv
#   - metadata_p10_canonical_modeling_configuration.json
# ==============================================================

import json
from pathlib import Path

import pandas as pd

print("=" * 70)
print("B5 — Define Canonical Modeling Configuration Layer")
print("=" * 70)

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Core benchmark constants
# --------------------------------------------------------------
RANDOM_SEED = 42
TEST_SIZE = 0.30
EARLY_WINDOW_WEEKS = 4
MAIN_ENROLLMENT_WINDOW_WEEKS = EARLY_WINDOW_WEEKS

# --------------------------------------------------------------
# 2) Feature groups
# --------------------------------------------------------------
# Keep names aligned with the current notebook convention.

STATIC_FEATURES = [
    "studied_credits",
    "num_of_prev_attempts",
]

TEMPORAL_FEATURES_DISCRETE = [
    "week",
    "total_clicks",
]

# Canonical enrollment-level early-window feature set
MAIN_ENROLLMENT_WINDOW_FEATURES = [
    "studied_credits",
    "num_of_prev_attempts",
    "clicks_first_4_weeks",
]

COMPARABLE_ARM_FEATURES = MAIN_ENROLLMENT_WINDOW_FEATURES.copy()

DYNAMIC_ARM_FEATURES_LINEAR = [
    "week",
    "total_clicks",
    "studied_credits",
    "num_of_prev_attempts",
]

DYNAMIC_ARM_FEATURES_NEURAL = [
    "week",
    "total_clicks",
    "studied_credits",
    "num_of_prev_attempts",
]

# Optional extra comparable-window features that may exist in some
# upstream tables but are not required in the minimal canonical set.
OPTIONAL_COMPARABLE_WINDOW_FEATURES = [
    "active_weeks_first_4",
    "mean_clicks_first_4_weeks",
]

# --------------------------------------------------------------
# 3) Canonical benchmark architecture config
# --------------------------------------------------------------
BENCHMARK_ARMS = {
    "comparable_arm": {
        "description": "Enrollment-level / static-after-early-window comparable benchmark arm.",
        "representation_type": "enrollment_level_or_early_window_summary",
        "update_regime": "static_after_early_window",
        "families": [
            "continuous_time",
            "continuous_time_neural",
            "discrete_time_comparable_minimal",
        ],
    },
    "dynamic_arm": {
        "description": "Weekly-updating person-period dynamic benchmark arm.",
        "representation_type": "dynamic_weekly_person_period",
        "update_regime": "weekly_updating",
        "families": [
            "discrete_time",
            "discrete_time_neural",
        ],
    },
}

EVALUATION_OVERLAYS = {
    "main_split": {
        "description": "Main benchmark split used by the original benchmark.",
        "table_or_protocol": "existing train/test ready tables",
        "currently_active": True,
    },
    "context_heldout_v1": {
        "description": "Alternative grouped context-held-out split for transportability analysis.",
        "table_or_protocol": "enrollment_survival_ready_context_split",
        "currently_active": False,
    },
}

# --------------------------------------------------------------
# 4) Canonical model input variants
# --------------------------------------------------------------
MODEL_INPUT_VARIANTS = {
    "continuous_time": {
        "benchmark_arm": "comparable_arm",
        "family": "continuous_time",
        "representation_type": "enrollment_level_or_early_window_summary",
        "update_regime": "static_after_early_window",
        "input_table_train": "enrollment_cox_ready_train",
        "input_table_test": "enrollment_cox_ready_test",
        "feature_set": COMPARABLE_ARM_FEATURES,
        "optional_feature_set": OPTIONAL_COMPARABLE_WINDOW_FEATURES,
        "currently_materialized": True,
        "included_in_main_results": True,
    },
    "continuous_time_neural": {
        "benchmark_arm": "comparable_arm",
        "family": "continuous_time_neural",
        "representation_type": "enrollment_level_or_early_window_summary",
        "update_regime": "static_after_early_window",
        "input_table_train": "enrollment_deepsurv_ready_train",
        "input_table_test": "enrollment_deepsurv_ready_test",
        "feature_set": COMPARABLE_ARM_FEATURES,
        "optional_feature_set": OPTIONAL_COMPARABLE_WINDOW_FEATURES,
        "currently_materialized": True,
        "included_in_main_results": True,
    },
    "discrete_time_comparable_minimal": {
        "benchmark_arm": "comparable_arm",
        "family": "discrete_time_comparable_minimal",
        "representation_type": "enrollment_level_or_early_window_summary",
        "update_regime": "static_after_early_window",
        "input_table_train": "to_be_materialized",
        "input_table_test": "to_be_materialized",
        "feature_set": COMPARABLE_ARM_FEATURES,
        "optional_feature_set": OPTIONAL_COMPARABLE_WINDOW_FEATURES,
        "currently_materialized": False,
        "included_in_main_results": False,
    },
    "discrete_time": {
        "benchmark_arm": "dynamic_arm",
        "family": "discrete_time",
        "representation_type": "dynamic_weekly_person_period",
        "update_regime": "weekly_updating",
        "input_table_train": "pp_linear_hazard_ready_train",
        "input_table_test": "pp_linear_hazard_ready_test",
        "feature_set": DYNAMIC_ARM_FEATURES_LINEAR,
        "optional_feature_set": [],
        "currently_materialized": True,
        "included_in_main_results": True,
    },
    "discrete_time_neural": {
        "benchmark_arm": "dynamic_arm",
        "family": "discrete_time_neural",
        "representation_type": "dynamic_weekly_person_period",
        "update_regime": "weekly_updating",
        "input_table_train": "pp_neural_hazard_ready_train",
        "input_table_test": "pp_neural_hazard_ready_test",
        "feature_set": DYNAMIC_ARM_FEATURES_NEURAL,
        "optional_feature_set": [],
        "currently_materialized": True,
        "included_in_main_results": True,
    },
}

# --------------------------------------------------------------
# 5) Backward-compatible high-level config object
# --------------------------------------------------------------
MODELING_CONFIG = {
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "early_window_weeks": EARLY_WINDOW_WEEKS,
    "main_enrollment_window_weeks": MAIN_ENROLLMENT_WINDOW_WEEKS,
    "static_features": STATIC_FEATURES,
    "temporal_features_discrete": TEMPORAL_FEATURES_DISCRETE,
    "main_enrollment_window_features": MAIN_ENROLLMENT_WINDOW_FEATURES,
    "optional_comparable_window_features": OPTIONAL_COMPARABLE_WINDOW_FEATURES,
    "benchmark_arms": BENCHMARK_ARMS,
    "evaluation_overlays": EVALUATION_OVERLAYS,
    "model_input_variants": MODEL_INPUT_VARIANTS,
    "main_interpretation_note": (
        "Family-level ranking must be interpreted within benchmark arm. "
        "Cross-arm comparisons are informative but not pure architecture-only comparisons."
    ),
}

# --------------------------------------------------------------
# 6) Tabular export for auditability
# --------------------------------------------------------------
rows = []
for variant_name, cfg in MODEL_INPUT_VARIANTS.items():
    rows.append({
        "variant_name": variant_name,
        "benchmark_arm": cfg["benchmark_arm"],
        "family": cfg["family"],
        "representation_type": cfg["representation_type"],
        "update_regime": cfg["update_regime"],
        "input_table_train": cfg["input_table_train"],
        "input_table_test": cfg["input_table_test"],
        "feature_set": " | ".join(cfg["feature_set"]),
        "optional_feature_set": " | ".join(cfg["optional_feature_set"]),
        "currently_materialized": cfg["currently_materialized"],
        "included_in_main_results": cfg["included_in_main_results"],
    })

table_p10_canonical_modeling_configuration = pd.DataFrame(rows).sort_values(
    ["benchmark_arm", "variant_name"]
).reset_index(drop=True)

path_table = OUT_TABLES / "table_p10_canonical_modeling_configuration.csv"
path_metadata = OUT_METADATA / "metadata_p10_canonical_modeling_configuration.json"

table_p10_canonical_modeling_configuration.to_csv(path_table, index=False)

metadata = {
    "step": "B5",
    "title": "Define Canonical Modeling Configuration Layer",
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "early_window_weeks": EARLY_WINDOW_WEEKS,
    "main_enrollment_window_weeks": MAIN_ENROLLMENT_WINDOW_WEEKS,
    "benchmark_arms": list(BENCHMARK_ARMS.keys()),
    "evaluation_overlays": list(EVALUATION_OVERLAYS.keys()),
    "model_variants": list(MODEL_INPUT_VARIANTS.keys()),
    "canonical_comparable_click_feature": "clicks_first_4_weeks",
    "notes": [
        "B5 now follows the corrected benchmark architecture.",
        "discrete_time_comparable_minimal is now officially part of the canonical configuration.",
        "The canonical comparable click feature is clicks_first_4_weeks, aligned with the notebook.",
        "This step defines configuration only; materialization of the new comparable discrete-time arm occurs downstream."
    ],
    "output_tables": [
        path_table.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 7) Display
# --------------------------------------------------------------
print("\nCanonical model input variants:")
display(table_p10_canonical_modeling_configuration)

print("\nKey config objects created:")
print("- RANDOM_SEED")
print("- TEST_SIZE")
print("- EARLY_WINDOW_WEEKS")
print("- MAIN_ENROLLMENT_WINDOW_WEEKS")
print("- STATIC_FEATURES")
print("- TEMPORAL_FEATURES_DISCRETE")
print("- MAIN_ENROLLMENT_WINDOW_FEATURES")
print("- OPTIONAL_COMPARABLE_WINDOW_FEATURES")
print("- BENCHMARK_ARMS")
print("- EVALUATION_OVERLAYS")
print("- MODEL_INPUT_VARIANTS")
print("- MODELING_CONFIG")

print("\nSaved:")
print("-", path_table.resolve())
print("-", path_metadata.resolve())

### B5.1 — Define Corrected Benchmark Architecture


### Funcao: add_row

Definicao isolada de add_row para reutilizacao nas etapas seguintes.


In [ ]:
# ============================================================
# B5.1 — Define Corrected Benchmark Architecture
# ============================================================
# Purpose:
#   Formalize the corrected benchmark architecture after the main configuration layer.
#   This step does NOT retrain models. It defines:
#     - benchmark arms
#     - family membership by arm
#     - expected input tables
#     - comparability scope
#     - evaluation overlays
#
# Main outputs:
#   - table_p10_1_benchmark_architecture.csv
#   - table_p10_1_benchmark_arm_summary.csv
#   - table_p10_1_benchmark_overlay_summary.csv
#   - metadata_p10_1_benchmark_architecture.json
# ============================================================

import json
from pathlib import Path

import pandas as pd

print("=" * 70)
print("B5.1 — Define Corrected Benchmark Architecture")
print("=" * 70)

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

available_tables = sorted(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

# ------------------------------------------------------------
# 1) Build official architecture table
# ------------------------------------------------------------
rows = []

def add_row(
    benchmark_arm,
    arm_role,
    family,
    model_label,
    status_in_project,
    current_or_future,
    representation_type,
    update_regime,
    expected_input_table,
    expected_split_protocol,
    comparability_scope,
    included_in_current_main_results,
    notes
):
    rows.append({
        "benchmark_arm": benchmark_arm,
        "arm_role": arm_role,
        "family": family,
        "model_label": model_label,
        "status_in_project": status_in_project,
        "current_or_future": current_or_future,
        "representation_type": representation_type,
        "update_regime": update_regime,
        "expected_input_table": expected_input_table,
        "expected_split_protocol": expected_split_protocol,
        "comparability_scope": comparability_scope,
        "included_in_current_main_results": included_in_current_main_results,
        "notes": notes,
    })

# ------------------------------------------------------------
# 1A) Comparable benchmark arm
# ------------------------------------------------------------
add_row(
    benchmark_arm="comparable_arm",
    arm_role="model_family",
    family="continuous_time",
    model_label="Comparable Cox arm",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="enrollment_level_or_early_window_summary",
    update_regime="static_after_early_window",
    expected_input_table="enrollment_cox_*",
    expected_split_protocol="main_split and future context-held-out reruns",
    comparability_scope="within comparable_arm",
    included_in_current_main_results=True,
    notes="Continuous-time comparable arm already exists and operates on static/enrollment-level inputs."
)

add_row(
    benchmark_arm="comparable_arm",
    arm_role="model_family",
    family="continuous_time_neural",
    model_label="Comparable DeepSurv arm",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="enrollment_level_or_early_window_summary",
    update_regime="static_after_early_window",
    expected_input_table="enrollment_deepsurv_*",
    expected_split_protocol="main_split and future context-held-out reruns",
    comparability_scope="within comparable_arm",
    included_in_current_main_results=True,
    notes="Neural continuous-time comparable arm already exists and operates on static/enrollment-level inputs."
)

add_row(
    benchmark_arm="comparable_arm",
    arm_role="model_family",
    family="discrete_time_comparable_minimal",
    model_label="Comparable discrete-time minimal arm",
    status_in_project="to_be_materialized",
    current_or_future="future",
    representation_type="enrollment_level_or_early_window_summary",
    update_regime="static_after_early_window",
    expected_input_table="to_be_defined_from_enrollment_survival_ready / enrollment-level comparable features",
    expected_split_protocol="main_split first, context-held-out optionally later",
    comparability_scope="within comparable_arm",
    included_in_current_main_results=False,
    notes="This future arm is the minimum structural correction needed to compare discrete-time models under the same representation regime as Cox/DeepSurv comparable arms."
)

# ------------------------------------------------------------
# 1B) Dynamic benchmark arm
# ------------------------------------------------------------
add_row(
    benchmark_arm="dynamic_arm",
    arm_role="model_family",
    family="discrete_time",
    model_label="Dynamic linear hazard arm",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="dynamic_weekly_person_period",
    update_regime="weekly_updating",
    expected_input_table="pp_linear_hazard_*",
    expected_split_protocol="main_split and future context-held-out reruns if materialized",
    comparability_scope="within dynamic_arm",
    included_in_current_main_results=True,
    notes="Dynamic discrete-time linear hazard arm already exists and uses person-period rows."
)

add_row(
    benchmark_arm="dynamic_arm",
    arm_role="model_family",
    family="discrete_time_neural",
    model_label="Dynamic neural hazard arm",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="dynamic_weekly_person_period",
    update_regime="weekly_updating",
    expected_input_table="pp_neural_hazard_*",
    expected_split_protocol="main_split and future context-held-out reruns if materialized",
    comparability_scope="within dynamic_arm",
    included_in_current_main_results=True,
    notes="Dynamic discrete-time neural hazard arm already exists and uses person-period rows."
)

# ------------------------------------------------------------
# 1C) Upstream backbones
# ------------------------------------------------------------
add_row(
    benchmark_arm="shared_backbone",
    arm_role="upstream_data",
    family="enrollment_survival_backbone",
    model_label="Enrollment survival backbone",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="enrollment_level_survival",
    update_regime="not_a_model",
    expected_input_table="enrollment_survival_ready",
    expected_split_protocol="feeds all benchmark arms",
    comparability_scope="shared_input_backbone",
    included_in_current_main_results=False,
    notes="Canonical event/time enrollment-level backbone."
)

add_row(
    benchmark_arm="shared_backbone",
    arm_role="upstream_data",
    family="person_period_backbone",
    model_label="Person-period dynamic backbone",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="dynamic_weekly_person_period",
    update_regime="not_a_model",
    expected_input_table="person_period_* / pp_*",
    expected_split_protocol="feeds dynamic benchmark arm",
    comparability_scope="shared_input_backbone",
    included_in_current_main_results=False,
    notes="Canonical dynamic weekly backbone for person-period models."
)

# ------------------------------------------------------------
# 1D) Evaluation overlays
# ------------------------------------------------------------
add_row(
    benchmark_arm="evaluation_overlay",
    arm_role="evaluation_protocol",
    family="main_random_enrollment_split",
    model_label="Main benchmark split",
    status_in_project="already_present",
    current_or_future="current",
    representation_type="split_protocol",
    update_regime="not_a_model",
    expected_input_table="existing train/test ready tables",
    expected_split_protocol="main_split",
    comparability_scope="applies_across_arms",
    included_in_current_main_results=True,
    notes="Main benchmark protocol currently used in the notebook and paper."
)

if "enrollment_survival_ready_context_split" in available_tables:
    add_row(
        benchmark_arm="evaluation_overlay",
        arm_role="evaluation_protocol",
        family="context_heldout_split",
        model_label="Context-held-out split",
        status_in_project="already_present",
        current_or_future="current",
        representation_type="split_protocol",
        update_regime="not_a_model",
        expected_input_table="enrollment_survival_ready_context_split",
        expected_split_protocol="context_heldout_v1",
        comparability_scope="applies_across_arms",
        included_in_current_main_results=False,
        notes="Alternative transportability-oriented split protocol holding out entire (module, presentation) groups."
    )

table_p10_1_benchmark_architecture = pd.DataFrame(rows)

# ------------------------------------------------------------
# 2) Arm-level summary
# ------------------------------------------------------------
table_p10_1_benchmark_arm_summary = (
    table_p10_1_benchmark_architecture
    .groupby(
        ["benchmark_arm", "representation_type", "update_regime", "comparability_scope"],
        dropna=False
    )
    .size()
    .reset_index(name="n_entries")
    .sort_values(["benchmark_arm", "representation_type", "update_regime"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3) Overlay summary
# ------------------------------------------------------------
table_p10_1_benchmark_overlay_summary = (
    table_p10_1_benchmark_architecture
    .loc[table_p10_1_benchmark_architecture["arm_role"] == "evaluation_protocol"]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4) Save outputs
# ------------------------------------------------------------
path_arch = OUT_TABLES / "table_p10_1_benchmark_architecture.csv"
path_arm_summary = OUT_TABLES / "table_p10_1_benchmark_arm_summary.csv"
path_overlay_summary = OUT_TABLES / "table_p10_1_benchmark_overlay_summary.csv"
path_metadata = OUT_METADATA / "metadata_p10_1_benchmark_architecture.json"

table_p10_1_benchmark_architecture.to_csv(path_arch, index=False)
table_p10_1_benchmark_arm_summary.to_csv(path_arm_summary, index=False)
table_p10_1_benchmark_overlay_summary.to_csv(path_overlay_summary, index=False)

metadata = {
    "step": "B5.1",
    "title": "Define Corrected Benchmark Architecture",
    "main_decision": (
        "Benchmark is formally separated into a comparable arm and a dynamic arm, "
        "with evaluation overlays treated as protocols rather than model families."
    ),
    "comparable_arm_definition": (
        "Enrollment-level or early-window summary representations; "
        "static-after-early-window updating regime."
    ),
    "dynamic_arm_definition": (
        "Dynamic weekly person-period representations; weekly-updating regime."
    ),
    "future_structural_task": (
        "Materialize a minimal comparable discrete-time arm so that discrete-time "
        "models can also be compared under the same representation regime as "
        "the comparable Cox and DeepSurv arms."
    ),
    "output_tables": [
        path_arch.as_posix(),
        path_arm_summary.as_posix(),
        path_overlay_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 5) Display
# ------------------------------------------------------------
print("\nBenchmark architecture:")
display(table_p10_1_benchmark_architecture)

print("\nBenchmark arm summary:")
display(table_p10_1_benchmark_arm_summary)

print("\nBenchmark overlay summary:")
display(table_p10_1_benchmark_overlay_summary)

print("\nSaved:")
print("-", path_arch.resolve())
print("-", path_arm_summary.resolve())
print("-", path_overlay_summary.resolve())
print("-", path_metadata.resolve())

### B5.2 — Materialize Canonical Enrollment Window Features

In [ ]:
# ==============================================================
# B5.2 — Materialize Canonical Enrollment Window Features
# --------------------------------------------------------------
# Purpose:
#   Materialize the canonical enrollment-level early-window
#   features used by the comparable enrollment-level benchmark arms.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - MAIN_ENROLLMENT_WINDOW_WEEKS
#   - person_period_enriched
#
# Outputs:
#   - DuckDB table: enrollment_window_features
#   - enrollment_window_features.parquet
#   - enrollment_window_features.csv
#   - table_enrollment_window_features_audit.csv
#   - table_enrollment_window_features_missing_check.csv
# ==============================================================

print("\n" + "=" * 70)
print("B5.2 — Materialize Canonical Enrollment Window Features")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR", "MAIN_ENROLLMENT_WINDOW_WEEKS"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

required_tables = ["person_period_enriched"]
missing_tables = [t for t in required_tables if t not in available_tables]
if missing_tables:
    raise RuntimeError(
        "Missing required DuckDB table(s)/view(s): "
        + ", ".join(missing_tables)
        + ". Please run the required upstream cells first."
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

window_weeks = int(MAIN_ENROLLMENT_WINDOW_WEEKS)

# ------------------------------
# 2) Materialize canonical table
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_window_features")

con.execute(f"""
CREATE TABLE enrollment_window_features AS
WITH early_window AS (
    SELECT
        enrollment_id,
        week,
        total_clicks_week,
        active_this_week,
        n_distinct_sites_week
    FROM person_period_enriched
    WHERE week >= 0
      AND week < {window_weeks}
),
agg AS (
    SELECT
        enrollment_id,
        SUM(COALESCE(total_clicks_week, 0)) AS clicks_first_{window_weeks}_weeks,
        SUM(COALESCE(active_this_week, 0)) AS active_weeks_first_{window_weeks},
        AVG(COALESCE(total_clicks_week, 0)) AS mean_clicks_first_{window_weeks}_weeks,
        AVG(COALESCE(n_distinct_sites_week, 0)) AS mean_distinct_sites_first_{window_weeks}_weeks,
        MAX(week) AS max_week_observed_in_window
    FROM early_window
    GROUP BY enrollment_id
)
SELECT
    enrollment_id,
    clicks_first_{window_weeks}_weeks,
    active_weeks_first_{window_weeks},
    mean_clicks_first_{window_weeks}_weeks,
    mean_distinct_sites_first_{window_weeks}_weeks,
    max_week_observed_in_window
FROM agg
""")

window_df = con.execute("SELECT * FROM enrollment_window_features").fetchdf()

# ------------------------------
# 3) Audits
# ------------------------------
audit_df = pd.DataFrame([{
    "table_name": "enrollment_window_features",
    "window_weeks": window_weeks,
    "n_rows": int(len(window_df)),
    "n_distinct_enrollments": int(window_df["enrollment_id"].nunique()) if len(window_df) > 0 else 0,
    "is_unique_by_enrollment_id": bool(
        len(window_df) == window_df["enrollment_id"].nunique()
    ) if len(window_df) > 0 else True
}])

missing_df = pd.DataFrame([{
    "n_missing_clicks": int(window_df[f"clicks_first_{window_weeks}_weeks"].isna().sum()) if len(window_df) > 0 else 0,
    "n_missing_active_weeks": int(window_df[f"active_weeks_first_{window_weeks}"].isna().sum()) if len(window_df) > 0 else 0,
    "n_missing_mean_clicks": int(window_df[f"mean_clicks_first_{window_weeks}_weeks"].isna().sum()) if len(window_df) > 0 else 0,
    "n_missing_mean_distinct_sites": int(window_df[f"mean_distinct_sites_first_{window_weeks}_weeks"].isna().sum()) if len(window_df) > 0 else 0,
}])

# ------------------------------
# 4) Exports
# ------------------------------
parquet_path = DATA_OUTPUT_DIR / "enrollment_window_features.parquet"
csv_path = DATA_OUTPUT_DIR / "enrollment_window_features.csv"
audit_path = TABLES_DIR / "table_enrollment_window_features_audit.csv"
missing_path = TABLES_DIR / "table_enrollment_window_features_missing_check.csv"

con.execute(f"COPY enrollment_window_features TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
con.execute(f"COPY enrollment_window_features TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_df.to_csv(audit_path, index=False)
missing_df.to_csv(missing_path, index=False)

# ------------------------------
# 5) Output
# ------------------------------
print("\nEnrollment window feature audit:")
display(audit_df)

print("\nMissing check:")
display(missing_df)

print("\nSample rows:")
display(window_df.head(10))

print("\nSaved:")
print("-", parquet_path.resolve())
print("-", csv_path.resolve())
print("-", audit_path.resolve())
print("-", missing_path.resolve())

## B6 — Build Model-Specific Input Views (Revised)


In [ ]:
# ==============================================================
# B6 — Build Model-Specific Input Views (Revised)
# --------------------------------------------------------------
# Purpose:
#   Build model-specific input views aligned with the revised
#   benchmark configuration, including:
#     - person-period inputs for discrete-time models
#     - enrollment-level early-window comparable inputs for Cox
#       and DeepSurv
#
# Methodological note:
#   This revised version keeps the discrete-time inputs unchanged,
#   but refactors the enrollment-level model inputs so that Cox and
#   DeepSurv rely on:
#     - structural static covariates
#     - early-window temporal summaries
#
#   This avoids using retrospective whole-course temporal summaries
#   in the main enrollment-level benchmark track.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - DATA_OUTPUT_DIR
#   - MODEL_INPUT_VARIANTS
#   - MAIN_ENROLLMENT_WINDOW_WEEKS
#
# Outputs:
#   - pp_linear_hazard_input
#   - pp_neural_hazard_input
#   - enrollment_cox_input
#   - enrollment_deepsurv_input
#   - table_model_input_views_audit.csv
# ==============================================================

print("\n" + "=" * 70)
print("B6 — Build Model-Specific Input Views (Revised)")
print("=" * 70)
print("Methodological note: enrollment-level inputs now use the")
print("early-window comparable design instead of whole-course summaries.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "DATA_OUTPUT_DIR",
    "MODEL_INPUT_VARIANTS", "MAIN_ENROLLMENT_WINDOW_WEEKS"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun prior setup cells first."
    )

import pandas as pd
import numpy as np

# ------------------------------
# 2) Determine main window columns
# ------------------------------
if MAIN_ENROLLMENT_WINDOW_WEEKS == 4:
    main_window_cols = [
        "clicks_first_4_weeks",
        "active_weeks_first_4",
        "mean_clicks_first_4_weeks",
    ]
elif MAIN_ENROLLMENT_WINDOW_WEEKS == 2:
    main_window_cols = [
        "clicks_first_2_weeks",
        "active_weeks_first_2",
        "mean_clicks_first_2_weeks",
    ]
else:
    raise ValueError(
        f"Unsupported MAIN_ENROLLMENT_WINDOW_WEEKS={MAIN_ENROLLMENT_WINDOW_WEEKS} in B6."
    )

# ------------------------------
# 3) Discrete-time input views
# ------------------------------
con.execute("DROP TABLE IF EXISTS pp_linear_hazard_input")
con.execute("""
CREATE TABLE pp_linear_hazard_input AS
SELECT
    enrollment_id,
    id_student,
    code_module,
    code_presentation,
    week,
    event_t,
    event_observed,
    t_event_week,
    t_final_week,
    used_zero_week_fallback_for_censoring,
    gender,
    region,
    highest_education,
    imd_band,
    age_band,
    num_of_prev_attempts,
    studied_credits,
    disability,
    total_clicks_week,
    active_this_week,
    n_vle_rows_week,
    n_distinct_sites_week,
    cum_clicks_until_t,
    recency,
    streak
FROM person_period_enriched
""")

con.execute("DROP TABLE IF EXISTS pp_neural_hazard_input")
con.execute("""
CREATE TABLE pp_neural_hazard_input AS
SELECT * FROM pp_linear_hazard_input
""")

# ------------------------------
# 4) Enrollment-level early-window comparable inputs
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_cox_input")
con.execute(f"""
CREATE TABLE enrollment_cox_input AS
SELECT
    b.enrollment_id,
    b.id_student,
    b.code_module,
    b.code_presentation,
    b.event,
    b.duration,
    b.duration_raw,
    b.used_zero_week_fallback_for_censoring,
    b.gender,
    b.region,
    b.highest_education,
    b.imd_band,
    b.age_band,
    b.num_of_prev_attempts,
    b.studied_credits,
    b.disability,
    w.{main_window_cols[0]},
    w.{main_window_cols[1]},
    w.{main_window_cols[2]}
FROM enrollment_model_table b
LEFT JOIN enrollment_window_features w
    ON b.enrollment_id = w.enrollment_id
""")

con.execute("DROP TABLE IF EXISTS enrollment_deepsurv_input")
con.execute("""
CREATE TABLE enrollment_deepsurv_input AS
SELECT * FROM enrollment_cox_input
""")

# ------------------------------
# 5) Audit tables
# ------------------------------
audit_rows = []

for table_name, family, target_col, duration_col in [
    ("pp_linear_hazard_input", "discrete_time", "event_t", None),
    ("pp_neural_hazard_input", "discrete_time", "event_t", None),
    ("enrollment_cox_input", "continuous_time", "event", "duration"),
    ("enrollment_deepsurv_input", "continuous_time", "event", "duration"),
]:
    q = f"SELECT * FROM {table_name}"
    df = con.execute(q).fetchdf()

    row = {
        "table_name": table_name,
        "family": family,
        "n_rows": int(df.shape[0]),
        "n_distinct_enrollments": int(df['enrollment_id'].nunique()) if 'enrollment_id' in df.columns else np.nan,
        "n_columns": int(df.shape[1]),
        "target_column": target_col,
        "target_sum": float(df[target_col].sum()) if target_col in df.columns else np.nan,
        "duration_column": duration_col,
        "min_duration": float(df[duration_col].min()) if duration_col and duration_col in df.columns else np.nan,
        "max_duration": float(df[duration_col].max()) if duration_col and duration_col in df.columns else np.nan,
    }
    audit_rows.append(row)

audit_df = pd.DataFrame(audit_rows)

# sample tables
sample_pp_linear = con.execute("""
SELECT *
FROM pp_linear_hazard_input
ORDER BY enrollment_id, week
LIMIT 10
""").fetchdf()

sample_cox = con.execute("""
SELECT *
FROM enrollment_cox_input
ORDER BY enrollment_id
LIMIT 10
""").fetchdf()

# ------------------------------
# 6) Save outputs
# ------------------------------
audit_path = TABLES_DIR / "table_model_input_views_audit.csv"
sample_pp_path = TABLES_DIR / "table_pp_linear_hazard_input_sample.csv"
sample_cox_path = TABLES_DIR / "table_enrollment_cox_input_sample.csv"

audit_df.to_csv(audit_path, index=False)
sample_pp_linear.to_csv(sample_pp_path, index=False)
sample_cox.to_csv(sample_cox_path, index=False)

# ------------------------------
# 7) Output for feedback
# ------------------------------
print("\nModel input views audit summary:")
display(audit_df)

print("\nSample — pp_linear_hazard_input:")
display(sample_pp_linear)

print("\nSample — enrollment_cox_input:")
display(sample_cox)

print("\nSaved audit:")
print("-", audit_path.resolve())
print("-", sample_pp_path.resolve())
print("-", sample_cox_path.resolve())

## B7 — Attach Canonical Window Features and Build Configured Inputs


In [ ]:
# ==============================================================
# B7 — Attach Canonical Window Features and Build Configured Inputs
# 
# ==============================================================
# Purpose:
#   Attach canonical enrollment-level window features driven by the
#   active configuration variables and add only the
#   non-duplicated columns to model-specific
#   enrollment-level inputs.
#
# Methodological note:
#   This version supports either:
#     - a scalar EARLY_WINDOW_WEEKS
#     - an iterable EARLY_WINDOW_WEEKS
#
#   It prevents duplicated columns when the base enrollment-level
#   inputs already contain the main comparable early-window
#   features.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - EARLY_WINDOW_WEEKS
#   - MODELING_CONFIG
#
# Outputs:
#   - DuckDB tables:
#       * enrollment_window_features
#       * enrollment_cox_input_configured
#       * enrollment_deepsurv_input_configured
#   - exported parquet/csv files
#   - table_enrollment_window_features_audit.csv
#   - table_enrollment_window_features_missing_check.csv
#   - table_enrollment_window_features_created_columns.csv
#   - table_enrollment_window_feature_attach_plan.csv
# ==============================================================

print("\n" + "=" * 70)
print("B7 — Build Window-Based Features and Experimental Variants")
print("=" * 70)
print("Methodological note: window-based features are generated from configuration variables.")
print("Convention adopted: first_w_weeks means rows where week < w.")
print("This version supports scalar or iterable EARLY_WINDOW_WEEKS.")
print("This version avoids duplicated columns in configured enrollment-level inputs.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR", "EARLY_WINDOW_WEEKS", "MODELING_CONFIG"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd
import numpy as np
from pathlib import Path

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Validate and normalize windows
# ------------------------------
if isinstance(EARLY_WINDOW_WEEKS, (list, tuple, set)):
    WINDOWS = sorted(set(int(w) for w in EARLY_WINDOW_WEEKS))
else:
    WINDOWS = [int(EARLY_WINDOW_WEEKS)]

if len(WINDOWS) == 0:
    raise ValueError("No valid EARLY_WINDOW_WEEKS were provided.")

if any(w <= 0 for w in WINDOWS):
    raise ValueError(f"All EARLY_WINDOW_WEEKS must be positive integers. Got: {WINDOWS}")

print(f"\nConfigured EARLY_WINDOW_WEEKS: {WINDOWS}")


### Funcao: get_table_columns

Definicao isolada de get_table_columns para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 3) Helper utilities
# ------------------------------
def get_table_columns(table_name: str) -> list[str]:
    df_cols = con.execute(f"SELECT * FROM {table_name} LIMIT 0").df()
    return list(df_cols.columns)


### Funcao: duplicated_names

Definicao isolada de duplicated_names para reutilizacao nas etapas seguintes.


In [ ]:

def duplicated_names(cols: list[str]) -> list[str]:
    seen = set()
    dup = []
    for c in cols:
        if c in seen and c not in dup:
            dup.append(c)
        seen.add(c)
    return dup


### Funcao: build_window_select_fragment

Definicao isolada de build_window_select_fragment para reutilizacao nas etapas seguintes.


In [ ]:

def build_window_select_fragment(cols: list[str], alias: str = "ewf") -> str:
    if not cols:
        return ""
    return ",\n    " + ",\n    ".join(f"{alias}.{c}" for c in cols)


In [ ]:

# ------------------------------
# 4) Build dynamic SQL for window features
# ------------------------------
window_feature_sql_parts = []
created_feature_names = []

for w in WINDOWS:
    clicks_col = f"clicks_first_{w}_weeks"
    active_col = f"active_weeks_first_{w}"
    mean_col = f"mean_clicks_first_{w}_weeks"

    created_feature_names.extend([clicks_col, active_col, mean_col])

    window_feature_sql_parts.append(f"""
    COALESCE(SUM(CASE WHEN week < {w} THEN total_clicks_week ELSE 0 END), 0) AS {clicks_col},
    COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0) AS {active_col},
    CASE
        WHEN COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0) > 0
        THEN
            COALESCE(SUM(CASE WHEN week < {w} THEN total_clicks_week ELSE 0 END), 0) * 1.0
            / COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0)
        ELSE 0
    END AS {mean_col}
    """)

window_feature_sql = ",\n".join(window_feature_sql_parts)

# ------------------------------
# 5) Build enrollment_window_features
# ------------------------------
required_source_table = "person_period_enriched"
if required_source_table not in con.execute("SHOW TABLES").fetchdf()["name"].tolist():
    raise RuntimeError(
        "Required source table 'person_period_enriched' not found. "
        "Run prior preprocessing/person-period steps first."
    )

required_pp_cols = {
    "enrollment_id",
    "week",
    "total_clicks_week",
    "active_this_week",
}
pp_cols = set(get_table_columns(required_source_table))
missing_pp_cols = sorted(required_pp_cols - pp_cols)
if missing_pp_cols:
    raise KeyError(
        "person_period_enriched is missing required columns for B7: "
        + ", ".join(missing_pp_cols)
    )

con.execute("DROP TABLE IF EXISTS enrollment_window_features")
con.execute(f"""
CREATE TABLE enrollment_window_features AS
SELECT
    enrollment_id,
    {window_feature_sql}
FROM person_period_enriched
GROUP BY enrollment_id
""")

# ------------------------------
# 6) Discover base/input columns dynamically
# ------------------------------
for base_table in ["enrollment_cox_input", "enrollment_deepsurv_input"]:
    if base_table not in con.execute("SHOW TABLES").fetchdf()["name"].tolist():
        raise RuntimeError(
            f"Required table '{base_table}' not found. Run B6 first."
        )

cox_base_cols = get_table_columns("enrollment_cox_input")
deepsurv_base_cols = get_table_columns("enrollment_deepsurv_input")
window_cols = [c for c in get_table_columns("enrollment_window_features") if c != "enrollment_id"]

cox_extra_window_cols = [c for c in window_cols if c not in cox_base_cols]
deepsurv_extra_window_cols = [c for c in window_cols if c not in deepsurv_base_cols]

cox_window_fragment = build_window_select_fragment(cox_extra_window_cols, alias="ewf")
deepsurv_window_fragment = build_window_select_fragment(deepsurv_extra_window_cols, alias="ewf")

attach_plan_df = pd.DataFrame([
    {
        "target_table": "enrollment_cox_input_configured",
        "base_table": "enrollment_cox_input",
        "n_base_columns": len(cox_base_cols),
        "n_window_columns_available": len(window_cols),
        "n_window_columns_attached": len(cox_extra_window_cols),
        "attached_window_columns": ", ".join(cox_extra_window_cols),
        "n_window_columns_skipped_as_already_present": len([c for c in window_cols if c in cox_base_cols]),
        "skipped_window_columns": ", ".join([c for c in window_cols if c in cox_base_cols]),
    },
    {
        "target_table": "enrollment_deepsurv_input_configured",
        "base_table": "enrollment_deepsurv_input",
        "n_base_columns": len(deepsurv_base_cols),
        "n_window_columns_available": len(window_cols),
        "n_window_columns_attached": len(deepsurv_extra_window_cols),
        "attached_window_columns": ", ".join(deepsurv_extra_window_cols),
        "n_window_columns_skipped_as_already_present": len([c for c in window_cols if c in deepsurv_base_cols]),
        "skipped_window_columns": ", ".join([c for c in window_cols if c in deepsurv_base_cols]),
    },
])

# ------------------------------
# 7) Build configured Cox and DeepSurv inputs without duplication
# ------------------------------
con.execute("DROP TABLE IF EXISTS enrollment_cox_input_configured")
con.execute(f"""
CREATE TABLE enrollment_cox_input_configured AS
SELECT
    eci.*{cox_window_fragment}
FROM enrollment_cox_input AS eci
LEFT JOIN enrollment_window_features AS ewf
    ON eci.enrollment_id = ewf.enrollment_id
""")

con.execute("DROP TABLE IF EXISTS enrollment_deepsurv_input_configured")
con.execute(f"""
CREATE TABLE enrollment_deepsurv_input_configured AS
SELECT
    edi.*{deepsurv_window_fragment}
FROM enrollment_deepsurv_input AS edi
LEFT JOIN enrollment_window_features AS ewf
    ON edi.enrollment_id = ewf.enrollment_id
""")

# ------------------------------
# 8) Main audit summary
# ------------------------------
audit_summary = con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments
FROM enrollment_window_features
""").fetchdf()

audit_summary["n_windows_configured"] = len(WINDOWS)
audit_summary["window_list"] = [", ".join(str(w) for w in WINDOWS)]
audit_summary["n_created_features"] = len(created_feature_names)

# ------------------------------
# 9) Missing check for created features
# ------------------------------
missing_exprs = []
for feat in created_feature_names:
    missing_exprs.append(f"SUM(CASE WHEN {feat} IS NULL THEN 1 ELSE 0 END) AS n_missing_{feat}")

missing_sql = ",\n    ".join(missing_exprs)

missing_check = con.execute(f"""
SELECT
    {missing_sql}
FROM enrollment_window_features
""").fetchdf()

# ------------------------------
# 10) Duplicate safety checks
# ------------------------------
configured_cox_cols = get_table_columns("enrollment_cox_input_configured")
configured_deepsurv_cols = get_table_columns("enrollment_deepsurv_input_configured")

cox_dup_cols = duplicated_names(configured_cox_cols)
deepsurv_dup_cols = duplicated_names(configured_deepsurv_cols)

if cox_dup_cols:
    raise ValueError(f"Duplicated columns detected in enrollment_cox_input_configured: {cox_dup_cols}")

if deepsurv_dup_cols:
    raise ValueError(f"Duplicated columns detected in enrollment_deepsurv_input_configured: {deepsurv_dup_cols}")

suffixed_artifacts = [
    c for c in configured_cox_cols if c.endswith("_1")
] + [
    c for c in configured_deepsurv_cols if c.endswith("_1")
]
if suffixed_artifacts:
    raise ValueError(
        "Suffixed duplicate artifact columns detected after configured join: "
        + ", ".join(sorted(set(suffixed_artifacts)))
    )

# ------------------------------
# 11) Sample rows
# ------------------------------
sample_rows = con.execute("""
SELECT *
FROM enrollment_cox_input_configured
ORDER BY enrollment_id
LIMIT 20
""").fetchdf()

# ------------------------------
# 12) Export tables
# ------------------------------
export_tables = [
    "enrollment_window_features",
    "enrollment_cox_input_configured",
    "enrollment_deepsurv_input_configured",
]

for table_name in export_tables:
    parquet_path = DATA_OUTPUT_DIR / f"{table_name}.parquet"
    csv_path = DATA_OUTPUT_DIR / f"{table_name}.csv"
    con.execute(f"COPY {table_name} TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
    con.execute(f"COPY {table_name} TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

audit_path = TABLES_DIR / "table_enrollment_window_features_audit.csv"
missing_path = TABLES_DIR / "table_enrollment_window_features_missing_check.csv"
feature_list_path = TABLES_DIR / "table_enrollment_window_features_created_columns.csv"
attach_plan_path = TABLES_DIR / "table_enrollment_window_feature_attach_plan.csv"
sample_path = TABLES_DIR / "table_enrollment_cox_input_configured_sample.csv"

audit_summary.to_csv(audit_path, index=False)
missing_check.to_csv(missing_path, index=False)
pd.DataFrame({"created_feature_name": created_feature_names}).to_csv(feature_list_path, index=False)
attach_plan_df.to_csv(attach_plan_path, index=False)
sample_rows.to_csv(sample_path, index=False)

# ------------------------------
# 13) Output for feedback
# ------------------------------
print("\nWindow configuration used:")
print(" - EARLY_WINDOW_WEEKS:", WINDOWS)

print("\nCreated window-based feature columns:")
for feat in created_feature_names:
    print(" -", feat)

print("\nWindow attachment plan:")
display(attach_plan_df)

print("\nEnrollment window features audit summary:")
display(audit_summary)

print("\nMissing check for created features:")
display(missing_check)

print("\nSample — enrollment_cox_input_configured:")
display(sample_rows)

print("\nSaved:")
print("-", audit_path.resolve())
print("-", missing_path.resolve())
print("-", feature_list_path.resolve())
print("-", attach_plan_path.resolve())
print("-", sample_path.resolve())

print("\nExported tables:")
for table_name in export_tables:
    print("-", (DATA_OUTPUT_DIR / f"{table_name}.parquet").resolve())
    print("-", (DATA_OUTPUT_DIR / f"{table_name}.csv").resolve())


### B7.1 — Early-Window Length Sensitivity Design


In [ ]:
# ==============================================================
# B7.1 — Early-Window Length Sensitivity Design
# ==============================================================
# Purpose:
#   Build and audit a canonical early-window sensitivity design
#   across multiple window lengths, using enrollment-level
#   comparable inputs as the main structural target.
#
# Design:
#   - materialize window features for 2, 4, and 6 weeks
#   - build canonical comparable enrollment-level variants
#   - avoid duplicated columns
#
# Main outputs:
#   - enrollment_window_features_sensitivity
#   - enrollment_cox_input_w2
#   - enrollment_cox_input_w4
#   - enrollment_cox_input_w6
#   - enrollment_deepsurv_input_w2
#   - enrollment_deepsurv_input_w4
#   - enrollment_deepsurv_input_w6
#   - table_p12_1_window_sensitivity_design.csv
#   - table_p12_1_window_sensitivity_attach_plan.csv
#   - metadata_p12_1_window_sensitivity_design.json
# ==============================================================

print("\n" + "=" * 70)
print("B7.1 — Early-Window Length Sensitivity Design")
print("=" * 70)

required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import json
from pathlib import Path
import pandas as pd

OUT_BASE = Path(OUTPUT_DIR)
DATA_OUTPUT_DIR = OUT_BASE / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOWS_SENSITIVITY = [2, 4, 6]

# --------------------------------------------------------------
# 1) Basic source checks
# --------------------------------------------------------------
available_tables = set(con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist())

required_tables = [
    "person_period_enriched",
    "enrollment_cox_input",
    "enrollment_deepsurv_input",
]
missing_tables = [t for t in required_tables if t not in available_tables]
if missing_tables:
    raise RuntimeError(
        "Missing required tables for B7.1: " + ", ".join(missing_tables)
    )


### Funcao: get_table_columns

Definicao isolada de get_table_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_table_columns(table_name: str) -> list[str]:
    return list(con.execute(f"SELECT * FROM {table_name} LIMIT 0").df().columns)


In [ ]:

pp_cols = set(get_table_columns("person_period_enriched"))
required_pp_cols = {"enrollment_id", "week", "total_clicks_week", "active_this_week"}
missing_pp_cols = sorted(required_pp_cols - pp_cols)
if missing_pp_cols:
    raise KeyError(
        "person_period_enriched is missing required columns for B7.1: "
        + ", ".join(missing_pp_cols)
    )

# --------------------------------------------------------------
# 2) Build full sensitivity window feature table
# --------------------------------------------------------------
window_feature_sql_parts = []
created_feature_names = []

for w in WINDOWS_SENSITIVITY:
    clicks_col = f"clicks_first_{w}_weeks"
    active_col = f"active_weeks_first_{w}"
    mean_col = f"mean_clicks_first_{w}_weeks"

    created_feature_names.extend([clicks_col, active_col, mean_col])

    window_feature_sql_parts.append(f"""
    COALESCE(SUM(CASE WHEN week < {w} THEN total_clicks_week ELSE 0 END), 0) AS {clicks_col},
    COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0) AS {active_col},
    CASE
        WHEN COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0) > 0
        THEN
            COALESCE(SUM(CASE WHEN week < {w} THEN total_clicks_week ELSE 0 END), 0) * 1.0
            / COALESCE(SUM(CASE WHEN week < {w} THEN active_this_week ELSE 0 END), 0)
        ELSE 0
    END AS {mean_col}
    """)

window_feature_sql = ",\n".join(window_feature_sql_parts)

con.execute("DROP TABLE IF EXISTS enrollment_window_features_sensitivity")
con.execute(f"""
CREATE TABLE enrollment_window_features_sensitivity AS
SELECT
    enrollment_id,
    {window_feature_sql}
FROM person_period_enriched
GROUP BY enrollment_id
""")


### Funcao: build_window_variant

Definicao isolada de build_window_variant para reutilizacao nas etapas seguintes.


In [ ]:

# --------------------------------------------------------------
# 3) Materialize window-specific comparable variants
# --------------------------------------------------------------
def build_window_variant(base_table: str, out_table: str, window_w: int):
    base_cols = get_table_columns(base_table)

    target_cols = [
        f"clicks_first_{window_w}_weeks",
        f"active_weeks_first_{window_w}",
        f"mean_clicks_first_{window_w}_weeks",
    ]

    extra_cols = [c for c in target_cols if c not in base_cols]

    fragment = ""
    if extra_cols:
        fragment = ",\n    " + ",\n    ".join(f"ewf.{c}" for c in extra_cols)

    con.execute(f"DROP TABLE IF EXISTS {out_table}")
    con.execute(f"""
    CREATE TABLE {out_table} AS
    SELECT
        b.*{fragment}
    FROM {base_table} AS b
    LEFT JOIN enrollment_window_features_sensitivity AS ewf
        ON b.enrollment_id = ewf.enrollment_id
    """)

    final_cols = get_table_columns(out_table)
    duplicated = [c for i, c in enumerate(final_cols) if c in final_cols[:i]]
    if duplicated:
        raise ValueError(f"Duplicated columns detected in {out_table}: {duplicated}")

    return {
        "target_table": out_table,
        "base_table": base_table,
        "window_weeks": window_w,
        "n_base_columns": len(base_cols),
        "n_target_window_columns": len(target_cols),
        "n_window_columns_attached": len(extra_cols),
        "attached_window_columns": ", ".join(extra_cols),
        "n_window_columns_already_present": len([c for c in target_cols if c in base_cols]),
        "already_present_window_columns": ", ".join([c for c in target_cols if c in base_cols]),
    }


In [ ]:

attach_rows = []

for w in WINDOWS_SENSITIVITY:
    attach_rows.append(build_window_variant("enrollment_cox_input", f"enrollment_cox_input_w{w}", w))
    attach_rows.append(build_window_variant("enrollment_deepsurv_input", f"enrollment_deepsurv_input_w{w}", w))

table_p12_1_window_sensitivity_attach_plan = pd.DataFrame(attach_rows)

# --------------------------------------------------------------
# 4) Build design summary
# --------------------------------------------------------------
summary_rows = []

ewf_audit = con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT enrollment_id) AS n_distinct_enrollments
FROM enrollment_window_features_sensitivity
""").fetchdf().iloc[0]

for w in WINDOWS_SENSITIVITY:
    for family, table_name in [
        ("continuous_time_cox", f"enrollment_cox_input_w{w}"),
        ("continuous_time_deepsurv", f"enrollment_deepsurv_input_w{w}"),
    ]:
        df = con.execute(f"SELECT * FROM {table_name}").fetchdf()
        summary_rows.append({
            "window_weeks": w,
            "family": family,
            "table_name": table_name,
            "n_rows": int(df.shape[0]),
            "n_distinct_enrollments": int(df["enrollment_id"].nunique()) if "enrollment_id" in df.columns else None,
            "n_columns": int(df.shape[1]),
            "has_clicks_feature": f"clicks_first_{w}_weeks" in df.columns,
            "has_active_feature": f"active_weeks_first_{w}" in df.columns,
            "has_mean_feature": f"mean_clicks_first_{w}_weeks" in df.columns,
        })

table_p12_1_window_sensitivity_design = pd.DataFrame(summary_rows)

# --------------------------------------------------------------
# 5) Save outputs
# --------------------------------------------------------------
path_design = TABLES_DIR / "table_p12_1_window_sensitivity_design.csv"
path_attach = TABLES_DIR / "table_p12_1_window_sensitivity_attach_plan.csv"
path_created = TABLES_DIR / "table_p12_1_window_sensitivity_created_columns.csv"
path_metadata = Path(OUTPUT_DIR) / "metadata" / "metadata_p12_1_window_sensitivity_design.json"
path_metadata.parent.mkdir(parents=True, exist_ok=True)

table_p12_1_window_sensitivity_design.to_csv(path_design, index=False)
table_p12_1_window_sensitivity_attach_plan.to_csv(path_attach, index=False)
pd.DataFrame({"created_feature_name": created_feature_names}).to_csv(path_created, index=False)

metadata = {
    "step": "B7.1",
    "title": "Early-Window Length Sensitivity Design",
    "windows_sensitivity": WINDOWS_SENSITIVITY,
    "n_rows_window_feature_table": int(ewf_audit["n_rows"]),
    "n_distinct_enrollments_window_feature_table": int(ewf_audit["n_distinct_enrollments"]),
    "output_tables": [
        path_design.as_posix(),
        path_attach.as_posix(),
        path_created.as_posix(),
    ],
    "materialized_duckdb_tables": [
        "enrollment_window_features_sensitivity",
        "enrollment_cox_input_w2",
        "enrollment_cox_input_w4",
        "enrollment_cox_input_w6",
        "enrollment_deepsurv_input_w2",
        "enrollment_deepsurv_input_w4",
        "enrollment_deepsurv_input_w6",
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 6) Display
# --------------------------------------------------------------
print("\nSensitivity windows used:")
print(WINDOWS_SENSITIVITY)

print("\nWindow sensitivity attach plan:")
display(table_p12_1_window_sensitivity_attach_plan)

print("\nWindow sensitivity design summary:")
display(table_p12_1_window_sensitivity_design)

print("\nSaved:")
print("-", path_design.resolve())
print("-", path_attach.resolve())
print("-", path_created.resolve())
print("-", path_metadata.resolve())


### B7.2 — Early-Window Length Sensitivity Comparison


### Funcao: get_table_columns

Definicao isolada de get_table_columns para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# B7.2 — Early-Window Length Sensitivity Comparison
# ==============================================================
# Purpose:
#   Consolidate the early-window sensitivity design into a
#   canonical comparison table for the feature-engineering section.
#
# Main outputs:
#   - table_p12_2_window_sensitivity_comparison.csv
#   - table_p12_2_window_sensitivity_summary.csv
#   - table_p12_2_window_sensitivity_registry.csv
#   - metadata_p12_2_window_sensitivity_comparison.json
# ==============================================================

print("\n" + "=" * 70)
print("B7.2 — Early-Window Length Sensitivity Comparison")
print("=" * 70)

required_names = ["con", "TABLES_DIR", "OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import json
from pathlib import Path
import pandas as pd
import numpy as np

OUT_BASE = Path(OUTPUT_DIR)
OUT_METADATA = OUT_BASE / "metadata"
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Resolve current materialized sensitivity variants
# --------------------------------------------------------------
available_tables = set(con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist())

windows = [2, 4, 6]
families = [
    ("continuous_time_cox", "enrollment_cox_input_w{w}"),
    ("continuous_time_deepsurv", "enrollment_deepsurv_input_w{w}"),
]

registry_rows = []
comparison_rows = []

def get_table_columns(table_name: str) -> list[str]:
    return list(con.execute(f"SELECT * FROM {table_name} LIMIT 0").df().columns)

for family_name, template in families:
    for w in windows:
        table_name = template.format(w=w)
        exists = table_name in available_tables

        row = {
            "window_weeks": w,
            "family": family_name,
            "table_name": table_name,
            "table_exists": exists,
            "n_rows": np.nan,
            "n_distinct_enrollments": np.nan,
            "n_columns": np.nan,
            "has_clicks_feature": False,
            "has_active_feature": False,
            "has_mean_feature": False,
            "event_sum": np.nan,
            "duration_min": np.nan,
            "duration_max": np.nan,
            "ready_for_model_training": False,
            "notes": "",
        }

        if exists:
            df = con.execute(f"SELECT * FROM {table_name}").fetchdf()
            cols = set(df.columns)

            clicks_col = f"clicks_first_{w}_weeks"
            active_col = f"active_weeks_first_{w}"
            mean_col = f"mean_clicks_first_{w}_weeks"

            row["n_rows"] = int(df.shape[0])
            row["n_distinct_enrollments"] = int(df["enrollment_id"].nunique()) if "enrollment_id" in df.columns else np.nan
            row["n_columns"] = int(df.shape[1])
            row["has_clicks_feature"] = clicks_col in cols
            row["has_active_feature"] = active_col in cols
            row["has_mean_feature"] = mean_col in cols

            if "event" in cols:
                row["event_sum"] = float(pd.to_numeric(df["event"], errors="coerce").fillna(0).sum())
            if "duration" in cols:
                row["duration_min"] = float(pd.to_numeric(df["duration"], errors="coerce").min())
                row["duration_max"] = float(pd.to_numeric(df["duration"], errors="coerce").max())

            row["ready_for_model_training"] = bool(
                row["has_clicks_feature"]
                and row["has_active_feature"]
                and row["has_mean_feature"]
                and ("event" in cols)
                and ("duration" in cols)
            )

            if w == 4:
                row["notes"] = "Main comparable early-window benchmark design."
            else:
                row["notes"] = "Sensitivity variant materialized and structurally ready."
        else:
            row["notes"] = "Variant table not materialized."

        registry_rows.append(row)

# canonical registry
table_p12_2_window_sensitivity_registry = (
    pd.DataFrame(registry_rows)
    .sort_values(["family", "window_weeks"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 2) Comparison table
# --------------------------------------------------------------
for family_name, g in table_p12_2_window_sensitivity_registry.groupby("family", dropna=False):
    main_row = g.loc[g["window_weeks"] == 4].copy()
    if len(main_row) == 0:
        for _, r in g.iterrows():
            comparison_rows.append({
                "family": family_name,
                "window_weeks": r["window_weeks"],
                "table_name": r["table_name"],
                "table_exists": r["table_exists"],
                "ready_for_model_training": r["ready_for_model_training"],
                "delta_n_columns_vs_w4": np.nan,
                "delta_event_sum_vs_w4": np.nan,
                "delta_duration_max_vs_w4": np.nan,
                "comparison_status": "missing_w4_reference",
                "notes": "No 4-week reference row available for comparison.",
            })
        continue

    main = main_row.iloc[0]

    for _, r in g.iterrows():
        comparison_rows.append({
            "family": family_name,
            "window_weeks": r["window_weeks"],
            "table_name": r["table_name"],
            "table_exists": r["table_exists"],
            "ready_for_model_training": r["ready_for_model_training"],
            "delta_n_columns_vs_w4": (
                pd.to_numeric(r["n_columns"], errors="coerce")
                - pd.to_numeric(main["n_columns"], errors="coerce")
            ) if pd.notna(r["n_columns"]) and pd.notna(main["n_columns"]) else np.nan,
            "delta_event_sum_vs_w4": (
                pd.to_numeric(r["event_sum"], errors="coerce")
                - pd.to_numeric(main["event_sum"], errors="coerce")
            ) if pd.notna(r["event_sum"]) and pd.notna(main["event_sum"]) else np.nan,
            "delta_duration_max_vs_w4": (
                pd.to_numeric(r["duration_max"], errors="coerce")
                - pd.to_numeric(main["duration_max"], errors="coerce")
            ) if pd.notna(r["duration_max"]) and pd.notna(main["duration_max"]) else np.nan,
            "comparison_status": (
                "reference_main_window"
                if r["window_weeks"] == 4 else
                "sensitivity_variant_ready"
                if bool(r["ready_for_model_training"]) else
                "not_ready"
            ),
            "notes": (
                "4-week reference window."
                if r["window_weeks"] == 4 else
                "Sensitivity variant can be used for downstream window-length comparison."
            ),
        })

table_p12_2_window_sensitivity_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(["family", "window_weeks"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 3) Summary table
# --------------------------------------------------------------
summary_rows = []
for family_name, g in table_p12_2_window_sensitivity_registry.groupby("family", dropna=False):
    summary_rows.append({
        "family": family_name,
        "n_windows_expected": len(windows),
        "n_windows_materialized": int(pd.to_numeric(g["table_exists"], errors="coerce").fillna(False).sum()),
        "n_windows_ready_for_model_training": int(pd.to_numeric(g["ready_for_model_training"], errors="coerce").fillna(False).sum()),
        "all_windows_materialized": bool(g["table_exists"].all()),
        "all_windows_ready_for_model_training": bool(g["ready_for_model_training"].all()),
        "window_list_materialized": ", ".join(str(int(w)) for w in g.loc[g["table_exists"], "window_weeks"].tolist()),
    })

table_p12_2_window_sensitivity_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("family")
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 4) Save outputs
# --------------------------------------------------------------
path_registry = TABLES_DIR / "table_p12_2_window_sensitivity_registry.csv"
path_comparison = TABLES_DIR / "table_p12_2_window_sensitivity_comparison.csv"
path_summary = TABLES_DIR / "table_p12_2_window_sensitivity_summary.csv"
path_metadata = OUT_METADATA / "metadata_p12_2_window_sensitivity_comparison.json"

table_p12_2_window_sensitivity_registry.to_csv(path_registry, index=False)
table_p12_2_window_sensitivity_comparison.to_csv(path_comparison, index=False)
table_p12_2_window_sensitivity_summary.to_csv(path_summary, index=False)

metadata = {
    "step": "B7.2",
    "title": "Early-Window Length Sensitivity Comparison",
    "windows_evaluated_structurally": windows,
    "families": [f[0] for f in families],
    "purpose": (
        "Consolidate the currently materialized early-window sensitivity variants "
        "into a canonical comparison table for the feature-engineering section."
    ),
    "output_tables": [
        path_registry.as_posix(),
        path_comparison.as_posix(),
        path_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 5) Display
# --------------------------------------------------------------
print("\nWindow sensitivity registry:")
display(table_p12_2_window_sensitivity_registry)

print("\nWindow sensitivity comparison:")
display(table_p12_2_window_sensitivity_comparison)

print("\nWindow sensitivity summary:")
display(table_p12_2_window_sensitivity_summary)

print("\nSaved:")
print("-", path_registry.resolve())
print("-", path_comparison.resolve())
print("-", path_summary.resolve())
print("-", path_metadata.resolve())

# C — Split design, audit, and modeling-ready partitions

## C1 — Event, Time, and Censoring Audit

In [ ]:
# ============================================================
# C1 — Event, Time, and Censoring Audit
# FINAL SUBSTITUTE VERSION
# ============================================================
# Purpose:
#   Audit the current survival operationalization before introducing
#   stronger split designs. This step does not alter the benchmark
#   data. It generates auditable evidence about:
#     - unit of analysis
#     - event definition
#     - censoring conventions
#     - zero-activity and pre-start activity cases
#     - temporal consistency
#     - course/presentation duration variation
#     - alignment between enrollment-level and person-period data
#     - concrete examples of flagged cases
#
# Expected inputs from previous cells:
#   - con
#   - enrollment_survival_ready table (from P5)
#   - courses table (if available)
#   - some person-period table (if available)
#
# Main outputs:
#   - table_p13_0_event_time_censoring_summary.csv
#   - table_p13_0_event_definition_breakdown.csv
#   - table_p13_0_temporal_consistency_checks.csv
#   - table_p13_0_zero_activity_breakdown.csv
#   - table_p13_0_course_duration_audit.csv
#   - table_p13_0_person_period_alignment.csv
#   - table_p13_0_duration_overrun_summary.csv
#   - table_p13_0_problem_examples.csv
#   - metadata_p13_0_event_time_censoring_audit.json
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C1 — Event, Time, and Censoring Audit")
print("=" * 70)
print("Substitute/final audit version.")
print("This step audits the current survival operationalization.")
print("It does not modify the event logic or the split.")
print("It generates auditable evidence for event definition,")
print("censoring conventions, temporal consistency, duration")
print("overruns, and enrollment/person-period alignment.")

# ------------------------------------------------------------
# 0) Basic checks
# ------------------------------------------------------------
required_globals = ["con"]
missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise NameError(
        f"Missing required objects from previous steps: {missing_globals}. "
        "Run prior setup cells first."
    )

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1) Resolve required enrollment-level survival table
# ------------------------------------------------------------
available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

if "enrollment_survival_ready" not in available_tables:
    raise RuntimeError(
        "DuckDB table 'enrollment_survival_ready' not found. "
        "Run P5 before P13.0."
    )

enr = con.execute("""
    SELECT *
    FROM enrollment_survival_ready
""").fetchdf()

required_enr_cols = [
    "id_student", "code_module", "code_presentation",
    "final_result", "date_registration", "date_unregistration",
    "is_withdrawn", "has_valid_unregistration_date",
    "event_observed", "is_withdrawn_without_valid_unregistration",
    "has_any_vle_activity", "max_vle_day", "n_vle_rows",
    "total_clicks_all_time", "t_event_week", "t_last_obs_week",
    "t_final_week", "used_zero_week_fallback_for_censoring"
]
missing_enr_cols = [c for c in required_enr_cols if c not in enr.columns]
if missing_enr_cols:
    raise KeyError(
        f"'enrollment_survival_ready' is missing required columns: {missing_enr_cols}"
    )

enr = enr.copy()
enr["enrollment_id"] = (
    enr["id_student"].astype(str)
    + "||" + enr["code_module"].astype(str)
    + "||" + enr["code_presentation"].astype(str)
)

# numeric coercions
date_reg = pd.to_numeric(enr["date_registration"], errors="coerce")
date_unreg = pd.to_numeric(enr["date_unregistration"], errors="coerce")
t_event = pd.to_numeric(enr["t_event_week"], errors="coerce")
t_last_obs = pd.to_numeric(enr["t_last_obs_week"], errors="coerce")
t_final = pd.to_numeric(enr["t_final_week"], errors="coerce")
max_vle_day = pd.to_numeric(enr["max_vle_day"], errors="coerce")
n_vle_rows = pd.to_numeric(enr["n_vle_rows"], errors="coerce")
total_clicks_all_time = pd.to_numeric(enr["total_clicks_all_time"], errors="coerce")

# ------------------------------------------------------------
# 2) Enrollment-level core summary
# ------------------------------------------------------------
n_total = len(enr)
n_unique_enrollment = enr["enrollment_id"].nunique()
n_unique_student = enr["id_student"].nunique()

n_withdrawn_total = int(pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0).sum())
n_event_observed = int(pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0).sum())
n_withdrawn_without_valid_unreg = int(
    pd.to_numeric(enr["is_withdrawn_without_valid_unregistration"], errors="coerce").fillna(0).sum()
)
n_nonwithdrawn = int((pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0) == 0).sum())

n_any_vle = int(pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0).sum())
n_no_vle = int((pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 0).sum())
n_zero_week_fallback = int(
    pd.to_numeric(enr["used_zero_week_fallback_for_censoring"], errors="coerce").fillna(0).sum()
)

n_negative_max_vle_day = int((max_vle_day < 0).fillna(False).sum())
n_negative_t_last_obs = int((t_last_obs < 0).fillna(False).sum())
n_negative_t_final = int((t_final < 0).fillna(False).sum())

table_p13_0_event_time_censoring_summary = pd.DataFrame([{
    "n_total_enrollments": int(n_total),
    "n_unique_enrollments": int(n_unique_enrollment),
    "n_unique_students": int(n_unique_student),
    "enrollment_key_unique": bool(n_total == n_unique_enrollment),
    "n_withdrawn_total": n_withdrawn_total,
    "n_event_observed": n_event_observed,
    "event_rate": float(pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0).mean()),
    "n_withdrawn_without_valid_unregistration": n_withdrawn_without_valid_unreg,
    "share_withdrawn_without_valid_unregistration_among_withdrawn":
        float(n_withdrawn_without_valid_unreg / n_withdrawn_total) if n_withdrawn_total > 0 else np.nan,
    "n_nonwithdrawn": n_nonwithdrawn,
    "n_with_any_vle_activity": n_any_vle,
    "n_without_vle_activity": n_no_vle,
    "share_without_vle_activity": float(n_no_vle / n_total) if n_total > 0 else np.nan,
    "n_used_zero_week_fallback_for_censoring": n_zero_week_fallback,
    "share_zero_week_fallback": float(n_zero_week_fallback / n_total) if n_total > 0 else np.nan,
    "n_negative_max_vle_day": n_negative_max_vle_day,
    "n_negative_t_last_obs_week": n_negative_t_last_obs,
    "n_negative_t_final_week": n_negative_t_final,
    "max_t_event_week": t_event.max(),
    "max_t_last_obs_week": t_last_obs.max(),
    "max_t_final_week": t_final.max(),
}])

# ------------------------------------------------------------
# 3) Event-definition breakdown
# ------------------------------------------------------------
table_p13_0_event_definition_breakdown = (
    enr.groupby(
        [
            "final_result",
            "is_withdrawn",
            "has_valid_unregistration_date",
            "event_observed",
            "is_withdrawn_without_valid_unregistration",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n")
    .sort_values(
        [
            "final_result",
            "is_withdrawn",
            "has_valid_unregistration_date",
            "event_observed",
            "is_withdrawn_without_valid_unregistration",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4) Temporal consistency checks
# ------------------------------------------------------------
checks = []


### Funcao: add_check

Definicao isolada de add_check para reutilizacao nas etapas seguintes.


In [ ]:

def add_check(name: str, mask: pd.Series):
    checks.append({
        "check_name": name,
        "n_flagged": int(mask.fillna(False).sum()),
        "share_flagged": float(mask.fillna(False).mean()) if len(mask) else np.nan
    })


In [ ]:

mask_event_observed_but_not_withdrawn = (
    (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 1)
    & (pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0) != 1)
)
mask_event_observed_but_missing_t_event_week = (
    (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 1)
    & (t_event.isna())
)
mask_non_event_but_has_t_event_week = (
    (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 0)
    & (t_event.notna())
)
mask_withdrawn_without_valid_unregistration_but_event_observed = (
    (pd.to_numeric(enr["is_withdrawn_without_valid_unregistration"], errors="coerce").fillna(0) == 1)
    & (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 1)
)
mask_valid_unregistration_before_registration = (
    date_unreg.notna() & date_reg.notna() & (date_unreg < date_reg)
)
mask_event_week_negative = t_event.notna() & (t_event < 0)
mask_last_obs_week_negative = t_last_obs.notna() & (t_last_obs < 0)
mask_final_week_negative = t_final.notna() & (t_final < 0)
mask_event_after_final_week = t_event.notna() & t_final.notna() & (t_event > t_final)
mask_last_obs_after_final_week = t_last_obs.notna() & t_final.notna() & (t_last_obs > t_final)
mask_has_any_vle_activity_but_missing_max_vle_day = (
    (pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 1)
    & (max_vle_day.isna())
)
mask_no_vle_activity_but_positive_last_obs_week = (
    (pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 0)
    & t_last_obs.notna() & (t_last_obs > 0)
)
mask_zero_week_fallback_used_but_has_vle_activity = (
    (pd.to_numeric(enr["used_zero_week_fallback_for_censoring"], errors="coerce").fillna(0) == 1)
    & (pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 1)
)
mask_zero_week_fallback_used_but_event_observed = (
    (pd.to_numeric(enr["used_zero_week_fallback_for_censoring"], errors="coerce").fillna(0) == 1)
    & (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 1)
)
mask_negative_max_vle_day = max_vle_day.notna() & (max_vle_day < 0)
mask_fail_with_withdrawn_flag = (
    enr["final_result"].astype(str).str.strip().eq("Fail")
    & (pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0) == 1)
)
mask_withdrawn_without_withdrawn_label = (
    enr["final_result"].astype(str).str.strip().eq("Withdrawn")
    & (pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0) != 1)
)

add_check("event_observed_but_not_withdrawn", mask_event_observed_but_not_withdrawn)
add_check("event_observed_but_missing_t_event_week", mask_event_observed_but_missing_t_event_week)
add_check("non_event_but_has_t_event_week", mask_non_event_but_has_t_event_week)
add_check(
    "withdrawn_without_valid_unregistration_but_event_observed",
    mask_withdrawn_without_valid_unregistration_but_event_observed
)
add_check("valid_unregistration_before_registration", mask_valid_unregistration_before_registration)
add_check("event_week_negative", mask_event_week_negative)
add_check("last_obs_week_negative", mask_last_obs_week_negative)
add_check("final_week_negative", mask_final_week_negative)
add_check("event_after_final_week", mask_event_after_final_week)
add_check("last_obs_after_final_week", mask_last_obs_after_final_week)
add_check("has_any_vle_activity_but_missing_max_vle_day", mask_has_any_vle_activity_but_missing_max_vle_day)
add_check("no_vle_activity_but_positive_last_obs_week", mask_no_vle_activity_but_positive_last_obs_week)
add_check("zero_week_fallback_used_but_has_vle_activity", mask_zero_week_fallback_used_but_has_vle_activity)
add_check("zero_week_fallback_used_but_event_observed", mask_zero_week_fallback_used_but_event_observed)
add_check("negative_max_vle_day", mask_negative_max_vle_day)
add_check("fail_with_withdrawn_flag", mask_fail_with_withdrawn_flag)
add_check("withdrawn_without_withdrawn_flag", mask_withdrawn_without_withdrawn_label)

table_p13_0_temporal_consistency_checks = pd.DataFrame(checks)

# ------------------------------------------------------------
# 5) Zero-activity / sparse-activity / pre-start activity breakdown
# ------------------------------------------------------------
enr["n_vle_rows_num"] = n_vle_rows.fillna(0)
enr["total_clicks_all_time_num"] = total_clicks_all_time.fillna(0)

enr["activity_profile"] = np.select(
    [
        pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 0,
        max_vle_day.notna() & (max_vle_day < 0),
        enr["n_vle_rows_num"] <= 1,
        enr["n_vle_rows_num"].between(2, 3, inclusive="both"),
        enr["n_vle_rows_num"] >= 4,
    ],
    [
        "no_vle_activity",
        "pre_start_only_activity",
        "1_vle_row",
        "2_to_3_vle_rows",
        "4plus_vle_rows",
    ],
    default="other"
)

table_p13_0_zero_activity_breakdown = (
    enr.groupby(
        ["activity_profile", "event_observed", "used_zero_week_fallback_for_censoring"],
        dropna=False
    )
    .size()
    .reset_index(name="n")
    .sort_values(["activity_profile", "event_observed", "used_zero_week_fallback_for_censoring"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6) Course/presentation duration audit
# ------------------------------------------------------------
if "courses" in available_tables:
    courses_df = con.execute("""
        SELECT *
        FROM courses
    """).fetchdf()

    possible_duration_cols = [
        "module_presentation_length",
        "length",
        "course_length",
        "presentation_length",
    ]
    duration_col = next((c for c in possible_duration_cols if c in courses_df.columns), None)

    if duration_col is not None:
        course_duration_df = courses_df[["code_module", "code_presentation", duration_col]].copy()
        course_duration_df = course_duration_df.rename(columns={duration_col: "official_course_length_days"})
        course_duration_df["official_course_length_weeks_floor"] = np.floor(
            pd.to_numeric(course_duration_df["official_course_length_days"], errors="coerce") / 7.0
        )
    else:
        course_duration_df = courses_df[["code_module", "code_presentation"]].drop_duplicates().copy()
        course_duration_df["official_course_length_days"] = np.nan
        course_duration_df["official_course_length_weeks_floor"] = np.nan
else:
    course_duration_df = enr[["code_module", "code_presentation"]].drop_duplicates().copy()
    course_duration_df["official_course_length_days"] = np.nan
    course_duration_df["official_course_length_weeks_floor"] = np.nan

obs_duration_df = (
    enr.groupby(["code_module", "code_presentation"], dropna=False)
    .agg(
        n_enrollments=("enrollment_id", "nunique"),
        n_events=("event_observed", "sum"),
        event_rate=("event_observed", "mean"),
        max_observed_t_final_week=("t_final_week", "max"),
        median_observed_t_final_week=("t_final_week", "median"),
        max_observed_t_event_week=("t_event_week", "max"),
        median_observed_t_last_obs_week=("t_last_obs_week", "median"),
    )
    .reset_index()
)

table_p13_0_course_duration_audit = (
    obs_duration_df.merge(
        course_duration_df,
        on=["code_module", "code_presentation"],
        how="left",
        validate="one_to_one"
    )
    .sort_values(["code_module", "code_presentation"])
    .reset_index(drop=True)
)

table_p13_0_duration_overrun_summary = table_p13_0_course_duration_audit.copy()
table_p13_0_duration_overrun_summary["observed_minus_official_weeks"] = (
    pd.to_numeric(table_p13_0_duration_overrun_summary["max_observed_t_final_week"], errors="coerce")
    - pd.to_numeric(table_p13_0_duration_overrun_summary["official_course_length_weeks_floor"], errors="coerce")
)
table_p13_0_duration_overrun_summary["has_duration_overrun"] = (
    table_p13_0_duration_overrun_summary["observed_minus_official_weeks"] > 0
)
table_p13_0_duration_overrun_summary = (
    table_p13_0_duration_overrun_summary
    .sort_values(
        ["has_duration_overrun", "observed_minus_official_weeks", "code_module", "code_presentation"],
        ascending=[False, False, True, True]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7) Enrollment vs person-period alignment audit
# ------------------------------------------------------------
# Search more robustly for a candidate person-period table
candidate_pp_tables = []
for name in sorted(available_tables):
    lname = name.lower()
    if ("person" in lname and "period" in lname) or ("pp_" in lname) or ("hazard_ready" in lname):
        candidate_pp_tables.append(name)


### Funcao: audit_person_period_table

Definicao isolada de audit_person_period_table para reutilizacao nas etapas seguintes.


In [ ]:

def audit_person_period_table(table_name: str, enr_df: pd.DataFrame):
    pp = con.execute(f"SELECT * FROM {table_name}").fetchdf()

    required_pp_cols = ["id_student", "code_module", "code_presentation", "week"]
    missing_pp_cols = [c for c in required_pp_cols if c not in pp.columns]

    result = {
        "person_period_table_found": True,
        "person_period_table_name": table_name,
        "required_columns_present": len(missing_pp_cols) == 0,
        "missing_columns": ", ".join(missing_pp_cols),
        "n_person_period_rows": int(len(pp)),
        "n_unique_person_period_enrollments": np.nan,
        "max_week_in_person_period": np.nan,
        "max_t_final_week_in_enrollment_table": pd.to_numeric(enr_df["t_final_week"], errors="coerce").max(),
        "all_enrollment_ids_covered_in_person_period": np.nan,
        "n_missing_enrollment_ids_in_person_period": np.nan,
        "share_enrollments_with_pp_max_week_equal_t_final_week": np.nan,
    }

    if len(missing_pp_cols) > 0:
        return pd.DataFrame([result])

    pp = pp.copy()
    pp["enrollment_id"] = (
        pp["id_student"].astype(str)
        + "||" + pp["code_module"].astype(str)
        + "||" + pp["code_presentation"].astype(str)
    )

    pp_cov = (
        pp.groupby("enrollment_id", dropna=False)
        .agg(
            pp_min_week=("week", "min"),
            pp_max_week=("week", "max"),
            pp_n_rows=("week", "size"),
        )
        .reset_index()
    )

    enr_cov = enr_df[["enrollment_id", "t_final_week", "event_observed"]].copy()
    enr_cov["t_final_week"] = pd.to_numeric(enr_cov["t_final_week"], errors="coerce")

    merged_cov = enr_cov.merge(pp_cov, on="enrollment_id", how="left", validate="one_to_one")
    missing_pp_enrollments = merged_cov["pp_n_rows"].isna()

    aligned_final_week = (
        merged_cov["pp_max_week"].notna()
        & merged_cov["t_final_week"].notna()
        & (merged_cov["pp_max_week"] == merged_cov["t_final_week"])
    )

    result.update({
        "n_unique_person_period_enrollments": int(pp["enrollment_id"].nunique()),
        "max_week_in_person_period": pd.to_numeric(pp["week"], errors="coerce").max(),
        "all_enrollment_ids_covered_in_person_period": bool((~missing_pp_enrollments).all()),
        "n_missing_enrollment_ids_in_person_period": int(missing_pp_enrollments.sum()),
        "share_enrollments_with_pp_max_week_equal_t_final_week":
            float(aligned_final_week.mean()) if len(aligned_final_week) else np.nan,
    })
    return pd.DataFrame([result])


In [ ]:

pp_alignment_frames = []
for tbl in candidate_pp_tables:
    try:
        pp_alignment_frames.append(audit_person_period_table(tbl, enr))
    except Exception as e:
        pp_alignment_frames.append(pd.DataFrame([{
            "person_period_table_found": True,
            "person_period_table_name": tbl,
            "required_columns_present": False,
            "missing_columns": f"ERROR: {str(e)}",
            "n_person_period_rows": np.nan,
            "n_unique_person_period_enrollments": np.nan,
            "max_week_in_person_period": np.nan,
            "max_t_final_week_in_enrollment_table": pd.to_numeric(enr["t_final_week"], errors="coerce").max(),
            "all_enrollment_ids_covered_in_person_period": np.nan,
            "n_missing_enrollment_ids_in_person_period": np.nan,
            "share_enrollments_with_pp_max_week_equal_t_final_week": np.nan,
        }]))

if len(pp_alignment_frames) > 0:
    table_p13_0_person_period_alignment = pd.concat(pp_alignment_frames, ignore_index=True)
else:
    table_p13_0_person_period_alignment = pd.DataFrame([{
        "person_period_table_found": False,
        "person_period_table_name": "",
        "required_columns_present": False,
        "missing_columns": "",
        "n_person_period_rows": np.nan,
        "n_unique_person_period_enrollments": np.nan,
        "max_week_in_person_period": np.nan,
        "max_t_final_week_in_enrollment_table": pd.to_numeric(enr["t_final_week"], errors="coerce").max(),
        "all_enrollment_ids_covered_in_person_period": np.nan,
        "n_missing_enrollment_ids_in_person_period": np.nan,
        "share_enrollments_with_pp_max_week_equal_t_final_week": np.nan,
    }])

# ------------------------------------------------------------
# 8) Problem examples
# ------------------------------------------------------------
problem_frames = []


### Funcao: collect_examples

Definicao isolada de collect_examples para reutilizacao nas etapas seguintes.


In [ ]:

def collect_examples(mask: pd.Series, label: str, max_rows: int = 20):
    cols = [
        "id_student", "code_module", "code_presentation", "final_result",
        "date_registration", "date_unregistration",
        "is_withdrawn", "has_valid_unregistration_date", "event_observed",
        "is_withdrawn_without_valid_unregistration",
        "has_any_vle_activity", "max_vle_day", "n_vle_rows",
        "total_clicks_all_time", "t_event_week", "t_last_obs_week", "t_final_week",
        "used_zero_week_fallback_for_censoring"
    ]
    use_cols = [c for c in cols if c in enr.columns]
    tmp = enr.loc[mask.fillna(False), use_cols].copy().head(max_rows)
    if len(tmp) > 0:
        tmp.insert(0, "problem_type", label)
        problem_frames.append(tmp)


In [ ]:

collect_examples(mask_last_obs_week_negative, "last_obs_week_negative")
collect_examples(mask_final_week_negative, "final_week_negative")
collect_examples(mask_last_obs_after_final_week, "last_obs_after_final_week")
collect_examples(mask_negative_max_vle_day, "negative_max_vle_day")
collect_examples(mask_fail_with_withdrawn_flag, "fail_with_withdrawn_flag")
collect_examples(mask_withdrawn_without_withdrawn_label, "withdrawn_without_withdrawn_flag")

duration_overrun_pairs = table_p13_0_duration_overrun_summary.loc[
    table_p13_0_duration_overrun_summary["has_duration_overrun"] == True,
    ["code_module", "code_presentation"]
].drop_duplicates()

if len(duration_overrun_pairs) > 0:
    overrun_keys = set(
        duration_overrun_pairs["code_module"].astype(str) + "||" + duration_overrun_pairs["code_presentation"].astype(str)
    )
    pair_mask = (
        enr["code_module"].astype(str) + "||" + enr["code_presentation"].astype(str)
    ).isin(overrun_keys)
    collect_examples(pair_mask, "duration_overrun_context_examples", max_rows=20)

if len(problem_frames) > 0:
    table_p13_0_problem_examples = pd.concat(problem_frames, ignore_index=True)
else:
    table_p13_0_problem_examples = pd.DataFrame(columns=[
        "problem_type",
        "id_student", "code_module", "code_presentation", "final_result",
        "date_registration", "date_unregistration",
        "is_withdrawn", "has_valid_unregistration_date", "event_observed",
        "is_withdrawn_without_valid_unregistration",
        "has_any_vle_activity", "max_vle_day", "n_vle_rows",
        "total_clicks_all_time", "t_event_week", "t_last_obs_week", "t_final_week",
        "used_zero_week_fallback_for_censoring"
    ])

# ------------------------------------------------------------
# 9) Save outputs
# ------------------------------------------------------------
path_summary = OUT_TABLES / "table_p13_0_event_time_censoring_summary.csv"
path_event_breakdown = OUT_TABLES / "table_p13_0_event_definition_breakdown.csv"
path_temporal_checks = OUT_TABLES / "table_p13_0_temporal_consistency_checks.csv"
path_zero_activity = OUT_TABLES / "table_p13_0_zero_activity_breakdown.csv"
path_course_duration = OUT_TABLES / "table_p13_0_course_duration_audit.csv"
path_pp_alignment = OUT_TABLES / "table_p13_0_person_period_alignment.csv"
path_duration_overrun = OUT_TABLES / "table_p13_0_duration_overrun_summary.csv"
path_problem_examples = OUT_TABLES / "table_p13_0_problem_examples.csv"
path_metadata = OUT_METADATA / "metadata_p13_0_event_time_censoring_audit.json"

table_p13_0_event_time_censoring_summary.to_csv(path_summary, index=False)
table_p13_0_event_definition_breakdown.to_csv(path_event_breakdown, index=False)
table_p13_0_temporal_consistency_checks.to_csv(path_temporal_checks, index=False)
table_p13_0_zero_activity_breakdown.to_csv(path_zero_activity, index=False)
table_p13_0_course_duration_audit.to_csv(path_course_duration, index=False)
table_p13_0_person_period_alignment.to_csv(path_pp_alignment, index=False)
table_p13_0_duration_overrun_summary.to_csv(path_duration_overrun, index=False)
table_p13_0_problem_examples.to_csv(path_problem_examples, index=False)

metadata_p13_0 = {
    "step": "P13.0",
    "title": "Event, Time, and Censoring Audit",
    "version": "final_substitute",
    "purpose": (
        "Audit the current survival operationalization before stronger split "
        "design changes. This step does not alter the benchmark."
    ),
    "event_definition_under_audit": "event_observed = 1 iff final_result='Withdrawn' and valid date_unregistration >= 0",
    "censoring_convention_under_audit": (
        "Non-event enrollments are censored at the last observed VLE week; "
        "if no VLE activity is observed and no event is observed, a week-0 "
        "fallback is used."
    ),
    "candidate_person_period_tables_detected": candidate_pp_tables,
    "output_tables": [
        path_summary.as_posix(),
        path_event_breakdown.as_posix(),
        path_temporal_checks.as_posix(),
        path_zero_activity.as_posix(),
        path_course_duration.as_posix(),
        path_pp_alignment.as_posix(),
        path_duration_overrun.as_posix(),
        path_problem_examples.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata_p13_0, f, indent=2)

# ------------------------------------------------------------
# 10) Display key outputs
# ------------------------------------------------------------
print("\nMain summary:")
display(table_p13_0_event_time_censoring_summary)

print("\nEvent definition breakdown:")
display(table_p13_0_event_definition_breakdown)

print("\nTemporal consistency checks:")
display(table_p13_0_temporal_consistency_checks)

print("\nZero / sparse / pre-start activity breakdown:")
display(table_p13_0_zero_activity_breakdown)

print("\nCourse/presentation duration audit:")
display(table_p13_0_course_duration_audit.head(20))

print("\nDuration overrun summary (top rows):")
display(table_p13_0_duration_overrun_summary.head(20))

print("\nPerson-period alignment audit:")
display(table_p13_0_person_period_alignment)

print("\nProblem examples:")
display(table_p13_0_problem_examples.head(50))

print("\nSaved:")
print("-", path_summary.resolve())
print("-", path_event_breakdown.resolve())
print("-", path_temporal_checks.resolve())
print("-", path_zero_activity.resolve())
print("-", path_course_duration.resolve())
print("-", path_pp_alignment.resolve())
print("-", path_duration_overrun.resolve())
print("-", path_problem_examples.resolve())
print("-", path_metadata.resolve())


## C2 — Apply Temporal Split and Leakage Control

In [ ]:
# ============================================================
# C2 — Apply Temporal Split and Leakage Control
# ============================================================
# Methodological note:
# This step defines the main benchmark split at the enrollment level,
# stratified by event status and coarse event-time bucket.
# The leakage audit in this step is restricted to enrollment-level
# identity leakage and propagation consistency across derived benchmark
# tables. Module/presentation overlap is not treated here as leakage by
# definition; it will be audited separately in a dedicated contextual audit.

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

print("=" * 70)
print("C2 — Apply Temporal Split and Leakage Control")
print("=" * 70)
print("Methodological note: this step defines the main benchmark split at the")
print("enrollment level, stratified by event status and coarse event-time bucket.")
print("The leakage audit in this step is restricted to enrollment-level identity")
print("leakage and propagation consistency across derived benchmark tables.")
print("Module/presentation overlap is not treated here as leakage by definition;")
print("it will be audited separately in a dedicated contextual split audit.")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TEST_SIZE = 0.30
RANDOM_STATE = 42
N_DURATION_BUCKETS = 4
MIN_ROWS_FOR_CANONICAL_BACKBONE = 1000

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------
OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)


### Funcao: build_enrollment_id

Definicao isolada de build_enrollment_id para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def build_enrollment_id(df: pd.DataFrame) -> pd.Series:
    required_cols = ["id_student", "code_module", "code_presentation"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns required to build enrollment_id: {missing}")
    return (
        df["id_student"].astype(str)
        + "||" + df["code_module"].astype(str)
        + "||" + df["code_presentation"].astype(str)
    )


### Funcao: make_duration_buckets

Definicao isolada de make_duration_buckets para reutilizacao nas etapas seguintes.


In [ ]:


def make_duration_buckets(duration_series: pd.Series, n_buckets: int = 4) -> pd.Series:
    s = pd.to_numeric(duration_series, errors="coerce").fillna(-1)

    try:
        buckets = pd.qcut(s, q=n_buckets, duplicates="drop")
        return buckets.astype(str)
    except Exception:
        ranks = s.rank(method="average", pct=True)
        edges = np.linspace(0, 1, n_buckets + 1)
        labels = [f"bucket_{i+1}" for i in range(n_buckets)]
        return pd.cut(
            ranks,
            bins=edges,
            labels=labels,
            include_lowest=True
        ).astype(str)


### Funcao: attach_split_assignment

Definicao isolada de attach_split_assignment para reutilizacao nas etapas seguintes.


In [ ]:


def attach_split_assignment(df: pd.DataFrame, split_assignment: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "enrollment_id" not in out.columns:
        out["enrollment_id"] = build_enrollment_id(out)
    return out.merge(
        split_assignment[["enrollment_id", "split"]],
        on="enrollment_id",
        how="left",
        validate="many_to_one",
    )


### Funcao: summarize_split

Definicao isolada de summarize_split para reutilizacao nas etapas seguintes.


In [ ]:


def summarize_split(df_enroll: pd.DataFrame) -> pd.DataFrame:
    return (
        df_enroll.groupby("split", dropna=False)
        .agg(
            n_enrollments=("enrollment_id", "nunique"),
            n_events=("event", "sum"),
            event_rate=("event", "mean"),
            min_duration=("duration", "min"),
            median_duration=("duration", "median"),
            max_duration=("duration", "max"),
        )
        .reset_index()
    )


### Funcao: summarize_split_by_bucket

Definicao isolada de summarize_split_by_bucket para reutilizacao nas etapas seguintes.


In [ ]:


def summarize_split_by_bucket(df_enroll: pd.DataFrame) -> pd.DataFrame:
    return (
        df_enroll.groupby(["split", "event", "duration_bucket"], dropna=False)
        .agg(n_enrollments=("enrollment_id", "nunique"))
        .reset_index()
        .sort_values(["split", "event", "duration_bucket"])
        .reset_index(drop=True)
    )


### Funcao: name_penalty

Larger penalty = less likely to be canonical benchmark backbone.


In [ ]:


def name_penalty(name: str) -> int:
    """
    Larger penalty = less likely to be canonical benchmark backbone.
    """
    n = name.lower()

    penalty = 0

    # Strong negative cues
    bad_patterns = [
        r"^sample",
        r"sample_",
        r"^cols_",
        r"^perm_",
        r"perm_",
        r"debug",
        r"toy",
        r"mock",
        r"tmp",
        r"temp",
        r"copy",
        r"preview",
        r"head",
        r"subset",
    ]
    for pat in bad_patterns:
        if re.search(pat, n):
            penalty += 100

    # Test/train specific views are less canonical than a full backbone
    if n.startswith("train_") or n.endswith("_train") or "_train_" in n:
        penalty += 20
    if n.startswith("test_") or n.endswith("_test") or "_test_" in n:
        penalty += 20

    # Positive cues
    good_patterns = [
        r"backbone",
        r"enrollment",
        r"benchmark",
        r"master",
        r"base_df",
        r"analysis_df",
    ]
    for pat in good_patterns:
        if re.search(pat, n):
            penalty -= 10

    return penalty


### Funcao: find_candidate_enrollment_backbones

Definicao isolada de find_candidate_enrollment_backbones para reutilizacao nas etapas seguintes.


In [ ]:


def find_candidate_enrollment_backbones(globals_dict: dict) -> pd.DataFrame:
    required_cols = {
        "id_student", "code_module", "code_presentation", "event", "duration"
    }

    candidates = []

    for name, obj in globals_dict.items():
        if not isinstance(obj, pd.DataFrame):
            continue

        cols = set(obj.columns)
        if not required_cols.issubset(cols):
            continue

        try:
            tmp = obj.copy()
            tmp["enrollment_id"] = build_enrollment_id(tmp)

            n_rows = len(tmp)
            n_unique_enrollments = tmp["enrollment_id"].nunique()
            duplicate_rows = n_rows - n_unique_enrollments
            duplicate_rate = duplicate_rows / n_rows if n_rows > 0 else np.nan

            event_numeric_share = pd.to_numeric(tmp["event"], errors="coerce").notna().mean()
            duration_numeric_share = pd.to_numeric(tmp["duration"], errors="coerce").notna().mean()

            candidates.append({
                "name": name,
                "n_rows": n_rows,
                "n_unique_enrollments": n_unique_enrollments,
                "duplicate_rows": duplicate_rows,
                "duplicate_rate": duplicate_rate,
                "event_numeric_share": event_numeric_share,
                "duration_numeric_share": duration_numeric_share,
                "name_penalty": name_penalty(name),
                "large_enough": n_rows >= MIN_ROWS_FOR_CANONICAL_BACKBONE,
            })
        except Exception:
            continue

    if len(candidates) == 0:
        return pd.DataFrame()

    cand_df = pd.DataFrame(candidates)

    # Prefer plausible real benchmark tables first
    plausible_df = cand_df[cand_df["large_enough"]].copy()

    if not plausible_df.empty:
        cand_df = plausible_df

    cand_df = cand_df.sort_values(
        by=[
            "event_numeric_share",
            "duration_numeric_share",
            "duplicate_rate",
            "name_penalty",
            "n_rows",
            "n_unique_enrollments",
        ],
        ascending=[False, False, True, True, False, False]
    ).reset_index(drop=True)

    return cand_df


In [ ]:


# ------------------------------------------------------------
# Resolve main enrollment-level dataframe
# ------------------------------------------------------------
candidate_summary = find_candidate_enrollment_backbones(globals())

if candidate_summary.empty:
    raise ValueError(
        "Could not find any plausible enrollment-level benchmark DataFrame in globals() "
        "with columns {id_student, code_module, code_presentation, event, duration}."
    )

print("\nCandidate enrollment backbones found:")
print(candidate_summary.head(20).to_string(index=False))

resolved_name = candidate_summary.iloc[0]["name"]
enroll_df = globals()[resolved_name].copy()

print(f"\nResolved canonical enrollment backbone from: {resolved_name}")
print(f"Rows in resolved enrollment backbone before deduplication check: {len(enroll_df):,}")

# ------------------------------------------------------------
# Enforce one row per enrollment
# ------------------------------------------------------------
if "enrollment_id" not in enroll_df.columns:
    enroll_df["enrollment_id"] = build_enrollment_id(enroll_df)

dup_counts = enroll_df["enrollment_id"].value_counts()
n_duplicate_ids = int((dup_counts > 1).sum())

if n_duplicate_ids > 0:
    print(f"Detected {n_duplicate_ids:,} duplicated enrollment_ids; keeping first occurrence.")
    enroll_df = (
        enroll_df.sort_values(["enrollment_id"])
        .drop_duplicates(subset=["enrollment_id"], keep="first")
        .reset_index(drop=True)
    )

print(f"Unique enrollments used for splitting: {enroll_df['enrollment_id'].nunique():,}")

# ------------------------------------------------------------
# Basic cleaning
# ------------------------------------------------------------
enroll_df["event"] = pd.to_numeric(enroll_df["event"], errors="coerce").fillna(0).astype(int)
enroll_df["duration"] = pd.to_numeric(enroll_df["duration"], errors="coerce")

before_clean = len(enroll_df)
enroll_df = enroll_df.dropna(subset=["duration"]).copy()
enroll_df["duration"] = enroll_df["duration"].astype(int)
after_clean = len(enroll_df)

if before_clean != after_clean:
    print(f"Dropped {before_clean - after_clean:,} enrollments due to missing duration.")

if len(enroll_df) < 10:
    raise ValueError(
        f"Resolved enrollment backbone '{resolved_name}' has only {len(enroll_df)} rows "
        "after cleaning, which is not plausible for the benchmark split."
    )

# ------------------------------------------------------------
# Create stratification label
# ------------------------------------------------------------
enroll_df["duration_bucket"] = make_duration_buckets(
    enroll_df["duration"],
    n_buckets=N_DURATION_BUCKETS
)
enroll_df["strata_label"] = (
    "event_" + enroll_df["event"].astype(str)
    + "__"
    + enroll_df["duration_bucket"].astype(str)
)

print("\nStratification label distribution (top 10):")
print(enroll_df["strata_label"].value_counts().head(10))

# ------------------------------------------------------------
# Split at enrollment level
# ------------------------------------------------------------
sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

X_dummy = np.zeros(len(enroll_df))
y_strata = enroll_df["strata_label"].astype(str).to_numpy()

train_idx, test_idx = next(sss.split(X_dummy, y_strata))

enroll_df["split"] = "train"
enroll_df.loc[enroll_df.index[test_idx], "split"] = "test"

split_assignment = enroll_df[
    ["enrollment_id", "id_student", "code_module", "code_presentation",
     "event", "duration", "duration_bucket", "strata_label", "split"]
].copy()

# ------------------------------------------------------------
# Main split summaries
# ------------------------------------------------------------
table_split_summary = summarize_split(enroll_df)
table_split_bucket_summary = summarize_split_by_bucket(enroll_df)

print("\nEnrollment split summary:")
print(table_split_summary.to_string(index=False))

print("\nEnrollment split bucket summary (head):")
print(table_split_bucket_summary.head(20).to_string(index=False))

# ------------------------------------------------------------
# Identity leakage audit (enrollment-level only)
# ------------------------------------------------------------
train_ids = set(split_assignment.loc[split_assignment["split"] == "train", "enrollment_id"])
test_ids = set(split_assignment.loc[split_assignment["split"] == "test", "enrollment_id"])
overlap_ids = train_ids.intersection(test_ids)

table_leakage_check = pd.DataFrame(
    [{
        "n_train_enrollments": len(train_ids),
        "n_test_enrollments": len(test_ids),
        "n_enrollments_checked": len(train_ids.union(test_ids)),
        "n_enrollments_with_leakage": len(overlap_ids),
        "identity_leakage_detected": len(overlap_ids) > 0,
    }]
)

print("\nEnrollment identity leakage check:")
print(table_leakage_check.to_string(index=False))

print("\nInterpretation:")
print("- The main benchmark split is enrollment-level.")
print("- Zero enrollment identity leakage was detected across train/test.")
print("- This check does not yet evaluate contextual overlap in code_module,")
print("  code_presentation, or module-presentation combinations.")
print("- Those contextual overlap diagnostics belong to the next dedicated step.")

# ------------------------------------------------------------
# Propagation check to derived dataframes (if available)
# ------------------------------------------------------------
derived_df_candidates = [
    "pp_df",
    "person_period_df",
    "df_person_period",
    "linear_df",
    "neural_df",
    "cox_df",
    "deepsurv_df",
]

propagation_rows = []

for name in derived_df_candidates:
    if name not in globals():
        continue
    df_obj = globals()[name]
    if not isinstance(df_obj, pd.DataFrame):
        continue

    try:
        df_tmp = df_obj.copy()

        if "enrollment_id" not in df_tmp.columns:
            required = {"id_student", "code_module", "code_presentation"}
            if required.issubset(df_tmp.columns):
                df_tmp["enrollment_id"] = build_enrollment_id(df_tmp)
            else:
                propagation_rows.append({
                    "table_name": name,
                    "n_rows": len(df_tmp),
                    "n_unique_enrollments": np.nan,
                    "n_missing_split_labels": np.nan,
                    "propagation_ok": False,
                    "note": "Could not derive enrollment_id"
                })
                continue

        df_attached = attach_split_assignment(df_tmp, split_assignment)
        n_missing = int(df_attached["split"].isna().sum())

        propagation_rows.append({
            "table_name": name,
            "n_rows": len(df_attached),
            "n_unique_enrollments": df_attached["enrollment_id"].nunique(),
            "n_missing_split_labels": n_missing,
            "propagation_ok": n_missing == 0,
            "note": "ok" if n_missing == 0 else "Some rows missing split label"
        })

    except Exception as e:
        propagation_rows.append({
            "table_name": name,
            "n_rows": len(df_obj),
            "n_unique_enrollments": np.nan,
            "n_missing_split_labels": np.nan,
            "propagation_ok": False,
            "note": f"Error: {type(e).__name__}: {e}"
        })

table_propagation_check = pd.DataFrame(propagation_rows)

if not table_propagation_check.empty:
    print("\nPropagation check across derived tables:")
    print(table_propagation_check.to_string(index=False))
else:
    print("\nPropagation check: no eligible derived tables found in globals().")

# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------
path_split_assignment = OUT_TABLES / "table_enrollment_split_assignment.csv"
path_split_summary = OUT_TABLES / "table_enrollment_split_summary.csv"
path_split_bucket_summary = OUT_TABLES / "table_enrollment_split_bucket_summary.csv"
path_leakage_check = OUT_TABLES / "table_enrollment_split_leakage_check.csv"
path_propagation_check = OUT_TABLES / "table_enrollment_split_propagation_check.csv"
path_candidate_backbones = OUT_TABLES / "table_enrollment_backbone_candidates.csv"
path_protocol_note = OUT_METADATA / "enrollment_split_protocol_note.json"

split_assignment.to_csv(path_split_assignment, index=False)
table_split_summary.to_csv(path_split_summary, index=False)
table_split_bucket_summary.to_csv(path_split_bucket_summary, index=False)
table_leakage_check.to_csv(path_leakage_check, index=False)
candidate_summary.to_csv(path_candidate_backbones, index=False)

if not table_propagation_check.empty:
    table_propagation_check.to_csv(path_propagation_check, index=False)

split_protocol_note = {
    "step_id": "P13",
    "step_title": "Apply Temporal Split and Leakage Control (Revised v5)",
    "split_unit": "enrollment",
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "stratification": "event_status + coarse_event_time_bucket",
    "n_duration_buckets": N_DURATION_BUCKETS,
    "identity_leakage_check": "performed",
    "identity_leakage_result": (
        "no enrollment leakage detected"
        if len(overlap_ids) == 0
        else f"{len(overlap_ids)} overlapping enrollment_ids detected"
    ),
    "contextual_module_presentation_audit": "not performed in this step",
    "intended_followup_step": "C2.1 — Module/Presentation Leakage Audit",
    "resolved_enrollment_backbone": resolved_name,
    "n_enrollments_used": int(enroll_df["enrollment_id"].nunique()),
    "n_train_enrollments": int((enroll_df["split"] == "train").sum()),
    "n_test_enrollments": int((enroll_df["split"] == "test").sum()),
    "min_rows_for_canonical_backbone": MIN_ROWS_FOR_CANONICAL_BACKBONE,
}

with open(path_protocol_note, "w", encoding="utf-8") as f:
    json.dump(split_protocol_note, f, indent=2)

print("\nSaved:")
print(f"- {path_split_assignment}")
print(f"- {path_split_summary}")
print(f"- {path_split_bucket_summary}")
print(f"- {path_leakage_check}")
print(f"- {path_candidate_backbones}")
if not table_propagation_check.empty:
    print(f"- {path_propagation_check}")
print(f"- {path_protocol_note}")


### C2.1 — Module/Presentation Leakage Audit

In [ ]:
# ============================================================
# C2.1 — Module/Presentation Leakage Audit
# ============================================================
# Methodological note:
# This step does not redefine the benchmark split and does not classify
# module/presentation overlap as identity leakage. Its purpose is purely
# diagnostic: to characterize how much contextual overlap exists across
# train and test with respect to code_module, code_presentation, and the
# combined module-presentation context.

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C2.1 — Module/Presentation Leakage Audit")
print("=" * 70)
print("Methodological note: this step is diagnostic only.")
print("It does not change the benchmark split and does not classify")
print("module/presentation overlap as identity leakage.")
print("Its purpose is to quantify contextual overlap between train and test")
print("for code_module, code_presentation, and module-presentation pairs.")

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------
OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Resolve split assignment generated in P13
# ------------------------------------------------------------
if "split_assignment" not in globals():
    raise NameError(
        "split_assignment not found in globals(). Run P13 before P13.1."
    )

split_df = split_assignment.copy()

required_cols = [
    "enrollment_id", "id_student", "code_module", "code_presentation", "split"
]
missing_cols = [c for c in required_cols if c not in split_df.columns]
if missing_cols:
    raise KeyError(f"split_assignment is missing required columns: {missing_cols}")

split_df["code_module"] = split_df["code_module"].astype(str)
split_df["code_presentation"] = split_df["code_presentation"].astype(str)
split_df["module_presentation"] = (
    split_df["code_module"] + "||" + split_df["code_presentation"]
)

train_df = split_df.loc[split_df["split"] == "train"].copy()
test_df = split_df.loc[split_df["split"] == "test"].copy()

print(f"\nTrain enrollments: {len(train_df):,}")
print(f"Test enrollments:  {len(test_df):,}")

# ------------------------------------------------------------
# Set-based contextual overlap
# ------------------------------------------------------------
train_modules = set(train_df["code_module"].unique())
test_modules = set(test_df["code_module"].unique())

train_presentations = set(train_df["code_presentation"].unique())
test_presentations = set(test_df["code_presentation"].unique())

train_modpres = set(train_df["module_presentation"].unique())
test_modpres = set(test_df["module_presentation"].unique())

shared_modules = train_modules.intersection(test_modules)
shared_presentations = train_presentations.intersection(test_presentations)
shared_modpres = train_modpres.intersection(test_modpres)

table_context_overlap_summary = pd.DataFrame([
    {
        "context_level": "code_module",
        "n_train_unique": len(train_modules),
        "n_test_unique": len(test_modules),
        "n_shared": len(shared_modules),
        "train_share_seen_in_test": len(shared_modules) / len(train_modules) if len(train_modules) else np.nan,
        "test_share_seen_in_train": len(shared_modules) / len(test_modules) if len(test_modules) else np.nan,
    },
    {
        "context_level": "code_presentation",
        "n_train_unique": len(train_presentations),
        "n_test_unique": len(test_presentations),
        "n_shared": len(shared_presentations),
        "train_share_seen_in_test": len(shared_presentations) / len(train_presentations) if len(train_presentations) else np.nan,
        "test_share_seen_in_train": len(shared_presentations) / len(test_presentations) if len(test_presentations) else np.nan,
    },
    {
        "context_level": "module_presentation",
        "n_train_unique": len(train_modpres),
        "n_test_unique": len(test_modpres),
        "n_shared": len(shared_modpres),
        "train_share_seen_in_test": len(shared_modpres) / len(train_modpres) if len(train_modpres) else np.nan,
        "test_share_seen_in_train": len(shared_modpres) / len(test_modpres) if len(test_modpres) else np.nan,
    },
])

print("\nContextual overlap summary:")
print(table_context_overlap_summary.to_string(index=False))

# ------------------------------------------------------------
# Enrollment-level exposure to seen/unseen contexts
# ------------------------------------------------------------
test_df["module_seen_in_train"] = test_df["code_module"].isin(train_modules)
test_df["presentation_seen_in_train"] = test_df["code_presentation"].isin(train_presentations)
test_df["module_presentation_seen_in_train"] = test_df["module_presentation"].isin(train_modpres)

train_df["module_seen_in_test"] = train_df["code_module"].isin(test_modules)
train_df["presentation_seen_in_test"] = train_df["code_presentation"].isin(test_presentations)
train_df["module_presentation_seen_in_test"] = train_df["module_presentation"].isin(test_modpres)

table_enrollment_context_exposure = pd.DataFrame([
    {
        "split": "test",
        "n_enrollments": len(test_df),
        "share_module_seen_in_other_split": test_df["module_seen_in_train"].mean(),
        "share_presentation_seen_in_other_split": test_df["presentation_seen_in_train"].mean(),
        "share_module_presentation_seen_in_other_split": test_df["module_presentation_seen_in_train"].mean(),
    },
    {
        "split": "train",
        "n_enrollments": len(train_df),
        "share_module_seen_in_other_split": train_df["module_seen_in_test"].mean(),
        "share_presentation_seen_in_other_split": train_df["presentation_seen_in_test"].mean(),
        "share_module_presentation_seen_in_other_split": train_df["module_presentation_seen_in_test"].mean(),
    },
])

print("\nEnrollment-level contextual exposure:")
print(table_enrollment_context_exposure.to_string(index=False))

# ------------------------------------------------------------
# Breakdown by module-presentation pair
# ------------------------------------------------------------
table_module_presentation_split_counts = (
    split_df.groupby(["code_module", "code_presentation", "split"])
    .agg(n_enrollments=("enrollment_id", "nunique"))
    .reset_index()
    .pivot(index=["code_module", "code_presentation"], columns="split", values="n_enrollments")
    .fillna(0)
    .reset_index()
)

for col in ["train", "test"]:
    if col not in table_module_presentation_split_counts.columns:
        table_module_presentation_split_counts[col] = 0

table_module_presentation_split_counts["appears_in_both_splits"] = (
    (table_module_presentation_split_counts["train"] > 0)
    & (table_module_presentation_split_counts["test"] > 0)
)

table_module_presentation_split_counts = (
    table_module_presentation_split_counts
    .sort_values(
        ["appears_in_both_splits", "train", "test", "code_module", "code_presentation"],
        ascending=[False, False, False, True, True]
    )
    .reset_index(drop=True)
)

print("\nModule-presentation split counts (head):")
print(table_module_presentation_split_counts.head(20).to_string(index=False))

# ------------------------------------------------------------
# Interpretation block
# ------------------------------------------------------------
n_modpres_both = int(table_module_presentation_split_counts["appears_in_both_splits"].sum())
n_modpres_total = int(len(table_module_presentation_split_counts))

print("\nInterpretation:")
print("- This is a contextual overlap audit, not an identity leakage audit.")
print("- Enrollment identity leakage remains defined only at the enrollment_id level.")
print(f"- Module-presentation contexts appearing in both splits: {n_modpres_both:,} / {n_modpres_total:,}.")
print("- If overlap is high, the benchmark should be described as enrollment-level")
print("  with shared curricular context across splits, not as fully context-disjoint.")
print("- If overlap is low, this strengthens the claim that test evaluation includes")
print("  more contextual novelty at the module-presentation level.")

# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------
path_context_overlap_summary = OUT_TABLES / "table_module_presentation_overlap_summary.csv"
path_context_exposure = OUT_TABLES / "table_module_presentation_context_exposure.csv"
path_modpres_counts = OUT_TABLES / "table_module_presentation_split_counts.csv"
path_context_metadata = OUT_METADATA / "module_presentation_overlap_audit_summary.json"

table_context_overlap_summary.to_csv(path_context_overlap_summary, index=False)
table_enrollment_context_exposure.to_csv(path_context_exposure, index=False)
table_module_presentation_split_counts.to_csv(path_modpres_counts, index=False)

context_metadata = {
    "step_id": "P13.1",
    "step_title": "Module/Presentation Leakage Audit",
    "diagnostic_only": True,
    "redefines_main_split": False,
    "identity_leakage_definition_changed": False,
    "n_train_enrollments": int(len(train_df)),
    "n_test_enrollments": int(len(test_df)),
    "n_unique_train_modules": int(len(train_modules)),
    "n_unique_test_modules": int(len(test_modules)),
    "n_shared_modules": int(len(shared_modules)),
    "n_unique_train_presentations": int(len(train_presentations)),
    "n_unique_test_presentations": int(len(test_presentations)),
    "n_shared_presentations": int(len(shared_presentations)),
    "n_unique_train_module_presentations": int(len(train_modpres)),
    "n_unique_test_module_presentations": int(len(test_modpres)),
    "n_shared_module_presentations": int(len(shared_modpres)),
    "n_module_presentations_in_both_splits": n_modpres_both,
    "n_total_module_presentations": n_modpres_total,
}
with open(path_context_metadata, "w", encoding="utf-8") as f:
    json.dump(context_metadata, f, indent=2)

print("\nSaved:")
print(f"- {path_context_overlap_summary}")
print(f"- {path_context_exposure}")
print(f"- {path_modpres_counts}")
print(f"- {path_context_metadata}")

### C2.2 — Consolidate Split and Context Audit for Paper Integration

In [ ]:
# ============================================================
# C2.2 — Consolidate Split and Context Audit for Paper Integration
# ============================================================
# Methodological note:
# This step does not alter the benchmark split. It consolidates the
# results of P13 and P13.1 into compact, paper-oriented artifacts that
# summarize (i) the enrollment-level split design, (ii) the identity
# leakage result, and (iii) the degree of shared curricular context
# across train and test.

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C2.2 — Consolidate Split and Context Audit for Paper Integration")
print("=" * 70)
print("Methodological note: this step does not alter the benchmark split.")
print("It consolidates the outputs of P13 and P13.1 into compact")
print("paper-oriented artifacts for manuscript integration.")

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------
OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_TABLES_PAPER_APPENDIX = OUT_TABLES / "paper_appendix"
OUT_METADATA = OUT_BASE / "metadata"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_TABLES_PAPER_APPENDIX.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Resolve required inputs from prior steps
# ------------------------------------------------------------
required_globals = ["split_assignment"]
missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise NameError(
        f"Missing required objects from prior steps: {missing_globals}. "
        "Run P13 and P13.1 before P13.2."
    )

split_df = split_assignment.copy()

required_split_cols = [
    "enrollment_id", "id_student", "code_module", "code_presentation",
    "event", "duration", "duration_bucket", "split"
]
missing_split_cols = [c for c in required_split_cols if c not in split_df.columns]
if missing_split_cols:
    raise KeyError(f"split_assignment is missing required columns: {missing_split_cols}")

split_df["code_module"] = split_df["code_module"].astype(str)
split_df["code_presentation"] = split_df["code_presentation"].astype(str)
split_df["module_presentation"] = (
    split_df["code_module"] + "||" + split_df["code_presentation"]
)

train_df = split_df.loc[split_df["split"] == "train"].copy()
test_df = split_df.loc[split_df["split"] == "test"].copy()

# ------------------------------------------------------------
# Identity leakage summary
# ------------------------------------------------------------
train_ids = set(train_df["enrollment_id"])
test_ids = set(test_df["enrollment_id"])
overlap_ids = train_ids.intersection(test_ids)

identity_leakage_detected = len(overlap_ids) > 0

# ------------------------------------------------------------
# Contextual overlap summary
# ------------------------------------------------------------
train_modules = set(train_df["code_module"].unique())
test_modules = set(test_df["code_module"].unique())

train_presentations = set(train_df["code_presentation"].unique())
test_presentations = set(test_df["code_presentation"].unique())

train_modpres = set(train_df["module_presentation"].unique())
test_modpres = set(test_df["module_presentation"].unique())

shared_modules = train_modules.intersection(test_modules)
shared_presentations = train_presentations.intersection(test_presentations)
shared_modpres = train_modpres.intersection(test_modpres)

# ------------------------------------------------------------
# Paper-oriented compact summary table
# ------------------------------------------------------------
table_paper_appendix_split_context_audit = pd.DataFrame([{
    "split_unit": "enrollment",
    "stratification": "event_status + coarse_event_time_bucket",
    "n_total_enrollments": int(split_df["enrollment_id"].nunique()),
    "n_train_enrollments": int(train_df["enrollment_id"].nunique()),
    "n_test_enrollments": int(test_df["enrollment_id"].nunique()),
    "train_event_rate": float(pd.to_numeric(train_df["event"], errors="coerce").mean()),
    "test_event_rate": float(pd.to_numeric(test_df["event"], errors="coerce").mean()),
    "identity_leakage_detected": "no" if not identity_leakage_detected else "yes",
    "shared_modules": f"{len(shared_modules)}/{len(train_modules)}",
    "shared_presentations": f"{len(shared_presentations)}/{len(train_presentations)}",
    "shared_module_presentations": f"{len(shared_modpres)}/{len(train_modpres)}",
    "contextual_interpretation": (
        "shared curricular context across train and test"
        if len(shared_modpres) > 0
        else "context-disjoint curricular split"
    )
}])

print("\nPaper-oriented compact split/context audit:")
print(table_paper_appendix_split_context_audit.to_string(index=False))

# ------------------------------------------------------------
# Optional long-form commentary object for later freeze use
# ------------------------------------------------------------
table_paper_appendix_split_context_commentary = pd.DataFrame([{
    "object_id": "Appendix Table A5",
    "object_type": "table",
    "suggested_name": "Split and contextual overlap audit",
    "objective_comment_english": (
        "Document the benchmark split unit, stratification rule, identity leakage result, "
        "and degree of curricular-context overlap across train and test."
    ),
    "what_it_shows_comment_english": (
        "This table shows that the benchmark uses an enrollment-level split with no "
        "enrollment identity leakage, but with fully shared curricular context across "
        "train and test at the module, presentation, and module-presentation levels."
    )
}])

# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------
path_compact = OUT_TABLES / "table_appendix_split_context_audit_compact.csv"
path_compact_paper = OUT_TABLES_PAPER_APPENDIX / "table_paper_appendix_split_context_audit.csv"
path_commentary = OUT_TABLES / "table_paper_appendix_split_context_commentary.csv"
path_metadata = OUT_METADATA / "split_context_audit_summary.json"

table_paper_appendix_split_context_audit.to_csv(path_compact, index=False)
table_paper_appendix_split_context_audit.to_csv(path_compact_paper, index=False)
table_paper_appendix_split_context_commentary.to_csv(path_commentary, index=False)

metadata = {
    "step_id": "P13.2",
    "step_title": "Consolidate Split and Context Audit for Paper Integration",
    "split_unit": "enrollment",
    "stratification": "event_status + coarse_event_time_bucket",
    "n_total_enrollments": int(split_df["enrollment_id"].nunique()),
    "n_train_enrollments": int(train_df["enrollment_id"].nunique()),
    "n_test_enrollments": int(test_df["enrollment_id"].nunique()),
    "identity_leakage_detected": bool(identity_leakage_detected),
    "n_shared_modules": int(len(shared_modules)),
    "n_train_modules": int(len(train_modules)),
    "n_shared_presentations": int(len(shared_presentations)),
    "n_train_presentations": int(len(train_presentations)),
    "n_shared_module_presentations": int(len(shared_modpres)),
    "n_train_module_presentations": int(len(train_modpres)),
    "contextual_interpretation": "shared curricular context across train and test",
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved:")
print(f"- {path_compact}")
print(f"- {path_compact_paper}")
print(f"- {path_commentary}")
print(f"- {path_metadata}")

### C2.3 — Context-Held-Out Split Design

### Funcao: module_coverage_penalty

Definicao isolada de module_coverage_penalty para reutilizacao nas etapas seguintes.


In [ ]:
# ============================================================
# C2.3 — Context-Held-Out Split Design
# REWRITTEN VERSION
# ============================================================
# Purpose:
#   Define a stronger grouped split based on curricular context.
#   This step does NOT yet materialize the final train/test tables.
#   It designs and audits a grouped holdout strategy using a
#   constrained search over candidate group subsets.
#
# Grouping unit:
#   (code_module, code_presentation)
#
# Main outputs:
#   - table_p13_3_group_inventory.csv
#   - table_p13_3_group_split_design.csv
#   - table_p13_3_split_design_summary.csv
#   - table_p13_3_candidate_solutions.csv
#   - metadata_p13_3_context_heldout_split_design.json
# ============================================================

import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C2.3 — Context-Held-Out Split Design")
print("=" * 70)
print("Rewritten version with constrained subset search.")

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

if "enrollment_survival_ready" not in available_tables:
    raise RuntimeError(
        "DuckDB table 'enrollment_survival_ready' not found. "
        "Run the corrected P5 first."
    )

df = con.execute("""
    SELECT *
    FROM enrollment_survival_ready
""").fetchdf()

required_cols = [
    "id_student", "code_module", "code_presentation",
    "event_observed", "t_final_week"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(
        f"'enrollment_survival_ready' is missing required columns: {missing_cols}"
    )

df = df.copy()
df["group_id"] = (
    df["code_module"].astype(str) + "||" + df["code_presentation"].astype(str)
)

# ------------------------------------------------------------
# 1) Inventory of presentation groups
# ------------------------------------------------------------
group_inventory = (
    df.groupby(["code_module", "code_presentation", "group_id"], dropna=False)
    .agg(
        n_enrollments=("id_student", "size"),
        n_unique_students=("id_student", "nunique"),
        n_events=("event_observed", "sum"),
        event_rate=("event_observed", "mean"),
        max_t_final_week=("t_final_week", "max"),
        median_t_final_week=("t_final_week", "median"),
    )
    .reset_index()
    .sort_values(["code_module", "code_presentation"])
    .reset_index(drop=True)
)

group_inventory["event_rate"] = group_inventory["event_rate"].astype(float)
group_inventory["max_t_final_week"] = pd.to_numeric(group_inventory["max_t_final_week"], errors="coerce")
group_inventory["median_t_final_week"] = pd.to_numeric(group_inventory["median_t_final_week"], errors="coerce")

total_enrollments = int(len(df))
total_events = int(pd.to_numeric(df["event_observed"], errors="coerce").fillna(0).sum())
overall_event_rate = total_events / total_enrollments
all_modules = sorted(group_inventory["code_module"].astype(str).unique().tolist())
n_total_modules = len(all_modules)

print(f"Total enrollments: {total_enrollments}")
print(f"Overall event rate: {overall_event_rate:.6f}")
print(f"Total groups: {len(group_inventory)}")
print(f"Distinct modules: {n_total_modules}")

# ------------------------------------------------------------
# 2) Search configuration
# ------------------------------------------------------------
target_test_share = 0.30
target_test_n = int(round(total_enrollments * target_test_share))

# Candidate number of groups in the test split.
# With 22 groups total, this is still manageable combinatorially.
candidate_k_values = [4, 5, 6, 7]

# Score weights
w_share = 1.00
w_event = 1.00
w_module = 0.35
w_overrun = 0.15

# Optional penalty if the known duration-overrun context goes to test
special_overrun_group = "FFF||2013J"

# Module coverage target for the test set:
# reward broader curricular coverage.
def module_coverage_penalty(test_modules_count: int, total_modules_count: int) -> float:
    # penalty decreases as the number of distinct modules in test increases
    return 1.0 - (test_modules_count / total_modules_count)

# ------------------------------------------------------------
# 3) Evaluate candidate subsets
# ------------------------------------------------------------
records = []

groups_as_records = group_inventory.to_dict(orient="records")

for k in candidate_k_values:
    for combo in itertools.combinations(groups_as_records, k):
        test_group_ids = [x["group_id"] for x in combo]
        test_modules = sorted({str(x["code_module"]) for x in combo})

        test_n = int(sum(int(x["n_enrollments"]) for x in combo))
        test_events = int(sum(int(x["n_events"]) for x in combo))
        test_share = test_n / total_enrollments
        test_event_rate = test_events / test_n if test_n > 0 else np.nan

        share_error = abs(test_share - target_test_share)
        event_error = abs(test_event_rate - overall_event_rate) if pd.notna(test_event_rate) else np.inf
        module_pen = module_coverage_penalty(len(test_modules), n_total_modules)
        overrun_pen = 1.0 if special_overrun_group in test_group_ids else 0.0

        # hard guardrails to avoid clearly poor solutions
        if not (0.25 <= test_share <= 0.35):
            continue

        score = (
            w_share * share_error
            + w_event * event_error
            + w_module * module_pen
            + w_overrun * overrun_pen
        )

        records.append({
            "k_groups": k,
            "test_group_ids": " | ".join(sorted(test_group_ids)),
            "test_modules": " | ".join(test_modules),
            "n_test_groups": len(test_group_ids),
            "n_test_modules": len(test_modules),
            "test_n": test_n,
            "test_share": test_share,
            "test_events": test_events,
            "test_event_rate": test_event_rate,
            "share_error_abs": share_error,
            "event_rate_error_abs": event_error,
            "module_coverage_penalty": module_pen,
            "includes_special_overrun_group": int(special_overrun_group in test_group_ids),
            "score": score,
        })

candidate_solutions = pd.DataFrame(records)

if candidate_solutions.empty:
    raise RuntimeError(
        "No candidate grouped split satisfied the search constraints. "
        "Relax the candidate_k_values or the test_share guardrails."
    )

candidate_solutions = (
    candidate_solutions
    .sort_values(
        [
            "score",
            "share_error_abs",
            "event_rate_error_abs",
            "includes_special_overrun_group",
            "n_test_modules",
            "test_n",
        ],
        ascending=[True, True, True, True, False, True]
    )
    .reset_index(drop=True)
)

best_solution = candidate_solutions.iloc[0].to_dict()
best_test_group_ids = best_solution["test_group_ids"].split(" | ")

print("\nBest solution found:")
for k, v in best_solution.items():
    print(f"{k}: {v}")

# ------------------------------------------------------------
# 4) Materialize design table (design only, not split tables yet)
# ------------------------------------------------------------
design = group_inventory.copy()
design["is_test_group"] = design["group_id"].isin(best_test_group_ids).astype(int)
design["split_role"] = np.where(design["is_test_group"] == 1, "test", "train")
design = design.sort_values(["split_role", "code_module", "code_presentation"]).reset_index(drop=True)

# ------------------------------------------------------------
# 5) Summary of selected design
# ------------------------------------------------------------
summary_rows = []

for role in ["train", "test"]:
    tmp = design.loc[design["split_role"] == role].copy()
    summary_rows.append({
        "split_role": role,
        "n_groups": int(len(tmp)),
        "n_modules": int(tmp["code_module"].astype(str).nunique()),
        "n_enrollments": int(tmp["n_enrollments"].sum()),
        "n_unique_students_sum_over_groups": int(tmp["n_unique_students"].sum()),
        "n_events": int(tmp["n_events"].sum()),
        "event_rate_weighted": float(tmp["n_events"].sum() / tmp["n_enrollments"].sum()) if tmp["n_enrollments"].sum() > 0 else np.nan,
        "min_group_size": int(tmp["n_enrollments"].min()) if len(tmp) > 0 else np.nan,
        "median_group_size": float(tmp["n_enrollments"].median()) if len(tmp) > 0 else np.nan,
        "max_group_size": int(tmp["n_enrollments"].max()) if len(tmp) > 0 else np.nan,
        "max_followup_week": pd.to_numeric(tmp["max_t_final_week"], errors="coerce").max() if len(tmp) > 0 else np.nan,
    })

split_design_summary_role = pd.DataFrame(summary_rows)

overall_summary = pd.DataFrame([{
    "total_enrollments": int(total_enrollments),
    "overall_event_rate": float(overall_event_rate),
    "target_test_share": float(target_test_share),
    "target_test_n": int(target_test_n),
    "actual_test_n": int(design.loc[design["split_role"] == "test", "n_enrollments"].sum()),
    "actual_test_share": float(
        design.loc[design["split_role"] == "test", "n_enrollments"].sum() / total_enrollments
    ),
    "n_total_groups": int(len(design)),
    "n_test_groups": int((design["split_role"] == "test").sum()),
    "n_train_groups": int((design["split_role"] == "train").sum()),
    "n_total_modules": int(n_total_modules),
    "n_test_modules": int(design.loc[design["split_role"] == "test", "code_module"].astype(str).nunique()),
    "n_train_modules": int(design.loc[design["split_role"] == "train", "code_module"].astype(str).nunique()),
    "best_score": float(best_solution["score"]),
    "best_share_error_abs": float(best_solution["share_error_abs"]),
    "best_event_rate_error_abs": float(best_solution["event_rate_error_abs"]),
    "best_includes_special_overrun_group": int(best_solution["includes_special_overrun_group"]),
}])

table_p13_3_split_design_summary = pd.concat(
    [overall_summary.assign(section="overall"), split_design_summary_role.assign(section="by_role")],
    ignore_index=True,
    sort=False
)

# keep top candidate solutions for audit
table_p13_3_candidate_solutions = candidate_solutions.head(25).copy()

# ------------------------------------------------------------
# 6) Save outputs
# ------------------------------------------------------------
path_inventory = OUT_TABLES / "table_p13_3_group_inventory.csv"
path_design = OUT_TABLES / "table_p13_3_group_split_design.csv"
path_summary = OUT_TABLES / "table_p13_3_split_design_summary.csv"
path_candidates = OUT_TABLES / "table_p13_3_candidate_solutions.csv"
path_metadata = OUT_METADATA / "metadata_p13_3_context_heldout_split_design.json"

group_inventory.to_csv(path_inventory, index=False)
design.to_csv(path_design, index=False)
table_p13_3_split_design_summary.to_csv(path_summary, index=False)
table_p13_3_candidate_solutions.to_csv(path_candidates, index=False)

metadata = {
    "step": "P13.3",
    "title": "Context-Held-Out Split Design",
    "version": "rewritten_subset_search",
    "grouping_unit": ["code_module", "code_presentation"],
    "design_type": "subset_search_group_holdout",
    "target_test_share": target_test_share,
    "target_test_n": target_test_n,
    "overall_event_rate": float(overall_event_rate),
    "candidate_k_values": candidate_k_values,
    "score_weights": {
        "w_share": w_share,
        "w_event": w_event,
        "w_module": w_module,
        "w_overrun": w_overrun,
    },
    "special_overrun_group": special_overrun_group,
    "selected_test_group_ids": best_test_group_ids,
    "actual_test_n": int(design.loc[design["split_role"] == "test", "n_enrollments"].sum()),
    "actual_test_share": float(
        design.loc[design["split_role"] == "test", "n_enrollments"].sum() / total_enrollments
    ),
    "n_total_groups": int(len(design)),
    "n_test_groups": int((design["split_role"] == "test").sum()),
    "n_train_groups": int((design["split_role"] == "train").sum()),
    "notes": [
        "whole (code_module, code_presentation) groups are held out together",
        "test groups are never seen in training",
        "subset search optimizes approximate test share, event-rate proximity, and module diversity",
        "this step defines the split design but does not yet materialize split-specific benchmark tables"
    ],
    "output_tables": [
        path_inventory.as_posix(),
        path_design.as_posix(),
        path_summary.as_posix(),
        path_candidates.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 7) Display
# ------------------------------------------------------------
print("\nGroup inventory:")
display(group_inventory)

print("\nSelected grouped split design:")
display(design)

print("\nSplit design summary:")
display(table_p13_3_split_design_summary)

print("\nTop candidate solutions:")
display(table_p13_3_candidate_solutions)

print("\nSaved:")
print("-", path_inventory.resolve())
print("-", path_design.resolve())
print("-", path_summary.resolve())
print("-", path_candidates.resolve())
print("-", path_metadata.resolve())

### C2.4 — Materialize Context-Held-Out Partitions

In [ ]:
# ============================================================
# C2.4 — Materialize Context-Held-Out Partitions
# ============================================================
# Purpose:
#   Materialize the grouped context-held-out split approved in P13.3.
#
# Required inputs:
#   - DuckDB table: enrollment_survival_ready
#   - CSV from P13.3: table_p13_3_group_split_design.csv
#
# Main outputs:
#   - DuckDB table: enrollment_survival_ready_context_split
#   - table_p13_4_context_split_assignment_audit.csv
#   - table_p13_4_context_split_overlap_audit.csv
#   - table_p13_4_context_split_group_summary.csv
#   - table_p13_4_context_split_partition_summary.csv
#   - metadata_p13_4_context_split_materialization.json
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C2.4 — Materialize Context-Held-Out Partitions")
print("=" * 70)

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

if "enrollment_survival_ready" not in available_tables:
    raise RuntimeError(
        "DuckDB table 'enrollment_survival_ready' not found. "
        "Run the corrected P5 first."
    )

path_design = OUT_TABLES / "table_p13_3_group_split_design.csv"
if not path_design.exists():
    raise FileNotFoundError(
        f"Required design file not found: {path_design.resolve()}. "
        "Run P13.3 first."
    )

# ------------------------------------------------------------
# 1) Load canonical enrollment table and approved design
# ------------------------------------------------------------
enr = con.execute("""
    SELECT *
    FROM enrollment_survival_ready
""").fetchdf()

design = pd.read_csv(path_design)

required_enr_cols = [
    "id_student", "code_module", "code_presentation",
    "event_observed", "t_final_week"
]
missing_enr_cols = [c for c in required_enr_cols if c not in enr.columns]
if missing_enr_cols:
    raise KeyError(
        f"'enrollment_survival_ready' is missing required columns: {missing_enr_cols}"
    )

required_design_cols = [
    "code_module", "code_presentation", "group_id", "is_test_group", "split_role"
]
missing_design_cols = [c for c in required_design_cols if c not in design.columns]
if missing_design_cols:
    raise KeyError(
        f"P13.3 design file is missing required columns: {missing_design_cols}"
    )

enr = enr.copy()
enr["group_id"] = (
    enr["code_module"].astype(str) + "||" + enr["code_presentation"].astype(str)
)

design = design.copy()
design["group_id"] = design["group_id"].astype(str)
design["split_role"] = design["split_role"].astype(str)
design["is_test_group"] = pd.to_numeric(design["is_test_group"], errors="coerce").fillna(0).astype(int)

# ------------------------------------------------------------
# 2) Merge split assignment onto enrollment table
# ------------------------------------------------------------
context_split = enr.merge(
    design[["group_id", "code_module", "code_presentation", "is_test_group", "split_role"]],
    on=["group_id", "code_module", "code_presentation"],
    how="left",
    validate="many_to_one"
)

missing_assignment_mask = context_split["split_role"].isna()
n_missing_assignments = int(missing_assignment_mask.sum())

if n_missing_assignments > 0:
    raise RuntimeError(
        f"{n_missing_assignments} enrollments did not receive a context split assignment. "
        "P13.3 design and enrollment_survival_ready are not aligned."
    )

context_split["context_split_name"] = "context_heldout_v1"
context_split["context_is_test"] = (context_split["split_role"] == "test").astype(int)
context_split["context_is_train"] = (context_split["split_role"] == "train").astype(int)

# ------------------------------------------------------------
# 3) Persist canonical DuckDB table
# ------------------------------------------------------------
con.register("tmp_enrollment_survival_ready_context_split_df", context_split)
con.execute("DROP TABLE IF EXISTS enrollment_survival_ready_context_split")
con.execute("""
    CREATE TABLE enrollment_survival_ready_context_split AS
    SELECT *
    FROM tmp_enrollment_survival_ready_context_split_df
""")

# ------------------------------------------------------------
# 4) Partition summary
# ------------------------------------------------------------
partition_summary = (
    context_split.groupby("split_role", dropna=False)
    .agg(
        n_enrollments=("id_student", "size"),
        n_unique_students=("id_student", "nunique"),
        n_groups=("group_id", "nunique"),
        n_modules=("code_module", "nunique"),
        n_events=("event_observed", "sum"),
        event_rate=("event_observed", "mean"),
        max_t_final_week=("t_final_week", "max"),
        median_t_final_week=("t_final_week", "median"),
    )
    .reset_index()
    .sort_values("split_role")
    .reset_index(drop=True)
)

overall_partition_summary = pd.DataFrame([{
    "context_split_name": "context_heldout_v1",
    "n_total_enrollments": int(len(context_split)),
    "n_total_groups": int(context_split["group_id"].nunique()),
    "n_total_modules": int(context_split["code_module"].nunique()),
    "n_missing_assignments": n_missing_assignments,
    "test_share": float((context_split["split_role"] == "test").mean()),
    "overall_event_rate": float(pd.to_numeric(context_split["event_observed"], errors="coerce").fillna(0).mean()),
}])

table_p13_4_context_split_partition_summary = pd.concat(
    [overall_partition_summary.assign(section="overall"),
     partition_summary.assign(section="by_role")],
    ignore_index=True,
    sort=False
)

# ------------------------------------------------------------
# 5) Group-level summary
# ------------------------------------------------------------
table_p13_4_context_split_group_summary = (
    context_split.groupby(
        ["split_role", "code_module", "code_presentation", "group_id"],
        dropna=False
    )
    .agg(
        n_enrollments=("id_student", "size"),
        n_unique_students=("id_student", "nunique"),
        n_events=("event_observed", "sum"),
        event_rate=("event_observed", "mean"),
        max_t_final_week=("t_final_week", "max"),
        median_t_final_week=("t_final_week", "median"),
    )
    .reset_index()
    .sort_values(["split_role", "code_module", "code_presentation"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6) Overlap audit
# ------------------------------------------------------------
train_groups = set(
    context_split.loc[context_split["split_role"] == "train", "group_id"].astype(str).unique().tolist()
)
test_groups = set(
    context_split.loc[context_split["split_role"] == "test", "group_id"].astype(str).unique().tolist()
)
group_overlap = sorted(train_groups.intersection(test_groups))

train_modules = set(
    context_split.loc[context_split["split_role"] == "train", "code_module"].astype(str).unique().tolist()
)
test_modules = set(
    context_split.loc[context_split["split_role"] == "test", "code_module"].astype(str).unique().tolist()
)
module_overlap = sorted(train_modules.intersection(test_modules))

table_p13_4_context_split_overlap_audit = pd.DataFrame([{
    "context_split_name": "context_heldout_v1",
    "n_train_groups": len(train_groups),
    "n_test_groups": len(test_groups),
    "n_group_overlap_between_train_and_test": len(group_overlap),
    "group_overlap_values": " | ".join(group_overlap),
    "n_train_modules": len(train_modules),
    "n_test_modules": len(test_modules),
    "n_module_overlap_between_train_and_test": len(module_overlap),
    "module_overlap_values": " | ".join(module_overlap),
    "group_overlap_is_zero": len(group_overlap) == 0,
    "module_overlap_expected_and_allowed": True,
}])

# ------------------------------------------------------------
# 7) Assignment audit
# ------------------------------------------------------------
assignment_audit = pd.DataFrame([{
    "context_split_name": "context_heldout_v1",
    "n_rows_in_context_table": int(len(context_split)),
    "n_unique_groups_in_context_table": int(context_split["group_id"].nunique()),
    "n_unique_modules_in_context_table": int(context_split["code_module"].nunique()),
    "n_missing_assignments": n_missing_assignments,
    "all_rows_assigned": n_missing_assignments == 0,
    "n_test_rows": int((context_split["split_role"] == "test").sum()),
    "n_train_rows": int((context_split["split_role"] == "train").sum()),
    "test_share": float((context_split["split_role"] == "test").mean()),
    "event_rate_train": float(
        pd.to_numeric(
            context_split.loc[context_split["split_role"] == "train", "event_observed"],
            errors="coerce"
        ).fillna(0).mean()
    ),
    "event_rate_test": float(
        pd.to_numeric(
            context_split.loc[context_split["split_role"] == "test", "event_observed"],
            errors="coerce"
        ).fillna(0).mean()
    ),
}])

table_p13_4_context_split_assignment_audit = assignment_audit

# ------------------------------------------------------------
# 8) Save outputs
# ------------------------------------------------------------
path_assignment_audit = OUT_TABLES / "table_p13_4_context_split_assignment_audit.csv"
path_overlap_audit = OUT_TABLES / "table_p13_4_context_split_overlap_audit.csv"
path_group_summary = OUT_TABLES / "table_p13_4_context_split_group_summary.csv"
path_partition_summary = OUT_TABLES / "table_p13_4_context_split_partition_summary.csv"
path_metadata = OUT_METADATA / "metadata_p13_4_context_split_materialization.json"

table_p13_4_context_split_assignment_audit.to_csv(path_assignment_audit, index=False)
table_p13_4_context_split_overlap_audit.to_csv(path_overlap_audit, index=False)
table_p13_4_context_split_group_summary.to_csv(path_group_summary, index=False)
table_p13_4_context_split_partition_summary.to_csv(path_partition_summary, index=False)

metadata = {
    "step": "P13.4",
    "title": "Materialize Context-Held-Out Partitions",
    "context_split_name": "context_heldout_v1",
    "source_design_file": path_design.as_posix(),
    "source_enrollment_table": "enrollment_survival_ready",
    "output_duckdb_table": "enrollment_survival_ready_context_split",
    "n_total_rows": int(len(context_split)),
    "n_total_groups": int(context_split["group_id"].nunique()),
    "n_total_modules": int(context_split["code_module"].nunique()),
    "n_missing_assignments": n_missing_assignments,
    "group_overlap_is_zero": len(group_overlap) == 0,
    "selected_test_groups": sorted(test_groups),
    "notes": [
        "whole (code_module, code_presentation) groups are assigned together",
        "group overlap between train and test must be zero",
        "module overlap is expected and allowed because the design is presentation-held-out, not module-held-out",
        "this table does not replace the main benchmark split"
    ],
    "output_tables": [
        path_assignment_audit.as_posix(),
        path_overlap_audit.as_posix(),
        path_group_summary.as_posix(),
        path_partition_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 9) Display
# ------------------------------------------------------------
print("\nContext split assignment audit:")
display(table_p13_4_context_split_assignment_audit)

print("\nContext split overlap audit:")
display(table_p13_4_context_split_overlap_audit)

print("\nContext split partition summary:")
display(table_p13_4_context_split_partition_summary)

print("\nContext split group summary:")
display(table_p13_4_context_split_group_summary)

print("\nSaved:")
print("-", path_assignment_audit.resolve())
print("-", path_overlap_audit.resolve())
print("-", path_group_summary.resolve())
print("-", path_partition_summary.resolve())
print("-", path_metadata.resolve())

## C3 — Prepare the Preprocessing Layer for Modeling

In [ ]:
# ==============================================================
# C3 — Prepare the Preprocessing Layer for Modeling
# --------------------------------------------------------------
# Purpose:
#   Materialize train/test datasets for modeling from the
#   split-propagated modeling input tables.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - pp_linear_hazard_input
#   - pp_neural_hazard_input
#   - enrollment_cox_input_configured
#   - enrollment_deepsurv_input_configured
#
# Canonical naming policy in this step:
#   - pp_linear_hazard_input_split
#   - pp_neural_hazard_input_split
#   - enrollment_cox_input_configured_split
#   - enrollment_deepsurv_input_configured_split
#
# Outputs:
#   - DuckDB tables:
#       * pp_linear_hazard_ready_train
#       * pp_linear_hazard_ready_test
#       * pp_neural_hazard_ready_train
#       * pp_neural_hazard_ready_test
#       * enrollment_cox_ready_train
#       * enrollment_cox_ready_test
#       * enrollment_deepsurv_ready_train
#       * enrollment_deepsurv_ready_test
#   - audit summary tables
# ==============================================================

print("\n" + "=" * 70)
print("C3 — Prepare the Preprocessing Layer for Modeling")
print("=" * 70)

print("Methodological note: this step materializes train/test datasets only.")
print("No learned preprocessing transformation is fitted here.")
print("All datasets are derived from split-propagated input tables.")

import pandas as pd
from pathlib import Path

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

# ------------------------------
# 2) Canonicalize split table names (no fallback in materialization)
# ------------------------------
base_to_split = {
    "pp_linear_hazard_input": "pp_linear_hazard_input_split",
    "pp_neural_hazard_input": "pp_neural_hazard_input_split",
    "enrollment_cox_input_configured": "enrollment_cox_input_configured_split",
    "enrollment_deepsurv_input_configured": "enrollment_deepsurv_input_configured_split",
}

missing_base = [t for t in base_to_split if t not in available_tables]
if missing_base:
    raise RuntimeError(
        "Missing required upstream input table(s): " + ", ".join(missing_base)
    )

for base_table, split_table in base_to_split.items():
    con.execute(f"DROP TABLE IF EXISTS {split_table}")
    con.execute(f"""
    CREATE TABLE {split_table} AS
    SELECT *
    FROM {base_table}
    """)

# Strict requirement: only canonical *_split names from here onward
required_tables = [
    "pp_linear_hazard_input_split",
    "pp_neural_hazard_input_split",
    "enrollment_cox_input_configured_split",
    "enrollment_deepsurv_input_configured_split",
]

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)
missing_tables = [t for t in required_tables if t not in available_tables]
if missing_tables:
    raise RuntimeError(
        "Missing required split-propagated input table(s): "
        + ", ".join(missing_tables)
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 3) Materialize ready train/test tables
# ------------------------------
materialization_specs = [
    ("pp_linear_hazard_input_split", "pp_linear_hazard_ready_train", "train"),
    ("pp_linear_hazard_input_split", "pp_linear_hazard_ready_test", "test"),
    ("pp_neural_hazard_input_split", "pp_neural_hazard_ready_train", "train"),
    ("pp_neural_hazard_input_split", "pp_neural_hazard_ready_test", "test"),
    ("enrollment_cox_input_configured_split", "enrollment_cox_ready_train", "train"),
    ("enrollment_cox_input_configured_split", "enrollment_cox_ready_test", "test"),
    ("enrollment_deepsurv_input_configured_split", "enrollment_deepsurv_ready_train", "train"),
    ("enrollment_deepsurv_input_configured_split", "enrollment_deepsurv_ready_test", "test"),
]

for source_table, target_table, split_value in materialization_specs:
    con.execute(f"DROP TABLE IF EXISTS {target_table}")
    con.execute(f"""
    CREATE TABLE {target_table} AS
    SELECT *
    FROM {source_table}
    WHERE split = '{split_value}'
    """)

# ------------------------------
# 4) Audit summary
# ------------------------------
ready_tables = [
    "pp_linear_hazard_ready_train",
    "pp_linear_hazard_ready_test",
    "pp_neural_hazard_ready_train",
    "pp_neural_hazard_ready_test",
    "enrollment_cox_ready_train",
    "enrollment_cox_ready_test",
    "enrollment_deepsurv_ready_train",
    "enrollment_deepsurv_ready_test",
]

audit_rows = []

for table_name in ready_tables:
    cols = con.execute(f"PRAGMA table_info('{table_name}')").fetchdf()["name"].astype(str).tolist()
    n_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    n_distinct_enrollments = con.execute(
        f"SELECT COUNT(DISTINCT enrollment_id) FROM {table_name}"
    ).fetchone()[0]

    split_values = con.execute(f"""
    SELECT STRING_AGG(DISTINCT split, ', ' ORDER BY split)
    FROM {table_name}
    """).fetchone()[0]

    target_col = None
    target_sum = None
    duration_min = None
    duration_max = None

    if "event_t" in cols:
        target_col = "event_t"
        target_sum = con.execute(f"SELECT COALESCE(SUM(event_t), 0) FROM {table_name}").fetchone()[0]

    elif "event" in cols:
        target_col = "event"
        target_sum = con.execute(f"SELECT COALESCE(SUM(event), 0) FROM {table_name}").fetchone()[0]

    if "duration" in cols:
        duration_min, duration_max = con.execute(
            f"SELECT MIN(duration), MAX(duration) FROM {table_name}"
        ).fetchone()

    audit_rows.append({
        "table_name": table_name,
        "n_rows": int(n_rows),
        "n_distinct_enrollments": int(n_distinct_enrollments),
        "n_columns": len(cols),
        "split_values_present": split_values,
        "target_column": target_col,
        "target_sum": float(target_sum) if target_sum is not None else None,
        "duration_min": float(duration_min) if duration_min is not None else None,
        "duration_max": float(duration_max) if duration_max is not None else None,
    })

audit_df = pd.DataFrame(audit_rows)

audit_path = TABLES_DIR / "table_p14_ready_dataset_audit.csv"
audit_df.to_csv(audit_path, index=False)

# ------------------------------
# 5) Export ready tables
# ------------------------------
for table_name in ready_tables:
    parquet_path = DATA_OUTPUT_DIR / f"{table_name}.parquet"
    csv_path = DATA_OUTPUT_DIR / f"{table_name}.csv"
    con.execute(f"COPY {table_name} TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
    con.execute(f"COPY {table_name} TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

# ------------------------------
# 6) Output
# ------------------------------
print("\nCanonical split tables materialized:")
for base_table, split_table in base_to_split.items():
    print(f"- {base_table} -> {split_table}")

print("\nReady dataset audit summary:")
display(audit_df)

print("\nSaved:")
print("-", audit_path.resolve())

print("\nExported tables:")
for table_name in ready_tables:
    print("-", (DATA_OUTPUT_DIR / f"{table_name}.parquet").resolve())
    print("-", (DATA_OUTPUT_DIR / f"{table_name}.csv").resolve())

# D — Modeling

Scope:
- benchmark protocol
- model-specific preprocessing
- performance-oriented tuned models
- consolidated benchmark comparison
- expanded calibration interpretation


## D1 — Define Benchmark Metrics and Evaluation Protocol (Revised v3)


In [ ]:
# ==============================================================
# D1 — Define Benchmark Metrics and Evaluation Protocol (Revised v3)
# --------------------------------------------------------------
# Purpose:
#   Define the formal benchmark evaluation protocol for the study,
#   including:
#     - metric hierarchy
#     - official benchmark horizons
#     - common unit of comparison
#     - censoring treatment
#     - concordance convention
#     - dynamic-vs-static prediction rules
#     - calibration protocol, including strengthened diagnostics
#
# Methodological note:
#   This benchmark is intentionally survival-oriented. The goal is
#   not to compare models using a single classification score, but
#   to assess them under a unified time-to-event evaluation frame.
#
#   This revised version makes explicit:
#     - how censoring is handled in Brier / IBS,
#     - which concordance convention is used,
#     - how dynamic predictions are defined for discrete-time models,
#     - how leakage is prevented,
#     - how horizon-level calibration is computed and reported,
#     - and how the expanded calibration diagnostics from Group E
#       fit into the benchmark protocol.
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#
# Optional inputs:
#   - table_p26_4_expanded_calibration_protocol.csv
#   - table_p26_5_paper_summary_all_tuned_models.csv
#   - table_p26_6_calibration_model_rankings.csv
#
# Outputs:
#   - BENCHMARK_CONFIG dictionary
#   - benchmark_config.json
#   - table_benchmark_metrics_manifest.csv
#   - table_evaluation_protocol_audit.csv
# ==============================================================

print("\n" + "=" * 70)
print("D1 — Define Benchmark Metrics and Evaluation Protocol (Revised v3)")
print("=" * 70)
print("Methodological note: the benchmark is survival-oriented and")
print("will be evaluated under a common enrollment-level protocol.")
print("This revision makes censoring, concordance, prediction timing,")
print("and the expanded calibration protocol explicit.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

OUTPUT_DIR = Path(OUTPUT_DIR)
TABLES_DIR = Path(TABLES_DIR)
METADATA_DIR = Path(METADATA_DIR)

# ------------------------------
# 2) Detect official horizons
# ------------------------------
p26_4_protocol_path = TABLES_DIR / "table_p26_4_expanded_calibration_protocol.csv"

if p26_4_protocol_path.exists():
    p26_4_protocol_df = pd.read_csv(p26_4_protocol_path)
    shared_horizons_row = p26_4_protocol_df.loc[
        p26_4_protocol_df["protocol_component"] == "shared_horizons",
        "value"
    ]
    if not shared_horizons_row.empty:
        BENCHMARK_HORIZONS = [
            int(x.strip())
            for x in str(shared_horizons_row.iloc[0]).split(",")
            if str(x).strip()
        ]
    else:
        BENCHMARK_HORIZONS = [10, 20, 30]
else:
    BENCHMARK_HORIZONS = [10, 20, 30]

BENCHMARK_HORIZONS = sorted(list(dict.fromkeys(BENCHMARK_HORIZONS)))

# ------------------------------
# 3) Protocol core definitions
# ------------------------------
BENCHMARK_CONFIG = {
    "protocol_name": "harmonized_survival_benchmark",
    "protocol_version": "P15_revised_v3",
    "created_at": datetime.utcnow().isoformat() + "Z",

    # Unit and event frame
    "unit_of_analysis": "enrollment",
    "event_definition": "withdrawn_with_valid_date_unregistration",
    "time_granularity": "weekly",
    "official_benchmark_horizons": BENCHMARK_HORIZONS,

    # Family comparison frame
    "comparison_scope": "cross-family survival-oriented benchmark",
    "comparison_unit_for_reporting": "enrollment-level horizon-specific predictions",
    "comparable_reporting_horizons": BENCHMARK_HORIZONS,

    # Prediction regimes
    "prediction_regimes": {
        "continuous_time_comparable_arm": (
            "Enrollment-level static prediction using early-window summarized covariates, "
            "evaluated at shared fixed horizons."
        ),
        "discrete_time_dynamic_arm": (
            "Weekly person-period prediction with horizon-level aggregation to enrollment-level "
            "risk summaries at shared fixed horizons."
        ),
        "benchmark_rule": (
            "Cross-family comparison is performed only at shared fixed horizons under a common "
            "enrollment-level reporting frame."
        ),
    },

    # Leakage / train-test discipline
    "anti_leakage_rules": [
        "All learned preprocessing transformations are fit on training data only.",
        "Test data are transformed using the training-fitted objects without re-estimation.",
        "Model tuning and internal validation remain restricted to training-side information.",
        "Evaluation uses only test-set predictions generated after model freeze."
    ],

    # Censoring conventions
    "censoring_protocol": {
        "event_type": "withdrawn",
        "right_censoring": True,
        "censoring_definition": (
            "Enrollments without an observed withdrawal event by the end of follow-up are treated as right-censored."
        ),
        "brier_and_ibs_treatment": (
            "Brier-type metrics and integrated Brier score follow the benchmark-specific censoring-aware convention "
            "already implemented in the notebook outputs."
        ),
    },

    # Concordance conventions
    "concordance_protocol": {
        "primary_time_dependent_concordance": (
            "Time-dependent concordance under the benchmark-specific implementation."
        ),
        "secondary_horizon_specific_discrimination": (
            "AUC or horizon-specific discrimination summaries evaluated at the shared horizons."
        ),
    },

    # Calibration conventions
    "calibration_protocol": {
        "primary_calibration_metric": "weighted_absolute_calibration_gap_by_horizon",
        "primary_metric_definition": (
            "Sample-size-weighted absolute gap between mean predicted risk and observed event frequency across calibration bins."
        ),
        "primary_metric_role": (
            "Primary calibration ranking signal in the tuned cross-model comparison."
        ),
        "supporting_metrics": [
            "brier_at_horizon",
            "event_rate_at_horizon",
            "support_n_at_horizon"
        ],
        "expanded_strengthening_metrics": [
            "approx_calibration_intercept_by_horizon",
            "approx_calibration_slope_by_horizon"
        ],
        "strengthening_metric_interpretation": (
            "Intercept and slope are used as calibration-strengthening diagnostics and interpretive qualifiers, "
            "not as a standalone replacement for the primary calibration gap."
        ),
        "optional_metric_not_required": "ici_style_summary",
        "official_horizons": BENCHMARK_HORIZONS,
        "comparability_rule": (
            "Calibration comparison is restricted to shared horizons where all compared tuned models provide valid outputs."
        ),
    },

    # Metric hierarchy
    "metric_hierarchy": {
        "primary_global_metrics": [
            "integrated_brier_score",
            "time_dependent_concordance"
        ],
        "primary_horizon_metrics": [
            "brier_at_horizon",
            "weighted_absolute_calibration_gap_by_horizon"
        ],
        "secondary_metrics": [
            "horizon_specific_auc",
            "predicted_vs_observed_survival"
        ],
        "contextual_audit_metrics": [
            "support_n_at_horizon",
            "event_rate_at_horizon",
            "approx_calibration_intercept_by_horizon",
            "approx_calibration_slope_by_horizon"
        ],
    },

    # Sensitivity / robustness framing
    "sensitivity_design": {
        "early_window_length_sensitivity": [2, 4, 6],
        "horizon_choice_stress_test": BENCHMARK_HORIZONS,
        "ablation_available": True,
        "note": (
            "Window choice and horizon choice are treated as explicit sensitivity dimensions rather than hidden defaults."
        ),
    },
}

# ------------------------------
# 4) Benchmark metrics manifest
# ------------------------------
metrics_manifest_rows = [
    {
        "metric_name": "integrated_brier_score",
        "metric_family": "overall_prediction_quality",
        "metric_role": "primary_global",
        "lower_is_better": True,
        "time_scope": "integrated_over_followup",
        "used_in_primary_ranking": True,
        "interpretation": "Integrated Brier Score under the benchmark censoring-aware convention."
    },
    {
        "metric_name": "time_dependent_concordance",
        "metric_family": "discrimination",
        "metric_role": "primary_global",
        "lower_is_better": False,
        "time_scope": "integrated_over_followup",
        "used_in_primary_ranking": True,
        "interpretation": "Time-dependent concordance under the benchmark convention."
    },
    {
        "metric_name": "brier_at_horizon",
        "metric_family": "overall_prediction_quality",
        "metric_role": "primary_horizon",
        "lower_is_better": True,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": True,
        "interpretation": "Horizon-specific Brier score reported at the official benchmark horizons."
    },
    {
        "metric_name": "weighted_absolute_calibration_gap_by_horizon",
        "metric_family": "calibration",
        "metric_role": "primary_horizon",
        "lower_is_better": True,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": True,
        "interpretation": "Primary calibration ranking signal based on weighted absolute gap across bins."
    },
    {
        "metric_name": "horizon_specific_auc",
        "metric_family": "discrimination",
        "metric_role": "secondary",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Secondary discrimination metric reported at shared horizons."
    },
    {
        "metric_name": "support_n_at_horizon",
        "metric_family": "audit_context",
        "metric_role": "contextual",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Number of evaluable enrollments contributing to each horizon."
    },
    {
        "metric_name": "event_rate_at_horizon",
        "metric_family": "audit_context",
        "metric_role": "contextual",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Observed event prevalence at each shared benchmark horizon."
    },
    {
        "metric_name": "approx_calibration_intercept_by_horizon",
        "metric_family": "calibration_strengthening",
        "metric_role": "diagnostic",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Calibration-strengthening diagnostic for directional bias at each horizon."
    },
    {
        "metric_name": "approx_calibration_slope_by_horizon",
        "metric_family": "calibration_strengthening",
        "metric_role": "diagnostic",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Calibration-strengthening diagnostic for compression/dispersion of predictions."
    },
    {
        "metric_name": "predicted_vs_observed_survival",
        "metric_family": "diagnostic",
        "metric_role": "diagnostic",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons_or_curve_level",
        "used_in_primary_ranking": False,
        "interpretation": "Visual agreement between predicted and observed survival summaries."
    },
]

benchmark_metrics_manifest_df = pd.DataFrame(metrics_manifest_rows)

# ------------------------------
# 5) Protocol audit table
# ------------------------------
protocol_audit_rows = [
    {
        "component": "unit_of_analysis",
        "status": "defined",
        "protocol_value": BENCHMARK_CONFIG["unit_of_analysis"],
        "notes": "Common unit for cross-family reporting."
    },
    {
        "component": "event_definition",
        "status": "defined",
        "protocol_value": BENCHMARK_CONFIG["event_definition"],
        "notes": "Observed withdrawal event with valid time stamp."
    },
    {
        "component": "official_horizons",
        "status": "defined",
        "protocol_value": ",".join(map(str, BENCHMARK_HORIZONS)),
        "notes": "Shared benchmark horizons used for fixed-horizon comparison."
    },
    {
        "component": "dynamic_vs_static_prediction_rule",
        "status": "defined",
        "protocol_value": "shared_horizon_enrollment_level_reporting",
        "notes": "Different model families are compared only after horizon-level alignment."
    },
    {
        "component": "censoring_treatment",
        "status": "defined",
        "protocol_value": "right_censoring_enabled",
        "notes": "Brier/IBS follow the benchmark-specific censoring-aware implementation."
    },
    {
        "component": "primary_calibration_metric",
        "status": "defined",
        "protocol_value": "weighted_absolute_calibration_gap_by_horizon",
        "notes": "Primary calibration ranking signal."
    },
    {
        "component": "expanded_calibration_strengthening",
        "status": "defined",
        "protocol_value": "intercept_and_slope_by_horizon",
        "notes": "Strengthening diagnostics added after Group E."
    },
    {
        "component": "ici_style_metric",
        "status": "optional_not_required",
        "protocol_value": "not_canonical_for_core_benchmark",
        "notes": "Kept optional; not required for the benchmark core."
    },
    {
        "component": "sensitivity_window_length",
        "status": "defined",
        "protocol_value": "2_4_6",
        "notes": "Early-window length treated as explicit sensitivity dimension."
    },
    {
        "component": "sensitivity_horizon_choice",
        "status": "defined",
        "protocol_value": ",".join(map(str, BENCHMARK_HORIZONS)),
        "notes": "Horizon choice treated as explicit stress test dimension."
    },
]

evaluation_protocol_audit_df = pd.DataFrame(protocol_audit_rows)

# ------------------------------
# 6) Optional linkage to P26 outputs
# ------------------------------
p26_5_summary_path = TABLES_DIR / "table_p26_5_paper_summary_all_tuned_models.csv"
p26_6_rankings_path = TABLES_DIR / "table_p26_6_calibration_model_rankings.csv"

if p26_5_summary_path.exists():
    p26_5_summary_df = pd.read_csv(p26_5_summary_path)
    if not p26_5_summary_df.empty:
        best_calibration_model = p26_5_summary_df.sort_values(
            by=["mean_calib_gap", "mean_brier"],
            ascending=[True, True]
        ).iloc[0]
        BENCHMARK_CONFIG["current_calibration_leader_from_p26_5"] = {
            "model_id": str(best_calibration_model["model_id"]),
            "model": str(best_calibration_model["model"]),
            "mean_calib_gap": float(best_calibration_model["mean_calib_gap"]),
            "mean_brier": float(best_calibration_model["mean_brier"]),
        }

if p26_6_rankings_path.exists():
    p26_6_rankings_df = pd.read_csv(p26_6_rankings_path)
    if not p26_6_rankings_df.empty:
        top_ranked = p26_6_rankings_df.sort_values(
            by="rank_overall_primary",
            ascending=True
        ).iloc[0]
        BENCHMARK_CONFIG["current_primary_calibration_ranking_leader_from_p26_6"] = {
            "model_id": str(top_ranked["model_id"]),
            "model": str(top_ranked["model"]),
            "rank_overall_primary": int(top_ranked["rank_overall_primary"]),
            "overall_calibration_reading": str(top_ranked["overall_calibration_reading"]),
        }

# ------------------------------
# 7) Save outputs
# ------------------------------
benchmark_config_path = METADATA_DIR / "benchmark_config.json"
metrics_manifest_path = TABLES_DIR / "table_benchmark_metrics_manifest.csv"
protocol_audit_path = TABLES_DIR / "table_evaluation_protocol_audit.csv"

benchmark_metrics_manifest_df.to_csv(metrics_manifest_path, index=False)
evaluation_protocol_audit_df.to_csv(protocol_audit_path, index=False)
save_json(BENCHMARK_CONFIG, benchmark_config_path)

# ------------------------------
# 8) Display
# ------------------------------
print("\nBenchmark configuration (core fields):")
display(pd.DataFrame([
    {"key": "protocol_name", "value": BENCHMARK_CONFIG["protocol_name"]},
    {"key": "protocol_version", "value": BENCHMARK_CONFIG["protocol_version"]},
    {"key": "unit_of_analysis", "value": BENCHMARK_CONFIG["unit_of_analysis"]},
    {"key": "event_definition", "value": BENCHMARK_CONFIG["event_definition"]},
    {"key": "official_benchmark_horizons", "value": ",".join(map(str, BENCHMARK_CONFIG["official_benchmark_horizons"]))},
    {"key": "primary_calibration_metric", "value": BENCHMARK_CONFIG["calibration_protocol"]["primary_calibration_metric"]},
    {"key": "strengthening_metrics", "value": ", ".join(BENCHMARK_CONFIG["calibration_protocol"]["expanded_strengthening_metrics"])},
]))

print("\nBenchmark metrics manifest:")
display(benchmark_metrics_manifest_df)

print("\nEvaluation protocol audit:")
display(evaluation_protocol_audit_df)

print("\nSaved:")
print("-", benchmark_config_path.resolve())
print("-", metrics_manifest_path.resolve())
print("-", protocol_audit_path.resolve())

### D1.1 — Materialize Benchmark Runtime Aliases

In [ ]:
# ==============================================================
# D1.1 — Materialize Benchmark Runtime Aliases
# --------------------------------------------------------------
# Purpose:
#   Recreate the benchmark constants expected by downstream cells
#   under sequential execution after the protocol cell has run.
#
# Methodological note:
#   This cell does not recompute metrics.
#   It only materializes compatibility aliases and a guarded
#   calibration-bin constant from the canonical benchmark protocol.
# ==============================================================

print("\n" + "=" * 70)
print("D1.1 — Materialize Benchmark Runtime Aliases")
print("=" * 70)
print("Methodological note: this cell materializes benchmark constants only.")

required_names = ["BENCHMARK_CONFIG", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run D1 first."
    )

if "BENCHMARK_HORIZONS" in globals():
    detected_horizons = [int(h) for h in BENCHMARK_HORIZONS]
else:
    detected_horizons = [
        int(h)
        for h in BENCHMARK_CONFIG.get("official_benchmark_horizons", [10, 20, 30])
    ]

HORIZONS_WEEKS = sorted(list(dict.fromkeys(detected_horizons)))
HORIZON_WEEKS = list(HORIZONS_WEEKS)
BENCHMARK_HORIZONS = list(HORIZONS_WEEKS)

calibration_protocol = BENCHMARK_CONFIG.setdefault("calibration_protocol", {})
calibration_bin_candidates = [
    calibration_protocol.get("n_bins"),
    calibration_protocol.get("target_n_bins"),
    calibration_protocol.get("calibration_bins"),
    BENCHMARK_CONFIG.get("calibration_bins"),
]

CALIBRATION_BINS = next(
    (
        int(value)
        for value in calibration_bin_candidates
        if value is not None and str(value).strip() != ""
    ),
    10,
)

BENCHMARK_CONFIG["official_benchmark_horizons"] = list(HORIZONS_WEEKS)
BENCHMARK_CONFIG["comparable_reporting_horizons"] = list(HORIZONS_WEEKS)
calibration_protocol["official_horizons"] = list(HORIZONS_WEEKS)
calibration_protocol["target_n_bins"] = int(CALIBRATION_BINS)
BENCHMARK_CONFIG["calibration_bins"] = int(CALIBRATION_BINS)

benchmark_config_path = METADATA_DIR / "benchmark_config.json"
save_json(BENCHMARK_CONFIG, benchmark_config_path)

print("\nMaterialized runtime aliases:")
print("- HORIZONS_WEEKS:", HORIZONS_WEEKS)
print("- HORIZON_WEEKS:", HORIZON_WEEKS)
print("- CALIBRATION_BINS:", CALIBRATION_BINS)
print("\nSaved:")
print("-", benchmark_config_path.resolve())

## D2 — Treatment for the Linear Discrete-Time Hazard Model


In [ ]:
# ==============================================================
# D2 — Treatment for the Linear Discrete-Time Hazard Model
# CORRECTED VERSION (robust to canonical-vs-materialized feature names)
# --------------------------------------------------------------
# Purpose:
#   Prepare the train/test preprocessing pipeline for the linear
#   discrete-time hazard baseline, with all learned transformations
#   fitted on training data only.
#
# Methodological note:
#   This step performs model-specific preprocessing for the linear
#   discrete-time hazard baseline. The following learned operations
#   are fit on training data only:
#     - numeric imputation
#     - categorical imputation
#     - one-hot encoding
#     - numeric standardization
#
#   The same fitted preprocessing object is then applied to the
#   test data without re-estimating any parameter, preventing leakage.
#
# Design note:
#   After the benchmark-architecture refactor, P10 started exposing
#   a canonical minimal dynamic-arm config (e.g., "total_clicks"),
#   while the materialized person-period tables still use the richer
#   notebook-native names (e.g., "total_clicks_week").
#   This cell therefore resolves canonical aliases to the actual
#   materialized columns instead of requiring exact name identity.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - STATIC_FEATURES
#   - TEMPORAL_FEATURES_DISCRETE
#
# Outputs:
#   - X_train_linear, X_test_linear
#   - y_train_linear, y_test_linear
#   - linear_preprocessor
#   - linear_feature_names_out
#   - audit tables / metadata
# ==============================================================

print("\n" + "=" * 70)
print("D2 — Treatment for the Linear Discrete-Time Hazard Model")
print("=" * 70)
print("Methodological note: all learned preprocessing transformations")
print("are fit on training data only and then applied to test data.")

# ------------------------------
# 1) Dependency checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "METADATA_DIR", "save_json",
    "STATIC_FEATURES", "TEMPORAL_FEATURES_DISCRETE"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------
# 2) Load train/test tables
# ------------------------------
train_df = con.execute("SELECT * FROM pp_linear_hazard_ready_train").fetchdf()
test_df = con.execute("SELECT * FROM pp_linear_hazard_ready_test").fetchdf()

if train_df.empty:
    raise ValueError("pp_linear_hazard_ready_train is empty.")
if test_df.empty:
    raise ValueError("pp_linear_hazard_ready_test is empty.")

# ------------------------------
# 3) Define target and operational feature columns
# ------------------------------
target_col = "event_t"

# These are the notebook-native operational columns currently
# materialized in pp_linear_hazard_input / ready tables.
categorical_features = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
    "week",
    "total_clicks_week",
    "active_this_week",
    "n_vle_rows_week",
    "n_distinct_sites_week",
    "cum_clicks_until_t",
    "recency",
    "streak",
]

linear_feature_columns = categorical_features + numeric_features

# ------------------------------
# 4) Canonical-to-materialized alias resolution
# ------------------------------
# P10 may expose a canonical minimal config such as:
#   TEMPORAL_FEATURES_DISCRETE = ["week", "total_clicks"]
# while the actual discrete tables use "total_clicks_week".
feature_alias_map = {
    "total_clicks": "total_clicks_week",
}

expected_features_raw = list(STATIC_FEATURES) + list(TEMPORAL_FEATURES_DISCRETE)
expected_features_resolved = [
    feature_alias_map.get(col, col) for col in expected_features_raw
]

# Require only that the canonical configured features are covered
# by the operational treatment spec after alias resolution.
missing_from_defined = [
    c for c in expected_features_resolved
    if c not in linear_feature_columns
]

# Also verify that the operational columns truly exist in both splits.
missing_in_train = [c for c in linear_feature_columns if c not in train_df.columns]
missing_in_test = [c for c in linear_feature_columns if c not in test_df.columns]

if missing_from_defined:
    raise ValueError(
        "Configured canonical features are not covered by the operational "
        f"linear treatment spec after alias resolution: {missing_from_defined}. "
        f"Raw expected features were: {expected_features_raw}"
    )

if missing_in_train or missing_in_test:
    raise ValueError(
        "Operational linear feature columns are missing from the materialized "
        "train/test tables.\n"
        f"Missing in train: {missing_in_train}\n"
        f"Missing in test : {missing_in_test}"
    )

if target_col not in train_df.columns or target_col not in test_df.columns:
    raise ValueError(
        f"Target column '{target_col}' is missing from train/test tables."
    )

# ------------------------------
# 5) Split X / y
# ------------------------------
X_train_raw = train_df[linear_feature_columns].copy()
X_test_raw = test_df[linear_feature_columns].copy()

y_train_linear = train_df[target_col].astype(int).copy()
y_test_linear = test_df[target_col].astype(int).copy()

# ------------------------------
# 6) Build preprocessing pipelines
# ------------------------------
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# ------------------------------
# 7) Fit on train only, transform train/test
# ------------------------------
X_train_linear = linear_preprocessor.fit_transform(X_train_raw)
X_test_linear = linear_preprocessor.transform(X_test_raw)

linear_feature_names_out = linear_preprocessor.get_feature_names_out().tolist()

# ------------------------------
# 8) Audit tables
# ------------------------------
preproc_audit_df = pd.DataFrame({
    "component": [
        "target_column",
        "n_train_rows",
        "n_test_rows",
        "n_operational_features",
        "n_numeric_features",
        "n_categorical_features",
        "n_output_features_after_preprocessing",
        "train_event_rate",
        "test_event_rate",
    ],
    "value": [
        target_col,
        int(len(train_df)),
        int(len(test_df)),
        int(len(linear_feature_columns)),
        int(len(numeric_features)),
        int(len(categorical_features)),
        int(len(linear_feature_names_out)),
        float(y_train_linear.mean()),
        float(y_test_linear.mean()),
    ],
})

feature_manifest_df = pd.DataFrame({
    "feature_name_operational": linear_feature_columns,
    "feature_role": [
        "categorical" if c in categorical_features else "numeric"
        for c in linear_feature_columns
    ],
    "present_in_train": [c in train_df.columns for c in linear_feature_columns],
    "present_in_test": [c in test_df.columns for c in linear_feature_columns],
    "covered_by_canonical_config_after_alias": [
        c in expected_features_resolved for c in linear_feature_columns
    ],
    "canonical_source_name": [
        next(
            (raw for raw, resolved in zip(expected_features_raw, expected_features_resolved) if resolved == c),
            ""
        )
        for c in linear_feature_columns
    ],
})

canonical_alignment_df = pd.DataFrame({
    "canonical_feature_raw": expected_features_raw,
    "canonical_feature_resolved": expected_features_resolved,
    "covered_by_operational_treatment_spec": [
        c in linear_feature_columns for c in expected_features_resolved
    ],
})

output_feature_manifest_df = pd.DataFrame({
    "preprocessed_feature_name": linear_feature_names_out
})

# ------------------------------
# 9) Save outputs
# ------------------------------
preproc_audit_path = TABLES_DIR / "table_p16_linear_treatment_audit.csv"
feature_manifest_path = TABLES_DIR / "table_p16_linear_feature_manifest.csv"
canonical_alignment_path = TABLES_DIR / "table_p16_linear_canonical_alignment.csv"
output_manifest_path = TABLES_DIR / "table_p16_linear_output_feature_manifest.csv"
metadata_path = METADATA_DIR / "metadata_p16_linear_treatment.json"

preproc_audit_df.to_csv(preproc_audit_path, index=False)
feature_manifest_df.to_csv(feature_manifest_path, index=False)
canonical_alignment_df.to_csv(canonical_alignment_path, index=False)
output_feature_manifest_df.to_csv(output_manifest_path, index=False)

save_json({
    "step": "P16",
    "model_family": "linear_discrete_time_hazard",
    "train_table": "pp_linear_hazard_ready_train",
    "test_table": "pp_linear_hazard_ready_test",
    "target_column": target_col,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "operational_feature_columns": linear_feature_columns,
    "canonical_expected_features_raw": expected_features_raw,
    "canonical_expected_features_resolved": expected_features_resolved,
    "feature_alias_map": feature_alias_map,
    "n_train_rows": int(len(train_df)),
    "n_test_rows": int(len(test_df)),
    "n_output_features_after_preprocessing": int(len(linear_feature_names_out)),
    "train_event_rate": float(y_train_linear.mean()),
    "test_event_rate": float(y_test_linear.mean()),
    "methodological_note": (
        "All learned preprocessing operations were fit on training data only "
        "and then applied unchanged to test data."
    ),
    "design_note": (
        "Canonical config features were resolved to notebook-native materialized "
        "column names before validation."
    ),
}, metadata_path)

# ------------------------------
# 10) Console summary
# ------------------------------
print("\nOperational feature columns used by the linear model:")
for col in linear_feature_columns:
    print(" -", col)

print("\nCanonical config alignment:")
display(canonical_alignment_df)

print("\nTreatment audit:")
display(preproc_audit_df)

print("\nSaved:")
print("-", preproc_audit_path.resolve())
print("-", feature_manifest_path.resolve())
print("-", canonical_alignment_path.resolve())
print("-", output_manifest_path.resolve())
print("-", metadata_path.resolve())

### D2.1 — Performance-Oriented Tuning for the Linear Discrete-Time Hazard Model (Revised)


### Funcao: get_pred_at_horizon

Definicao isolada de get_pred_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D2.1 — Performance-Oriented Tuning for the Linear Discrete-Time Hazard Model (Revised)
# --------------------------------------------------------------
# Purpose:
#   Perform a controlled hyperparameter search for the linear
#   discrete-time hazard model, select the best candidate, refit
#   it on the full training set, and evaluate it under the same
#   survival-oriented benchmark protocol defined in D1.
#
# Methodological note:
#   This revised version preserves the tuning step but restores
#   full benchmark comparability by computing:
#     - IBS
#     - C-index
#     - horizon-specific Brier score
#     - calibration at horizons
#     - risk AUC at horizons
#     - predicted vs observed survival
#
#   The model remains a discrete-time hazard model:
#   weekly probabilities are interpreted as hazards and enrollment-
#   level survival is reconstructed from hazard trajectories.
# ==============================================================

print("\n" + "=" * 70)
print("D2.1 — Performance-Oriented Tuning for the Linear Discrete-Time Hazard Model (Revised)")
print("=" * 70)
print("Methodological note: this step performs controlled tuning and")
print("evaluates the best candidate under the same survival-oriented")
print("benchmark protocol defined in D1.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json",
    "X_train_linear", "X_test_linear",
    "y_train_linear", "y_test_linear",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun the required prior steps first."
    )

import itertools
import joblib
import numpy as np
import pandas as pd
import scipy
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
from sklearn.model_selection import GroupShuffleSplit

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.util import Surv
    SKSURV_AVAILABLE = True
except Exception:
    SKSURV_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for the revised P17.2 evaluation.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"

try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Resolve source dataframes
# ------------------------------
if "train_df_linear" in globals():
    train_df_linear_local = train_df_linear.copy()
elif "pp_linear_hazard_ready_train" in globals():
    train_df_linear_local = pp_linear_hazard_ready_train.copy()
elif "con" in globals():
    train_df_linear_local = con.execute("SELECT * FROM pp_linear_hazard_ready_train").fetchdf()
else:
    raise NameError("Could not resolve train_df_linear / pp_linear_hazard_ready_train.")

if "test_df_linear" in globals():
    test_df_linear_local = test_df_linear.copy()
elif "pp_linear_hazard_ready_test" in globals():
    test_df_linear_local = pp_linear_hazard_ready_test.copy()
elif "con" in globals():
    test_df_linear_local = con.execute("SELECT * FROM pp_linear_hazard_ready_test").fetchdf()
else:
    raise NameError("Could not resolve test_df_linear / pp_linear_hazard_ready_test.")

if "linear_preprocessor" in globals():
    linear_preprocessor_obj = linear_preprocessor
elif "preprocessor_linear" in globals():
    linear_preprocessor_obj = preprocessor_linear
else:
    linear_preprocessor_obj = None

# ------------------------------
# 4) Train/validation split at enrollment level
# ------------------------------
groups = train_df_linear_local["enrollment_id"].astype(str).to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
subtrain_idx, val_idx = next(gss.split(X_train_linear, y_train_linear, groups=groups))

X_subtrain = X_train_linear[subtrain_idx]
X_val = X_train_linear[val_idx]
y_subtrain = y_train_linear[subtrain_idx]
y_val = y_train_linear[val_idx]

# ------------------------------
# 5) Tuning config
# ------------------------------
LINEAR_TUNING_CONFIG = {
    "model_name": "linear_discrete_time_hazard_tuned",
    "search_space": {
        "penalty": ["l1", "l2"],
        "C": [0.01, 0.1, 1.0, 10.0],
    },
    "solver": "liblinear",
    "class_weight": None,
    "max_iter": 2000,
    "selection_metric": "val_log_loss",
    "benchmark_positioning_note": (
        "Performance-oriented tuned linear discrete-time hazard benchmark."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
}

# ------------------------------
# 6) Run tuning grid
# ------------------------------
candidate_rows = []
candidate_models = []

candidate_id = 0
for penalty, C in itertools.product(
    LINEAR_TUNING_CONFIG["search_space"]["penalty"],
    LINEAR_TUNING_CONFIG["search_space"]["C"]
):
    candidate_id += 1

    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver=LINEAR_TUNING_CONFIG["solver"],
        class_weight=LINEAR_TUNING_CONFIG["class_weight"],
        max_iter=LINEAR_TUNING_CONFIG["max_iter"],
        random_state=RANDOM_SEED,
    )

    model.fit(X_subtrain, y_subtrain)
    val_pred = np.clip(model.predict_proba(X_val)[:, 1], 1e-8, 1 - 1e-8)

    val_logloss = log_loss(y_val, val_pred, labels=[0, 1])
    val_brier = brier_score_loss(y_val, val_pred)

    if len(np.unique(y_val)) >= 2:
        val_roc_auc = roc_auc_score(y_val, val_pred)
        val_pr_auc = average_precision_score(y_val, val_pred)
    else:
        val_roc_auc = np.nan
        val_pr_auc = np.nan

    candidate_rows.append({
        "candidate_id": candidate_id,
        "penalty": penalty,
        "C": C,
        "val_log_loss": float(val_logloss),
        "val_brier": float(val_brier),
        "val_roc_auc": float(val_roc_auc) if pd.notna(val_roc_auc) else np.nan,
        "val_pr_auc": float(val_pr_auc) if pd.notna(val_pr_auc) else np.nan,
    })

tuning_results_df = pd.DataFrame(candidate_rows).sort_values(
    by=["val_log_loss", "val_brier", "candidate_id"],
    ascending=[True, True, True]
).reset_index(drop=True)

best_row = tuning_results_df.iloc[0]
best_candidate_df = pd.DataFrame([best_row.to_dict()])

best_penalty = best_row["penalty"]
best_C = float(best_row["C"])

# ------------------------------
# 7) Refit best model on full train
# ------------------------------
best_model = LogisticRegression(
    penalty=best_penalty,
    C=best_C,
    solver=LINEAR_TUNING_CONFIG["solver"],
    class_weight=LINEAR_TUNING_CONFIG["class_weight"],
    max_iter=LINEAR_TUNING_CONFIG["max_iter"],
    random_state=RANDOM_SEED,
)
best_model.fit(X_train_linear, y_train_linear)

# ------------------------------
# 8) Predict row-level hazards on test
# ------------------------------
test_pred_row = np.clip(best_model.predict_proba(X_test_linear)[:, 1], 1e-8, 1 - 1e-8)

test_pred_df = test_df_linear_local.copy()
test_pred_df["pred_hazard"] = test_pred_row

required_cols = ["enrollment_id", "week", "event_t"]
missing_eval_cols = [c for c in required_cols if c not in test_pred_df.columns]
if missing_eval_cols:
    raise ValueError(f"Missing required columns in test dataframe: {missing_eval_cols}")

test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
    lambda s: (1.0 - s).cumprod()
)
test_pred_df["pred_risk"] = 1.0 - test_pred_df["pred_survival"]

# enrollment-level truth
truth_test = (
    test_pred_df.groupby("enrollment_id", as_index=False)
    .agg(
        event=("event_observed", "max") if "event_observed" in test_pred_df.columns else ("event_t", "max"),
        duration=("time_for_split", "max") if "time_for_split" in test_pred_df.columns else ("week", "max"),
    )
)

truth_train = (
    train_df_linear_local.groupby("enrollment_id", as_index=False)
    .agg(
        event=("event_observed", "max") if "event_observed" in train_df_linear_local.columns else ("event_t", "max"),
        duration=("time_for_split", "max") if "time_for_split" in train_df_linear_local.columns else ("week", "max"),
    )
)

durations_test = truth_test["duration"].astype(int).to_numpy()
events_test = truth_test["event"].astype(int).to_numpy()

# ------------------------------
# 9) Build survival matrix for pycox evaluation
# ------------------------------
surv_wide = (
    test_pred_df[["enrollment_id", "week", "pred_survival"]]
    .drop_duplicates(subset=["enrollment_id", "week"])
    .pivot(index="week", columns="enrollment_id", values="pred_survival")
    .sort_index()
)

max_week_test = int(test_pred_df["week"].max())
full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
surv_wide = surv_wide.reindex(full_week_index)
surv_wide = surv_wide.ffill()
surv_wide = surv_wide.fillna(1.0)
surv_df = surv_wide.copy()
surv_df.columns.name = "enrollment_id"

# ------------------------------
# 10) Primary survival metrics
# ------------------------------
eval_surv = EvalSurv(
    surv=surv_df,
    durations=durations_test,
    events=events_test,
    censor_surv="km",
)

primary_metrics_rows = []

try:
    c_index = float(eval_surv.concordance_td())
    c_index_note = "pycox_concordance_td"
except Exception as e:
    c_index = np.nan
    c_index_note = f"failed: {str(e)}"

primary_metrics_rows.append({
    "metric_name": "c_index",
    "metric_category": "co_primary",
    "metric_value": c_index,
    "notes": c_index_note,
})

try:
    max_requested_horizon = int(max(HORIZONS_WEEKS))
    ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
    ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    ibs_note = "pycox_integrated_brier_score"
except Exception as e:
    try:
        max_requested_horizon = int(max(HORIZONS_WEEKS))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        dense_brier = eval_surv.brier_score(ibs_grid)
        ibs_value = float(np.trapz(dense_brier.values.astype(float), ibs_grid) / (ibs_grid[-1] - ibs_grid[0]))
        ibs_note = f"fallback_np_trapz_after_pycox_failure: {str(e)}"
    except Exception as e2:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)} | fallback_failed: {str(e2)}"

primary_metrics_rows.insert(0, {
    "metric_name": "ibs",
    "metric_category": "primary",
    "metric_value": ibs_value,
    "notes": ibs_note,
})

primary_metrics_summary = pd.DataFrame(primary_metrics_rows)

# ------------------------------
# 11) Brier by horizon from EvalSurv
# ------------------------------
try:
    brier_h = eval_surv.brier_score(np.array(HORIZONS_WEEKS, dtype=int))
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": list(brier_h.index.astype(int)),
        "metric_name": ["brier_at_horizon"] * len(brier_h.index),
        "metric_category": ["primary"] * len(brier_h.index),
        "metric_value": list(brier_h.values.astype(float)),
        "notes": ["pycox_brier_score"] * len(brier_h.index),
    })
except Exception as e:
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": HORIZONS_WEEKS,
        "metric_name": ["brier_at_horizon"] * len(HORIZONS_WEEKS),
        "metric_category": ["primary"] * len(HORIZONS_WEEKS),
        "metric_value": [np.nan] * len(HORIZONS_WEEKS),
        "notes": [f"failed: {str(e)}"] * len(HORIZONS_WEEKS),
    })

# ------------------------------
# 12) Helpers for horizon extraction
# ------------------------------
def get_pred_at_horizon(df, h, value_col):
    eligible = df[df["week"] <= h].copy()
    if eligible.empty:
        return pd.DataFrame(columns=["enrollment_id", value_col])
    last_rows = (
        eligible.sort_values(["enrollment_id", "week"])
        .groupby("enrollment_id", as_index=False)
        .tail(1)
    )
    return last_rows[["enrollment_id", value_col]]

# ------------------------------
# 13) Support, calibration, risk AUC, predicted vs observed
# ------------------------------
support_rows = []
risk_auc_rows = []
calibration_rows = []
pred_vs_obs_rows = []

for h in HORIZONS_WEEKS:
    pred_surv_h = get_pred_at_horizon(test_pred_df, h, "pred_survival").rename(columns={"pred_survival": "pred_survival_h"})
    pred_risk_h = get_pred_at_horizon(test_pred_df, h, "pred_risk").rename(columns={"pred_risk": "pred_risk_h"})

    eval_df = truth_test.merge(pred_surv_h, on="enrollment_id", how="left").merge(pred_risk_h, on="enrollment_id", how="left")

    eval_df["is_evaluable_at_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
        (eval_df["duration"] >= h)
    ).astype(int)

    eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
    eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
    eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

    support_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
        "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df["observed_event_by_h"].nunique() >= 2:
        risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
    else:
        risk_auc = np.nan

    risk_auc_rows.append({
        "horizon_week": h,
        "metric_name": "risk_auc_at_horizon",
        "metric_category": "secondary",
        "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        "notes": "roc_auc_on_evaluable_subset",
    })

    pred_vs_obs_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df.shape[0] > 0:
        ranked = eval_df["pred_risk_h"].rank(method="first")
        n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))
        eval_df["calibration_bin"] = pd.qcut(
            ranked,
            q=n_bins_eff,
            labels=False,
            duplicates="drop"
        )

        calib_tab = (
            eval_df.groupby("calibration_bin")
            .agg(
                n=("enrollment_id", "count"),
                mean_predicted_risk=("pred_risk_h", "mean"),
                observed_event_rate=("observed_event_by_h", "mean"),
            )
            .reset_index()
        )
        calib_tab["horizon_week"] = h
        calib_tab["abs_calibration_gap"] = (
            calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
        ).abs()

        for _, row in calib_tab.iterrows():
            calibration_rows.append({
                "horizon_week": int(row["horizon_week"]),
                "calibration_bin": int(row["calibration_bin"]),
                "n": int(row["n"]),
                "mean_predicted_risk": float(row["mean_predicted_risk"]),
                "observed_event_rate": float(row["observed_event_rate"]),
                "abs_calibration_gap": float(row["abs_calibration_gap"]),
            })

support_by_horizon_df = pd.DataFrame(support_rows)
risk_auc_by_horizon_df = pd.DataFrame(risk_auc_rows)
predicted_vs_observed_survival_df = pd.DataFrame(pred_vs_obs_rows)
calibration_bins_df = pd.DataFrame(calibration_rows)

calibration_summary_rows = []
for h in HORIZONS_WEEKS:
    h_df = calibration_bins_df[calibration_bins_df["horizon_week"] == h].copy()
    if h_df.shape[0] > 0:
        ece_like = np.average(h_df["abs_calibration_gap"], weights=h_df["n"])
    else:
        ece_like = np.nan

    calibration_summary_rows.append({
        "horizon_week": h,
        "metric_name": "calibration_at_horizon",
        "metric_category": "primary",
        "metric_value": float(ece_like) if pd.notna(ece_like) else np.nan,
        "notes": "Weighted absolute calibration gap across bins",
    })

calibration_summary_df = pd.DataFrame(calibration_summary_rows)

# ------------------------------
# 14) Optional time-dependent AUC
# ------------------------------
td_auc_rows = []
if SKSURV_AVAILABLE:
    try:
        y_train_surv = Surv.from_arrays(
            event=truth_train["event"].astype(bool).to_numpy(),
            time=truth_train["duration"].astype(float).to_numpy(),
        )
        y_test_surv = Surv.from_arrays(
            event=truth_test["event"].astype(bool).to_numpy(),
            time=truth_test["duration"].astype(float).to_numpy(),
        )

        for h in HORIZONS_WEEKS:
            pred_risk_h = get_pred_at_horizon(test_pred_df, h, "pred_risk")
            merged = truth_test[["enrollment_id"]].merge(pred_risk_h, on="enrollment_id", how="left")
            auc_vals, mean_auc = cumulative_dynamic_auc(
                y_train_surv,
                y_test_surv,
                merged["pred_risk"].to_numpy(),
                np.array([h], dtype=float)
            )
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": float(auc_vals[0]),
                "notes": "sksurv cumulative_dynamic_auc",
            })
    except Exception as e:
        for h in HORIZONS_WEEKS:
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": np.nan,
                "notes": f"sksurv_available_but_failed: {str(e)}",
            })
else:
    for h in HORIZONS_WEEKS:
        td_auc_rows.append({
            "horizon_week": h,
            "metric_name": "time_dependent_auc",
            "metric_category": "secondary",
            "metric_value": np.nan,
            "notes": "sksurv_unavailable",
        })

time_dependent_auc_df = pd.DataFrame(td_auc_rows)
secondary_metrics_df = pd.concat([risk_auc_by_horizon_df, time_dependent_auc_df], ignore_index=True)

# ------------------------------
# 15) Row-level diagnostics
# ------------------------------
if len(np.unique(y_test_linear)) >= 2:
    row_roc_auc = roc_auc_score(y_test_linear, test_pred_row)
    row_pr_auc = average_precision_score(y_test_linear, test_pred_row)
else:
    row_roc_auc = np.nan
    row_pr_auc = np.nan

row_log_loss = log_loss(y_test_linear, test_pred_row, labels=[0, 1])
row_brier = brier_score_loss(y_test_linear, test_pred_row)

row_diagnostics_df = pd.DataFrame([{
    "model_name": "linear_discrete_time_hazard_tuned",
    "row_level_roc_auc": float(row_roc_auc) if pd.notna(row_roc_auc) else np.nan,
    "row_level_pr_auc": float(row_pr_auc) if pd.notna(row_pr_auc) else np.nan,
    "row_level_log_loss": float(row_log_loss),
    "row_level_brier": float(row_brier),
}])

# ------------------------------
# 16) Save artifacts
# ------------------------------
model_dir = OUTPUT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "linear_discrete_time_hazard_tuned.joblib"
preprocessor_path = model_dir / "linear_discrete_time_preprocessor.joblib"
tuning_results_path = TABLES_DIR / "table_linear_tuning_results.csv"
predictions_path = TABLES_DIR / "table_linear_tuned_test_predictions.csv"
primary_metrics_path = TABLES_DIR / "table_linear_tuned_primary_metrics.csv"
brier_path = TABLES_DIR / "table_linear_tuned_brier_by_horizon.csv"
secondary_metrics_path = TABLES_DIR / "table_linear_tuned_secondary_metrics.csv"
row_diag_path = TABLES_DIR / "table_linear_tuned_row_diagnostics.csv"
support_path = TABLES_DIR / "table_linear_tuned_support_by_horizon.csv"
cal_summary_path = TABLES_DIR / "table_linear_tuned_calibration_summary.csv"
cal_bins_path = TABLES_DIR / "table_linear_tuned_calibration_bins_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_linear_tuned_predicted_vs_observed_survival.csv"
config_path = METADATA_DIR / "linear_tuned_model_config.json"

joblib.dump(best_model, model_path)
if linear_preprocessor_obj is not None:
    joblib.dump(linear_preprocessor_obj, preprocessor_path)

tuning_results_df.to_csv(tuning_results_path, index=False)
test_pred_df.to_csv(predictions_path, index=False)
primary_metrics_summary.to_csv(primary_metrics_path, index=False)
brier_by_horizon_df.to_csv(brier_path, index=False)
secondary_metrics_df.to_csv(secondary_metrics_path, index=False)
row_diagnostics_df.to_csv(row_diag_path, index=False)
support_by_horizon_df.to_csv(support_path, index=False)
calibration_summary_df.to_csv(cal_summary_path, index=False)
calibration_bins_df.to_csv(cal_bins_path, index=False)
predicted_vs_observed_survival_df.to_csv(pred_vs_obs_path, index=False)

config_to_save = dict(LINEAR_TUNING_CONFIG)
config_to_save["best_candidate"] = {
    "penalty": best_penalty,
    "C": best_C,
}
save_json(config_to_save, config_path)

# ------------------------------
# 17) Output for feedback
# ------------------------------
print("\nTuning results:")
display(tuning_results_df)

print("\nBest candidate:")
display(best_candidate_df)

print("\nPrimary metrics summary:")
display(primary_metrics_summary)

print("\nBrier by horizon:")
display(brier_by_horizon_df)

print("\nCalibration summary by horizon:")
display(calibration_summary_df)

print("\nSecondary metrics:")
display(secondary_metrics_df)

print("\nRow-level diagnostics:")
display(row_diagnostics_df)

print("\nSupport by horizon:")
display(support_by_horizon_df)

print("\nPredicted vs observed survival:")
display(predicted_vs_observed_survival_df)

print("\nSaved artifacts:")
print("-", model_path.resolve())
if linear_preprocessor_obj is not None:
    print("-", preprocessor_path.resolve())
print("-", tuning_results_path.resolve())
print("-", predictions_path.resolve())
print("-", primary_metrics_path.resolve())
print("-", brier_path.resolve())
print("-", secondary_metrics_path.resolve())
print("-", row_diag_path.resolve())
print("-", support_path.resolve())
print("-", cal_summary_path.resolve())
print("-", cal_bins_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", config_path.resolve())

## D3 — Treatment for the Neural Discrete-Time Survival Model


In [ ]:
# ==============================================================
# D3 — Treatment for the Neural Discrete-Time Survival Model
# CORRECTED VERSION (robust to canonical-vs-materialized feature names)
# --------------------------------------------------------------
# Purpose:
#   Prepare the train/test preprocessing pipeline for the neural
#   discrete-time survival model, with all learned transformations
#   fitted on training data only.
#
# Methodological note:
#   This step intentionally uses the same conceptual feature set
#   adopted for the linear discrete-time hazard baseline, so that
#   the comparison between the linear and neural formulations is
#   driven primarily by model class rather than by a different
#   feature engineering design.
#
#   The following learned operations are fit on training data only:
#     - numeric imputation
#     - categorical imputation
#     - one-hot encoding
#     - numeric standardization
#
#   The same fitted preprocessing object is then applied to the
#   test data without re-estimating any parameter, preventing leakage.
#
# Design note:
#   After the benchmark-architecture refactor, P10 started exposing
#   a canonical minimal dynamic-arm config (e.g., "total_clicks"),
#   while the materialized person-period tables still use the richer
#   notebook-native names (e.g., "total_clicks_week").
#   This cell therefore resolves canonical aliases to the actual
#   materialized columns instead of requiring exact name identity.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - STATIC_FEATURES
#   - TEMPORAL_FEATURES_DISCRETE
#
# Outputs:
#   - X_train_neural, X_test_neural
#   - y_train_neural, y_test_neural
#   - neural_preprocessor
#   - neural_feature_names_out
#   - audit tables / metadata
# ==============================================================

print("\n" + "=" * 70)
print("D3 — Treatment for the Neural Discrete-Time Survival Model")
print("=" * 70)
print("Methodological note: the neural model uses the same conceptual")
print("feature set as the linear baseline for comparability.")
print("All learned preprocessing transformations are fit on training data only.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "METADATA_DIR", "save_json",
    "STATIC_FEATURES", "TEMPORAL_FEATURES_DISCRETE"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------
# 2) Load train/test tables
# ------------------------------
train_df_neural = con.execute("SELECT * FROM pp_neural_hazard_ready_train").fetchdf()
test_df_neural = con.execute("SELECT * FROM pp_neural_hazard_ready_test").fetchdf()

if train_df_neural.empty:
    raise ValueError("pp_neural_hazard_ready_train is empty.")
if test_df_neural.empty:
    raise ValueError("pp_neural_hazard_ready_test is empty.")

# ------------------------------
# 3) Define target and operational feature columns
# ------------------------------
target_col = "event_t"

# Notebook-native materialized columns currently used in the
# discrete-time neural person-period tables.
categorical_features = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
    "week",
    "total_clicks_week",
    "active_this_week",
    "n_vle_rows_week",
    "n_distinct_sites_week",
    "cum_clicks_until_t",
    "recency",
    "streak",
]

neural_feature_columns = categorical_features + numeric_features

# ------------------------------
# 4) Canonical-to-materialized alias resolution
# ------------------------------
feature_alias_map = {
    "total_clicks": "total_clicks_week",
}

expected_features_raw = list(STATIC_FEATURES) + list(TEMPORAL_FEATURES_DISCRETE)
expected_features_resolved = [
    feature_alias_map.get(col, col) for col in expected_features_raw
]

missing_from_defined = [
    c for c in expected_features_resolved
    if c not in neural_feature_columns
]

missing_in_train = [c for c in neural_feature_columns if c not in train_df_neural.columns]
missing_in_test = [c for c in neural_feature_columns if c not in test_df_neural.columns]

if missing_from_defined:
    raise ValueError(
        "Configured canonical features are not covered by the operational "
        f"neural treatment spec after alias resolution: {missing_from_defined}. "
        f"Raw expected features were: {expected_features_raw}"
    )

if missing_in_train or missing_in_test:
    raise ValueError(
        "Operational neural feature columns are missing from the materialized "
        "train/test tables.\n"
        f"Missing in train: {missing_in_train}\n"
        f"Missing in test : {missing_in_test}"
    )

if target_col not in train_df_neural.columns or target_col not in test_df_neural.columns:
    raise ValueError(
        f"Target column '{target_col}' is missing from train/test tables."
    )

# ------------------------------
# 5) Split X / y
# ------------------------------
X_train_raw_neural = train_df_neural[neural_feature_columns].copy()
X_test_raw_neural = test_df_neural[neural_feature_columns].copy()

y_train_neural = train_df_neural[target_col].astype(np.float32).to_numpy()
y_test_neural = test_df_neural[target_col].astype(np.float32).to_numpy()

# ------------------------------
# 6) Build preprocessing pipelines
# ------------------------------
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

neural_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# ------------------------------
# 7) Fit on train only, transform train/test
# ------------------------------
X_train_neural = neural_preprocessor.fit_transform(X_train_raw_neural).astype(np.float32)
X_test_neural = neural_preprocessor.transform(X_test_raw_neural).astype(np.float32)

neural_feature_names_out = neural_preprocessor.get_feature_names_out().tolist()

# ------------------------------
# 8) Build preprocessing summaries
# ------------------------------
summary_rows = [{
    "model_family": "neural_discrete_time_survival",
    "train_rows": int(X_train_neural.shape[0]),
    "test_rows": int(X_test_neural.shape[0]),
    "n_input_features_raw": int(len(neural_feature_columns)),
    "n_numeric_features": int(len(numeric_features)),
    "n_categorical_features": int(len(categorical_features)),
    "n_output_features_after_preprocessing": int(len(neural_feature_names_out)),
    "train_event_rate": float(y_train_neural.mean()),
    "test_event_rate": float(y_test_neural.mean()),
}]

neural_preprocessing_summary_df = pd.DataFrame(summary_rows)

feature_manifest_df = pd.DataFrame({
    "feature_name_operational": neural_feature_columns,
    "feature_role": [
        "categorical" if c in categorical_features else "numeric"
        for c in neural_feature_columns
    ],
    "present_in_train": [c in train_df_neural.columns for c in neural_feature_columns],
    "present_in_test": [c in test_df_neural.columns for c in neural_feature_columns],
    "covered_by_canonical_config_after_alias": [
        c in expected_features_resolved for c in neural_feature_columns
    ],
    "canonical_source_name": [
        next(
            (raw for raw, resolved in zip(expected_features_raw, expected_features_resolved) if resolved == c),
            ""
        )
        for c in neural_feature_columns
    ],
})

canonical_alignment_df = pd.DataFrame({
    "canonical_feature_raw": expected_features_raw,
    "canonical_feature_resolved": expected_features_resolved,
    "covered_by_operational_treatment_spec": [
        c in neural_feature_columns for c in expected_features_resolved
    ],
})

output_feature_manifest_df = pd.DataFrame({
    "preprocessed_feature_name": neural_feature_names_out
})

# ------------------------------
# 9) Save outputs
# ------------------------------
summary_path = TABLES_DIR / "table_p18_neural_treatment_summary.csv"
feature_manifest_path = TABLES_DIR / "table_p18_neural_feature_manifest.csv"
canonical_alignment_path = TABLES_DIR / "table_p18_neural_canonical_alignment.csv"
output_manifest_path = TABLES_DIR / "table_p18_neural_output_feature_manifest.csv"
config_path = METADATA_DIR / "metadata_p18_neural_treatment.json"

neural_preprocessing_summary_df.to_csv(summary_path, index=False)
feature_manifest_df.to_csv(feature_manifest_path, index=False)
canonical_alignment_df.to_csv(canonical_alignment_path, index=False)
output_feature_manifest_df.to_csv(output_manifest_path, index=False)

save_json(
    {
        "step": "P18",
        "model_family": "neural_discrete_time_survival",
        "train_table": "pp_neural_hazard_ready_train",
        "test_table": "pp_neural_hazard_ready_test",
        "target_column": target_col,
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
        "operational_feature_columns": neural_feature_columns,
        "canonical_expected_features_raw": expected_features_raw,
        "canonical_expected_features_resolved": expected_features_resolved,
        "feature_alias_map": feature_alias_map,
        "n_train_rows": int(X_train_neural.shape[0]),
        "n_test_rows": int(X_test_neural.shape[0]),
        "n_output_features_after_preprocessing": int(len(neural_feature_names_out)),
        "train_event_rate": float(y_train_neural.mean()),
        "test_event_rate": float(y_test_neural.mean()),
        "methodological_note": (
            "All learned preprocessing operations were fit on training data only "
            "and then applied unchanged to test data."
        ),
        "design_note": (
            "Canonical config features were resolved to notebook-native materialized "
            "column names before validation."
        ),
    },
    config_path,
)

# ------------------------------
# 10) Console summary
# ------------------------------
print("\nOperational feature columns used by the neural model:")
for col in neural_feature_columns:
    print(" -", col)

print("\nCanonical config alignment:")
display(canonical_alignment_df)

print("\nNeural preprocessing summary:")
display(neural_preprocessing_summary_df)

print("\nSaved:")
print("-", summary_path.resolve())
print("-", feature_manifest_path.resolve())
print("-", canonical_alignment_path.resolve())
print("-", output_manifest_path.resolve())
print("-", config_path.resolve())

### D3.1 — Performance-Oriented Neural Tuning Benchmark


In [ ]:
# ==============================================================
# D3.1 — Performance-Oriented Neural Tuning Benchmark
# --------------------------------------------------------------
# Purpose:
#   Run a controlled neural tuning benchmark to obtain a stronger
#   performance-oriented discrete-time survival model, while keeping
#   the evaluation protocol aligned with the benchmark design.
#
# Methodological note:
#   This step materializes the official tuned neural benchmark.
#   It introduces a controlled tuning phase intended to improve performance without
#   turning the benchmark into an unbounded optimization exercise.
#
#   The search remains deliberately limited and reproducible.
#   Candidate models are compared under the same enrollment-level
#   internal validation strategy used to avoid leakage.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - X_train_neural, X_test_neural
#   - y_train_neural, y_test_neural
#   - train_df_neural, test_df_neural
#   - neural_preprocessor
#   - neural_feature_names_out
#   - HORIZONS_WEEKS
#   - CALIBRATION_BINS
#
# Outputs:
#   - tuning results table
#   - best tuned neural model
#   - test-time hazard/survival predictions
#   - primary / secondary / diagnostic metric tables
#   - calibration tables by horizon
#   - saved model artifacts and evaluation outputs
# ==============================================================

print("\n" + "=" * 70)
print("D3.1 — Performance-Oriented Neural Tuning Benchmark")
print("=" * 70)
print("Methodological note: this step performs controlled neural tuning.")
print("This step is the official stronger neural benchmark under the")
print("same evaluation protocol and tuned-model consolidation strategy.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json",
    "X_train_neural", "X_test_neural", "y_train_neural", "y_test_neural",
    "train_df_neural", "test_df_neural", "neural_preprocessor", "neural_feature_names_out",
    "HORIZONS_WEEKS", "CALIBRATION_BINS"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun the required prior steps first."
    )

import itertools
import joblib
import numpy as np
import pandas as pd
import scipy
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)
from sklearn.model_selection import train_test_split

try:
    from pycox.evaluation import EvalSurv
    PYCOX_EVAL_AVAILABLE = True
except Exception:
    PYCOX_EVAL_AVAILABLE = False

try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.util import Surv
    SKSURV_AVAILABLE = True
except Exception:
    SKSURV_AVAILABLE = False

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"

try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Global tuning config
# ------------------------------
TUNING_CONFIG = {
    "model_name": "neural_discrete_time_survival_tuned",
    "search_strategy": "controlled_grid_search",
    "selection_criterion": "lowest_validation_loss",
    "validation_split_unit": "enrollment",
    "validation_fraction": 0.10,
    "batch_size": 1024,
    "max_epochs": 30,
    "early_stopping_patience": 5,
    "optimizer": "Adam",
    "loss": "BCEWithLogitsLoss",
    "random_state": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "benchmark_positioning_note": (
        "Performance-oriented neural benchmark following the simple benchmark "
        "version from P19, while preserving the same preprocessing and "
        "evaluation protocol."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
}

device = torch.device(TUNING_CONFIG["device"])

# ------------------------------
# 4) Internal validation split at enrollment level
# ------------------------------
train_enrollment_ids = train_df_neural["enrollment_id"].drop_duplicates().tolist()

enr_train_ids, enr_val_ids = train_test_split(
    train_enrollment_ids,
    test_size=TUNING_CONFIG["validation_fraction"],
    random_state=TUNING_CONFIG["random_state"],
)

enr_train_ids = set(enr_train_ids)
enr_val_ids = set(enr_val_ids)

internal_split = train_df_neural["enrollment_id"].map(
    lambda x: "train_internal" if x in enr_train_ids else "val_internal"
)

train_internal_mask = (internal_split == "train_internal").to_numpy()
val_internal_mask = (internal_split == "val_internal").to_numpy()

X_train_internal = X_train_neural[train_internal_mask]
y_train_internal = y_train_neural[train_internal_mask]

X_val_internal = X_train_neural[val_internal_mask]
y_val_internal = y_train_neural[val_internal_mask]

# torch tensors
X_train_tensor = torch.from_numpy(X_train_internal.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train_internal.astype(np.float32)).view(-1, 1)

X_val_tensor = torch.from_numpy(X_val_internal.astype(np.float32))
y_val_tensor = torch.from_numpy(y_val_internal.astype(np.float32)).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)


### Classe: TunedHazardMLP

Definicao isolada de TunedHazardMLP para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 5) Model class
# ------------------------------
class TunedHazardMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [ ]:

# ------------------------------
# 6) Tuning grid
# ------------------------------
grid_hidden_dims = [[64, 32], [128, 64]]
grid_dropout = [0.10, 0.30]
grid_learning_rate = [1e-3, 5e-4]
grid_weight_decay = [1e-5, 1e-4]

candidate_grid = list(itertools.product(
    grid_hidden_dims,
    grid_dropout,
    grid_learning_rate,
    grid_weight_decay
))

tuning_rows = []
best_candidate = None
best_val_loss = float("inf")
best_state_dict = None
best_model_obj = None

for candidate_id, (hidden_dims, dropout, lr, weight_decay) in enumerate(candidate_grid, start=1):
    torch.manual_seed(TUNING_CONFIG["random_state"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(TUNING_CONFIG["random_state"])

    model = TunedHazardMLP(
        input_dim=int(X_train_neural.shape[1]),
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=TUNING_CONFIG["batch_size"],
        shuffle=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=TUNING_CONFIG["batch_size"],
        shuffle=False,
        drop_last=False,
    )

    candidate_best_val = float("inf")
    candidate_best_epoch = None
    candidate_best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, TUNING_CONFIG["max_epochs"] + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())

        mean_train_loss = float(np.mean(train_losses))
        mean_val_loss = float(np.mean(val_losses))

        if mean_val_loss < candidate_best_val - 1e-6:
            candidate_best_val = mean_val_loss
            candidate_best_epoch = epoch
            candidate_best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= TUNING_CONFIG["early_stopping_patience"]:
            break

    tuning_rows.append({
        "candidate_id": candidate_id,
        "hidden_dims": str(hidden_dims),
        "dropout": dropout,
        "learning_rate": lr,
        "weight_decay": weight_decay,
        "best_val_loss": candidate_best_val,
        "best_epoch": candidate_best_epoch,
    })

    if candidate_best_val < best_val_loss:
        best_val_loss = candidate_best_val
        best_candidate = {
            "candidate_id": candidate_id,
            "hidden_dims": hidden_dims,
            "dropout": dropout,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "best_val_loss": candidate_best_val,
            "best_epoch": candidate_best_epoch,
        }
        best_state_dict = candidate_best_state
        best_model_obj = TunedHazardMLP(
            input_dim=int(X_train_neural.shape[1]),
            hidden_dims=hidden_dims,
            dropout=dropout,
        ).to(device)

tuning_results_df = pd.DataFrame(tuning_rows).sort_values(
    ["best_val_loss", "candidate_id"], ascending=[True, True]
).reset_index(drop=True)

# ------------------------------
# 7) Final model = best candidate loaded with best state
# ------------------------------
best_model_obj.load_state_dict(best_state_dict)
best_model_obj.eval()


### Funcao: predict_proba_in_batches

Definicao isolada de predict_proba_in_batches para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 8) Prediction helper
# ------------------------------
def predict_proba_in_batches(model, X_np: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, X_np.shape[0], batch_size):
            xb = torch.from_numpy(X_np[start:start+batch_size].astype(np.float32)).to(device)
            logits = model(xb)
            p = torch.sigmoid(logits).cpu().numpy().reshape(-1)
            probs.append(p)
    return np.concatenate(probs, axis=0)


In [ ]:

train_hazard_tuned = predict_proba_in_batches(best_model_obj, X_train_neural)
test_hazard_tuned = predict_proba_in_batches(best_model_obj, X_test_neural)

eps = 1e-8
train_hazard_tuned = np.clip(train_hazard_tuned, eps, 1 - eps)
test_hazard_tuned = np.clip(test_hazard_tuned, eps, 1 - eps)

# ------------------------------
# 9) Row-level diagnostic metrics
# ------------------------------
row_diagnostics = pd.DataFrame([{
    "model_name": "neural_discrete_time_survival_tuned",
    "row_level_roc_auc": roc_auc_score(y_test_neural, test_hazard_tuned),
    "row_level_pr_auc": average_precision_score(y_test_neural, test_hazard_tuned),
    "row_level_log_loss": log_loss(y_test_neural, test_hazard_tuned),
    "row_level_brier": brier_score_loss(y_test_neural, test_hazard_tuned),
}])

# ------------------------------
# 10) Reconstruct test hazard/survival trajectories
# ------------------------------
test_pred_df_tuned = test_df_neural.copy()
test_pred_df_tuned["pred_hazard"] = test_hazard_tuned
test_pred_df_tuned = test_pred_df_tuned.sort_values(["enrollment_id", "week"]).reset_index(drop=True)


### Funcao: compute_survival

Definicao isolada de compute_survival para reutilizacao nas etapas seguintes.


In [ ]:

def compute_survival(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy()
    g["pred_survival"] = (1.0 - g["pred_hazard"]).cumprod()
    g["pred_risk"] = 1.0 - g["pred_survival"]
    return g


In [ ]:

test_pred_df_tuned = (
    test_pred_df_tuned.groupby("enrollment_id", group_keys=False)
    .apply(compute_survival)
    .reset_index(drop=True)
)

# ------------------------------
# 11) Enrollment-level truth
# ------------------------------
truth_test = con.execute("""
SELECT
    enrollment_id,
    event,
    duration
FROM enrollment_cox_ready_test
ORDER BY enrollment_id
""").fetchdf()

truth_train = con.execute("""
SELECT
    enrollment_id,
    event,
    duration
FROM enrollment_cox_ready_train
ORDER BY enrollment_id
""").fetchdf()

truth_test["event"] = truth_test["event"].astype(int)
truth_test["duration"] = truth_test["duration"].astype(int)

truth_train["event"] = truth_train["event"].astype(int)
truth_train["duration"] = truth_train["duration"].astype(int)

# ------------------------------
# 12) Build survival/risk matrices
# ------------------------------
max_duration_test = int(truth_test["duration"].max())
eval_time_grid_full = np.arange(0, max_duration_test + 1, dtype=int)

surv_dict = {}
risk_dict = {}

for enrollment_id, group in test_pred_df_tuned.groupby("enrollment_id"):
    g = group.sort_values("week")

    surv_series = pd.Series(
        g["pred_survival"].values,
        index=g["week"].astype(int).values,
        name=enrollment_id,
    )
    risk_series = pd.Series(
        g["pred_risk"].values,
        index=g["week"].astype(int).values,
        name=enrollment_id,
    )

    surv_full = surv_series.reindex(eval_time_grid_full).ffill().fillna(1.0)
    risk_full = risk_series.reindex(eval_time_grid_full).ffill().fillna(0.0)

    surv_dict[enrollment_id] = surv_full.values
    risk_dict[enrollment_id] = risk_full.values

surv_df = pd.DataFrame(surv_dict, index=eval_time_grid_full)
risk_df = pd.DataFrame(risk_dict, index=eval_time_grid_full)

surv_df.columns.name = "enrollment_id"
risk_df.columns.name = "enrollment_id"

truth_test = truth_test.set_index("enrollment_id").loc[surv_df.columns].reset_index()
durations_test = truth_test["duration"].to_numpy(dtype=int)
events_test = truth_test["event"].to_numpy(dtype=int)


### Funcao: get_pred_risk_at_horizon

Definicao isolada de get_pred_risk_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 13) Helper functions
# ------------------------------
def get_pred_risk_at_horizon(h: int, risk_matrix: pd.DataFrame) -> pd.DataFrame:
    if h not in risk_matrix.index:
        raise ValueError(f"Horizon {h} not present in risk_df index.")
    return (
        risk_matrix.loc[h]
        .rename("pred_risk_h")
        .rename_axis("enrollment_id")
        .reset_index()
    )


### Funcao: get_pred_survival_at_horizon

Definicao isolada de get_pred_survival_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

def get_pred_survival_at_horizon(h: int, surv_matrix: pd.DataFrame) -> pd.DataFrame:
    if h not in surv_matrix.index:
        raise ValueError(f"Horizon {h} not present in surv_df index.")
    return (
        surv_matrix.loc[h]
        .rename("pred_survival_h")
        .rename_axis("enrollment_id")
        .reset_index()
    )


In [ ]:

# ------------------------------
# 14) Primary survival metrics
# ------------------------------
primary_metrics_rows = []
brier_at_horizon_df = pd.DataFrame(columns=[
    "horizon_week", "metric_name", "metric_category", "metric_value", "notes"
])

if PYCOX_EVAL_AVAILABLE:
    eval_surv = EvalSurv(
        surv=surv_df,
        durations=durations_test,
        events=events_test,
        censor_surv="km",
    )

    try:
        c_index = float(eval_surv.concordance_td())
        c_index_note = "pycox_concordance_td"
    except Exception as e:
        c_index = np.nan
        c_index_note = f"failed: {str(e)}"

    primary_metrics_rows.append({
        "metric_name": "c_index",
        "metric_category": "co_primary",
        "metric_value": c_index,
        "notes": c_index_note,
    })

    try:
        brier_h = eval_surv.brier_score(np.array(HORIZONS_WEEKS, dtype=int))
        brier_at_horizon_df = pd.DataFrame({
            "horizon_week": list(brier_h.index.astype(int)),
            "metric_name": ["brier_at_horizon"] * len(brier_h.index),
            "metric_category": ["primary"] * len(brier_h.index),
            "metric_value": list(brier_h.values.astype(float)),
            "notes": ["pycox_brier_score"] * len(brier_h.index),
        })
    except Exception as e:
        brier_at_horizon_df = pd.DataFrame({
            "horizon_week": HORIZONS_WEEKS,
            "metric_name": ["brier_at_horizon"] * len(HORIZONS_WEEKS),
            "metric_category": ["primary"] * len(HORIZONS_WEEKS),
            "metric_value": [np.nan] * len(HORIZONS_WEEKS),
            "notes": [f"failed: {str(e)}"] * len(HORIZONS_WEEKS),
        })

    try:
        max_requested_horizon = int(max(HORIZONS_WEEKS))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
        ibs_note = "pycox_integrated_brier_score"
    except Exception as e:
        try:
            max_requested_horizon = int(max(HORIZONS_WEEKS))
            ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
            dense_brier = eval_surv.brier_score(ibs_grid)
            ibs_value = float(np.trapz(dense_brier.values.astype(float), ibs_grid) / (ibs_grid[-1] - ibs_grid[0]))
            ibs_note = f"fallback_np_trapz_after_pycox_failure: {str(e)}"
        except Exception as e2:
            ibs_value = np.nan
            ibs_note = f"failed: {str(e)} | fallback_failed: {str(e2)}"

    primary_metrics_rows.insert(0, {
        "metric_name": "ibs",
        "metric_category": "primary",
        "metric_value": ibs_value,
        "notes": ibs_note,
    })
else:
    primary_metrics_rows.append({
        "metric_name": "ibs",
        "metric_category": "primary",
        "metric_value": np.nan,
        "notes": "pycox_unavailable",
    })
    primary_metrics_rows.append({
        "metric_name": "c_index",
        "metric_category": "co_primary",
        "metric_value": np.nan,
        "notes": "pycox_unavailable",
    })

primary_metrics_summary = pd.DataFrame(primary_metrics_rows)

# ------------------------------
# 15) Horizon-specific support, risk AUC, and calibration
# ------------------------------
support_rows = []
risk_auc_rows = []
calibration_rows = []

for h in HORIZONS_WEEKS:
    eval_df = truth_test.copy()
    eval_df["is_evaluable_at_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
        (eval_df["duration"] >= h)
    ).astype(int)

    eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()

    pred_h = get_pred_risk_at_horizon(h, risk_df)
    eval_df = eval_df.merge(pred_h, on="enrollment_id", how="left")

    eval_df["observed_event_by_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
    )

    support_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
        "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df["observed_event_by_h"].nunique() >= 2:
        risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
    else:
        risk_auc = np.nan

    risk_auc_rows.append({
        "horizon_week": h,
        "metric_name": "risk_auc_at_horizon",
        "metric_category": "secondary",
        "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        "notes": "roc_auc_on_evaluable_subset",
    })

    calib_df = eval_df[["enrollment_id", "pred_risk_h", "observed_event_by_h"]].copy()

    if calib_df.shape[0] > 0:
        ranked = calib_df["pred_risk_h"].rank(method="first")
        n_bins_eff = int(min(CALIBRATION_BINS, max(1, calib_df.shape[0])))

        calib_df["calibration_bin"] = pd.qcut(
            ranked,
            q=n_bins_eff,
            labels=False,
            duplicates="drop"
        )

        calib_tab = (
            calib_df.groupby("calibration_bin")
            .agg(
                n=("enrollment_id", "count"),
                mean_predicted_risk=("pred_risk_h", "mean"),
                observed_event_rate=("observed_event_by_h", "mean"),
            )
            .reset_index()
        )
        calib_tab["horizon_week"] = h
        calib_tab["abs_calibration_gap"] = (
            calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
        ).abs()

        for _, row in calib_tab.iterrows():
            calibration_rows.append({
                "horizon_week": int(row["horizon_week"]),
                "calibration_bin": int(row["calibration_bin"]),
                "n": int(row["n"]),
                "mean_predicted_risk": float(row["mean_predicted_risk"]),
                "observed_event_rate": float(row["observed_event_rate"]),
                "abs_calibration_gap": float(row["abs_calibration_gap"]),
            })

support_by_horizon_df = pd.DataFrame(support_rows)
risk_auc_by_horizon_df = pd.DataFrame(risk_auc_rows)
calibration_by_horizon_df = pd.DataFrame(calibration_rows)

calibration_summary_rows = []
for h in HORIZONS_WEEKS:
    h_df = calibration_by_horizon_df[calibration_by_horizon_df["horizon_week"] == h].copy()
    if h_df.shape[0] > 0:
        ece_like = np.average(h_df["abs_calibration_gap"], weights=h_df["n"])
    else:
        ece_like = np.nan

    calibration_summary_rows.append({
        "horizon_week": h,
        "metric_name": "calibration_at_horizon",
        "metric_category": "primary",
        "metric_value": float(ece_like) if pd.notna(ece_like) else np.nan,
        "notes": "Weighted absolute calibration gap across bins",
    })

calibration_summary_df = pd.DataFrame(calibration_summary_rows)

# ------------------------------
# 16) Optional time-dependent AUC
# ------------------------------
td_auc_rows = []

if SKSURV_AVAILABLE:
    try:
        y_train_surv = Surv.from_arrays(
            event=truth_train["event"].astype(bool).to_numpy(),
            time=truth_train["duration"].astype(float).to_numpy(),
        )
        y_test_surv = Surv.from_arrays(
            event=truth_test["event"].astype(bool).to_numpy(),
            time=truth_test["duration"].astype(float).to_numpy(),
        )

        for h in HORIZONS_WEEKS:
            risk_scores_h = risk_df.loc[h].to_numpy()
            auc_vals, mean_auc = cumulative_dynamic_auc(
                y_train_surv,
                y_test_surv,
                risk_scores_h,
                np.array([h], dtype=float)
            )
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": float(auc_vals[0]),
                "notes": "sksurv cumulative_dynamic_auc",
            })
    except Exception as e:
        for h in HORIZONS_WEEKS:
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": np.nan,
                "notes": f"sksurv_available_but_failed: {str(e)}",
            })
else:
    for h in HORIZONS_WEEKS:
        td_auc_rows.append({
            "horizon_week": h,
            "metric_name": "time_dependent_auc",
            "metric_category": "secondary",
            "metric_value": np.nan,
            "notes": "sksurv_unavailable",
        })

time_dependent_auc_df = pd.DataFrame(td_auc_rows)

# ------------------------------
# 17) Predicted vs observed survival by horizon
# ------------------------------
pred_vs_obs_rows = []

for h in HORIZONS_WEEKS:
    pred_surv_h = get_pred_survival_at_horizon(h, surv_df)
    tmp = truth_test.merge(pred_surv_h, on="enrollment_id", how="left")

    tmp["is_evaluable_at_h"] = (
        ((tmp["event"] == 1) & (tmp["duration"] <= h)) |
        (tmp["duration"] >= h)
    ).astype(int)
    tmp = tmp[tmp["is_evaluable_at_h"] == 1].copy()

    tmp["observed_survival_by_h"] = 1 - (
        ((tmp["event"] == 1) & (tmp["duration"] <= h)).astype(int)
    )

    pred_vs_obs_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(tmp.shape[0]),
        "mean_predicted_survival": float(tmp["pred_survival_h"].mean()) if tmp.shape[0] > 0 else np.nan,
        "mean_observed_survival": float(tmp["observed_survival_by_h"].mean()) if tmp.shape[0] > 0 else np.nan,
        "abs_gap": float(abs(tmp["pred_survival_h"].mean() - tmp["observed_survival_by_h"].mean())) if tmp.shape[0] > 0 else np.nan,
    })

predicted_vs_observed_survival_df = pd.DataFrame(pred_vs_obs_rows)

# ------------------------------
# 18) Hazard and survival audit tables
# ------------------------------
hazard_audit_summary = pd.DataFrame([{
    "mean_pred_hazard_test": float(np.mean(test_hazard_tuned)),
    "median_pred_hazard_test": float(np.median(test_hazard_tuned)),
    "p90_pred_hazard_test": float(np.quantile(test_hazard_tuned, 0.90)),
    "p99_pred_hazard_test": float(np.quantile(test_hazard_tuned, 0.99)),
    "min_pred_hazard_test": float(np.min(test_hazard_tuned)),
    "max_pred_hazard_test": float(np.max(test_hazard_tuned)),
}])

hazard_by_week_df = (
    test_pred_df_tuned.groupby("week")
    .agg(
        mean_pred_hazard=("pred_hazard", "mean"),
        median_pred_hazard=("pred_hazard", "median"),
        mean_pred_survival=("pred_survival", "mean"),
        mean_pred_risk=("pred_risk", "mean"),
        n_rows=("enrollment_id", "count"),
    )
    .reset_index()
    .sort_values("week")
)

# ------------------------------
# 19) Save model and outputs
# ------------------------------
model_dir = OUTPUT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "neural_discrete_time_survival_tuned.pt"
preprocessor_path = model_dir / "neural_discrete_time_preprocessor.joblib"
tuning_results_path = TABLES_DIR / "table_neural_tuning_results.csv"
history_path = TABLES_DIR / "table_neural_tuned_training_history.csv"
test_predictions_path = TABLES_DIR / "table_neural_tuned_test_predictions.csv"
primary_metrics_path = TABLES_DIR / "table_neural_tuned_primary_metrics.csv"
brier_horizon_path = TABLES_DIR / "table_neural_tuned_brier_by_horizon.csv"
secondary_metrics_path = TABLES_DIR / "table_neural_tuned_secondary_metrics.csv"
diagnostics_path = TABLES_DIR / "table_neural_tuned_row_diagnostics.csv"
support_path = TABLES_DIR / "table_neural_tuned_support_by_horizon.csv"
calibration_summary_path = TABLES_DIR / "table_neural_tuned_calibration_summary.csv"
calibration_bins_path = TABLES_DIR / "table_neural_tuned_calibration_bins_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_neural_tuned_predicted_vs_observed_survival.csv"
hazard_audit_path = TABLES_DIR / "table_neural_tuned_hazard_audit_summary.csv"
hazard_by_week_path = TABLES_DIR / "table_neural_tuned_hazard_by_week.csv"
model_config_path = METADATA_DIR / "neural_tuned_model_config.json"

torch.save(best_model_obj.state_dict(), model_path)
joblib.dump(neural_preprocessor, preprocessor_path)

tuning_results_df.to_csv(tuning_results_path, index=False)
# final chosen candidate history is not stored per-candidate during search; store best candidate summary
pd.DataFrame([best_candidate]).to_csv(history_path, index=False)
test_pred_df_tuned.to_csv(test_predictions_path, index=False)
primary_metrics_summary.to_csv(primary_metrics_path, index=False)
brier_at_horizon_df.to_csv(brier_horizon_path, index=False)

secondary_metrics_df = pd.concat(
    [risk_auc_by_horizon_df, time_dependent_auc_df],
    ignore_index=True
)
secondary_metrics_df.to_csv(secondary_metrics_path, index=False)

row_diagnostics.to_csv(diagnostics_path, index=False)
support_by_horizon_df.to_csv(support_path, index=False)
calibration_summary_df.to_csv(calibration_summary_path, index=False)
calibration_by_horizon_df.to_csv(calibration_bins_path, index=False)
predicted_vs_observed_survival_df.to_csv(pred_vs_obs_path, index=False)
hazard_audit_summary.to_csv(hazard_audit_path, index=False)
hazard_by_week_df.to_csv(hazard_by_week_path, index=False)

best_config_to_save = dict(TUNING_CONFIG)
best_config_to_save["best_candidate"] = best_candidate
save_json(best_config_to_save, model_config_path)

# ------------------------------
# 20) Output for feedback
# ------------------------------
print("\nTuning results:")
display(tuning_results_df)

print("\nBest candidate:")
display(pd.DataFrame([best_candidate]))

print("\nPrimary metrics summary:")
display(primary_metrics_summary)

print("\nBrier by horizon:")
display(brier_at_horizon_df)

print("\nCalibration summary by horizon:")
display(calibration_summary_df)

print("\nSecondary metrics:")
display(secondary_metrics_df)

print("\nRow-level diagnostics:")
display(row_diagnostics)

print("\nSupport by horizon:")
display(support_by_horizon_df)

print("\nPredicted vs observed survival:")
display(predicted_vs_observed_survival_df)

print("\nHazard audit summary:")
display(hazard_audit_summary)

print("\nHazard by week (first rows):")
display(hazard_by_week_df.head(20))

print("\nSaved artifacts:")
print("-", model_path.resolve())
print("-", preprocessor_path.resolve())
print("-", tuning_results_path.resolve())
print("-", history_path.resolve())
print("-", test_predictions_path.resolve())
print("-", primary_metrics_path.resolve())
print("-", brier_horizon_path.resolve())
print("-", secondary_metrics_path.resolve())
print("-", diagnostics_path.resolve())
print("-", support_path.resolve())
print("-", calibration_summary_path.resolve())
print("-", calibration_bins_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", hazard_audit_path.resolve())
print("-", hazard_by_week_path.resolve())
print("-", model_config_path.resolve())


## D4 — Treatment for Cox (Revised)


In [ ]:
# ==============================================================
# D4 — Treatment for Cox (Revised)
# --------------------------------------------------------------
# Purpose:
#   Prepare the train/test preprocessing pipeline for the revised
#   Cox benchmark, with all learned transformations fitted on
#   training data only.
#
# Methodological note:
#   This revised Cox treatment aligns with the early-window
#   comparable benchmark design.
#
#   Therefore, the Cox model now uses:
#     - static covariates
#     - early-window temporal summaries (main window = 4 weeks)
#
#   This avoids retrospective whole-course summaries in the main
#   Cox benchmark track.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - STATIC_FEATURES
#   - MAIN_ENROLLMENT_WINDOW_FEATURES
#
# Outputs:
#   - X_train_cox, X_test_cox
#   - duration_train_cox, duration_test_cox
#   - event_train_cox, event_test_cox
#   - cox_preprocessor
#   - cox_feature_names_out
#   - audit tables / metadata
# ==============================================================

print("\n" + "=" * 70)
print("D4 — Treatment for Cox (Revised)")
print("=" * 70)
print("Methodological note: this Cox treatment now follows the")
print("early-window comparable benchmark design.")
print("All learned preprocessing transformations are fit on training data only.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "METADATA_DIR", "save_json",
    "STATIC_FEATURES", "MAIN_ENROLLMENT_WINDOW_FEATURES"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------
# 2) Load train/test tables
# ------------------------------
train_df_cox = con.execute("SELECT * FROM enrollment_cox_ready_train").fetchdf()
test_df_cox = con.execute("SELECT * FROM enrollment_cox_ready_test").fetchdf()

# ------------------------------
# 3) Define targets and feature columns
# ------------------------------
duration_col = "duration"
event_col = "event"

categorical_features = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
    "clicks_first_4_weeks",
    "active_weeks_first_4",
    "mean_clicks_first_4_weeks",
]

cox_feature_columns = categorical_features + numeric_features

expected_features = STATIC_FEATURES + MAIN_ENROLLMENT_WINDOW_FEATURES
missing_from_defined = [c for c in expected_features if c not in cox_feature_columns]
if missing_from_defined:
    raise ValueError(f"Missing configured Cox features in treatment spec: {missing_from_defined}")

# ------------------------------
# 4) Split X / targets
# ------------------------------
X_train_raw_cox = train_df_cox[cox_feature_columns].copy()
X_test_raw_cox = test_df_cox[cox_feature_columns].copy()

duration_train_cox = train_df_cox[duration_col].astype(np.float32).to_numpy()
duration_test_cox = test_df_cox[duration_col].astype(np.float32).to_numpy()

event_train_cox = train_df_cox[event_col].astype(np.int32).to_numpy()
event_test_cox = test_df_cox[event_col].astype(np.int32).to_numpy()

# ------------------------------
# 5) Build preprocessing pipelines
# ------------------------------
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

cox_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# ------------------------------
# 6) Fit on train only, transform train/test
# ------------------------------
X_train_cox = cox_preprocessor.fit_transform(X_train_raw_cox).astype(np.float32)
X_test_cox = cox_preprocessor.transform(X_test_raw_cox).astype(np.float32)

cox_feature_names_out = cox_preprocessor.get_feature_names_out().tolist()

# ------------------------------
# 7) Build preprocessing summaries
# ------------------------------
summary_rows = [{
    "model_family": "cox",
    "train_rows": int(X_train_cox.shape[0]),
    "test_rows": int(X_test_cox.shape[0]),
    "n_input_features_raw": int(len(cox_feature_columns)),
    "n_numeric_features": int(len(numeric_features)),
    "n_categorical_features": int(len(categorical_features)),
    "n_features_after_transform": int(X_train_cox.shape[1]),
    "n_events_train": int(event_train_cox.sum()),
    "n_events_test": int(event_test_cox.sum()),
    "mean_duration_train": float(np.mean(duration_train_cox)),
    "mean_duration_test": float(np.mean(duration_test_cox)),
    "numeric_imputation": "median",
    "categorical_imputation": "constant_missing",
    "categorical_encoding": "one_hot_handle_unknown_ignore",
    "numeric_scaling": "standard_scaler_fit_on_train_only",
    "output_dtype": "float32",
    "cox_positioning_note": "early-window comparable Cox benchmark",
    "window_weeks": 4,
}]

cox_preprocessing_summary = pd.DataFrame(summary_rows)
cox_feature_names_df = pd.DataFrame({
    "feature_name_out": cox_feature_names_out
})

# ------------------------------
# 8) Save metadata and audit tables
# ------------------------------
summary_path = TABLES_DIR / "table_cox_preprocessing_summary.csv"
feature_names_path = TABLES_DIR / "table_cox_feature_names_out.csv"
config_path = METADATA_DIR / "cox_preprocessing_config.json"

cox_preprocessing_summary.to_csv(summary_path, index=False)
cox_feature_names_df.to_csv(feature_names_path, index=False)

cox_preprocessing_config = {
    "model_family": "cox",
    "duration_column": duration_col,
    "event_column": event_col,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "raw_feature_columns": cox_feature_columns,
    "numeric_imputation": "median",
    "categorical_imputation": "constant_missing",
    "categorical_encoding": "one_hot_handle_unknown_ignore",
    "numeric_scaling": "standard_scaler_fit_on_train_only",
    "output_dtype": "float32",
    "cox_positioning_note": "early-window comparable Cox benchmark",
    "window_weeks": 4,
    "n_features_after_transform": int(X_train_cox.shape[1]),
}

save_json(cox_preprocessing_config, config_path)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nCox preprocessing summary:")
display(cox_preprocessing_summary)

print("\nFirst transformed feature names:")
display(cox_feature_names_df.head(40))

print("\nArray shapes:")
print("X_train_cox     :", X_train_cox.shape, X_train_cox.dtype)
print("X_test_cox      :", X_test_cox.shape, X_test_cox.dtype)
print("duration_train  :", duration_train_cox.shape, duration_train_cox.dtype)
print("duration_test   :", duration_test_cox.shape, duration_test_cox.dtype)
print("event_train     :", event_train_cox.shape, event_train_cox.dtype)
print("event_test      :", event_test_cox.shape, event_test_cox.dtype)

print("\nSaved:")
print("-", summary_path.resolve())
print("-", feature_names_path.resolve())
print("-", config_path.resolve())

### D4.1 — Performance-Oriented Cox Tuning


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D4.1 — Performance-Oriented Cox Tuning
# --------------------------------------------------------------
# Purpose:
#   Perform a controlled hyperparameter search for the comparable
#   Cox benchmark, select the best candidate, refit it on the full
#   training set, and evaluate it under the same survival-oriented
#   benchmark protocol defined in D1.
#
# Methodological note:
#   This step provides the official tuned Cox benchmark under the same early-window
#   comparable design:
#     - static covariates
#     - early-window temporal summaries (main window = 4 weeks)
#
#   The search is intentionally limited and reproducible.
# ==============================================================

print("\n" + "=" * 70)
print("D4.1 — Performance-Oriented Cox Tuning")
print("=" * 70)
print("Methodological note: this step performs controlled tuning for the")
print("early-window comparable Cox benchmark under the same survival-oriented")
print("benchmark protocol defined in D1.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json",
    "X_train_cox", "X_test_cox",
    "duration_train_cox", "duration_test_cox",
    "event_train_cox", "event_test_cox",
    "train_df_cox", "test_df_cox",
    "cox_preprocessor", "cox_feature_names_out",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun the required prior steps first."
    )

import itertools
import joblib
import numpy as np
import pandas as pd
import scipy
from sklearn.model_selection import train_test_split

try:
    from lifelines import CoxPHFitter
    LIFELINES_AVAILABLE = True
except Exception:
    LIFELINES_AVAILABLE = False

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.util import Surv
    SKSURV_AVAILABLE = True
except Exception:
    SKSURV_AVAILABLE = False

if not LIFELINES_AVAILABLE:
    raise ImportError("lifelines is required for P22.2.")

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P22.2.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"

try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Build train/test frames
# ------------------------------
X_train_cox_df_full = pd.DataFrame(X_train_cox, columns=cox_feature_names_out)
X_test_cox_df_full = pd.DataFrame(X_test_cox, columns=cox_feature_names_out)

# zero-variance filter based on full train
train_std_full = X_train_cox_df_full.std(axis=0, ddof=0)
keep_cols = train_std_full[train_std_full > 0].index.tolist()
dropped_zero_var_cols = train_std_full[train_std_full <= 0].index.tolist()

X_train_cox_df_full = X_train_cox_df_full[keep_cols].copy()
X_test_cox_df_full = X_test_cox_df_full[keep_cols].copy()

train_cox_full_df = X_train_cox_df_full.copy()
train_cox_full_df["duration"] = duration_train_cox
train_cox_full_df["event"] = event_train_cox
train_cox_full_df["enrollment_id"] = train_df_cox["enrollment_id"].values

test_cox_eval_df = X_test_cox_df_full.copy()
test_cox_eval_df["duration"] = duration_test_cox
test_cox_eval_df["event"] = event_test_cox
test_cox_eval_df["enrollment_id"] = test_df_cox["enrollment_id"].values

# ------------------------------
# 4) Enrollment-level train/validation split
# ------------------------------
unique_train_enrollments = train_cox_full_df["enrollment_id"].drop_duplicates().to_numpy()

# stratify by event if possible
enrollment_event_df = (
    train_cox_full_df[["enrollment_id", "event"]]
    .drop_duplicates(subset=["enrollment_id"])
    .copy()
)

stratify_labels = enrollment_event_df["event"] if enrollment_event_df["event"].nunique() > 1 else None

train_ids, val_ids = train_test_split(
    enrollment_event_df["enrollment_id"].to_numpy(),
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=stratify_labels
)

subtrain_df = train_cox_full_df[train_cox_full_df["enrollment_id"].isin(train_ids)].copy()
val_df = train_cox_full_df[train_cox_full_df["enrollment_id"].isin(val_ids)].copy()

# Remove enrollment_id from fit frames
subtrain_fit_df = subtrain_df.drop(columns=["enrollment_id"]).copy()
val_fit_df = val_df.drop(columns=["enrollment_id"]).copy()

# ------------------------------
# 5) Tuning config
# ------------------------------
COX_TUNING_CONFIG = {
    "model_name": "cox_early_window_tuned",
    "penalizer_grid": [0.001, 0.01, 0.1, 1.0],
    "l1_ratio_grid": [0.0, 0.5, 1.0],
    "selection_metric": "val_neg_partial_log_likelihood",
    "window_weeks": 4,
    "benchmark_positioning_note": (
        "Performance-oriented tuned Cox benchmark under the same early-window comparable input design."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
}

# ------------------------------
# 6) Run tuning grid
# ------------------------------
candidate_rows = []

for candidate_id, (penalizer, l1_ratio) in enumerate(
    itertools.product(COX_TUNING_CONFIG["penalizer_grid"], COX_TUNING_CONFIG["l1_ratio_grid"]),
    start=1
):
    try:
        model = CoxPHFitter(
            penalizer=penalizer,
            l1_ratio=l1_ratio,
        )

        model.fit(
            subtrain_fit_df,
            duration_col="duration",
            event_col="event",
            show_progress=False,
        )

        # Lifelines reports log_likelihood_ on fitted data; use val score via score()
        val_score = model.score(val_fit_df, scoring_method="log_likelihood")
        val_c_index = model.score(val_fit_df, scoring_method="concordance_index")

        candidate_rows.append({
            "candidate_id": candidate_id,
            "penalizer": float(penalizer),
            "l1_ratio": float(l1_ratio),
            "val_neg_partial_log_likelihood": float(-val_score),
            "val_concordance_index": float(val_c_index),
            "fit_status": "ok",
        })
    except Exception as e:
        candidate_rows.append({
            "candidate_id": candidate_id,
            "penalizer": float(penalizer),
            "l1_ratio": float(l1_ratio),
            "val_neg_partial_log_likelihood": np.nan,
            "val_concordance_index": np.nan,
            "fit_status": f"failed: {str(e)}",
        })

tuning_results_df = pd.DataFrame(candidate_rows)

valid_tuning_df = tuning_results_df[tuning_results_df["fit_status"] == "ok"].copy()
if valid_tuning_df.empty:
    raise RuntimeError("All Cox tuning candidates failed. Review convergence and preprocessing.")

tuning_results_df = tuning_results_df.sort_values(
    by=["fit_status", "val_neg_partial_log_likelihood", "candidate_id"],
    ascending=[True, True, True]
).reset_index(drop=True)

best_valid_row = valid_tuning_df.sort_values(
    by=["val_neg_partial_log_likelihood", "candidate_id"],
    ascending=[True, True]
).iloc[0]

best_candidate_df = pd.DataFrame([best_valid_row.to_dict()])

best_penalizer = float(best_valid_row["penalizer"])
best_l1_ratio = float(best_valid_row["l1_ratio"])

# ------------------------------
# 7) Refit best model on full train
# ------------------------------
train_fit_df_full = train_cox_full_df.drop(columns=["enrollment_id"]).copy()

best_cox_model = CoxPHFitter(
    penalizer=best_penalizer,
    l1_ratio=best_l1_ratio,
)

best_cox_model.fit(
    train_fit_df_full,
    duration_col="duration",
    event_col="event",
    show_progress=False,
)

# ------------------------------
# 8) Predict survival on test
# ------------------------------
pred_surv = best_cox_model.predict_survival_function(
    X_test_cox_df_full,
    times=np.arange(0, int(np.max(duration_test_cox)) + 1, dtype=int)
)

surv_df = pred_surv.copy()
surv_df.columns = test_df_cox["enrollment_id"].tolist()
surv_df.columns.name = "enrollment_id"
surv_df.index = surv_df.index.astype(int)

truth_test = pd.DataFrame({
    "enrollment_id": test_df_cox["enrollment_id"].values,
    "event": event_test_cox.astype(int),
    "duration": duration_test_cox.astype(int),
})

truth_train = pd.DataFrame({
    "enrollment_id": train_df_cox["enrollment_id"].values,
    "event": event_train_cox.astype(int),
    "duration": duration_train_cox.astype(int),
})

durations_test = truth_test["duration"].to_numpy(dtype=int)
events_test = truth_test["event"].to_numpy(dtype=int)

# ------------------------------
# 9) Primary survival metrics
# ------------------------------
eval_surv = EvalSurv(
    surv=surv_df,
    durations=durations_test,
    events=events_test,
    censor_surv="km",
)

primary_metrics_rows = []

try:
    c_index = float(eval_surv.concordance_td())
    c_index_note = "pycox_concordance_td"
except Exception as e:
    c_index = np.nan
    c_index_note = f"failed: {str(e)}"

primary_metrics_rows.append({
    "metric_name": "c_index",
    "metric_category": "co_primary",
    "metric_value": c_index,
    "notes": c_index_note,
})

try:
    max_requested_horizon = int(max(HORIZONS_WEEKS))
    ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
    ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    ibs_note = "pycox_integrated_brier_score"
except Exception as e:
    try:
        max_requested_horizon = int(max(HORIZONS_WEEKS))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        dense_brier = eval_surv.brier_score(ibs_grid)
        ibs_value = float(np.trapz(dense_brier.values.astype(float), ibs_grid) / (ibs_grid[-1] - ibs_grid[0]))
        ibs_note = f"fallback_np_trapz_after_pycox_failure: {str(e)}"
    except Exception as e2:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)} | fallback_failed: {str(e2)}"

primary_metrics_rows.insert(0, {
    "metric_name": "ibs",
    "metric_category": "primary",
    "metric_value": ibs_value,
    "notes": ibs_note,
})

primary_metrics_summary = pd.DataFrame(primary_metrics_rows)

# ------------------------------
# 10) Brier by horizon
# ------------------------------
try:
    brier_h = eval_surv.brier_score(np.array(HORIZONS_WEEKS, dtype=int))
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": list(brier_h.index.astype(int)),
        "metric_name": ["brier_at_horizon"] * len(brier_h.index),
        "metric_category": ["primary"] * len(brier_h.index),
        "metric_value": list(brier_h.values.astype(float)),
        "notes": ["pycox_brier_score"] * len(brier_h.index),
    })
except Exception as e:
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": HORIZONS_WEEKS,
        "metric_name": ["brier_at_horizon"] * len(HORIZONS_WEEKS),
        "metric_category": ["primary"] * len(HORIZONS_WEEKS),
        "metric_value": [np.nan] * len(HORIZONS_WEEKS),
        "notes": [f"failed: {str(e)}"] * len(HORIZONS_WEEKS),
    })

# ------------------------------
# 11) Helpers
# ------------------------------
def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]

# ------------------------------
# 12) Support, calibration, risk AUC, predicted vs observed
# ------------------------------
support_rows = []
risk_auc_rows = []
calibration_rows = []
pred_vs_obs_rows = []
test_pred_rows = []

for h in HORIZONS_WEEKS:
    pred_surv_h = get_surv_at_horizon(surv_df, h)
    pred_risk_h = 1.0 - pred_surv_h

    eval_df = truth_test.copy()
    eval_df["pred_survival_h"] = eval_df["enrollment_id"].map(pred_surv_h.to_dict())
    eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

    eval_df["is_evaluable_at_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
        (eval_df["duration"] >= h)
    ).astype(int)
    eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()

    eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
    eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

    support_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
        "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df["observed_event_by_h"].nunique() >= 2:
        risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
    else:
        risk_auc = np.nan

    risk_auc_rows.append({
        "horizon_week": h,
        "metric_name": "risk_auc_at_horizon",
        "metric_category": "secondary",
        "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        "notes": "roc_auc_on_evaluable_subset",
    })

    pred_vs_obs_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df.shape[0] > 0:
        ranked = eval_df["pred_risk_h"].rank(method="first")
        n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))

        eval_df["calibration_bin"] = pd.qcut(
            ranked,
            q=n_bins_eff,
            labels=False,
            duplicates="drop"
        )

        calib_tab = (
            eval_df.groupby("calibration_bin")
            .agg(
                n=("enrollment_id", "count"),
                mean_predicted_risk=("pred_risk_h", "mean"),
                observed_event_rate=("observed_event_by_h", "mean"),
            )
            .reset_index()
        )
        calib_tab["horizon_week"] = h
        calib_tab["abs_calibration_gap"] = (
            calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
        ).abs()

        for _, row in calib_tab.iterrows():
            calibration_rows.append({
                "horizon_week": int(row["horizon_week"]),
                "calibration_bin": int(row["calibration_bin"]),
                "n": int(row["n"]),
                "mean_predicted_risk": float(row["mean_predicted_risk"]),
                "observed_event_rate": float(row["observed_event_rate"]),
                "abs_calibration_gap": float(row["abs_calibration_gap"]),
            })

    tmp_preds = truth_test[["enrollment_id", "event", "duration"]].copy()
    tmp_preds["horizon_week"] = h
    tmp_preds["pred_survival_h"] = tmp_preds["enrollment_id"].map(pred_surv_h.to_dict())
    tmp_preds["pred_risk_h"] = tmp_preds["enrollment_id"].map(pred_risk_h.to_dict())
    test_pred_rows.append(tmp_preds)

support_by_horizon_df = pd.DataFrame(support_rows)
risk_auc_by_horizon_df = pd.DataFrame(risk_auc_rows)
predicted_vs_observed_survival_df = pd.DataFrame(pred_vs_obs_rows)
calibration_bins_df = pd.DataFrame(calibration_rows)
test_predictions_df = pd.concat(test_pred_rows, ignore_index=True)

calibration_summary_rows = []
for h in HORIZONS_WEEKS:
    h_df = calibration_bins_df[calibration_bins_df["horizon_week"] == h].copy()
    if h_df.shape[0] > 0:
        ece_like = np.average(h_df["abs_calibration_gap"], weights=h_df["n"])
    else:
        ece_like = np.nan
    calibration_summary_rows.append({
        "horizon_week": h,
        "metric_name": "calibration_at_horizon",
        "metric_category": "primary",
        "metric_value": float(ece_like) if pd.notna(ece_like) else np.nan,
        "notes": "Weighted absolute calibration gap across bins",
    })

calibration_summary_df = pd.DataFrame(calibration_summary_rows)

# ------------------------------
# 13) Optional time-dependent AUC
# ------------------------------
td_auc_rows = []
if SKSURV_AVAILABLE:
    try:
        y_train_surv = Surv.from_arrays(
            event=truth_train["event"].astype(bool).to_numpy(),
            time=truth_train["duration"].astype(float).to_numpy(),
        )
        y_test_surv = Surv.from_arrays(
            event=truth_test["event"].astype(bool).to_numpy(),
            time=truth_test["duration"].astype(float).to_numpy(),
        )

        for h in HORIZONS_WEEKS:
            pred_risk_h = 1.0 - get_surv_at_horizon(surv_df, h)
            auc_vals, mean_auc = cumulative_dynamic_auc(
                y_train_surv,
                y_test_surv,
                pred_risk_h.to_numpy(),
                np.array([h], dtype=float)
            )
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": float(auc_vals[0]),
                "notes": "sksurv cumulative_dynamic_auc",
            })
    except Exception as e:
        for h in HORIZONS_WEEKS:
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": np.nan,
                "notes": f"sksurv_available_but_failed: {str(e)}",
            })
else:
    for h in HORIZONS_WEEKS:
        td_auc_rows.append({
            "horizon_week": h,
            "metric_name": "time_dependent_auc",
            "metric_category": "secondary",
            "metric_value": np.nan,
            "notes": "sksurv_unavailable",
        })

time_dependent_auc_df = pd.DataFrame(td_auc_rows)
secondary_metrics_df = pd.concat([risk_auc_by_horizon_df, time_dependent_auc_df], ignore_index=True)

# ------------------------------
# 14) Save coefficient/stability tables
# ------------------------------
cox_coefficients_df = best_cox_model.summary.reset_index().rename(columns={"covariate": "feature_name"})

stability_notes_df = pd.DataFrame([{
    "n_input_features_before_filter": len(cox_feature_names_out),
    "n_input_features_after_filter": len(keep_cols),
    "n_zero_variance_features_dropped": len(dropped_zero_var_cols),
    "dropped_zero_variance_features": "; ".join(dropped_zero_var_cols) if dropped_zero_var_cols else "",
    "best_penalizer": best_penalizer,
    "best_l1_ratio": best_l1_ratio,
    "window_weeks": COX_TUNING_CONFIG["window_weeks"],
}])

# ------------------------------
# 15) Save artifacts
# ------------------------------
model_dir = OUTPUT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "cox_early_window_tuned.joblib"
preprocessor_path = model_dir / "cox_preprocessor.joblib"
tuning_results_path = TABLES_DIR / "table_cox_tuning_results.csv"
predictions_path = TABLES_DIR / "table_cox_tuned_test_predictions.csv"
primary_metrics_path = TABLES_DIR / "table_cox_tuned_primary_metrics.csv"
brier_path = TABLES_DIR / "table_cox_tuned_brier_by_horizon.csv"
secondary_metrics_path = TABLES_DIR / "table_cox_tuned_secondary_metrics.csv"
support_path = TABLES_DIR / "table_cox_tuned_support_by_horizon.csv"
cal_summary_path = TABLES_DIR / "table_cox_tuned_calibration_summary.csv"
cal_bins_path = TABLES_DIR / "table_cox_tuned_calibration_bins_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_cox_tuned_predicted_vs_observed_survival.csv"
coef_path = TABLES_DIR / "table_cox_tuned_coefficients.csv"
stability_path = TABLES_DIR / "table_cox_tuned_stability_notes.csv"
config_path = METADATA_DIR / "cox_tuned_model_config.json"

joblib.dump(best_cox_model, model_path)
joblib.dump(cox_preprocessor, preprocessor_path)

tuning_results_df.to_csv(tuning_results_path, index=False)
test_predictions_df.to_csv(predictions_path, index=False)
primary_metrics_summary.to_csv(primary_metrics_path, index=False)
brier_by_horizon_df.to_csv(brier_path, index=False)
secondary_metrics_df.to_csv(secondary_metrics_path, index=False)
support_by_horizon_df.to_csv(support_path, index=False)
calibration_summary_df.to_csv(cal_summary_path, index=False)
calibration_bins_df.to_csv(cal_bins_path, index=False)
predicted_vs_observed_survival_df.to_csv(pred_vs_obs_path, index=False)
cox_coefficients_df.to_csv(coef_path, index=False)
stability_notes_df.to_csv(stability_path, index=False)

config_to_save = dict(COX_TUNING_CONFIG)
config_to_save["best_candidate"] = {
    "penalizer": best_penalizer,
    "l1_ratio": best_l1_ratio,
}
save_json(config_to_save, config_path)

# ------------------------------
# 16) Output for feedback
# ------------------------------
print("\nTuning results:")
display(tuning_results_df)

print("\nBest candidate:")
display(best_candidate_df)

print("\nPrimary metrics summary:")
display(primary_metrics_summary)

print("\nBrier by horizon:")
display(brier_by_horizon_df)

print("\nCalibration summary by horizon:")
display(calibration_summary_df)

print("\nSecondary metrics:")
display(secondary_metrics_df)

print("\nSupport by horizon:")
display(support_by_horizon_df)

print("\nPredicted vs observed survival:")
display(predicted_vs_observed_survival_df)

print("\nTop Cox coefficients:")
display(cox_coefficients_df.head(20))

print("\nStability notes:")
display(stability_notes_df)

print("\nSaved artifacts:")
print("-", model_path.resolve())
print("-", preprocessor_path.resolve())
print("-", tuning_results_path.resolve())
print("-", predictions_path.resolve())
print("-", primary_metrics_path.resolve())
print("-", brier_path.resolve())
print("-", secondary_metrics_path.resolve())
print("-", support_path.resolve())
print("-", cal_summary_path.resolve())
print("-", cal_bins_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", coef_path.resolve())
print("-", stability_path.resolve())
print("-", config_path.resolve())

## D5 — Treatment for DeepSurv


In [ ]:
# ==============================================================
# D5 — Treatment for DeepSurv
# --------------------------------------------------------------
# Purpose:
#   Prepare the train/test preprocessing pipeline for the revised
#   DeepSurv benchmark, with all learned transformations fitted on
#   training data only.
#
# Methodological note:
#   This DeepSurv treatment aligns with the same early-window
#   comparable benchmark design used by the Cox model.
#
#   Therefore, the DeepSurv model uses:
#     - static covariates
#     - early-window temporal summaries (main window = 4 weeks)
#
#   This preserves comparability between Cox and DeepSurv:
#   both models receive the same conceptual input information,
#   differing only in modeling strategy.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - STATIC_FEATURES
#   - MAIN_ENROLLMENT_WINDOW_FEATURES
#
# Outputs:
#   - X_train_deepsurv, X_test_deepsurv
#   - duration_train_deepsurv, duration_test_deepsurv
#   - event_train_deepsurv, event_test_deepsurv
#   - deepsurv_preprocessor
#   - deepsurv_feature_names_out
#   - audit tables / metadata
# ==============================================================

print("\n" + "=" * 70)
print("D5 — Treatment for DeepSurv")
print("=" * 70)
print("Methodological note: this DeepSurv treatment follows the")
print("same early-window comparable benchmark design used by Cox.")
print("All learned preprocessing transformations are fit on training data only.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "METADATA_DIR", "save_json",
    "STATIC_FEATURES", "MAIN_ENROLLMENT_WINDOW_FEATURES"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------
# 2) Load train/test tables
# ------------------------------
train_df_deepsurv = con.execute("SELECT * FROM enrollment_deepsurv_ready_train").fetchdf()
test_df_deepsurv = con.execute("SELECT * FROM enrollment_deepsurv_ready_test").fetchdf()

# ------------------------------
# 3) Define targets and feature columns
# ------------------------------
duration_col = "duration"
event_col = "event"

categorical_features = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
    "clicks_first_4_weeks",
    "active_weeks_first_4",
    "mean_clicks_first_4_weeks",
]

deepsurv_feature_columns = categorical_features + numeric_features

expected_features = STATIC_FEATURES + MAIN_ENROLLMENT_WINDOW_FEATURES
missing_from_defined = [c for c in expected_features if c not in deepsurv_feature_columns]
if missing_from_defined:
    raise ValueError(f"Missing configured DeepSurv features in treatment spec: {missing_from_defined}")

# ------------------------------
# 4) Split X / targets
# ------------------------------
X_train_raw_deepsurv = train_df_deepsurv[deepsurv_feature_columns].copy()
X_test_raw_deepsurv = test_df_deepsurv[deepsurv_feature_columns].copy()

duration_train_deepsurv = train_df_deepsurv[duration_col].astype(np.float32).to_numpy()
duration_test_deepsurv = test_df_deepsurv[duration_col].astype(np.float32).to_numpy()

event_train_deepsurv = train_df_deepsurv[event_col].astype(np.int32).to_numpy()
event_test_deepsurv = test_df_deepsurv[event_col].astype(np.int32).to_numpy()

# ------------------------------
# 5) Build preprocessing pipelines
# ------------------------------
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

deepsurv_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# ------------------------------
# 6) Fit on train only, transform train/test
# ------------------------------
X_train_deepsurv = deepsurv_preprocessor.fit_transform(X_train_raw_deepsurv).astype(np.float32)
X_test_deepsurv = deepsurv_preprocessor.transform(X_test_raw_deepsurv).astype(np.float32)

deepsurv_feature_names_out = deepsurv_preprocessor.get_feature_names_out().tolist()

# ------------------------------
# 7) Build preprocessing summaries
# ------------------------------
summary_rows = [{
    "model_family": "deepsurv",
    "train_rows": int(X_train_deepsurv.shape[0]),
    "test_rows": int(X_test_deepsurv.shape[0]),
    "n_input_features_raw": int(len(deepsurv_feature_columns)),
    "n_numeric_features": int(len(numeric_features)),
    "n_categorical_features": int(len(categorical_features)),
    "n_features_after_transform": int(X_train_deepsurv.shape[1]),
    "n_events_train": int(event_train_deepsurv.sum()),
    "n_events_test": int(event_test_deepsurv.sum()),
    "mean_duration_train": float(np.mean(duration_train_deepsurv)),
    "mean_duration_test": float(np.mean(duration_test_deepsurv)),
    "numeric_imputation": "median",
    "categorical_imputation": "constant_missing",
    "categorical_encoding": "one_hot_handle_unknown_ignore",
    "numeric_scaling": "standard_scaler_fit_on_train_only",
    "output_dtype": "float32",
    "comparability_note": "same conceptual input design as Cox benchmark",
    "window_weeks": 4,
}]

deepsurv_preprocessing_summary = pd.DataFrame(summary_rows)
deepsurv_feature_names_df = pd.DataFrame({
    "feature_name_out": deepsurv_feature_names_out
})

# ------------------------------
# 8) Save metadata and audit tables
# ------------------------------
summary_path = TABLES_DIR / "table_deepsurv_preprocessing_summary.csv"
feature_names_path = TABLES_DIR / "table_deepsurv_feature_names_out.csv"
config_path = METADATA_DIR / "deepsurv_preprocessing_config.json"

deepsurv_preprocessing_summary.to_csv(summary_path, index=False)
deepsurv_feature_names_df.to_csv(feature_names_path, index=False)

deepsurv_preprocessing_config = {
    "model_family": "deepsurv",
    "duration_column": duration_col,
    "event_column": event_col,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "raw_feature_columns": deepsurv_feature_columns,
    "numeric_imputation": "median",
    "categorical_imputation": "constant_missing",
    "categorical_encoding": "one_hot_handle_unknown_ignore",
    "numeric_scaling": "standard_scaler_fit_on_train_only",
    "output_dtype": "float32",
    "comparability_note": "same conceptual input design as Cox benchmark",
    "window_weeks": 4,
    "n_features_after_transform": int(X_train_deepsurv.shape[1]),
}

save_json(deepsurv_preprocessing_config, config_path)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nDeepSurv preprocessing summary:")
display(deepsurv_preprocessing_summary)

print("\nFirst transformed feature names:")
display(deepsurv_feature_names_df.head(40))

print("\nArray shapes:")
print("X_train_deepsurv     :", X_train_deepsurv.shape, X_train_deepsurv.dtype)
print("X_test_deepsurv      :", X_test_deepsurv.shape, X_test_deepsurv.dtype)
print("duration_train       :", duration_train_deepsurv.shape, duration_train_deepsurv.dtype)
print("duration_test        :", duration_test_deepsurv.shape, duration_test_deepsurv.dtype)
print("event_train          :", event_train_deepsurv.shape, event_train_deepsurv.dtype)
print("event_test           :", event_test_deepsurv.shape, event_test_deepsurv.dtype)

print("\nSaved:")
print("-", summary_path.resolve())
print("-", feature_names_path.resolve())
print("-", config_path.resolve())

### D5.1 — Performance-Oriented DeepSurv Tuning


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D5.1 — Performance-Oriented DeepSurv Tuning
# --------------------------------------------------------------
# Purpose:
#   Perform a controlled hyperparameter search for DeepSurv and
#   evaluate the best candidate under the benchmark protocol.
#
# Methodological note:
#   This step provides the official stronger performance-oriented DeepSurv benchmark
#   under the same information constraints:
#     - static covariates
#     - early-window temporal summaries (main window = 4 weeks)
#
#   The search is intentionally limited and reproducible.
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - X_train_deepsurv, X_test_deepsurv
#   - duration_train_deepsurv, duration_test_deepsurv
#   - event_train_deepsurv, event_test_deepsurv
#   - train_df_deepsurv, test_df_deepsurv
#   - deepsurv_preprocessor
#   - deepsurv_feature_names_out
#   - HORIZONS_WEEKS
#   - CALIBRATION_BINS
#   - RANDOM_SEED
#
# Outputs:
#   - tuned DeepSurv model
#   - tuning results
#   - training history for best model
#   - test predictions
#   - benchmark metrics and diagnostics
# ==============================================================

print("\n" + "=" * 70)
print("D5.1 — Performance-Oriented DeepSurv Tuning")
print("=" * 70)
print("Methodological note: this step performs controlled DeepSurv tuning.")
print("This step is the official stronger DeepSurv benchmark under the")
print("performance-oriented DeepSurv benchmark under the same evaluation protocol.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json",
    "X_train_deepsurv", "X_test_deepsurv",
    "duration_train_deepsurv", "duration_test_deepsurv",
    "event_train_deepsurv", "event_test_deepsurv",
    "train_df_deepsurv", "test_df_deepsurv",
    "deepsurv_preprocessor", "deepsurv_feature_names_out",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun the required prior steps first."
    )

import copy
import itertools
import joblib
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

try:
    from pycox.models import CoxPH
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.util import Surv
    SKSURV_AVAILABLE = True
except Exception:
    SKSURV_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P25.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"

try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Reproducibility
# ------------------------------
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ------------------------------
# 4) Tuning config
# ------------------------------
DEEPSURV_TUNING_CONFIG = {
    "model_name": "deepsurv_tuned",
    "window_weeks": 4,
    "hidden_dims_grid": [[32, 16], [64, 32], [128, 64]],
    "dropout_grid": [0.1, 0.3],
    "learning_rate_grid": [5e-4, 1e-3],
    "weight_decay_grid": [1e-5, 1e-4],
    "batch_norm": True,
    "output_bias": False,
    "batch_size": 256,
    "epochs": 100,
    "patience": 10,
    "validation_fraction": 0.20,
    "selection_metric": "best_val_loss",
    "benchmark_positioning_note": (
        "Performance-oriented tuned DeepSurv benchmark under the same early-window comparable input design."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
}

# ------------------------------
# 5) Train/validation split at enrollment level
# ------------------------------
n_train = X_train_deepsurv.shape[0]
perm = np.random.RandomState(RANDOM_SEED).permutation(n_train)
val_size = int(np.floor(DEEPSURV_TUNING_CONFIG["validation_fraction"] * n_train))
val_idx = perm[:val_size]
subtrain_idx = perm[val_size:]

X_subtrain = X_train_deepsurv[subtrain_idx]
X_val = X_train_deepsurv[val_idx]

dur_subtrain = duration_train_deepsurv[subtrain_idx]
dur_val = duration_train_deepsurv[val_idx]

evt_subtrain = event_train_deepsurv[subtrain_idx]
evt_val = event_train_deepsurv[val_idx]

y_subtrain = (dur_subtrain.astype(np.float32), evt_subtrain.astype(np.float32))
y_val = (dur_val.astype(np.float32), evt_val.astype(np.float32))
y_train_full = (
    duration_train_deepsurv.astype(np.float32),
    event_train_deepsurv.astype(np.float32),
)

# ------------------------------
# 6) Run tuning grid
# ------------------------------
candidate_rows = []
candidate_artifacts = []

param_grid = list(itertools.product(
    DEEPSURV_TUNING_CONFIG["hidden_dims_grid"],
    DEEPSURV_TUNING_CONFIG["dropout_grid"],
    DEEPSURV_TUNING_CONFIG["learning_rate_grid"],
    DEEPSURV_TUNING_CONFIG["weight_decay_grid"],
))

in_features = X_train_deepsurv.shape[1]

for candidate_id, (hidden_dims, dropout, lr, wd) in enumerate(param_grid, start=1):
    torch.manual_seed(RANDOM_SEED + candidate_id)
    np.random.seed(RANDOM_SEED + candidate_id)

    net = tt.practical.MLPVanilla(
        in_features=in_features,
        num_nodes=hidden_dims,
        out_features=1,
        batch_norm=DEEPSURV_TUNING_CONFIG["batch_norm"],
        dropout=dropout,
        output_bias=DEEPSURV_TUNING_CONFIG["output_bias"],
    )

    model = CoxPH(net, tt.optim.AdamW)
    model.optimizer.set_lr(lr)

    callbacks = [tt.callbacks.EarlyStopping(patience=DEEPSURV_TUNING_CONFIG["patience"])]

    log = model.fit(
        X_subtrain,
        y_subtrain,
        batch_size=DEEPSURV_TUNING_CONFIG["batch_size"],
        epochs=DEEPSURV_TUNING_CONFIG["epochs"],
        callbacks=callbacks,
        verbose=False,
        val_data=(X_val, y_val),
    )

    history_df = log.to_pandas().reset_index().rename(columns={"index": "epoch"})
    if "train_loss" not in history_df.columns and "loss" in history_df.columns:
        history_df = history_df.rename(columns={"loss": "train_loss"})
    history_df["epoch"] = history_df["epoch"] + 1

    if "val_loss" in history_df.columns:
        best_row_idx = history_df["val_loss"].idxmin()
        best_val_loss = float(history_df.loc[best_row_idx, "val_loss"])
        best_epoch = int(history_df.loc[best_row_idx, "epoch"])
    else:
        best_val_loss = np.nan
        best_epoch = int(history_df["epoch"].max())

    candidate_rows.append({
        "candidate_id": candidate_id,
        "hidden_dims": str(hidden_dims),
        "dropout": dropout,
        "learning_rate": lr,
        "weight_decay": wd,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
    })

    candidate_artifacts.append({
        "candidate_id": candidate_id,
        "hidden_dims": hidden_dims,
        "dropout": dropout,
        "learning_rate": lr,
        "weight_decay": wd,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "history_df": history_df.copy(),
    })

tuning_results_df = pd.DataFrame(candidate_rows).sort_values(
    by=["best_val_loss", "candidate_id"],
    ascending=[True, True]
).reset_index(drop=True)

best_candidate = candidate_artifacts[
    int(tuning_results_df.iloc[0]["candidate_id"]) - 1
]

best_candidate_df = pd.DataFrame([{
    "candidate_id": best_candidate["candidate_id"],
    "hidden_dims": str(best_candidate["hidden_dims"]),
    "dropout": best_candidate["dropout"],
    "learning_rate": best_candidate["learning_rate"],
    "weight_decay": best_candidate["weight_decay"],
    "best_val_loss": best_candidate["best_val_loss"],
    "best_epoch": best_candidate["best_epoch"],
}])

# ------------------------------
# 7) Refit best model on full train
# ------------------------------
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

net_final = tt.practical.MLPVanilla(
    in_features=in_features,
    num_nodes=best_candidate["hidden_dims"],
    out_features=1,
    batch_norm=DEEPSURV_TUNING_CONFIG["batch_norm"],
    dropout=best_candidate["dropout"],
    output_bias=DEEPSURV_TUNING_CONFIG["output_bias"],
)

model_final = CoxPH(net_final, tt.optim.AdamW)
model_final.optimizer.set_lr(best_candidate["learning_rate"])

_ = model_final.fit(
    X_train_deepsurv,
    y_train_full,
    batch_size=DEEPSURV_TUNING_CONFIG["batch_size"],
    epochs=int(best_candidate["best_epoch"]),
    verbose=False,
)

model_final.compute_baseline_hazards()

best_history_df = best_candidate["history_df"].copy()

# ------------------------------
# 8) Predict survival on test
# ------------------------------
surv_df = model_final.predict_surv_df(X_test_deepsurv)
surv_df.columns = test_df_deepsurv["enrollment_id"].tolist()
surv_df.columns.name = "enrollment_id"
surv_df.index = surv_df.index.astype(float)

def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]

truth_test = pd.DataFrame({
    "enrollment_id": test_df_deepsurv["enrollment_id"].values,
    "event": event_test_deepsurv.astype(int),
    "duration": duration_test_deepsurv.astype(int),
})

truth_train = pd.DataFrame({
    "enrollment_id": train_df_deepsurv["enrollment_id"].values,
    "event": event_train_deepsurv.astype(int),
    "duration": duration_train_deepsurv.astype(int),
})

durations_test = truth_test["duration"].to_numpy(dtype=int)
events_test = truth_test["event"].to_numpy(dtype=int)

# ------------------------------
# 9) Primary survival metrics
# ------------------------------
eval_surv = EvalSurv(
    surv=surv_df,
    durations=durations_test,
    events=events_test,
    censor_surv="km",
)

primary_metrics_rows = []

try:
    c_index = float(eval_surv.concordance_td())
    c_index_note = "pycox_concordance_td"
except Exception as e:
    c_index = np.nan
    c_index_note = f"failed: {str(e)}"

primary_metrics_rows.append({
    "metric_name": "c_index",
    "metric_category": "co_primary",
    "metric_value": c_index,
    "notes": c_index_note,
})

try:
    max_requested_horizon = int(max(HORIZONS_WEEKS))
    ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
    ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    ibs_note = "pycox_integrated_brier_score"
except Exception as e:
    try:
        max_requested_horizon = int(max(HORIZONS_WEEKS))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        dense_brier = eval_surv.brier_score(ibs_grid)
        ibs_value = float(np.trapz(dense_brier.values.astype(float), ibs_grid) / (ibs_grid[-1] - ibs_grid[0]))
        ibs_note = f"fallback_np_trapz_after_pycox_failure: {str(e)}"
    except Exception as e2:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)} | fallback_failed: {str(e2)}"

primary_metrics_rows.insert(0, {
    "metric_name": "ibs",
    "metric_category": "primary",
    "metric_value": ibs_value,
    "notes": ibs_note,
})

primary_metrics_summary = pd.DataFrame(primary_metrics_rows)

# ------------------------------
# 10) Brier by horizon
# ------------------------------
try:
    brier_h = eval_surv.brier_score(np.array(HORIZONS_WEEKS, dtype=int))
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": list(brier_h.index.astype(int)),
        "metric_name": ["brier_at_horizon"] * len(brier_h.index),
        "metric_category": ["primary"] * len(brier_h.index),
        "metric_value": list(brier_h.values.astype(float)),
        "notes": ["pycox_brier_score"] * len(brier_h.index),
    })
except Exception as e:
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": HORIZONS_WEEKS,
        "metric_name": ["brier_at_horizon"] * len(HORIZONS_WEEKS),
        "metric_category": ["primary"] * len(HORIZONS_WEEKS),
        "metric_value": [np.nan] * len(HORIZONS_WEEKS),
        "notes": [f"failed: {str(e)}"] * len(HORIZONS_WEEKS),
    })

# ------------------------------
# 11) Support, calibration, risk AUC, predicted vs observed
# ------------------------------
support_rows = []
risk_auc_rows = []
calibration_rows = []
pred_vs_obs_rows = []
test_pred_rows = []

for h in HORIZONS_WEEKS:
    pred_surv_h = get_surv_at_horizon(surv_df, h)
    pred_risk_h = 1.0 - pred_surv_h

    eval_df = truth_test.copy()
    eval_df["pred_survival_h"] = eval_df["enrollment_id"].map(pred_surv_h.to_dict())
    eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

    eval_df["is_evaluable_at_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
        (eval_df["duration"] >= h)
    ).astype(int)

    eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
    eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
    eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

    support_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
        "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df["observed_event_by_h"].nunique() >= 2:
        risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
    else:
        risk_auc = np.nan

    risk_auc_rows.append({
        "horizon_week": h,
        "metric_name": "risk_auc_at_horizon",
        "metric_category": "secondary",
        "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        "notes": "roc_auc_on_evaluable_subset",
    })

    pred_vs_obs_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df.shape[0] > 0:
        ranked = eval_df["pred_risk_h"].rank(method="first")
        n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))

        eval_df["calibration_bin"] = pd.qcut(
            ranked,
            q=n_bins_eff,
            labels=False,
            duplicates="drop"
        )

        calib_tab = (
            eval_df.groupby("calibration_bin")
            .agg(
                n=("enrollment_id", "count"),
                mean_predicted_risk=("pred_risk_h", "mean"),
                observed_event_rate=("observed_event_by_h", "mean"),
            )
            .reset_index()
        )
        calib_tab["horizon_week"] = h
        calib_tab["abs_calibration_gap"] = (
            calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
        ).abs()

        for _, row in calib_tab.iterrows():
            calibration_rows.append({
                "horizon_week": int(row["horizon_week"]),
                "calibration_bin": int(row["calibration_bin"]),
                "n": int(row["n"]),
                "mean_predicted_risk": float(row["mean_predicted_risk"]),
                "observed_event_rate": float(row["observed_event_rate"]),
                "abs_calibration_gap": float(row["abs_calibration_gap"]),
            })

    tmp_preds = truth_test[["enrollment_id", "event", "duration"]].copy()
    tmp_preds["horizon_week"] = h
    tmp_preds["pred_survival_h"] = tmp_preds["enrollment_id"].map(pred_surv_h.to_dict())
    tmp_preds["pred_risk_h"] = tmp_preds["enrollment_id"].map(pred_risk_h.to_dict())
    test_pred_rows.append(tmp_preds)

support_by_horizon_df = pd.DataFrame(support_rows)
risk_auc_by_horizon_df = pd.DataFrame(risk_auc_rows)
predicted_vs_observed_survival_df = pd.DataFrame(pred_vs_obs_rows)
calibration_bins_df = pd.DataFrame(calibration_rows)
test_predictions_df = pd.concat(test_pred_rows, ignore_index=True)

calibration_summary_rows = []
for h in HORIZONS_WEEKS:
    h_df = calibration_bins_df[calibration_bins_df["horizon_week"] == h].copy()
    if h_df.shape[0] > 0:
        ece_like = np.average(h_df["abs_calibration_gap"], weights=h_df["n"])
    else:
        ece_like = np.nan
    calibration_summary_rows.append({
        "horizon_week": h,
        "metric_name": "calibration_at_horizon",
        "metric_category": "primary",
        "metric_value": float(ece_like) if pd.notna(ece_like) else np.nan,
        "notes": "Weighted absolute calibration gap across bins",
    })

calibration_summary_df = pd.DataFrame(calibration_summary_rows)

# ------------------------------
# 12) Optional time-dependent AUC
# ------------------------------
td_auc_rows = []
if SKSURV_AVAILABLE:
    try:
        y_train_surv = Surv.from_arrays(
            event=truth_train["event"].astype(bool).to_numpy(),
            time=truth_train["duration"].astype(float).to_numpy(),
        )
        y_test_surv = Surv.from_arrays(
            event=truth_test["event"].astype(bool).to_numpy(),
            time=truth_test["duration"].astype(float).to_numpy(),
        )

        for h in HORIZONS_WEEKS:
            pred_risk_h = 1.0 - get_surv_at_horizon(surv_df, h)
            auc_vals, mean_auc = cumulative_dynamic_auc(
                y_train_surv,
                y_test_surv,
                pred_risk_h.to_numpy(),
                np.array([h], dtype=float)
            )
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": float(auc_vals[0]),
                "notes": "sksurv cumulative_dynamic_auc",
            })
    except Exception as e:
        for h in HORIZONS_WEEKS:
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": np.nan,
                "notes": f"sksurv_available_but_failed: {str(e)}",
            })
else:
    for h in HORIZONS_WEEKS:
        td_auc_rows.append({
            "horizon_week": h,
            "metric_name": "time_dependent_auc",
            "metric_category": "secondary",
            "metric_value": np.nan,
            "notes": "sksurv_unavailable",
        })

time_dependent_auc_df = pd.DataFrame(td_auc_rows)
secondary_metrics_df = pd.concat([risk_auc_by_horizon_df, time_dependent_auc_df], ignore_index=True)

# ------------------------------
# 13) Row-level diagnostics (auxiliary proxy only)
# ------------------------------
test_event_times = truth_test["duration"].astype(int).tolist()
row_pred_surv = []
for eid, h in zip(truth_test["enrollment_id"], test_event_times):
    row_pred_surv.append(float(get_surv_at_horizon(surv_df, h).get(eid, np.nan)))
row_pred_surv = np.asarray(row_pred_surv)
row_pred_risk = 1.0 - row_pred_surv

valid_mask = np.isfinite(row_pred_risk)
y_true_row = truth_test.loc[valid_mask, "event"].to_numpy().astype(int)
y_prob_row = np.clip(row_pred_risk[valid_mask], 1e-8, 1 - 1e-8)

if len(np.unique(y_true_row)) >= 2:
    row_roc_auc = roc_auc_score(y_true_row, y_prob_row)
    row_pr_auc = average_precision_score(y_true_row, y_prob_row)
else:
    row_roc_auc = np.nan
    row_pr_auc = np.nan

row_log_loss = log_loss(y_true_row, y_prob_row, labels=[0, 1]) if len(y_true_row) > 0 else np.nan
row_brier = np.mean((y_prob_row - y_true_row) ** 2) if len(y_true_row) > 0 else np.nan

row_diagnostics_df = pd.DataFrame([{
    "model_name": "deepsurv_tuned",
    "row_level_roc_auc": float(row_roc_auc) if pd.notna(row_roc_auc) else np.nan,
    "row_level_pr_auc": float(row_pr_auc) if pd.notna(row_pr_auc) else np.nan,
    "row_level_log_loss": float(row_log_loss) if pd.notna(row_log_loss) else np.nan,
    "row_level_brier": float(row_brier) if pd.notna(row_brier) else np.nan,
    "diagnostic_note": "auxiliary proxy only; not a primary cross-family benchmark metric",
}])

# ------------------------------
# 14) Save artifacts
# ------------------------------
model_dir = OUTPUT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "deepsurv_tuned.pt"
preprocessor_path = model_dir / "deepsurv_preprocessor.joblib"
tuning_results_path = TABLES_DIR / "table_deepsurv_tuning_results.csv"
training_history_path = TABLES_DIR / "table_deepsurv_tuned_training_history.csv"
predictions_path = TABLES_DIR / "table_deepsurv_tuned_test_predictions.csv"
primary_metrics_path = TABLES_DIR / "table_deepsurv_tuned_primary_metrics.csv"
brier_path = TABLES_DIR / "table_deepsurv_tuned_brier_by_horizon.csv"
secondary_metrics_path = TABLES_DIR / "table_deepsurv_tuned_secondary_metrics.csv"
row_diag_path = TABLES_DIR / "table_deepsurv_tuned_row_diagnostics.csv"
support_path = TABLES_DIR / "table_deepsurv_tuned_support_by_horizon.csv"
cal_summary_path = TABLES_DIR / "table_deepsurv_tuned_calibration_summary.csv"
cal_bins_path = TABLES_DIR / "table_deepsurv_tuned_calibration_bins_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_deepsurv_tuned_predicted_vs_observed_survival.csv"
config_path = METADATA_DIR / "deepsurv_tuned_model_config.json"

torch.save(model_final.net.state_dict(), model_path)
joblib.dump(deepsurv_preprocessor, preprocessor_path)

tuning_results_df.to_csv(tuning_results_path, index=False)
best_history_df.to_csv(training_history_path, index=False)
test_predictions_df.to_csv(predictions_path, index=False)
primary_metrics_summary.to_csv(primary_metrics_path, index=False)
brier_by_horizon_df.to_csv(brier_path, index=False)
secondary_metrics_df.to_csv(secondary_metrics_path, index=False)
row_diagnostics_df.to_csv(row_diag_path, index=False)
support_by_horizon_df.to_csv(support_path, index=False)
calibration_summary_df.to_csv(cal_summary_path, index=False)
calibration_bins_df.to_csv(cal_bins_path, index=False)
predicted_vs_observed_survival_df.to_csv(pred_vs_obs_path, index=False)

tuned_config_to_save = copy.deepcopy(DEEPSURV_TUNING_CONFIG)
tuned_config_to_save["best_candidate"] = {
    "candidate_id": int(best_candidate["candidate_id"]),
    "hidden_dims": best_candidate["hidden_dims"],
    "dropout": float(best_candidate["dropout"]),
    "learning_rate": float(best_candidate["learning_rate"]),
    "weight_decay": float(best_candidate["weight_decay"]),
    "best_val_loss": float(best_candidate["best_val_loss"]),
    "best_epoch": int(best_candidate["best_epoch"]),
}
save_json(tuned_config_to_save, config_path)

# ------------------------------
# 15) Output for feedback
# ------------------------------
print("\nTuning results:")
display(tuning_results_df)

print("\nBest candidate:")
display(best_candidate_df)

print("\nPrimary metrics summary:")
display(primary_metrics_summary)

print("\nBrier by horizon:")
display(brier_by_horizon_df)

print("\nCalibration summary by horizon:")
display(calibration_summary_df)

print("\nSecondary metrics:")
display(secondary_metrics_df)

print("\nRow-level diagnostics:")
display(row_diagnostics_df)

print("\nSupport by horizon:")
display(support_by_horizon_df)

print("\nPredicted vs observed survival:")
display(predicted_vs_observed_survival_df)

print("\nSaved artifacts:")
print("-", model_path.resolve())
print("-", preprocessor_path.resolve())
print("-", tuning_results_path.resolve())
print("-", training_history_path.resolve())
print("-", predictions_path.resolve())
print("-", primary_metrics_path.resolve())
print("-", brier_path.resolve())
print("-", secondary_metrics_path.resolve())
print("-", row_diag_path.resolve())
print("-", support_path.resolve())
print("-", cal_summary_path.resolve())
print("-", cal_bins_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", config_path.resolve())

## D6 — Consolidated Benchmark Comparison and Leaderboard


### Funcao: read_metric_value

Definicao isolada de read_metric_value para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D6 — Consolidated Benchmark Comparison and Leaderboard
# --------------------------------------------------------------
# Purpose:
#   Consolidate the benchmark outputs from the selected official
#   model variants and build the main leaderboard tables.
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - tables generated by the official selected variants
#
# Official variants consolidated here:
#   - linear_tuned
#   - neural_tuned
#   - cox_tuned
#   - deepsurv_tuned
#
# Outputs:
#   - table_benchmark_leaderboard_main.csv
#   - table_benchmark_leaderboard_by_horizon.csv
#   - table_benchmark_selected_sources.csv
# ==============================================================

print("\n" + "=" * 70)
print("D6 — Consolidated Benchmark Comparison and Leaderboard")
print("=" * 70)

print("Methodological note: this step consolidates benchmark outputs only.")
print("No model is trained here.")
print("This version uses the official selected model variants explicitly.")

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------
# 1) Basic paths
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

OUT_TABLES = TABLES_DIR
OUT_TABLES.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Official source registry
# ------------------------------
MODEL_SOURCES = {
    "linear_tuned": {
        "display_name": "Linear Discrete-Time (Tuned)",
        "family": "discrete_time_linear",
        "primary": OUT_TABLES / "table_linear_tuned_primary_metrics.csv",
        "brier": OUT_TABLES / "table_linear_tuned_brier_by_horizon.csv",
        "calibration": OUT_TABLES / "table_linear_tuned_calibration_summary.csv",
        "secondary": OUT_TABLES / "table_linear_tuned_secondary_metrics.csv",
        "support": OUT_TABLES / "table_linear_tuned_support_by_horizon.csv",
        "pred_vs_obs": OUT_TABLES / "table_linear_tuned_predicted_vs_observed_survival.csv",
    },
    "neural_tuned": {
        "display_name": "Neural Discrete-Time (Tuned)",
        "family": "discrete_time_neural",
        "primary": OUT_TABLES / "table_neural_tuned_primary_metrics.csv",
        "brier": OUT_TABLES / "table_neural_tuned_brier_by_horizon.csv",
        "calibration": OUT_TABLES / "table_neural_tuned_calibration_summary.csv",
        "secondary": OUT_TABLES / "table_neural_tuned_secondary_metrics.csv",
        "support": OUT_TABLES / "table_neural_tuned_support_by_horizon.csv",
        "pred_vs_obs": OUT_TABLES / "table_neural_tuned_predicted_vs_observed_survival.csv",
    },
    "cox_tuned": {
        "display_name": "Cox (Tuned)",
        "family": "continuous_time_cox",
        "primary": OUT_TABLES / "table_cox_tuned_primary_metrics.csv",
        "brier": OUT_TABLES / "table_cox_tuned_brier_by_horizon.csv",
        "calibration": OUT_TABLES / "table_cox_tuned_calibration_summary.csv",
        "secondary": OUT_TABLES / "table_cox_tuned_secondary_metrics.csv",
        "support": OUT_TABLES / "table_cox_tuned_support_by_horizon.csv",
        "pred_vs_obs": OUT_TABLES / "table_cox_tuned_predicted_vs_observed_survival.csv",
    },
    "deepsurv_tuned": {
        "display_name": "DeepSurv (Tuned)",
        "family": "continuous_time_neural",
        "primary": OUT_TABLES / "table_deepsurv_tuned_primary_metrics.csv",
        "brier": OUT_TABLES / "table_deepsurv_tuned_brier_by_horizon.csv",
        "calibration": OUT_TABLES / "table_deepsurv_tuned_calibration_summary.csv",
        "secondary": OUT_TABLES / "table_deepsurv_tuned_secondary_metrics.csv",
        "support": OUT_TABLES / "table_deepsurv_tuned_support_by_horizon.csv",
        "pred_vs_obs": OUT_TABLES / "table_deepsurv_tuned_predicted_vs_observed_survival.csv",
    },
}

# ------------------------------
# 3) Validate files
# ------------------------------
missing_files = []
selected_source_rows = []

for model_key, cfg in MODEL_SOURCES.items():
    for artifact_name, path in cfg.items():
        if artifact_name in ["display_name", "family"]:
            continue
        exists = path.exists()
        selected_source_rows.append({
            "model_key": model_key,
            "display_name": cfg["display_name"],
            "family": cfg["family"],
            "artifact_name": artifact_name,
            "path": str(path),
            "exists": exists,
        })
        if not exists:
            missing_files.append(str(path))

selected_sources_df = pd.DataFrame(selected_source_rows)

if missing_files:
    raise FileNotFoundError(
        "Some benchmark result files are missing for the selected official variants:\n- "
        + "\n- ".join(missing_files)
    )

# ------------------------------
# 4) Read helper functions
# ------------------------------
def read_metric_value(primary_path: Path, metric_name: str):
    df = pd.read_csv(primary_path)
    row = df.loc[df["metric_name"] == metric_name]
    if row.empty:
        return np.nan
    return float(row["metric_value"].iloc[0])

# ------------------------------
# 5) Main leaderboard
# ------------------------------
leader_rows = []

for model_key, cfg in MODEL_SOURCES.items():
    ibs = read_metric_value(cfg["primary"], "ibs")
    c_index = read_metric_value(cfg["primary"], "c_index")

    calib_df = pd.read_csv(cfg["calibration"])
    mean_calibration_gap = float(calib_df["metric_value"].mean()) if not calib_df.empty else np.nan

    leader_rows.append({
        "model_key": model_key,
        "display_name": cfg["display_name"],
        "family": cfg["family"],
        "ibs": ibs,
        "c_index": c_index,
        "mean_calibration_gap": mean_calibration_gap,
    })

leaderboard_main_df = pd.DataFrame(leader_rows)

leaderboard_main_df["rank_ibs"] = leaderboard_main_df["ibs"].rank(method="min", ascending=True)
leaderboard_main_df["rank_c_index"] = leaderboard_main_df["c_index"].rank(method="min", ascending=False)
leaderboard_main_df["rank_mean_calibration_gap"] = leaderboard_main_df["mean_calibration_gap"].rank(method="min", ascending=True)

leaderboard_main_df = leaderboard_main_df.sort_values(
    ["rank_ibs", "rank_mean_calibration_gap", "rank_c_index", "display_name"]
).reset_index(drop=True)

# ------------------------------
# 6) Horizon leaderboard
# ------------------------------
horizon_rows = []

for model_key, cfg in MODEL_SOURCES.items():
    brier_df = pd.read_csv(cfg["brier"]).copy()
    calib_df = pd.read_csv(cfg["calibration"]).copy()
    support_df = pd.read_csv(cfg["support"]).copy()
    pred_obs_df = pd.read_csv(cfg["pred_vs_obs"]).copy()

    brier_df = brier_df.rename(columns={"metric_value": "brier_value"})
    calib_df = calib_df.rename(columns={"metric_value": "calibration_gap"})
    pred_obs_df = pred_obs_df.rename(columns={"abs_gap": "predicted_vs_observed_abs_gap"})

    merged = (
        brier_df[["horizon_week", "brier_value"]]
        .merge(calib_df[["horizon_week", "calibration_gap"]], on="horizon_week", how="outer")
        .merge(
            support_df[["horizon_week", "n_evaluable_enrollments", "n_events_by_horizon", "event_rate_by_horizon"]],
            on="horizon_week",
            how="left"
        )
        .merge(
            pred_obs_df[["horizon_week", "predicted_vs_observed_abs_gap"]],
            on="horizon_week",
            how="left"
        )
    )

    merged["model_key"] = model_key
    merged["display_name"] = cfg["display_name"]
    merged["family"] = cfg["family"]

    horizon_rows.append(merged)

leaderboard_horizon_df = pd.concat(horizon_rows, ignore_index=True)

leaderboard_horizon_df = leaderboard_horizon_df[
    [
        "model_key",
        "display_name",
        "family",
        "horizon_week",
        "brier_value",
        "calibration_gap",
        "predicted_vs_observed_abs_gap",
        "n_evaluable_enrollments",
        "n_events_by_horizon",
        "event_rate_by_horizon",
    ]
].sort_values(["horizon_week", "display_name"]).reset_index(drop=True)

# ------------------------------
# 7) Save outputs
# ------------------------------
selected_sources_path = OUT_TABLES / "table_benchmark_selected_sources.csv"
leaderboard_main_path = OUT_TABLES / "table_benchmark_leaderboard_main.csv"
leaderboard_horizon_path = OUT_TABLES / "table_benchmark_leaderboard_by_horizon.csv"

selected_sources_df.to_csv(selected_sources_path, index=False)
leaderboard_main_df.to_csv(leaderboard_main_path, index=False)
leaderboard_horizon_df.to_csv(leaderboard_horizon_path, index=False)

# ------------------------------
# 8) Output
# ------------------------------
print("\nSelected official benchmark sources:")
display(selected_sources_df)

print("\nMain leaderboard:")
display(leaderboard_main_df)

print("\nLeaderboard by horizon:")
display(leaderboard_horizon_df)

print("\nSaved:")
print("-", selected_sources_path.resolve())
print("-", leaderboard_main_path.resolve())
print("-", leaderboard_horizon_path.resolve())

### D6.1 — Define Expanded Survival Calibration Diagnostics


### Funcao: _safe_read_csv

Definicao isolada de _safe_read_csv para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D6.1 — Define Expanded Survival Calibration Diagnostics
# --------------------------------------------------------------
# Purpose:
#   Formalize the expanded calibration protocol used by the benchmark,
#   including:
#     - eligible models
#     - official benchmark horizons
#     - primary calibration metric
#     - required strengthening diagnostics
#     - support / comparability rules
#     - required downstream outputs for P26.5 and P26.6
#
# Design principle:
#   This is a protocol-definition step, not a metric-recomputation step.
#   It creates the canonical specification that later cells must follow.
#
# Notes:
#   - The benchmark already uses shared horizons 10/20/30.
#   - The benchmark already uses a horizon-specific weighted absolute
#     calibration gap across bins as the main calibration metric.
#   - The strengthened layer now explicitly requires approximate
#     calibration intercept and slope by horizon.
#   - An ICI-style metric is kept as optional / non-blocking unless
#     row-level predictions are available in a stable canonical form.
# ==============================================================

print("=" * 70)
print("D6.1 — Define Expanded Survival Calibration Diagnostics")
print("=" * 70)

# ------------------------------
# 0) Dependency checks
# ------------------------------
required_names = ["TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import json
import pandas as pd
from datetime import datetime

# Optional objects
duckdb_available = "con" in globals()

# ------------------------------
# 1) Benchmark horizons
# ------------------------------
# Prefer an existing notebook/global convention if present.
if "BENCHMARK_HORIZONS" in globals():
    benchmark_horizons = [int(x) for x in BENCHMARK_HORIZONS]
elif "HORIZON_WEEKS" in globals():
    benchmark_horizons = [int(x) for x in HORIZON_WEEKS]
else:
    benchmark_horizons = [10, 20, 30]

benchmark_horizons = sorted(list(dict.fromkeys(benchmark_horizons)))

# ------------------------------
# 2) Discover currently available calibration artifacts
# ------------------------------
table_dir = Path(TABLES_DIR)
metadata_dir = Path(METADATA_DIR)

artifact_candidates = {
    "benchmark_calibration_by_horizon_wide": table_dir / "table_benchmark_calibration_by_horizon_wide.csv",
    "all_tuned_calibration_audit": table_dir / "table_p26_5_all_tuned_calibration_audit.csv",
    "paper_summary_all_tuned_models": table_dir / "table_p26_5_paper_summary_all_tuned_models.csv",
    "reliability_bins_all_tuned_models": table_dir / "table_p26_5_reliability_bins_all_tuned_models.csv",
    "reliability_bins_h20_all_tuned_models": table_dir / "table_p26_5_reliability_bins_h20_all_tuned_models.csv",
    "p11_1_slope_intercept_main": table_dir / "table_p11_1_calibration_slope_intercept_by_horizon.csv",
    "p11_1_slope_intercept_summary": table_dir / "table_p11_1_calibration_slope_intercept_summary.csv",
    "p11_2_unified_strengthening": table_dir / "table_p11_2_unified_calibration_strengthening.csv",
}

artifact_inventory_rows = []
for artifact_name, artifact_path in artifact_candidates.items():
    artifact_inventory_rows.append({
        "artifact_name": artifact_name,
        "exists": artifact_path.exists(),
        "path": artifact_path.as_posix(),
    })

artifact_inventory_df = pd.DataFrame(artifact_inventory_rows)

# ------------------------------
# 3) Infer eligible model families, if possible
# ------------------------------
def _safe_read_csv(path_obj):
    try:
        return pd.read_csv(path_obj)
    except Exception:
        return None

eligible_models_inferred = []

# First preference: existing all-tuned audit
p26_5_audit_path = artifact_candidates["all_tuned_calibration_audit"]
if p26_5_audit_path.exists():
    df_tmp = _safe_read_csv(p26_5_audit_path)
    if df_tmp is not None:
        for candidate_col in ["model_label", "model_name", "model_id", "family_label", "family_name"]:
            if candidate_col in df_tmp.columns:
                vals = (
                    df_tmp[candidate_col]
                    .dropna()
                    .astype(str)
                    .str.strip()
                    .tolist()
                )
                eligible_models_inferred = sorted(list(dict.fromkeys([v for v in vals if v])))
                if eligible_models_inferred:
                    break

# Second preference: unified strengthening table
if not eligible_models_inferred:
    p11_2_path = artifact_candidates["p11_2_unified_strengthening"]
    if p11_2_path.exists():
        df_tmp = _safe_read_csv(p11_2_path)
        if df_tmp is not None:
            for candidate_col in ["model_label", "model_name", "model_id", "family_label", "family_name"]:
                if candidate_col in df_tmp.columns:
                    vals = (
                        df_tmp[candidate_col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .tolist()
                    )
                    eligible_models_inferred = sorted(list(dict.fromkeys([v for v in vals if v])))
                    if eligible_models_inferred:
                        break

# Conservative fallback if nothing canonical is found yet
if not eligible_models_inferred:
    eligible_models_inferred = [
        "cox_tuned",
        "deepsurv_tuned",
        "linear_discrete_time_tuned",
        "neural_discrete_time_tuned",
    ]

# ------------------------------
# 4) Define expanded calibration protocol
# ------------------------------
protocol_rows = [
    {
        "protocol_component": "target_scope",
        "value": "all_tuned_benchmark_families",
        "status": "required",
        "notes": (
            "Expanded calibration diagnostics apply to tuned benchmark families "
            "that produce enrollment-level risk or survival outputs at shared horizons."
        ),
    },
    {
        "protocol_component": "eligible_models_rule",
        "value": "tuned_models_with_horizon_level_predictions_and_reliability_bins",
        "status": "required",
        "notes": (
            "A model is eligible if it has stable test-set horizon predictions and can be "
            "represented in the calibration audit at the shared benchmark horizons."
        ),
    },
    {
        "protocol_component": "shared_horizons",
        "value": ",".join(map(str, benchmark_horizons)),
        "status": "required",
        "notes": "Official benchmark horizons for calibration comparison.",
    },
    {
        "protocol_component": "primary_metric_name",
        "value": "weighted_absolute_calibration_gap_by_horizon",
        "status": "required",
        "notes": (
            "Primary calibration metric. Computed as the sample-size-weighted mean absolute "
            "gap between mean predicted risk and empirical event rate across risk bins."
        ),
    },
    {
        "protocol_component": "primary_metric_role",
        "value": "main_calibration_ranking_signal",
        "status": "required",
        "notes": (
            "Lower values indicate better alignment between predicted event risk and "
            "observed event frequency at a given horizon."
        ),
    },
    {
        "protocol_component": "required_strengthening_metric_1",
        "value": "approx_calibration_intercept_by_horizon",
        "status": "required",
        "notes": (
            "Approximate intercept derived from weighted fit on reliability bins. "
            "Used to assess systematic under/over-prediction."
        ),
    },
    {
        "protocol_component": "required_strengthening_metric_2",
        "value": "approx_calibration_slope_by_horizon",
        "status": "required",
        "notes": (
            "Approximate slope derived from weighted fit on reliability bins. "
            "Used to assess spread/compression of predicted risk."
        ),
    },
    {
        "protocol_component": "secondary_metric_1",
        "value": "brier_at_horizon",
        "status": "required",
        "notes": (
            "Retained as complementary performance metric at shared horizons; "
            "not a substitute for calibration-specific diagnostics."
        ),
    },
    {
        "protocol_component": "secondary_metric_2",
        "value": "support_fraction_or_n_at_horizon",
        "status": "required",
        "notes": (
            "Used to determine whether a horizon remains interpretable and comparable."
        ),
    },
    {
        "protocol_component": "secondary_metric_3",
        "value": "event_rate_at_horizon",
        "status": "required",
        "notes": (
            "Provides outcome prevalence context for calibration interpretation."
        ),
    },
    {
        "protocol_component": "secondary_metric_4",
        "value": "reliability_bins_table",
        "status": "required",
        "notes": (
            "Canonical artifact backing calibration plots and bin-level audit."
        ),
    },
    {
        "protocol_component": "optional_metric",
        "value": "ici_style_summary_if_row_level_predictions_are_stably_available",
        "status": "optional_non_blocking",
        "notes": (
            "ICI-style summary is desirable, but not required for the canonical benchmark "
            "if row-level predictions are not yet available in a stable unified structure."
        ),
    },
    {
        "protocol_component": "support_rule",
        "value": "report_all_horizons_but_flag_low_support",
        "status": "required",
        "notes": (
            "Horizons should be reported, but low-support horizons must be explicitly flagged "
            "and should not dominate comparative interpretation."
        ),
    },
    {
        "protocol_component": "comparability_rule",
        "value": "compare_models_only_at_shared_horizons_with_available_predictions",
        "status": "required",
        "notes": (
            "Cross-family comparison must remain restricted to shared benchmark horizons "
            "where all compared models have valid outputs."
        ),
    },
    {
        "protocol_component": "estimation_rule_for_slope_intercept",
        "value": "weighted_logit_scale_fit_over_reliability_bins",
        "status": "required",
        "notes": (
            "Approximate calibration slope/intercept are estimated through a weighted fit "
            "over reliability-bin summaries, not through a separate retraining workflow."
        ),
    },
    {
        "protocol_component": "interpretation_rule",
        "value": "use_multi_metric_calibration_reading",
        "status": "required",
        "notes": (
            "Final calibration interpretation must jointly consider primary gap, "
            "slope/intercept, support, and Brier; no single scalar should dominate the narrative."
        ),
    },
]

protocol_df = pd.DataFrame(protocol_rows)

# ------------------------------
# 5) Register eligible models
# ------------------------------
eligible_models_df = pd.DataFrame({
    "model_label": eligible_models_inferred,
    "eligibility_status": "eligible_if_horizon_outputs_present",
    "eligibility_basis": (
        "Tuned benchmark family with expected horizon-level calibration audit compatibility."
    ),
})

# ------------------------------
# 6) Define required downstream outputs
# ------------------------------
downstream_outputs_rows = [
    {
        "downstream_step": "P26.5",
        "required_output_name": "table_p26_5_all_tuned_calibration_audit.csv",
        "required": True,
        "purpose": (
            "Long-form audit table with one row per model-horizon-metric combination."
        ),
    },
    {
        "downstream_step": "P26.5",
        "required_output_name": "table_p26_5_paper_summary_all_tuned_models.csv",
        "required": True,
        "purpose": (
            "Paper-facing compact summary of calibration across tuned benchmark families."
        ),
    },
    {
        "downstream_step": "P26.5",
        "required_output_name": "table_p26_5_reliability_bins_all_tuned_models.csv",
        "required": True,
        "purpose": (
            "Canonical reliability-bin table supporting bin-level calibration audit."
        ),
    },
    {
        "downstream_step": "P26.5",
        "required_output_name": "figure_p26_5_reliability_selected_horizon.png",
        "required": False,
        "purpose": (
            "Optional visualization for selected horizon(s), especially h20 or paper-chosen horizon."
        ),
    },
    {
        "downstream_step": "P26.6",
        "required_output_name": "table_p26_6_calibration_interpretation_summary.csv",
        "required": True,
        "purpose": (
            "Narrative-ready interpretation layer summarizing comparative calibration strengths/weaknesses."
        ),
    },
]

downstream_outputs_df = pd.DataFrame(downstream_outputs_rows)

# ------------------------------
# 7) Define metric registry
# ------------------------------
metric_registry_rows = [
    {
        "metric_name": "weighted_absolute_calibration_gap_by_horizon",
        "metric_family": "calibration",
        "metric_role": "primary",
        "required": True,
        "lower_is_better": True,
        "estimation_level": "horizon",
        "canonical_source": "reliability_bins",
        "notes": (
            "Sample-size-weighted mean absolute gap between predicted risk and observed event rate across bins."
        ),
    },
    {
        "metric_name": "approx_calibration_intercept_by_horizon",
        "metric_family": "calibration_strengthening",
        "metric_role": "required_secondary",
        "required": True,
        "lower_is_better": False,
        "estimation_level": "horizon",
        "canonical_source": "weighted_fit_on_reliability_bins",
        "notes": (
            "Approximate intercept; closeness to 0 is desirable."
        ),
    },
    {
        "metric_name": "approx_calibration_slope_by_horizon",
        "metric_family": "calibration_strengthening",
        "metric_role": "required_secondary",
        "required": True,
        "lower_is_better": False,
        "estimation_level": "horizon",
        "canonical_source": "weighted_fit_on_reliability_bins",
        "notes": (
            "Approximate slope; closeness to 1 is desirable."
        ),
    },
    {
        "metric_name": "brier_at_horizon",
        "metric_family": "overall_prediction_quality",
        "metric_role": "supporting",
        "required": True,
        "lower_is_better": True,
        "estimation_level": "horizon",
        "canonical_source": "benchmark_outputs",
        "notes": (
            "Complementary performance metric, not a direct substitute for calibration."
        ),
    },
    {
        "metric_name": "support_fraction_or_n_at_horizon",
        "metric_family": "audit_context",
        "metric_role": "supporting",
        "required": True,
        "lower_is_better": False,
        "estimation_level": "horizon",
        "canonical_source": "benchmark_outputs",
        "notes": (
            "Used to qualify interpretability and comparability of horizon-level summaries."
        ),
    },
    {
        "metric_name": "event_rate_at_horizon",
        "metric_family": "audit_context",
        "metric_role": "supporting",
        "required": True,
        "lower_is_better": False,
        "estimation_level": "horizon",
        "canonical_source": "benchmark_outputs_or_bins",
        "notes": (
            "Outcome prevalence context for horizon-level interpretation."
        ),
    },
    {
        "metric_name": "ici_style_summary",
        "metric_family": "calibration_strengthening",
        "metric_role": "optional_exploratory",
        "required": False,
        "lower_is_better": True,
        "estimation_level": "horizon_or_global",
        "canonical_source": "row_level_predictions_if_available",
        "notes": (
            "Optional. Implement only if row-level predictions are available in stable canonical form."
        ),
    },
]

metric_registry_df = pd.DataFrame(metric_registry_rows)

# ------------------------------
# 8) Save outputs
# ------------------------------
path_protocol = table_dir / "table_p26_4_expanded_calibration_protocol.csv"
path_eligible = table_dir / "table_p26_4_calibration_eligible_models.csv"
path_metric_registry = table_dir / "table_p26_4_calibration_metric_registry.csv"
path_artifacts = table_dir / "table_p26_4_existing_calibration_artifacts.csv"
path_downstream = table_dir / "table_p26_4_required_downstream_outputs.csv"
path_metadata = metadata_dir / "metadata_p26_4_expanded_calibration_protocol.json"

protocol_df.to_csv(path_protocol, index=False)
eligible_models_df.to_csv(path_eligible, index=False)
metric_registry_df.to_csv(path_metric_registry, index=False)
artifact_inventory_df.to_csv(path_artifacts, index=False)
downstream_outputs_df.to_csv(path_downstream, index=False)

metadata = {
    "step": "D6.1",
    "title": "Define Expanded Survival Calibration Diagnostics",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "benchmark_horizons": benchmark_horizons,
    "eligible_models_inferred": eligible_models_inferred,
    "protocol_design_note": (
        "This step formalizes the expanded calibration protocol but does not recompute metrics."
    ),
    "primary_metric": "weighted_absolute_calibration_gap_by_horizon",
    "required_strengthening_metrics": [
        "approx_calibration_intercept_by_horizon",
        "approx_calibration_slope_by_horizon",
    ],
    "optional_metric": "ici_style_summary",
    "support_rule": "report_all_horizons_but_flag_low_support",
    "comparability_rule": "compare_models_only_at_shared_horizons_with_available_predictions",
    "required_downstream_outputs": downstream_outputs_df["required_output_name"].tolist(),
    "artifact_inventory": artifact_inventory_rows,
}

save_json(metadata, path_metadata)

# ------------------------------
# 9) Console summary
# ------------------------------
print("\nOfficial benchmark horizons:")
print(benchmark_horizons)

print("\nEligible models (inferred / fallback):")
display(eligible_models_df)

print("\nExpanded calibration protocol:")
display(protocol_df)

print("\nCalibration metric registry:")
display(metric_registry_df)

print("\nExisting calibration artifacts detected:")
display(artifact_inventory_df)

print("\nRequired downstream outputs:")
display(downstream_outputs_df)

print("\nSaved:")
print("-", path_protocol.resolve())
print("-", path_eligible.resolve())
print("-", path_metric_registry.resolve())
print("-", path_artifacts.resolve())
print("-", path_downstream.resolve())
print("-", path_metadata.resolve())

#### D6.1.1 — Materialize Benchmark Wide Comparison Tables

In [ ]:
# ==============================================================
# D6.1.1 — Materialize Benchmark Wide Comparison Tables
# --------------------------------------------------------------
# Purpose:
#   Build the wide-format benchmark comparison tables required
#   by downstream expanded calibration audit steps.
#
# Inputs expected from previous cells:
#   - TABLES_DIR
#   - table_benchmark_leaderboard_by_horizon.csv
#
# Outputs:
#   - table_benchmark_calibration_by_horizon_wide.csv
#   - table_benchmark_brier_by_horizon_wide.csv
#   - table_benchmark_support_reference.csv
# ==============================================================

print("\n" + "=" * 70)
print("D6.1.1 — Materialize Benchmark Wide Comparison Tables")
print("=" * 70)

from pathlib import Path
import pandas as pd

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

leaderboard_horizon_path = TABLES_DIR / "table_benchmark_leaderboard_by_horizon.csv"
if not leaderboard_horizon_path.exists():
    raise FileNotFoundError(
        f"Missing required file: {leaderboard_horizon_path}. "
        "Run D6 first."
    )

# ------------------------------
# 2) Read benchmark by-horizon table
# ------------------------------
df = pd.read_csv(leaderboard_horizon_path)

required_cols = [
    "model_key",
    "display_name",
    "family",
    "horizon_week",
    "brier_value",
    "calibration_gap",
    "n_evaluable_enrollments",
    "n_events_by_horizon",
    "event_rate_by_horizon",
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(
        "Benchmark leaderboard by horizon is missing required columns: "
        + ", ".join(missing_cols)
    )

# ------------------------------
# 3) Calibration wide
# ------------------------------
calibration_wide_df = (
    df.pivot_table(
        index=["model_key", "display_name", "family"],
        columns="horizon_week",
        values="calibration_gap",
        aggfunc="first"
    )
    .reset_index()
)

calibration_wide_df.columns = [
    f"calibration_h{int(c)}" if isinstance(c, (int, float)) else c
    for c in calibration_wide_df.columns
]

# ------------------------------
# 4) Brier wide
# ------------------------------
brier_wide_df = (
    df.pivot_table(
        index=["model_key", "display_name", "family"],
        columns="horizon_week",
        values="brier_value",
        aggfunc="first"
    )
    .reset_index()
)

brier_wide_df.columns = [
    f"brier_h{int(c)}" if isinstance(c, (int, float)) else c
    for c in brier_wide_df.columns
]

# ------------------------------
# 5) Support reference
# ------------------------------
support_reference_df = (
    df[
        [
            "horizon_week",
            "n_evaluable_enrollments",
            "n_events_by_horizon",
            "event_rate_by_horizon",
        ]
    ]
    .drop_duplicates()
    .sort_values(["horizon_week", "n_evaluable_enrollments"], ascending=[True, False])
    .reset_index(drop=True)
)

# ------------------------------
# 6) Save outputs
# ------------------------------
calibration_wide_path = TABLES_DIR / "table_benchmark_calibration_by_horizon_wide.csv"
brier_wide_path = TABLES_DIR / "table_benchmark_brier_by_horizon_wide.csv"
support_reference_path = TABLES_DIR / "table_benchmark_support_reference.csv"

calibration_wide_df.to_csv(calibration_wide_path, index=False)
brier_wide_df.to_csv(brier_wide_path, index=False)
support_reference_df.to_csv(support_reference_path, index=False)

# ------------------------------
# 7) Output
# ------------------------------
print("\nCalibration wide table:")
display(calibration_wide_df)

print("\nBrier wide table:")
display(brier_wide_df)

print("\nSupport reference:")
display(support_reference_df)

print("\nSaved:")
print("-", calibration_wide_path.resolve())
print("-", brier_wide_path.resolve())
print("-", support_reference_path.resolve())

### D6.2 — Unified Expanded Calibration Audit Across Tuned Models


In [ ]:
# ==============================================================
# D6.2 — Unified Expanded Calibration Audit Across Tuned Models
# --------------------------------------------------------------
# Purpose:
#   Build a unified expanded calibration audit across the tuned
#   benchmark models using the canonical benchmark-wide tables.
#
# Design:
#   - benchmark-wide calibration, brier, and support tables are required
#   - slope/intercept artifacts from P11.1 are optional if not yet present
#   - legacy reliability tables are optional
#
# Outputs:
#   - table_p26_5_all_tuned_calibration_audit.csv
#   - table_p26_5_paper_summary_all_tuned_models.csv
#   - table_p26_5_reliability_bins_all_tuned_models.csv
# ==============================================================

print("\n" + "=" * 70)
print("D6.2 — Unified Expanded Calibration Audit Across Tuned Models")
print("=" * 70)

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Paths
# ------------------------------
benchmark_calibration_path = TABLES_DIR / "table_benchmark_calibration_by_horizon_wide.csv"
benchmark_brier_path = TABLES_DIR / "table_benchmark_brier_by_horizon_wide.csv"
support_reference_path = TABLES_DIR / "table_benchmark_support_reference.csv"

p11_1_by_horizon_path = TABLES_DIR / "table_p11_1_calibration_slope_intercept_by_horizon.csv"
legacy_reliability_all_path = TABLES_DIR / "table_calibration_reliability_bins_all_tuned_models.csv"
legacy_reliability_h20_path = TABLES_DIR / "table_calibration_reliability_bins_h20_all_tuned_models.csv"


### Funcao: read_required_csv

Definicao isolada de read_required_csv para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 3) Read helpers
# ------------------------------
def read_required_csv(path: Path, label: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file for {label}: {path}")
    return pd.read_csv(path)


### Funcao: read_optional_csv

Definicao isolada de read_optional_csv para reutilizacao nas etapas seguintes.


In [ ]:

def read_optional_csv(path: Path):
    if not path.exists():
        return None
    return pd.read_csv(path)


In [ ]:

benchmark_calibration_wide_df = read_required_csv(benchmark_calibration_path, "benchmark calibration wide")
benchmark_brier_wide_df = read_required_csv(benchmark_brier_path, "benchmark brier wide")
support_reference_df = read_required_csv(support_reference_path, "support reference")

p11_1_by_horizon_df = read_optional_csv(p11_1_by_horizon_path)
legacy_reliability_all_df = read_optional_csv(legacy_reliability_all_path)
legacy_reliability_h20_df = read_optional_csv(legacy_reliability_h20_path)

# ------------------------------
# 4) Reshape benchmark-wide tables to long format
# ------------------------------
id_cols = ["model_key", "display_name", "family"]

calibration_long = benchmark_calibration_wide_df.melt(
    id_vars=id_cols,
    var_name="metric_horizon",
    value_name="calibration_gap"
)
calibration_long["horizon_week"] = (
    calibration_long["metric_horizon"]
    .astype(str)
    .str.extract(r"h(\d+)", expand=False)
    .astype(float)
)
calibration_long = calibration_long.drop(columns=["metric_horizon"])

brier_long = benchmark_brier_wide_df.melt(
    id_vars=id_cols,
    var_name="metric_horizon",
    value_name="brier_value"
)
brier_long["horizon_week"] = (
    brier_long["metric_horizon"]
    .astype(str)
    .str.extract(r"h(\d+)", expand=False)
    .astype(float)
)
brier_long = brier_long.drop(columns=["metric_horizon"])

support_long = support_reference_df.copy()

support_long = (
    support_long.sort_values(["horizon_week", "n_evaluable_enrollments"], ascending=[True, False])
    .drop_duplicates(subset=["horizon_week"], keep="first")
    .reset_index(drop=True)
)

# ------------------------------
# 5) Merge core required pieces
# ------------------------------
audit_df = (
    calibration_long
    .merge(brier_long, on=["model_key", "display_name", "family", "horizon_week"], how="outer")
    .merge(support_long, on="horizon_week", how="left")
)

audit_df["metric_scope"] = "shared_benchmark_horizon"
audit_df["slope_available"] = False
audit_df["intercept_available"] = False
audit_df["approx_calibration_slope"] = np.nan
audit_df["approx_calibration_intercept"] = np.nan

# ------------------------------
# 6) Optionally enrich with P11.1 outputs if available
# ------------------------------
if p11_1_by_horizon_df is not None:
    p11 = p11_1_by_horizon_df.copy()

    rename_map = {}
    if "model_label" in p11.columns and "model_key" not in p11.columns:
        rename_map["model_label"] = "model_key"
    if "horizon" in p11.columns and "horizon_week" not in p11.columns:
        rename_map["horizon"] = "horizon_week"
    if "calibration_slope" in p11.columns and "approx_calibration_slope" not in p11.columns:
        rename_map["calibration_slope"] = "approx_calibration_slope"
    if "calibration_intercept" in p11.columns and "approx_calibration_intercept" not in p11.columns:
        rename_map["calibration_intercept"] = "approx_calibration_intercept"

    p11 = p11.rename(columns=rename_map)

    wanted = ["model_key", "horizon_week", "approx_calibration_slope", "approx_calibration_intercept"]
    available = [c for c in wanted if c in p11.columns]
    if set(["model_key", "horizon_week"]).issubset(available):
        p11 = p11[available].copy()
        audit_df = audit_df.merge(
            p11,
            on=["model_key", "horizon_week"],
            how="left",
            suffixes=("", "_p11")
        )

        if "approx_calibration_slope_p11" in audit_df.columns:
            audit_df["approx_calibration_slope"] = audit_df["approx_calibration_slope_p11"].combine_first(
                audit_df["approx_calibration_slope"]
            )
            audit_df = audit_df.drop(columns=["approx_calibration_slope_p11"])

        if "approx_calibration_intercept_p11" in audit_df.columns:
            audit_df["approx_calibration_intercept"] = audit_df["approx_calibration_intercept_p11"].combine_first(
                audit_df["approx_calibration_intercept"]
            )
            audit_df = audit_df.drop(columns=["approx_calibration_intercept_p11"])

        audit_df["slope_available"] = audit_df["approx_calibration_slope"].notna()
        audit_df["intercept_available"] = audit_df["approx_calibration_intercept"].notna()

# ------------------------------
# 7) Reliability bins output
# ------------------------------
if legacy_reliability_all_df is not None:
    reliability_bins_df = legacy_reliability_all_df.copy()
elif legacy_reliability_h20_df is not None:
    reliability_bins_df = legacy_reliability_h20_df.copy()
else:
    reliability_bins_df = pd.DataFrame({
        "note": ["No legacy reliability bin table was available at this stage."]
    })

# ------------------------------
# 8) Paper summary
# ------------------------------
paper_summary_df = (
    audit_df.groupby(["model_key", "display_name", "family"], dropna=False)
    .agg(
        mean_calibration_gap=("calibration_gap", "mean"),
        max_calibration_gap=("calibration_gap", "max"),
        mean_brier=("brier_value", "mean"),
        min_brier=("brier_value", "min"),
        n_horizons=("horizon_week", "nunique"),
        all_slope_available=("slope_available", "all"),
        all_intercept_available=("intercept_available", "all"),
    )
    .reset_index()
    .sort_values(["mean_calibration_gap", "mean_brier", "display_name"])
    .reset_index(drop=True)
)

# ------------------------------
# 9) Save outputs
# ------------------------------
audit_path = TABLES_DIR / "table_p26_5_all_tuned_calibration_audit.csv"
paper_summary_path = TABLES_DIR / "table_p26_5_paper_summary_all_tuned_models.csv"
reliability_bins_path = TABLES_DIR / "table_p26_5_reliability_bins_all_tuned_models.csv"

audit_df.to_csv(audit_path, index=False)
paper_summary_df.to_csv(paper_summary_path, index=False)
reliability_bins_df.to_csv(reliability_bins_path, index=False)

# ------------------------------
# 10) Output
# ------------------------------
print("\nUnified calibration audit:")
display(audit_df)

print("\nPaper summary across tuned models:")
display(paper_summary_df)

print("\nReliability bins artifact:")
display(reliability_bins_df.head(20))

print("\nOptional enrichment status:")
print(f"- P11.1 by horizon available: {p11_1_by_horizon_df is not None}")
print(f"- Legacy reliability bins (all) available: {legacy_reliability_all_df is not None}")
print(f"- Legacy reliability bins (h20) available: {legacy_reliability_h20_df is not None}")

print("\nSaved:")
print("-", audit_path.resolve())
print("-", paper_summary_path.resolve())
print("-", reliability_bins_path.resolve())


### D6.3 — Calibration Interpretation Summary for Paper Use


In [ ]:
# ==============================================================
# D6.3 — Calibration Interpretation Summary for Paper Use
# --------------------------------------------------------------
# Purpose:
#   Build a paper-oriented interpretation layer from the unified
#   expanded calibration audit across tuned benchmark models.
#
# Inputs expected from previous cells:
#   - TABLES_DIR
#   - table_p26_5_all_tuned_calibration_audit.csv
#   - table_p26_5_paper_summary_all_tuned_models.csv
#
# Outputs:
#   - table_p26_6_calibration_interpretation_summary.csv
#   - table_p26_6_horizon_leaders.csv
# ==============================================================

print("\n" + "=" * 70)
print("D6.3 — Calibration Interpretation Summary for Paper Use")
print("=" * 70)

from pathlib import Path
import pandas as pd

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

audit_path = TABLES_DIR / "table_p26_5_all_tuned_calibration_audit.csv"
paper_summary_path = TABLES_DIR / "table_p26_5_paper_summary_all_tuned_models.csv"

if not audit_path.exists():
    raise FileNotFoundError(f"Missing required file: {audit_path}")
if not paper_summary_path.exists():
    raise FileNotFoundError(f"Missing required file: {paper_summary_path}")

audit_df = pd.read_csv(audit_path)
paper_summary_df = pd.read_csv(paper_summary_path)

required_audit_cols = [
    "model_key",
    "display_name",
    "family",
    "horizon_week",
    "calibration_gap",
    "brier_value",
]
missing_audit_cols = [c for c in required_audit_cols if c not in audit_df.columns]
if missing_audit_cols:
    raise KeyError(
        "Unified calibration audit is missing required columns: "
        + ", ".join(missing_audit_cols)
    )

# ------------------------------
# 2) Horizon leaders
# ------------------------------
horizon_leader_rows = []

for horizon, g in audit_df.groupby("horizon_week"):
    g_sorted = g.sort_values(
        by=["calibration_gap", "brier_value", "display_name"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    best = g_sorted.iloc[0]

    horizon_leader_rows.append({
        "horizon_week": int(horizon),
        "best_model_key": best["model_key"],
        "best_display_name": best["display_name"],
        "best_family": best["family"],
        "best_calibration_gap": float(best["calibration_gap"]),
        "best_brier_value": float(best["brier_value"]),
        "n_models_compared": int(len(g_sorted)),
    })

horizon_leaders_df = pd.DataFrame(horizon_leader_rows).sort_values("horizon_week").reset_index(drop=True)

# ------------------------------
# 3) Narrative-ready interpretation summary
# ------------------------------
summary_rows = []

for _, row in paper_summary_df.iterrows():
    interp_parts = []

    mean_gap = row["mean_calibration_gap"]
    if pd.notna(mean_gap):
        if mean_gap <= 0.03:
            interp_parts.append("very strong average calibration")
        elif mean_gap <= 0.06:
            interp_parts.append("strong average calibration")
        elif mean_gap <= 0.10:
            interp_parts.append("moderate calibration")
        else:
            interp_parts.append("weaker calibration")

    max_gap = row["max_calibration_gap"]
    if pd.notna(max_gap):
        if max_gap <= 0.04:
            interp_parts.append("stable across horizons")
        elif max_gap <= 0.08:
            interp_parts.append("some horizon sensitivity")
        else:
            interp_parts.append("substantial horizon sensitivity")

    mean_brier = row["mean_brier"]
    if pd.notna(mean_brier):
        if mean_brier <= 0.13:
            interp_parts.append("strong overall horizon-level accuracy")
        elif mean_brier <= 0.17:
            interp_parts.append("competitive overall horizon-level accuracy")
        else:
            interp_parts.append("weaker overall horizon-level accuracy")

    if bool(row.get("all_slope_available", False)) and bool(row.get("all_intercept_available", False)):
        strengthening_status = "expanded slope/intercept diagnostics available"
    else:
        strengthening_status = "expanded slope/intercept diagnostics not yet available"

    summary_rows.append({
        "model_key": row["model_key"],
        "display_name": row["display_name"],
        "family": row["family"],
        "mean_calibration_gap": row["mean_calibration_gap"],
        "max_calibration_gap": row["max_calibration_gap"],
        "mean_brier": row["mean_brier"],
        "min_brier": row["min_brier"],
        "n_horizons": row["n_horizons"],
        "all_slope_available": row["all_slope_available"],
        "all_intercept_available": row["all_intercept_available"],
        "interpretation_summary": "; ".join(interp_parts),
        "strengthening_status": strengthening_status,
    })

interpretation_df = pd.DataFrame(summary_rows).sort_values(
    ["mean_calibration_gap", "mean_brier", "display_name"]
).reset_index(drop=True)

# ------------------------------
# 4) Save outputs
# ------------------------------
interpretation_path = TABLES_DIR / "table_p26_6_calibration_interpretation_summary.csv"
horizon_leaders_path = TABLES_DIR / "table_p26_6_horizon_leaders.csv"

interpretation_df.to_csv(interpretation_path, index=False)
horizon_leaders_df.to_csv(horizon_leaders_path, index=False)

# ------------------------------
# 5) Output
# ------------------------------
print("\nCalibration interpretation summary:")
display(interpretation_df)

print("\nHorizon leaders:")
display(horizon_leaders_df)

print("\nSaved:")
print("-", interpretation_path.resolve())
print("-", horizon_leaders_path.resolve())

# E — Post-hoc Audits and Dependency Map

This section keeps only the late-stage audit blocks that still matter for the manuscript pipeline or for reviewer-facing robustness checks.

Use the dependency map below before rerunning isolated cells. The paper-facing path is now: benchmark outputs -> calibration/ablation/explainability audits -> curated paper freeze.

In [ ]:
# ==============================================================
# E0 — Dependency Map for Late-Stage Audit Sections
# --------------------------------------------------------------
# Purpose:
#   Make late-stage dependencies explicit before the notebook is
#   rerun selectively after the editorial cleanup.
#
# Methodological note:
#   This cell documents execution contracts only.
#   It does not train models and does not recompute metrics.
# ==============================================================

print("\n" + "=" * 70)
print("E0 — Dependency Map for Late-Stage Audit Sections")
print("=" * 70)
print("Methodological note: this cell documents dependencies only.")

required_names = ["TABLES_DIR", "METADATA_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd

dependency_rows = [
    {
        "section_id": "E1",
        "title": "Benchmark comparability audit",
        "depends_on": "canonical tuned benchmark tables",
        "primary_outputs": "table_p10_0_benchmark_comparability_*",
        "paper_facing": False,
        "notes": "Standalone audit retained as optional post-hoc check."
    },
    {
        "section_id": "E2",
        "title": "Calibration artifact audit",
        "depends_on": "per-model calibration tables + benchmark-wide calibration tables",
        "primary_outputs": "table_p11_0_calibration_*",
        "paper_facing": True,
        "notes": "Feeds calibration strengthening and final paper freeze."
    },
    {
        "section_id": "E3",
        "title": "Calibration slope/intercept audit",
        "depends_on": "E2 calibration artifacts or legacy reliability bins",
        "primary_outputs": "table_p11_1_calibration_slope_intercept_*",
        "paper_facing": True,
        "notes": "Optional strengthening layer merged into calibration summaries when available."
    },
    {
        "section_id": "E4",
        "title": "Unified calibration strengthening summary",
        "depends_on": "E2 + E3",
        "primary_outputs": "table_p11_2_calibration_strengthening_*",
        "paper_facing": True,
        "notes": "Strengthens reviewer-facing calibration interpretation."
    },
    {
        "section_id": "E5",
        "title": "Sensitivity artifact audit",
        "depends_on": "sensitivity outputs if that branch is executed",
        "primary_outputs": "table_p12_0_sensitivity_*",
        "paper_facing": False,
        "notes": "Kept as optional post-hoc robustness layer."
    },
    {
        "section_id": "E6",
        "title": "Horizon-choice stress test",
        "depends_on": "E5",
        "primary_outputs": "table_p12_3_horizon_choice_*",
        "paper_facing": False,
        "notes": "Optional; not required by the manuscript freeze."
    },
    {
        "section_id": "E7",
        "title": "Unified sensitivity summary",
        "depends_on": "E5 + E6",
        "primary_outputs": "table_p12_4_sensitivity_*",
        "paper_facing": False,
        "notes": "Optional; retained for robustness documentation only."
    },
    {
        "section_id": "F1",
        "title": "Define ablation protocol",
        "depends_on": "benchmark-ready tuned datasets",
        "primary_outputs": "table_ablation_model_registry.csv and companions",
        "paper_facing": True,
        "notes": "Foundational registry for all downstream ablation cells."
    },
    {
        "section_id": "F2",
        "title": "Materialize ablation variants",
        "depends_on": "F1 + tuned ready datasets",
        "primary_outputs": "table_ablation_variant_registry.csv",
        "paper_facing": True,
        "notes": "Feeds both discrete-time and continuous-time ablation training cells."
    },
    {
        "section_id": "F3",
        "title": "Discrete-time tuned ablation runs",
        "depends_on": "F2",
        "primary_outputs": "table_ablation_discrete_*",
        "paper_facing": True,
        "notes": "Feeds F5 consolidated ablation outputs."
    },
    {
        "section_id": "F4",
        "title": "Continuous-time tuned ablation runs",
        "depends_on": "F2",
        "primary_outputs": "table_ablation_continuous_*",
        "paper_facing": True,
        "notes": "Feeds F5 consolidated ablation outputs."
    },
    {
        "section_id": "F5",
        "title": "Consolidate ablation results",
        "depends_on": "F3 + F4",
        "primary_outputs": "table_ablation_summary_by_model.csv",
        "paper_facing": True,
        "notes": "Canonical source for the paper ablation table and figure."
    },
    {
        "section_id": "F6",
        "title": "Preprocessing and tuning audit",
        "depends_on": "tuning result tables + preprocessing summaries",
        "primary_outputs": "table_preprocessing_and_tuning_audit.csv",
        "paper_facing": True,
        "notes": "Feeds appendix preprocessing/tuning table."
    },
    {
        "section_id": "F7",
        "title": "Bootstrap uncertainty audit",
        "depends_on": "saved tuned models + test-ready datasets",
        "primary_outputs": "table_appendix_bootstrap_uncertainty_compact.csv",
        "paper_facing": True,
        "notes": "Feeds appendix uncertainty table."
    },
    {
        "section_id": "F8",
        "title": "Comparable Cox PH audit",
        "depends_on": "cox tuned model + enrollment_cox_ready_train",
        "primary_outputs": "table_appendix_cox_ph_global_summary.csv + fig_appendix_cox_ph_diagnostics.png",
        "paper_facing": True,
        "notes": "Feeds appendix PH table and figure."
    },
    {
        "section_id": "G1",
        "title": "Define explainability protocol",
        "depends_on": "tuned benchmark families already frozen",
        "primary_outputs": "table_explainability_model_registry.csv",
        "paper_facing": True,
        "notes": "Registry cell for the full explainability layer."
    },
    {
        "section_id": "G2",
        "title": "Linear tuned explainability",
        "depends_on": "linear tuned model + preprocessor",
        "primary_outputs": "table_linear_explainability_*",
        "paper_facing": True,
        "notes": "Feeds G6 consolidated explainability outputs."
    },
    {
        "section_id": "G3",
        "title": "Neural tuned explainability",
        "depends_on": "neural tuned ready data + preprocessor",
        "primary_outputs": "table_neural_explainability_*",
        "paper_facing": True,
        "notes": "Feeds G6 consolidated explainability outputs."
    },
    {
        "section_id": "G4",
        "title": "Cox tuned explainability",
        "depends_on": "cox tuned model + preprocessor",
        "primary_outputs": "table_cox_explainability_*",
        "paper_facing": True,
        "notes": "Feeds G6 consolidated explainability outputs."
    },
    {
        "section_id": "G5",
        "title": "DeepSurv tuned explainability",
        "depends_on": "deepsurv tuned ready data + preprocessor",
        "primary_outputs": "table_deepsurv_explainability_*",
        "paper_facing": True,
        "notes": "Feeds G6 consolidated explainability outputs."
    },
    {
        "section_id": "G6",
        "title": "Consolidate explainability",
        "depends_on": "G2 + G3 + G4 + G5",
        "primary_outputs": "table_explainability_summary_by_model.csv + table_explainability_all_blocks_normalized.csv",
        "paper_facing": True,
        "notes": "Canonical source for the paper explainability table and figure."
    },
    {
        "section_id": "G7",
        "title": "Freeze curated paper artifacts",
        "depends_on": "benchmark leaderboard + E2/E3/E4 + F5/F6/F7/F8 + G6",
        "primary_outputs": "outputs_benchmark_survival/paper_main and paper_appendix",
        "paper_facing": True,
        "notes": "Curates only the CSV and PNG artifacts cited by the manuscript."
    },
    {
        "section_id": "G8",
        "title": "Display curated paper figures",
        "depends_on": "G7",
        "primary_outputs": "notebook displays only",
        "paper_facing": True,
        "notes": "Visual QA for exported PNG assets."
    },
    {
        "section_id": "G9",
        "title": "Preview curated paper evidence",
        "depends_on": "G7",
        "primary_outputs": "notebook prints and displays only",
        "paper_facing": True,
        "notes": "Notebook-side synthesis from curated paper artifacts only."
    },
]

dependency_map_df = pd.DataFrame(dependency_rows)
dependency_map_path = TABLES_DIR / "table_late_stage_dependency_map.csv"
dependency_map_df.to_csv(dependency_map_path, index=False)

paper_path_df = dependency_map_df[dependency_map_df["paper_facing"]].copy()

print("\nLate-stage dependency map:")
display(dependency_map_df)

print("\nPaper-facing execution path:")
display(paper_path_df[["section_id", "title", "depends_on", "primary_outputs"]])

print("\nCurated late-stage policy:")
print("- Auxiliary comparable discrete arm cells were removed from the notebook.")
print("- Narrative interpretation markdown cells were removed so conclusions come from executed outputs.")
print("- The curated manuscript path now terminates at outputs_benchmark_survival/paper_main and outputs_benchmark_survival/paper_appendix.")

print("\nSaved:")
print("-", dependency_map_path.resolve())

# E1 — Benchmark Comparability Audit

### Funcao: add_manual_row

Definicao isolada de add_manual_row para reutilizacao nas etapas seguintes.


In [ ]:
# ============================================================
# E1 — Benchmark Comparability Audit
# ============================================================
# Purpose:
#   Audit structural comparability across benchmark model families.
#   This step does not retrain anything. It creates an auditable map
#   of which inputs/tables/representations are used by each family.
#
# Main outputs:
#   - table_p10_0_benchmark_comparability_audit.csv
#   - table_p10_0_benchmark_comparability_summary.csv
#   - metadata_p10_0_benchmark_comparability_audit.json
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E1 — Benchmark Comparability Audit")
print("=" * 70)

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

available_tables = sorted(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

# ------------------------------------------------------------
# 1) Candidate benchmark-related tables
# ------------------------------------------------------------
candidate_rows = []

for tbl in available_tables:
    lname = tbl.lower()

    is_relevant = any(token in lname for token in [
        "pp_", "person_period", "hazard", "survival",
        "cox", "deepsurv", "discrete", "continuous",
        "neural", "linear", "input", "ready", "split"
    ])

    if not is_relevant:
        continue

    try:
        cols_df = con.execute(f"PRAGMA table_info('{tbl}')").fetchdf()
        cols = cols_df["name"].astype(str).tolist()
        n_rows = con.execute(f"SELECT COUNT(*) AS n FROM {tbl}").fetchdf()["n"].iloc[0]

        colset = set(cols)
        has_week = "week" in colset
        has_id_student = "id_student" in colset
        has_module = "code_module" in colset
        has_presentation = "code_presentation" in colset
        has_event = "event_observed" in colset or "event" in colset
        has_t_final = "t_final_week" in colset
        has_t_event = "t_event_week" in colset

        # crude representation heuristics
        if has_week and n_rows > 50000:
            temporal_representation = "dynamic_weekly"
        elif (not has_week) and has_t_final:
            temporal_representation = "enrollment_level_survival"
        elif (not has_week) and ("week" not in colset):
            temporal_representation = "static_or_summary_level"
        else:
            temporal_representation = "other"

        # family heuristics from table name
        if "cox" in lname:
            benchmark_family = "continuous_time"
        elif "deepsurv" in lname:
            benchmark_family = "continuous_time_neural"
        elif "linear_hazard" in lname or "neural_hazard" in lname or "hazard" in lname:
            benchmark_family = "discrete_time"
        elif "person_period" in lname or "pp_" in lname:
            benchmark_family = "person_period_upstream"
        elif "survival" in lname:
            benchmark_family = "enrollment_survival_upstream"
        else:
            benchmark_family = "other"

        candidate_rows.append({
            "table_name": tbl,
            "n_rows": int(n_rows),
            "n_columns": int(len(cols)),
            "has_id_student": has_id_student,
            "has_code_module": has_module,
            "has_code_presentation": has_presentation,
            "has_week": has_week,
            "has_event_signal": has_event,
            "has_t_event_week": has_t_event,
            "has_t_final_week": has_t_final,
            "temporal_representation_detected": temporal_representation,
            "benchmark_family_detected": benchmark_family,
            "columns_preview": ", ".join(cols[:20]),
        })

    except Exception as e:
        candidate_rows.append({
            "table_name": tbl,
            "n_rows": np.nan,
            "n_columns": np.nan,
            "has_id_student": np.nan,
            "has_code_module": np.nan,
            "has_code_presentation": np.nan,
            "has_week": np.nan,
            "has_event_signal": np.nan,
            "has_t_event_week": np.nan,
            "has_t_final_week": np.nan,
            "temporal_representation_detected": "error",
            "benchmark_family_detected": "error",
            "columns_preview": f"ERROR: {str(e)}",
        })

table_candidates = pd.DataFrame(candidate_rows)

if table_candidates.empty:
    raise RuntimeError(
        "No benchmark-related tables were detected. "
        "Run the benchmark notebook steps before P10.0."
    )

# ------------------------------------------------------------
# 2) Manual structural audit layer
# ------------------------------------------------------------
# This layer translates the current benchmark architecture into a
# model-family comparability map. It is intentionally explicit.
#
# You may edit labels later if the notebook architecture evolves.

manual_rows = []

def add_manual_row(
    model_label,
    family,
    likely_input_table,
    representation_type,
    update_regime,
    benchmark_role,
    comparability_status,
    rationale
):
    manual_rows.append({
        "model_or_family_label": model_label,
        "family": family,
        "likely_input_table": likely_input_table,
        "representation_type": representation_type,
        "update_regime": update_regime,
        "benchmark_role": benchmark_role,
        "comparability_status": comparability_status,
        "rationale": rationale,
    })

# Upstream enrollment-level table
add_manual_row(
    model_label="Enrollment survival backbone",
    family="upstream_enrollment_level",
    likely_input_table="enrollment_survival_ready",
    representation_type="enrollment_level_survival",
    update_regime="not_a_model",
    benchmark_role="upstream_backbone",
    comparability_status="not_applicable",
    rationale="Canonical enrollment-level event/time table used upstream."
)

# Person-period upstream tables
for tbl_name in ["person_period_min", "person_period_enriched", "pp_linear_hazard_input", "pp_neural_hazard_input"]:
    if tbl_name in table_candidates["table_name"].values:
        add_manual_row(
            model_label=tbl_name,
            family="discrete_time_upstream",
            likely_input_table=tbl_name,
            representation_type="dynamic_weekly",
            update_regime="weekly_updating",
            benchmark_role="upstream_discrete_time_input",
            comparability_status="supports_dynamic_regime",
            rationale="Person-period structure with week-level rows enables dynamic weekly updating."
        )

# Discrete-time linear hazard
if "pp_linear_hazard_ready_train" in table_candidates["table_name"].values or "pp_linear_hazard_input" in table_candidates["table_name"].values:
    add_manual_row(
        model_label="Discrete-time linear hazard",
        family="discrete_time",
        likely_input_table="pp_linear_hazard_*",
        representation_type="dynamic_weekly",
        update_regime="weekly_updating",
        benchmark_role="main_benchmark_model",
        comparability_status="structurally_asymmetric_vs_static_continuous",
        rationale="Uses person-period rows and week-indexed dynamic updating."
    )

# Discrete-time neural hazard
if "pp_neural_hazard_ready_train" in table_candidates["table_name"].values or "pp_neural_hazard_input" in table_candidates["table_name"].values:
    add_manual_row(
        model_label="Discrete-time neural hazard",
        family="discrete_time_neural",
        likely_input_table="pp_neural_hazard_*",
        representation_type="dynamic_weekly",
        update_regime="weekly_updating",
        benchmark_role="main_benchmark_model",
        comparability_status="structurally_asymmetric_vs_static_continuous",
        rationale="Uses person-period rows and week-indexed dynamic updating."
    )

# Continuous-time Cox comparable arm
cox_like_tables = [x for x in table_candidates["table_name"].tolist() if "cox" in x.lower()]
if len(cox_like_tables) > 0:
    add_manual_row(
        model_label="Comparable Cox-based continuous-time arm",
        family="continuous_time",
        likely_input_table=" / ".join(cox_like_tables[:5]),
        representation_type="early_window_summary_or_enrollment_level",
        update_regime="static_after_early_window",
        benchmark_role="main_benchmark_model",
        comparability_status="structurally_asymmetric_vs_dynamic_discrete",
        rationale="Continuous-time comparable models are represented at enrollment/static summary level rather than dynamic weekly person-period updating."
    )

# Deepsurv comparable arm
deepsurv_like_tables = [x for x in table_candidates["table_name"].tolist() if "deepsurv" in x.lower()]
if len(deepsurv_like_tables) > 0:
    add_manual_row(
        model_label="Comparable DeepSurv arm",
        family="continuous_time_neural",
        likely_input_table=" / ".join(deepsurv_like_tables[:5]),
        representation_type="early_window_summary_or_enrollment_level",
        update_regime="static_after_early_window",
        benchmark_role="main_benchmark_model",
        comparability_status="structurally_asymmetric_vs_dynamic_discrete",
        rationale="Neural continuous-time arm appears to rely on enrollment-level summarized inputs rather than dynamic weekly rows."
    )

# Context-held-out split is not a family, but relevant to benchmark design
if "enrollment_survival_ready_context_split" in available_tables:
    add_manual_row(
        model_label="Context-held-out split backbone",
        family="evaluation_protocol",
        likely_input_table="enrollment_survival_ready_context_split",
        representation_type="enrollment_level_split_protocol",
        update_regime="not_a_model",
        benchmark_role="alternative_transportability_protocol",
        comparability_status="not_applicable",
        rationale="Alternative split protocol for unseen curricular contexts; does not itself change family comparability."
    )

table_p10_0_benchmark_comparability_audit = pd.DataFrame(manual_rows)

# ------------------------------------------------------------
# 3) Family-level summary
# ------------------------------------------------------------
summary = (
    table_p10_0_benchmark_comparability_audit
    .groupby(["family", "representation_type", "update_regime", "comparability_status"], dropna=False)
    .size()
    .reset_index(name="n_rows")
    .sort_values(["family", "representation_type", "update_regime"])
    .reset_index(drop=True)
)

# overall interpretive summary
overall_findings = []

n_dynamic = int(
    (table_p10_0_benchmark_comparability_audit["representation_type"] == "dynamic_weekly").sum()
)
n_static_summary = int(
    (table_p10_0_benchmark_comparability_audit["representation_type"] == "early_window_summary_or_enrollment_level").sum()
)
n_asym = int(
    table_p10_0_benchmark_comparability_audit["comparability_status"]
    .astype(str)
    .str.contains("asymmetric", case=False, na=False)
    .sum()
)

overall_findings.append({
    "finding": "n_dynamic_weekly_entries",
    "value": n_dynamic,
})
overall_findings.append({
    "finding": "n_early_window_or_enrollment_level_entries",
    "value": n_static_summary,
})
overall_findings.append({
    "finding": "n_entries_flagged_as_structurally_asymmetric",
    "value": n_asym,
})
overall_findings.append({
    "finding": "benchmark_comparability_diagnosis",
    "value": (
        "Current benchmark includes both dynamic weekly discrete-time arms and "
        "static/early-window continuous-time comparable arms; therefore family-level "
        "ranking should not be interpreted as a pure architecture-only comparison."
    )
})

table_p10_0_benchmark_comparability_summary = pd.DataFrame(overall_findings)

# ------------------------------------------------------------
# 4) Save outputs
# ------------------------------------------------------------
path_candidates = OUT_TABLES / "table_p10_0_detected_benchmark_related_tables.csv"
path_audit = OUT_TABLES / "table_p10_0_benchmark_comparability_audit.csv"
path_summary = OUT_TABLES / "table_p10_0_benchmark_comparability_summary.csv"
path_metadata = OUT_METADATA / "metadata_p10_0_benchmark_comparability_audit.json"

table_candidates.to_csv(path_candidates, index=False)
table_p10_0_benchmark_comparability_audit.to_csv(path_audit, index=False)
table_p10_0_benchmark_comparability_summary.to_csv(path_summary, index=False)

metadata = {
    "step": "P10.0",
    "title": "Benchmark Comparability Audit",
    "purpose": "Audit structural comparability across benchmark families before redesign.",
    "main_diagnosis": (
        "The current benchmark mixes dynamic weekly discrete-time inputs with "
        "static or early-window continuous-time comparable inputs."
    ),
    "output_tables": [
        path_candidates.as_posix(),
        path_audit.as_posix(),
        path_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 5) Display
# ------------------------------------------------------------
print("\nDetected benchmark-related tables:")
display(table_candidates)

print("\nComparability audit:")
display(table_p10_0_benchmark_comparability_audit)

print("\nComparability summary:")
display(table_p10_0_benchmark_comparability_summary)

print("\nSaved:")
print("-", path_candidates.resolve())
print("-", path_audit.resolve())
print("-", path_summary.resolve())
print("-", path_metadata.resolve())

# P10.5 — Integrate Comparable Discrete-Time Minimal Arm into

In [ ]:
# ==============================================================
# P10.5 — Integrate Comparable Discrete-Time Minimal Arm into
#          Benchmark Results
# ==============================================================
# Purpose:
#   Consolidate benchmark results for the comparable arm,
#   incorporating the newly evaluated
#   discrete_time_comparable_minimal branch.
#
# Main outputs:
#   - table_p10_5_comparable_arm_benchmark_results.csv
#   - table_p10_5_comparable_arm_best_by_variant.csv
#   - table_p10_5_benchmark_result_inventory.csv
#   - metadata_p10_5_comparable_arm_integration.json
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("P10.5 — Integrate Comparable Discrete-Time Minimal Arm into Benchmark Results")
print("=" * 70)

required_names = ["con", "MODEL_INPUT_VARIANTS"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Run prior setup cells first."
    )

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Load currently available result artifacts
# --------------------------------------------------------------
path_new_discrete = OUT_TABLES / "table_p10_4_comparable_discrete_metrics_by_horizon.csv"
if not path_new_discrete.exists():
    raise FileNotFoundError(
        f"Required file not found: {path_new_discrete.resolve()}. Run P10.4 first."
    )

new_discrete_df = pd.read_csv(path_new_discrete)

# Optional existing comparable-arm result sources
candidate_existing_files = [
    OUT_TABLES / "table_enrollment_model_metrics.csv",
    OUT_TABLES / "table_benchmark_enrollment_model_metrics.csv",
    OUT_TABLES / "table_main_benchmark_metrics.csv",
    OUT_TABLES / "table_model_metrics.csv",
    OUT_TABLES / "table_benchmark_results.csv",
]

existing_loaded = []
for p in candidate_existing_files:
    if p.exists():
        try:
            df = pd.read_csv(p)
            df["source_file"] = p.name
            existing_loaded.append(df)
        except Exception:
            pass

# --------------------------------------------------------------
# 2) Normalize the new comparable discrete result table
# --------------------------------------------------------------
disc = new_discrete_df.copy()

# Normalize column names to a benchmark-friendly schema
disc_norm = pd.DataFrame({
    "benchmark_variant": disc["benchmark_variant"],
    "benchmark_arm": disc["benchmark_arm"],
    "model_family": disc["benchmark_variant"],
    "model_label": "Comparable discrete-time minimal arm",
    "representation_type": disc["representation_type"],
    "update_regime": disc["update_regime"],
    "model_class": disc["model_class"],
    "horizon": pd.to_numeric(disc["horizon"], errors="coerce"),
    "metric_primary_name": "auc_enrollment_score_by_horizon",
    "metric_primary_value": pd.to_numeric(disc["auc_enrollment_score_by_horizon"], errors="coerce"),
    "metric_ap_name": "average_precision_by_horizon",
    "metric_ap_value": pd.to_numeric(disc["average_precision_by_horizon"], errors="coerce"),
    "metric_brier_name": "brier_by_horizon_on_risk",
    "metric_brier_value": pd.to_numeric(disc["brier_by_horizon_on_risk"], errors="coerce"),
    "n_test_enrollments_total": pd.to_numeric(disc["n_test_enrollments_total"], errors="coerce"),
    "n_test_enrollments_with_support": pd.to_numeric(disc["n_test_enrollments_with_support"], errors="coerce"),
    "support_fraction": pd.to_numeric(disc["support_fraction"], errors="coerce"),
    "positive_rate_by_horizon_supported": pd.to_numeric(disc["positive_rate_by_horizon_supported"], errors="coerce"),
    "feature_set_used": disc["feature_set_used"].astype(str),
    "result_status": "materialized_now",
    "source_note": "P10.4 comparable discrete-time minimal arm",
})

# --------------------------------------------------------------
# 3) Build inventory of currently known benchmark-result files
# --------------------------------------------------------------
inventory_rows = [{
    "artifact_name": path_new_discrete.name,
    "artifact_exists": True,
    "artifact_role": "new_comparable_discrete_result",
    "n_rows": int(len(new_discrete_df)),
}]

for p in candidate_existing_files:
    inventory_rows.append({
        "artifact_name": p.name,
        "artifact_exists": p.exists(),
        "artifact_role": "optional_existing_benchmark_result_source",
        "n_rows": int(len(pd.read_csv(p))) if p.exists() else np.nan,
    })

table_p10_5_benchmark_result_inventory = pd.DataFrame(inventory_rows)

# --------------------------------------------------------------
# 4) Attempt soft integration with existing comparable-arm results
# --------------------------------------------------------------
# Since historical metric tables may vary in schema, do not force a brittle merge.
# Instead, keep the new result canonical and attach any existing sources as inventory.
#
# This table is the new canonical comparable-arm result table for downstream use.
table_p10_5_comparable_arm_benchmark_results = disc_norm.copy()

# Best row by variant according to primary metric (AUC)
best_idx = (
    table_p10_5_comparable_arm_benchmark_results
    .groupby("benchmark_variant")["metric_primary_value"]
    .idxmax()
    .dropna()
    .astype(int)
    .tolist()
)

table_p10_5_comparable_arm_best_by_variant = (
    table_p10_5_comparable_arm_benchmark_results
    .loc[best_idx]
    .sort_values(["metric_primary_value", "horizon"], ascending=[False, True])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 5) Persist a DuckDB table for downstream use
# --------------------------------------------------------------
con.register(
    "tmp_p10_5_comparable_arm_benchmark_results_df",
    table_p10_5_comparable_arm_benchmark_results
)
con.execute("DROP TABLE IF EXISTS comparable_arm_benchmark_results")
con.execute("""
    CREATE TABLE comparable_arm_benchmark_results AS
    SELECT *
    FROM tmp_p10_5_comparable_arm_benchmark_results_df
""")

# --------------------------------------------------------------
# 6) Save outputs
# --------------------------------------------------------------
path_results = OUT_TABLES / "table_p10_5_comparable_arm_benchmark_results.csv"
path_best = OUT_TABLES / "table_p10_5_comparable_arm_best_by_variant.csv"
path_inventory = OUT_TABLES / "table_p10_5_benchmark_result_inventory.csv"
path_metadata = OUT_METADATA / "metadata_p10_5_comparable_arm_integration.json"

table_p10_5_comparable_arm_benchmark_results.to_csv(path_results, index=False)
table_p10_5_comparable_arm_best_by_variant.to_csv(path_best, index=False)
table_p10_5_benchmark_result_inventory.to_csv(path_inventory, index=False)

metadata = {
    "step": "P10.5",
    "title": "Integrate Comparable Discrete-Time Minimal Arm into Benchmark Results",
    "new_variant_integrated": "discrete_time_comparable_minimal",
    "output_duckdb_table": "comparable_arm_benchmark_results",
    "notes": [
        "This step consolidates the comparable discrete-time minimal arm into the benchmark result structure.",
        "It does not retrain older branches.",
        "The result table created here is the canonical comparable-arm result table for downstream use.",
        "If historical comparable-arm metric tables exist in alternative schemas, they are tracked in the inventory rather than brittle-merged automatically."
    ],
    "output_tables": [
        path_results.as_posix(),
        path_best.as_posix(),
        path_inventory.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 7) Display
# --------------------------------------------------------------
print("\nBenchmark result inventory:")
display(table_p10_5_benchmark_result_inventory)

print("\nComparable-arm benchmark results:")
display(table_p10_5_comparable_arm_benchmark_results)

print("\nBest result by variant:")
display(table_p10_5_comparable_arm_best_by_variant)

print("\nSaved:")
print("-", path_results.resolve())
print("-", path_best.resolve())
print("-", path_inventory.resolve())
print("-", path_metadata.resolve())

# E2 — Survival Calibration Audit

In [ ]:
# ==============================================================
# E2 — Survival Calibration Audit
# CORRECTED VERSION ALIGNED WITH CURRENT NOTEBOOK CONVENTION
# ==============================================================
# Purpose:
#   Audit currently available calibration evidence in the notebook
#   and summarize what already exists vs. what is still missing,
#   using the current artifact naming convention.
#
# Main outputs:
#   - table_p11_0_calibration_artifact_inventory.csv
#   - table_p11_0_calibration_audit_summary.csv
#   - table_p11_0_calibration_missing_components.csv
#   - metadata_p11_0_survival_calibration_audit.json
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E2 — Survival Calibration Audit")
print("=" * 70)

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_FIGURES = OUT_BASE / "figures"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Inventory calibration-related files under current convention
# --------------------------------------------------------------
candidate_files = []

for folder, kind in [
    (OUT_TABLES, "table"),
    (OUT_FIGURES, "figure"),
    (OUT_METADATA, "metadata"),
]:
    if folder.exists():
        for p in sorted(folder.iterdir()):
            name_l = p.name.lower()
            if any(token in name_l for token in [
                "calibration",
                "reliability",
                "brier",
                "support_by_horizon",
                "by_horizon",
                "paper_summary",
                "audit_tuned_models",
                "benchmark_calibration",
            ]):
                candidate_files.append({
                    "artifact_name": p.name,
                    "artifact_path": str(p.resolve()),
                    "artifact_type": kind,
                    "exists": True,
                    "size_bytes": p.stat().st_size if p.exists() else np.nan,
                })

table_p11_0_calibration_artifact_inventory = pd.DataFrame(candidate_files)

if table_p11_0_calibration_artifact_inventory.empty:
    table_p11_0_calibration_artifact_inventory = pd.DataFrame([{
        "artifact_name": "(none found)",
        "artifact_path": "",
        "artifact_type": "none",
        "exists": False,
        "size_bytes": np.nan,
    }])

# --------------------------------------------------------------
# 2) Detect currently available calibration evidence classes
# --------------------------------------------------------------
table_names = set(
    table_p11_0_calibration_artifact_inventory.loc[
        table_p11_0_calibration_artifact_inventory["artifact_type"] == "table",
        "artifact_name"
    ].astype(str).tolist()
)


### Funcao: any_match

Definicao isolada de any_match para reutilizacao nas etapas seguintes.


In [ ]:

def any_match(prefixes_or_names):
    return any(name in table_names for name in prefixes_or_names)


### Funcao: any_contains

Definicao isolada de any_contains para reutilizacao nas etapas seguintes.


In [ ]:

def any_contains(substrs):
    return any(any(sub in name for sub in substrs) for name in table_names)


In [ ]:

has_model_level_bins = any_contains(["_calibration_bins_by_horizon.csv"])
has_model_level_summary = any_contains(["_calibration_summary.csv"])
has_model_level_brier = any_contains(["_brier_by_horizon.csv"])
has_model_level_support = any_contains(["_support_by_horizon.csv"])
has_benchmark_wide_cal = "table_benchmark_calibration_by_horizon_wide.csv" in table_names
has_benchmark_wide_brier = "table_benchmark_brier_by_horizon_wide.csv" in table_names
has_benchmark_wide_auc = "table_benchmark_risk_auc_by_horizon_wide.csv" in table_names
has_all_tuned_audit = "table_calibration_audit_all_tuned_models.csv" in table_names
has_paper_summary = "table_calibration_paper_summary_all_tuned_models.csv" in table_names
has_reliability_bins_all_tuned = "table_calibration_reliability_bins_all_tuned_models.csv" in table_names
has_reliability_bins_h20 = "table_calibration_reliability_bins_h20_all_tuned_models.csv" in table_names

# --------------------------------------------------------------
# 3) Build corrected audit summary
# --------------------------------------------------------------
summary_rows = [
    {
        "evidence_group": "model_level_calibration_bins_by_horizon",
        "available": has_model_level_bins,
        "notes": "Per-model calibration bins by horizon detected under current naming convention."
    },
    {
        "evidence_group": "model_level_calibration_summary",
        "available": has_model_level_summary,
        "notes": "Per-model calibration summary tables detected under current naming convention."
    },
    {
        "evidence_group": "model_level_brier_by_horizon",
        "available": has_model_level_brier,
        "notes": "Per-model Brier-by-horizon tables detected."
    },
    {
        "evidence_group": "model_level_support_by_horizon",
        "available": has_model_level_support,
        "notes": "Per-model support-by-horizon tables detected."
    },
    {
        "evidence_group": "benchmark_wide_calibration_table",
        "available": has_benchmark_wide_cal,
        "notes": "Wide benchmark calibration table by horizon detected."
    },
    {
        "evidence_group": "benchmark_wide_brier_table",
        "available": has_benchmark_wide_brier,
        "notes": "Wide benchmark Brier-by-horizon table detected."
    },
    {
        "evidence_group": "benchmark_wide_auc_table",
        "available": has_benchmark_wide_auc,
        "notes": "Wide benchmark risk-AUC-by-horizon table detected."
    },
    {
        "evidence_group": "all_tuned_calibration_audit",
        "available": has_all_tuned_audit,
        "notes": "Calibration audit across tuned models detected."
    },
    {
        "evidence_group": "paper_summary_all_tuned_models",
        "available": has_paper_summary,
        "notes": "Paper-facing calibration summary across tuned models detected."
    },
    {
        "evidence_group": "reliability_bins_all_tuned_models",
        "available": has_reliability_bins_all_tuned,
        "notes": "Reliability bins across tuned models detected."
    },
    {
        "evidence_group": "reliability_bins_h20_all_tuned_models",
        "available": has_reliability_bins_h20,
        "notes": "Reliability bins at horizon 20 across tuned models detected."
    },
]

table_p11_0_calibration_audit_summary = pd.DataFrame(summary_rows)

# --------------------------------------------------------------
# 4) Missing / stronger components audit
# --------------------------------------------------------------
missing_components = [
    {
        "component": "horizon_specific_calibration_summary",
        "status": "already_present",
        "why_it_matters": "Reviewer requested stronger survival calibration evidence.",
        "recommended_next_step": "Reuse and cite existing horizon-specific calibration tables.",
        "current_evidence_note": "Model-level and benchmark-wide horizon-specific calibration artifacts already exist."
    },
    {
        "component": "benchmark_calibration_comparison_table",
        "status": "already_present_but_may_need_final_paper_framing",
        "why_it_matters": "Needed for side-by-side reporting in the paper.",
        "recommended_next_step": "Map existing benchmark-wide calibration table to paper-ready narrative/table.",
        "current_evidence_note": "table_benchmark_calibration_by_horizon_wide.csv already exists."
    },
    {
        "component": "calibration_slope_and_intercept_by_horizon",
        "status": "still_missing",
        "why_it_matters": "Strengthens calibration evaluation beyond binned summaries.",
        "recommended_next_step": "Estimate horizon-specific slope/intercept diagnostics.",
        "current_evidence_note": "No canonical slope/intercept artifact detected."
    },
    {
        "component": "integrated_calibration_style_index",
        "status": "still_missing",
        "why_it_matters": "Provides a stronger integrated calibration summary than raw bin tables alone.",
        "recommended_next_step": "Implement horizon-wise ICI-style summary or nearest operational equivalent.",
        "current_evidence_note": "No canonical ICI-style artifact detected."
    },
    {
        "component": "explicit_calibration_strengthening_for_reviewer_response",
        "status": "partially_missing",
        "why_it_matters": "Reviewer specifically questioned robustness of calibration assessment.",
        "recommended_next_step": "Add at least one stronger calibration diagnostic beyond bin summaries.",
        "current_evidence_note": "Rich calibration evidence exists already, but stronger diagnostics are still needed."
    },
]

table_p11_0_calibration_missing_components = pd.DataFrame(missing_components)

# --------------------------------------------------------------
# 5) Save outputs
# --------------------------------------------------------------
path_inventory = OUT_TABLES / "table_p11_0_calibration_artifact_inventory.csv"
path_summary = OUT_TABLES / "table_p11_0_calibration_audit_summary.csv"
path_missing = OUT_TABLES / "table_p11_0_calibration_missing_components.csv"
path_metadata = OUT_METADATA / "metadata_p11_0_survival_calibration_audit.json"

table_p11_0_calibration_artifact_inventory.to_csv(path_inventory, index=False)
table_p11_0_calibration_audit_summary.to_csv(path_summary, index=False)
table_p11_0_calibration_missing_components.to_csv(path_missing, index=False)

metadata = {
    "step": "P11.0",
    "title": "Survival Calibration Audit",
    "purpose": (
        "Audit currently available calibration artifacts and identify the next "
        "calibration-strengthening tasks for Group C using the current notebook convention."
    ),
    "has_model_level_calibration_bins_by_horizon": bool(has_model_level_bins),
    "has_model_level_calibration_summary": bool(has_model_level_summary),
    "has_model_level_brier_by_horizon": bool(has_model_level_brier),
    "has_model_level_support_by_horizon": bool(has_model_level_support),
    "has_benchmark_wide_calibration_table": bool(has_benchmark_wide_cal),
    "has_benchmark_wide_brier_table": bool(has_benchmark_wide_brier),
    "has_benchmark_wide_auc_table": bool(has_benchmark_wide_auc),
    "has_all_tuned_calibration_audit": bool(has_all_tuned_audit),
    "has_paper_summary_all_tuned_models": bool(has_paper_summary),
    "output_tables": [
        path_inventory.as_posix(),
        path_summary.as_posix(),
        path_missing.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 6) Display
# --------------------------------------------------------------
print("\nCalibration artifact inventory:")
display(table_p11_0_calibration_artifact_inventory)

print("\nCalibration audit summary:")
display(table_p11_0_calibration_audit_summary)

print("\nMissing / stronger calibration components:")
display(table_p11_0_calibration_missing_components)

print("\nSaved:")
print("-", path_inventory.resolve())
print("-", path_summary.resolve())
print("-", path_missing.resolve())
print("-", path_metadata.resolve())


# E3 — Horizon-wise Calibration Slope and Intercept Audit

In [ ]:
# ==============================================================
# E3 — Horizon-wise Calibration Slope and Intercept Audit
# REVISED VERSION (compatible with n and n_in_bin schemas)
# ==============================================================
# Purpose:
#   Estimate horizon-specific calibration intercept and slope
#   from current benchmark calibration bin artifacts.
#
# Method:
#   Weighted linear fit on:
#       logit(observed_event_rate) = intercept + slope * logit(mean_predicted_risk)
#   using calibration-bin summaries within each model_id x horizon_week.
#
# Main outputs:
#   - table_p11_1_calibration_slope_intercept_by_horizon.csv
#   - table_p11_1_calibration_slope_intercept_summary.csv
#   - metadata_p11_1_calibration_slope_intercept_audit.json
#
# Revision note:
#   This version supports both legacy reliability-bin schema using `n`
#   and current consolidated schema using `n_in_bin`.
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E3 — Horizon-wise Calibration Slope and Intercept Audit")
print("=" * 70)

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

reliability_all_path = OUT_TABLES / "table_calibration_reliability_bins_all_tuned_models.csv"
reliability_h20_path = OUT_TABLES / "table_calibration_reliability_bins_h20_all_tuned_models.csv"

if reliability_all_path.exists():
    rel_df = pd.read_csv(reliability_all_path)
    source_type = "reliability_bins_all_tuned_models"
elif reliability_h20_path.exists():
    rel_df = pd.read_csv(reliability_h20_path)
    source_type = "reliability_bins_h20_all_tuned_models"
else:
    raise FileNotFoundError(
        "No reliability-bin calibration artifact found. "
        "Expected table_calibration_reliability_bins_all_tuned_models.csv "
        "or table_calibration_reliability_bins_h20_all_tuned_models.csv."
    )

# ------------------------------
# 1) Schema checks
# ------------------------------
required_cols = [
    "horizon_week",
    "calibration_bin",
    "mean_predicted_risk",
    "observed_event_rate",
    "model_id",
    "model",
    "family",
]
missing_cols = [c for c in required_cols if c not in rel_df.columns]
if missing_cols:
    raise KeyError(f"Reliability-bin table is missing required columns: {missing_cols}")

# weight column: current schema prefers n_in_bin; legacy schema used n
if "n_in_bin" in rel_df.columns:
    weight_col = "n_in_bin"
elif "n" in rel_df.columns:
    weight_col = "n"
else:
    raise KeyError(
        "Reliability-bin table is missing the weight column. "
        "Expected either 'n_in_bin' (current schema) or 'n' (legacy schema)."
    )


### Funcao: clip_prob

Definicao isolada de clip_prob para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 2) Helpers
# ------------------------------
def clip_prob(x):
    x = pd.to_numeric(x, errors="coerce").astype(float)
    return np.clip(x, 1e-6, 1 - 1e-6)


### Funcao: weighted_mean

Definicao isolada de weighted_mean para reutilizacao nas etapas seguintes.


In [ ]:

def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    den = np.sum(w)
    if den <= 0:
        return np.nan
    return np.sum(w * x) / den


### Funcao: weighted_var

Definicao isolada de weighted_var para reutilizacao nas etapas seguintes.


In [ ]:

def weighted_var(x, w):
    mx = weighted_mean(x, w)
    return weighted_mean((x - mx) ** 2, w)


### Funcao: weighted_cov

Definicao isolada de weighted_cov para reutilizacao nas etapas seguintes.


In [ ]:

def weighted_cov(x, y, w):
    mx = weighted_mean(x, w)
    my = weighted_mean(y, w)
    return weighted_mean((x - mx) * (y - my), w)


### Funcao: fit_weighted_line

Definicao isolada de fit_weighted_line para reutilizacao nas etapas seguintes.


In [ ]:

def fit_weighted_line(x, y, w):
    var_x = weighted_var(x, w)
    if pd.isna(var_x) or var_x <= 0:
        return np.nan, np.nan
    slope = weighted_cov(x, y, w) / var_x
    intercept = weighted_mean(y, w) - slope * weighted_mean(x, w)
    return intercept, slope


In [ ]:

# ------------------------------
# 3) Prepare inputs
# ------------------------------
rel_df = rel_df.copy()
rel_df["horizon_week"] = pd.to_numeric(rel_df["horizon_week"], errors="coerce")
rel_df[weight_col] = pd.to_numeric(rel_df[weight_col], errors="coerce")

group_cols = ["model_id", "model", "family", "horizon_week"]
results = []

# ------------------------------
# 4) Estimate by model x horizon
# ------------------------------
for (model_id, model, family, horizon_week), g in rel_df.groupby(group_cols, dropna=False):
    pred = clip_prob(g["mean_predicted_risk"])
    obs = clip_prob(g["observed_event_rate"])
    w = pd.to_numeric(g[weight_col], errors="coerce").fillna(0).astype(float)

    valid_mask = (
        np.isfinite(pred) &
        np.isfinite(obs) &
        np.isfinite(w) &
        (w > 0)
    )

    pred = pred[valid_mask]
    obs = obs[valid_mask]
    w = w[valid_mask]

    if len(pred) < 2:
        results.append({
            "model_id": str(model_id),
            "model": str(model),
            "family": str(family),
            "horizon_week": int(horizon_week) if pd.notna(horizon_week) else np.nan,
            "calibration_intercept_logit": np.nan,
            "calibration_slope_logit": np.nan,
            "n_bins_used": int(len(pred)),
            "total_weight": float(np.sum(w)) if len(w) > 0 else 0.0,
            "source_type": source_type,
            "method_note": (
                "Insufficient valid bins for weighted logit-scale calibration fit."
            ),
        })
        continue

    x = np.log(pred / (1 - pred))
    y = np.log(obs / (1 - obs))

    intercept, slope = fit_weighted_line(x, y, w)

    results.append({
        "model_id": str(model_id),
        "model": str(model),
        "family": str(family),
        "horizon_week": int(horizon_week),
        "calibration_intercept_logit": intercept,
        "calibration_slope_logit": slope,
        "n_bins_used": int(len(pred)),
        "total_weight": float(np.sum(w)),
        "source_type": source_type,
        "method_note": (
            "Weighted linear fit on logit(observed_event_rate) vs "
            "logit(mean_predicted_risk) across calibration bins."
        ),
    })

if len(results) == 0:
    raise RuntimeError("No calibration slope/intercept estimates could be produced.")

# ------------------------------
# 5) Main table
# ------------------------------
table_p11_1_calibration_slope_intercept_by_horizon = (
    pd.DataFrame(results)
    .sort_values(["model_id", "horizon_week"])
    .reset_index(drop=True)
)

# ------------------------------
# 6) Summary table
# ------------------------------
summary_rows = []
for model_id, g in table_p11_1_calibration_slope_intercept_by_horizon.groupby("model_id", dropna=False):
    summary_rows.append({
        "model_id": model_id,
        "model": g["model"].iloc[0],
        "family": g["family"].iloc[0],
        "n_horizons_estimated": int(g["horizon_week"].nunique()),
        "mean_calibration_intercept_logit": pd.to_numeric(g["calibration_intercept_logit"], errors="coerce").mean(),
        "mean_calibration_slope_logit": pd.to_numeric(g["calibration_slope_logit"], errors="coerce").mean(),
        "min_calibration_slope_logit": pd.to_numeric(g["calibration_slope_logit"], errors="coerce").min(),
        "max_calibration_slope_logit": pd.to_numeric(g["calibration_slope_logit"], errors="coerce").max(),
        "source_type": g["source_type"].iloc[0],
    })

table_p11_1_calibration_slope_intercept_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("model_id")
    .reset_index(drop=True)
)

# ------------------------------
# 7) Save
# ------------------------------
path_main = OUT_TABLES / "table_p11_1_calibration_slope_intercept_by_horizon.csv"
path_summary = OUT_TABLES / "table_p11_1_calibration_slope_intercept_summary.csv"
path_metadata = OUT_METADATA / "metadata_p11_1_calibration_slope_intercept_audit.json"

table_p11_1_calibration_slope_intercept_by_horizon.to_csv(path_main, index=False)
table_p11_1_calibration_slope_intercept_summary.to_csv(path_summary, index=False)

metadata = {
    "step": "P11.1",
    "title": "Horizon-wise Calibration Slope and Intercept Audit",
    "purpose": (
        "Estimate horizon-specific approximate calibration intercept and slope "
        "from current reliability-bin calibration artifacts."
    ),
    "source_type": source_type,
    "weight_column_used": weight_col,
    "estimation_note": (
        "Approximate weighted linear fit on "
        "logit(observed_event_rate) vs logit(mean_predicted_risk) across calibration bins."
    ),
    "output_tables": [
        path_main.as_posix(),
        path_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------
# 8) Display
# ------------------------------
print(f"\nWeight column used: {weight_col}")

print("\nCalibration slope/intercept by horizon:")
display(table_p11_1_calibration_slope_intercept_by_horizon)

print("\nCalibration slope/intercept summary:")
display(table_p11_1_calibration_slope_intercept_summary)

print("\nSaved:")
print("-", path_main.resolve())
print("-", path_summary.resolve())
print("-", path_metadata.resolve())


# E4 — Build Unified Calibration Strengthening Table

In [ ]:
# ==============================================================
# E4 — Build Unified Calibration Strengthening Table
# REVISED VERSION (aligned to current audit and P11.1 schemas)
# ==============================================================
# Purpose:
#   Consolidate current horizon-wise calibration evidence into
#   a single canonical table for paper/reviewer use.
#
# Inputs:
#   - table_calibration_audit_all_tuned_models.csv
#   - table_p11_1_calibration_slope_intercept_by_horizon.csv
#
# Main outputs:
#   - table_p11_2_calibration_strengthening_by_horizon.csv
#   - table_p11_2_calibration_strengthening_summary.csv
#   - metadata_p11_2_calibration_strengthening.json
#
# Revision note:
#   This version is aligned to the current benchmark schema, using:
#   - calibration_gap or weighted_absolute_calibration_gap_by_horizon
#   - support_n_at_horizon
#   - event_rate_at_horizon
#   - P11.1 logit-scale slope/intercept outputs
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E4 — Build Unified Calibration Strengthening Table")
print("=" * 70)

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Resolve required sources
# --------------------------------------------------------------
path_audit = OUT_TABLES / "table_calibration_audit_all_tuned_models.csv"
path_slope = OUT_TABLES / "table_p11_1_calibration_slope_intercept_by_horizon.csv"

if not path_audit.exists():
    raise FileNotFoundError(f"Missing required file: {path_audit.resolve()}")
if not path_slope.exists():
    raise FileNotFoundError(f"Missing required file: {path_slope.resolve()}")

audit_df = pd.read_csv(path_audit)
slope_df = pd.read_csv(path_slope)

# --------------------------------------------------------------
# 2) Resolve current audit schema
# --------------------------------------------------------------
required_audit_base = ["model_id", "model", "family", "horizon_week"]
missing_base = [c for c in required_audit_base if c not in audit_df.columns]
if missing_base:
    raise KeyError(f"Audit table missing required identifier columns: {missing_base}")

if "weighted_absolute_calibration_gap_by_horizon" in audit_df.columns:
    audit_df = audit_df.copy()
    audit_df["calibration_gap_resolved"] = pd.to_numeric(
        audit_df["weighted_absolute_calibration_gap_by_horizon"], errors="coerce"
    )
elif "calibration_gap" in audit_df.columns:
    audit_df = audit_df.copy()
    audit_df["calibration_gap_resolved"] = pd.to_numeric(
        audit_df["calibration_gap"], errors="coerce"
    )
else:
    raise KeyError(
        "Audit table must contain either "
        "'weighted_absolute_calibration_gap_by_horizon' or 'calibration_gap'."
    )

# Support/event-rate current naming
if "support_n_at_horizon" in audit_df.columns:
    audit_df["support_n_resolved"] = pd.to_numeric(audit_df["support_n_at_horizon"], errors="coerce")
elif "n_evaluable_enrollments" in audit_df.columns:
    audit_df["support_n_resolved"] = pd.to_numeric(audit_df["n_evaluable_enrollments"], errors="coerce")
else:
    raise KeyError(
        "Audit table must contain either 'support_n_at_horizon' or 'n_evaluable_enrollments'."
    )

if "event_rate_at_horizon" in audit_df.columns:
    audit_df["event_rate_resolved"] = pd.to_numeric(audit_df["event_rate_at_horizon"], errors="coerce")
elif "event_rate_by_horizon" in audit_df.columns:
    audit_df["event_rate_resolved"] = pd.to_numeric(audit_df["event_rate_by_horizon"], errors="coerce")
else:
    raise KeyError(
        "Audit table must contain either 'event_rate_at_horizon' or 'event_rate_by_horizon'."
    )

# Optional fields
if "n_events_by_horizon" not in audit_df.columns:
    audit_df["n_events_by_horizon"] = np.nan

if "brier_at_horizon" not in audit_df.columns:
    audit_df["brier_at_horizon"] = np.nan

# Optional auxiliary proxy from current pipeline
if "abs_gap_proxy_from_bins" not in audit_df.columns:
    audit_df["abs_gap_proxy_from_bins"] = np.nan

# --------------------------------------------------------------
# 3) Resolve P11.1 schema
# --------------------------------------------------------------
required_slope_cols = [
    "model_id",
    "model",
    "family",
    "horizon_week",
    "calibration_intercept_logit",
    "calibration_slope_logit",
    "n_bins_used",
]
missing_slope = [c for c in required_slope_cols if c not in slope_df.columns]
if missing_slope:
    raise KeyError(f"Slope/intercept table missing required columns: {missing_slope}")

slope_df = slope_df.copy()
slope_df["horizon_week"] = pd.to_numeric(slope_df["horizon_week"], errors="coerce")
slope_df["calibration_intercept_logit"] = pd.to_numeric(
    slope_df["calibration_intercept_logit"], errors="coerce"
)
slope_df["calibration_slope_logit"] = pd.to_numeric(
    slope_df["calibration_slope_logit"], errors="coerce"
)
slope_df["n_bins_used"] = pd.to_numeric(slope_df["n_bins_used"], errors="coerce")
if "total_weight" in slope_df.columns:
    slope_df["total_weight"] = pd.to_numeric(slope_df["total_weight"], errors="coerce")
else:
    slope_df["total_weight"] = np.nan

# --------------------------------------------------------------
# 4) Prepare canonical merge inputs
# --------------------------------------------------------------
audit_core = audit_df.copy()
audit_core["horizon_week"] = pd.to_numeric(audit_core["horizon_week"], errors="coerce")

# Keep one row per model x horizon
audit_keep_cols = [
    "model_id",
    "model",
    "family",
    "horizon_week",
    "calibration_gap_resolved",
    "support_n_resolved",
    "n_events_by_horizon",
    "event_rate_resolved",
    "brier_at_horizon",
    "abs_gap_proxy_from_bins",
]
audit_core = (
    audit_core[audit_keep_cols]
    .drop_duplicates(subset=["model_id", "model", "family", "horizon_week"])
    .copy()
)

slope_core = (
    slope_df[
        [
            "model_id",
            "model",
            "family",
            "horizon_week",
            "calibration_intercept_logit",
            "calibration_slope_logit",
            "n_bins_used",
            "total_weight",
            "source_type",
            "method_note",
        ]
    ]
    .drop_duplicates(subset=["model_id", "model", "family", "horizon_week"])
    .copy()
)

# --------------------------------------------------------------
# 5) Merge into unified strengthening table
# --------------------------------------------------------------
table_p11_2_calibration_strengthening_by_horizon = audit_core.merge(
    slope_core,
    on=["model_id", "model", "family", "horizon_week"],
    how="left"
)

table_p11_2_calibration_strengthening_by_horizon = table_p11_2_calibration_strengthening_by_horizon.rename(
    columns={
        "calibration_gap_resolved": "calibration_gap",
        "support_n_resolved": "support_n_at_horizon",
        "event_rate_resolved": "event_rate_at_horizon",
        "calibration_intercept_logit": "approx_calibration_intercept_by_horizon",
        "calibration_slope_logit": "approx_calibration_slope_by_horizon",
    }
)


### Funcao: primary_gap_reading

Definicao isolada de primary_gap_reading para reutilizacao nas etapas seguintes.


In [ ]:

# --------------------------------------------------------------
# 6) Diagnostic readings / flags
# --------------------------------------------------------------
def primary_gap_reading(gap):
    if pd.isna(gap):
        return "missing"
    if gap <= 0.04:
        return "strong"
    if gap <= 0.08:
        return "acceptable"
    return "weaker"


### Funcao: slope_reading_logit

Definicao isolada de slope_reading_logit para reutilizacao nas etapas seguintes.


In [ ]:

def slope_reading_logit(s):
    if pd.isna(s):
        return "missing"
    if 0.90 <= s <= 1.10:
        return "near_ideal"
    if 0.75 <= s < 0.90:
        return "mild_compression"
    if 1.10 < s <= 1.25:
        return "mild_overdispersion"
    if s < 0.75:
        return "strong_compression"
    return "strong_overdispersion"


### Funcao: intercept_magnitude_reading

Definicao isolada de intercept_magnitude_reading para reutilizacao nas etapas seguintes.


In [ ]:

def intercept_magnitude_reading(i):
    if pd.isna(i):
        return "missing"
    if abs(i) <= 0.15:
        return "near_ideal"
    if 0.15 < abs(i) <= 0.40:
        return "moderate_shift"
    return "large_shift"


### Funcao: intercept_direction

Definicao isolada de intercept_direction para reutilizacao nas etapas seguintes.


In [ ]:

def intercept_direction(i):
    if pd.isna(i):
        return "missing"
    if abs(i) <= 0.15:
        return "no_clear_bias"
    if i < 0:
        return "systematic_overprediction"
    return "systematic_underprediction"


### Funcao: strengthening_status

Definicao isolada de strengthening_status para reutilizacao nas etapas seguintes.


In [ ]:

def strengthening_status(intercept, slope):
    problems = 0
    if pd.notna(intercept) and abs(intercept) > 0.40:
        problems += 1
    elif pd.notna(intercept) and abs(intercept) > 0.15:
        problems += 0.5

    if pd.notna(slope) and (slope < 0.75 or slope > 1.25):
        problems += 1
    elif pd.notna(slope) and ((0.75 <= slope < 0.90) or (1.10 < slope <= 1.25)):
        problems += 0.5

    if problems == 0:
        return "supportive"
    if problems <= 1:
        return "caution"
    return "problematic"


### Funcao: combined_flag

Definicao isolada de combined_flag para reutilizacao nas etapas seguintes.


In [ ]:

def combined_flag(gap, intercept, slope):
    primary = primary_gap_reading(gap)
    strength = strengthening_status(intercept, slope)

    if primary == "missing":
        return "insufficient_information"
    if primary == "strong" and strength == "supportive":
        return "strong"
    if primary == "strong" and strength in ["caution", "problematic"]:
        return "acceptable_with_caveats"
    if primary == "acceptable" and strength in ["supportive", "caution"]:
        return "acceptable_with_caveats"
    return "weaker_or_problematic"


In [ ]:

table_p11_2_calibration_strengthening_by_horizon["primary_gap_reading"] = (
    table_p11_2_calibration_strengthening_by_horizon["calibration_gap"].apply(primary_gap_reading)
)
table_p11_2_calibration_strengthening_by_horizon["slope_reading"] = (
    table_p11_2_calibration_strengthening_by_horizon["approx_calibration_slope_by_horizon"].apply(slope_reading_logit)
)
table_p11_2_calibration_strengthening_by_horizon["intercept_magnitude_reading"] = (
    table_p11_2_calibration_strengthening_by_horizon["approx_calibration_intercept_by_horizon"].apply(intercept_magnitude_reading)
)
table_p11_2_calibration_strengthening_by_horizon["intercept_direction"] = (
    table_p11_2_calibration_strengthening_by_horizon["approx_calibration_intercept_by_horizon"].apply(intercept_direction)
)
table_p11_2_calibration_strengthening_by_horizon["strengthening_status"] = (
    table_p11_2_calibration_strengthening_by_horizon.apply(
        lambda row: strengthening_status(
            row["approx_calibration_intercept_by_horizon"],
            row["approx_calibration_slope_by_horizon"],
        ),
        axis=1,
    )
)
table_p11_2_calibration_strengthening_by_horizon["calibration_flag"] = (
    table_p11_2_calibration_strengthening_by_horizon.apply(
        lambda row: combined_flag(
            row["calibration_gap"],
            row["approx_calibration_intercept_by_horizon"],
            row["approx_calibration_slope_by_horizon"],
        ),
        axis=1,
    )
)

table_p11_2_calibration_strengthening_by_horizon = (
    table_p11_2_calibration_strengthening_by_horizon
    .sort_values(["horizon_week", "calibration_gap", "brier_at_horizon"], ascending=[True, True, True])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 7) Summary table
# --------------------------------------------------------------
summary_rows = []
for model_id, g in table_p11_2_calibration_strengthening_by_horizon.groupby("model_id", dropna=False):
    share_strong = (g["calibration_flag"] == "strong").mean()
    share_ok = (g["calibration_flag"].isin(["strong", "acceptable_with_caveats"])).mean()

    if share_strong >= 2/3:
        overall_reading = "strong"
    elif share_ok >= 2/3:
        overall_reading = "acceptable_with_caveats"
    else:
        overall_reading = "weaker_or_problematic"

    summary_rows.append({
        "model_id": model_id,
        "model": g["model"].iloc[0],
        "family": g["family"].iloc[0],
        "n_horizons_estimated": int(g["horizon_week"].nunique()),
        "mean_calibration_gap": pd.to_numeric(g["calibration_gap"], errors="coerce").mean(),
        "mean_brier_at_horizon": pd.to_numeric(g["brier_at_horizon"], errors="coerce").mean(),
        "mean_event_rate_at_horizon": pd.to_numeric(g["event_rate_at_horizon"], errors="coerce").mean(),
        "mean_support_n_at_horizon": pd.to_numeric(g["support_n_at_horizon"], errors="coerce").mean(),
        "mean_calibration_intercept_logit": pd.to_numeric(g["approx_calibration_intercept_by_horizon"], errors="coerce").mean(),
        "mean_calibration_slope_logit": pd.to_numeric(g["approx_calibration_slope_by_horizon"], errors="coerce").mean(),
        "min_calibration_slope_logit": pd.to_numeric(g["approx_calibration_slope_by_horizon"], errors="coerce").min(),
        "max_calibration_slope_logit": pd.to_numeric(g["approx_calibration_slope_by_horizon"], errors="coerce").max(),
        "best_horizon_by_calibration_gap": (
            g.loc[pd.to_numeric(g["calibration_gap"], errors="coerce").idxmin(), "horizon_week"]
            if g["calibration_gap"].notna().any() else np.nan
        ),
        "overall_calibration_reading": overall_reading,
        "source_type": g["source_type"].dropna().iloc[0] if g["source_type"].notna().any() else np.nan,
    })

table_p11_2_calibration_strengthening_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["mean_calibration_gap", "mean_brier_at_horizon"], ascending=[True, True])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 8) Save outputs
# --------------------------------------------------------------
path_main = OUT_TABLES / "table_p11_2_calibration_strengthening_by_horizon.csv"
path_summary = OUT_TABLES / "table_p11_2_calibration_strengthening_summary.csv"
path_metadata = OUT_METADATA / "metadata_p11_2_calibration_strengthening.json"

table_p11_2_calibration_strengthening_by_horizon.to_csv(path_main, index=False)
table_p11_2_calibration_strengthening_summary.to_csv(path_summary, index=False)

metadata = {
    "step": "P11.2",
    "title": "Build Unified Calibration Strengthening Table",
    "purpose": (
        "Consolidate calibration gap, support, event rate, "
        "and slope/intercept diagnostics into a single canonical table."
    ),
    "inputs": [
        path_audit.as_posix(),
        path_slope.as_posix(),
    ],
    "resolved_audit_fields": {
        "calibration_gap": "weighted_absolute_calibration_gap_by_horizon or calibration_gap",
        "support_n_at_horizon": "support_n_at_horizon or n_evaluable_enrollments",
        "event_rate_at_horizon": "event_rate_at_horizon or event_rate_by_horizon",
    },
    "output_tables": [
        path_main.as_posix(),
        path_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 9) Display
# --------------------------------------------------------------
print("\nUnified calibration strengthening table:")
display(table_p11_2_calibration_strengthening_by_horizon)

print("\nCalibration strengthening summary:")
display(table_p11_2_calibration_strengthening_summary)

print("\nSaved:")
print("-", path_main.resolve())
print("-", path_summary.resolve())
print("-", path_metadata.resolve())


# E5 — Sensitivity Design Audit

In [ ]:
# ==============================================================
# E5 — Sensitivity Design Audit
# ==============================================================
# Purpose:
#   Audit current sensitivity-design coverage in the notebook
#   and identify what is already present vs. what is still
#   missing for Group D.
#
# Main outputs:
#   - table_p12_0_sensitivity_artifact_inventory.csv
#   - table_p12_0_sensitivity_design_summary.csv
#   - table_p12_0_sensitivity_missing_components.csv
#   - metadata_p12_0_sensitivity_design_audit.json
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E5 — Sensitivity Design Audit")
print("=" * 70)

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_FIGURES = OUT_BASE / "figures"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Inventory potentially relevant sensitivity artifacts
# --------------------------------------------------------------
candidate_files = []

tokens = [
    "sensitivity",
    "ablation",
    "window",
    "horizon",
    "support_by_horizon",
    "brier_by_horizon",
    "risk_auc_by_horizon",
    "calibration_by_horizon",
    "benchmark_",
    "variant",
]

for folder, kind in [
    (OUT_TABLES, "table"),
    (OUT_FIGURES, "figure"),
    (OUT_METADATA, "metadata"),
]:
    if folder.exists():
        for p in sorted(folder.iterdir()):
            name_l = p.name.lower()
            if any(tok in name_l for tok in tokens):
                candidate_files.append({
                    "artifact_name": p.name,
                    "artifact_path": str(p.resolve()),
                    "artifact_type": kind,
                    "exists": True,
                    "size_bytes": p.stat().st_size if p.exists() else np.nan,
                })

table_p12_0_sensitivity_artifact_inventory = pd.DataFrame(candidate_files)

if table_p12_0_sensitivity_artifact_inventory.empty:
    table_p12_0_sensitivity_artifact_inventory = pd.DataFrame([{
        "artifact_name": "(none found)",
        "artifact_path": "",
        "artifact_type": "none",
        "exists": False,
        "size_bytes": np.nan,
    }])

table_names = set(
    table_p12_0_sensitivity_artifact_inventory.loc[
        table_p12_0_sensitivity_artifact_inventory["artifact_type"] == "table",
        "artifact_name"
    ].astype(str).tolist()
)


### Funcao: exists_exact

Definicao isolada de exists_exact para reutilizacao nas etapas seguintes.


In [ ]:

# --------------------------------------------------------------
# 2) Coverage checks for reviewer-requested sensitivity dimensions
# --------------------------------------------------------------
def exists_exact(name: str) -> bool:
    return name in table_names


### Funcao: exists_contains

Definicao isolada de exists_contains para reutilizacao nas etapas seguintes.


In [ ]:

def exists_contains(substr: str) -> bool:
    return any(substr in name for name in table_names)


In [ ]:

has_ablation_continuous = exists_exact("table_ablation_continuous_calibration_by_horizon.csv")
has_ablation_discrete = exists_exact("table_ablation_discrete_calibration_by_horizon.csv")

has_benchmark_auc_horizons = exists_exact("table_benchmark_risk_auc_by_horizon_wide.csv")
has_benchmark_brier_horizons = exists_exact("table_benchmark_brier_by_horizon_wide.csv")
has_benchmark_calib_horizons = exists_exact("table_benchmark_calibration_by_horizon_wide.csv")

has_window_features_logic = True  # by design, because P12 exists in notebook structure
has_horizon_evaluation = has_benchmark_auc_horizons or has_benchmark_brier_horizons or has_benchmark_calib_horizons

# More specific missing reviewer items
has_explicit_window_length_sensitivity = exists_contains("first_2") or exists_contains("first_4") or exists_contains("first_6")
has_explicit_discretization_scheme_sensitivity = exists_contains("km") or exists_contains("quantile") or exists_contains("equidistant")
has_explicit_horizon_choice_sensitivity = has_horizon_evaluation

# --------------------------------------------------------------
# 3) Build corrected design summary
# --------------------------------------------------------------
summary_rows = [
    {
        "sensitivity_dimension": "ablation_currently_present",
        "available": bool(has_ablation_continuous or has_ablation_discrete),
        "notes": "Ablation artifacts are already present in the notebook outputs."
    },
    {
        "sensitivity_dimension": "horizon_wise_benchmark_outputs",
        "available": bool(has_horizon_evaluation),
        "notes": "Benchmark-wide horizon outputs already exist for AUC/Brier/calibration."
    },
    {
        "sensitivity_dimension": "window_feature_design_present",
        "available": bool(has_window_features_logic),
        "notes": "Window-based design is already part of the notebook structure."
    },
    {
        "sensitivity_dimension": "explicit_window_length_sensitivity",
        "available": bool(has_explicit_window_length_sensitivity),
        "notes": "Evidence of multiple early-window lengths should be confirmed structurally, not inferred only from names."
    },
    {
        "sensitivity_dimension": "explicit_discretization_scheme_sensitivity",
        "available": bool(has_explicit_discretization_scheme_sensitivity),
        "notes": "No clear evidence yet of a canonical sensitivity comparison across discretization schemes."
    },
    {
        "sensitivity_dimension": "explicit_horizon_choice_sensitivity",
        "available": bool(has_explicit_horizon_choice_sensitivity),
        "notes": "Horizon-wise outputs exist, but stress-test framing may still need to be made explicit."
    },
]

table_p12_0_sensitivity_design_summary = pd.DataFrame(summary_rows)

# --------------------------------------------------------------
# 4) Missing / next components
# --------------------------------------------------------------
missing_components = [
    {
        "component": "early_window_length_sensitivity",
        "status": (
            "partially_present_but_not_yet_canonical"
            if has_explicit_window_length_sensitivity else
            "still_missing"
        ),
        "why_it_matters": "Reviewer explicitly questioned the choice of early-window length.",
        "recommended_next_step": "Create a canonical sensitivity comparison across at least 2/4/6-week early-window designs.",
        "current_evidence_note": (
            "Some window-specific naming may already exist, but not yet as a canonical sensitivity table."
            if has_explicit_window_length_sensitivity else
            "No canonical early-window sensitivity artifact identified."
        ),
    },
    {
        "component": "discretization_scheme_sensitivity",
        "status": (
            "partially_present_but_not_yet_canonical"
            if has_explicit_discretization_scheme_sensitivity else
            "still_missing"
        ),
        "why_it_matters": "Reviewer explicitly questioned discretization choices.",
        "recommended_next_step": "Create a canonical comparison across discretization schemes or document why one harmonized scheme is retained.",
        "current_evidence_note": (
            "Some discretization-related naming was detected, but not yet as a canonical stress-test artifact."
            if has_explicit_discretization_scheme_sensitivity else
            "No canonical discretization sensitivity artifact identified."
        ),
    },
    {
        "component": "horizon_choice_stress_test",
        "status": (
            "already_partially_present"
            if has_explicit_horizon_choice_sensitivity else
            "still_missing"
        ),
        "why_it_matters": "Reviewer explicitly questioned whether 10/20/30 was stress-tested.",
        "recommended_next_step": "Reframe existing horizon-wise benchmark outputs as an explicit horizon-choice sensitivity analysis.",
        "current_evidence_note": (
            "Horizon-wise outputs already exist, but may need explicit sensitivity framing."
            if has_explicit_horizon_choice_sensitivity else
            "No canonical horizon stress-test artifact identified."
        ),
    },
    {
        "component": "canonical_sensitivity_summary_table",
        "status": "still_missing",
        "why_it_matters": "Needed for clear reporting in the paper and reviewer response.",
        "recommended_next_step": "Build one unified table summarizing sensitivity results across the selected dimensions.",
        "current_evidence_note": "No canonical unified sensitivity table identified yet.",
    },
]

table_p12_0_sensitivity_missing_components = pd.DataFrame(missing_components)

# --------------------------------------------------------------
# 5) Save outputs
# --------------------------------------------------------------
path_inventory = OUT_TABLES / "table_p12_0_sensitivity_artifact_inventory.csv"
path_summary = OUT_TABLES / "table_p12_0_sensitivity_design_summary.csv"
path_missing = OUT_TABLES / "table_p12_0_sensitivity_missing_components.csv"
path_metadata = OUT_METADATA / "metadata_p12_0_sensitivity_design_audit.json"

table_p12_0_sensitivity_artifact_inventory.to_csv(path_inventory, index=False)
table_p12_0_sensitivity_design_summary.to_csv(path_summary, index=False)
table_p12_0_sensitivity_missing_components.to_csv(path_missing, index=False)

metadata = {
    "step": "P12.0",
    "title": "Sensitivity Design Audit",
    "purpose": (
        "Audit current sensitivity-design coverage and identify the next "
        "structural tasks for Group D."
    ),
    "has_ablation_artifacts": bool(has_ablation_continuous or has_ablation_discrete),
    "has_horizon_wise_benchmark_outputs": bool(has_horizon_evaluation),
    "has_window_feature_design_present": bool(has_window_features_logic),
    "output_tables": [
        path_inventory.as_posix(),
        path_summary.as_posix(),
        path_missing.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 6) Display
# --------------------------------------------------------------
print("\nSensitivity artifact inventory:")
display(table_p12_0_sensitivity_artifact_inventory)

print("\nSensitivity design summary:")
display(table_p12_0_sensitivity_design_summary)

print("\nMissing / next sensitivity components:")
display(table_p12_0_sensitivity_missing_components)

print("\nSaved:")
print("-", path_inventory.resolve())
print("-", path_summary.resolve())
print("-", path_missing.resolve())
print("-", path_metadata.resolve())


# E6 — Horizon-Choice Stress-Test Framing

### Funcao: wide_to_long_metric

Definicao isolada de wide_to_long_metric para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# E6 — Horizon-Choice Stress-Test Framing
# ==============================================================
# Purpose:
#   Reframe existing benchmark horizon-wise outputs as an explicit
#   horizon-choice sensitivity analysis for Group D.
#
# Main outputs:
#   - table_p12_3_horizon_choice_stress_test.csv
#   - table_p12_3_horizon_choice_summary.csv
#   - table_p12_3_horizon_choice_registry.csv
#   - metadata_p12_3_horizon_choice_stress_test.json
# ==============================================================

print("\n" + "=" * 70)
print("E6 — Horizon-Choice Stress-Test Framing")
print("=" * 70)

required_names = ["TABLES_DIR", "OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import json
from pathlib import Path
import pandas as pd
import numpy as np

OUT_BASE = Path(OUTPUT_DIR)
OUT_METADATA = OUT_BASE / "metadata"
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Resolve source files
# --------------------------------------------------------------
path_auc = TABLES_DIR / "table_benchmark_risk_auc_by_horizon_wide.csv"
path_brier = TABLES_DIR / "table_benchmark_brier_by_horizon_wide.csv"
path_calib = TABLES_DIR / "table_benchmark_calibration_by_horizon_wide.csv"
path_registry = TABLES_DIR / "table_benchmark_model_registry.csv"

required_files = [path_auc, path_brier, path_calib]
missing_files = [str(p.resolve()) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required benchmark horizon files for P12.3:\n- "
        + "\n- ".join(missing_files)
    )

auc_df = pd.read_csv(path_auc)
brier_df = pd.read_csv(path_brier)
calib_df = pd.read_csv(path_calib)
registry_df = pd.read_csv(path_registry) if path_registry.exists() else None

# --------------------------------------------------------------
# 2) Helper: wide-to-long by h10/h20/h30
# --------------------------------------------------------------
def wide_to_long_metric(df: pd.DataFrame, metric_name: str) -> pd.DataFrame:
    required_base = ["model_id", "display_name", "family", "variant"]
    missing = [c for c in required_base if c not in df.columns]
    if missing:
        raise KeyError(f"{metric_name} table is missing required columns: {missing}")

    horizon_cols = [c for c in df.columns if c.lower().startswith("h")]
    if not horizon_cols:
        raise ValueError(f"{metric_name} table has no horizon columns.")

    out = df.melt(
        id_vars=required_base,
        value_vars=horizon_cols,
        var_name="horizon_label",
        value_name=metric_name,
    ).copy()

    out["horizon_week"] = (
        out["horizon_label"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )
    return out.drop(columns=["horizon_label"])

auc_long = wide_to_long_metric(auc_df, "risk_auc")
brier_long = wide_to_long_metric(brier_df, "brier")
calib_long = wide_to_long_metric(calib_df, "calibration_gap")

# --------------------------------------------------------------
# 3) Merge canonical horizon-stress-test table
# --------------------------------------------------------------
merge_cols = ["model_id", "display_name", "family", "variant", "horizon_week"]

stress_df = auc_long.merge(
    brier_long,
    on=merge_cols,
    how="outer",
    validate="one_to_one",
).merge(
    calib_long,
    on=merge_cols,
    how="outer",
    validate="one_to_one",
)

stress_df = stress_df.sort_values(["model_id", "horizon_week"]).reset_index(drop=True)

# optional registry enrichment
if registry_df is not None and "model_id" in registry_df.columns:
    keep_cols = [c for c in ["model_id", "benchmark_arm", "representation_type", "update_regime", "model_class"] if c in registry_df.columns]
    if keep_cols:
        stress_df = stress_df.merge(
            registry_df[keep_cols].drop_duplicates(subset=["model_id"]),
            on="model_id",
            how="left",
        )

# stress-test framing columns
stress_df["horizon_choice_role"] = stress_df["horizon_week"].map({
    10.0: "short_horizon",
    20.0: "mid_horizon",
    30.0: "long_horizon",
}).fillna("other_horizon")

stress_df["horizon_choice_in_scope"] = stress_df["horizon_week"].isin([10, 20, 30])

# --------------------------------------------------------------
# 4) Registry table
# --------------------------------------------------------------
table_p12_3_horizon_choice_registry = (
    stress_df[
        [
            c for c in stress_df.columns
            if c in [
                "model_id", "display_name", "family", "variant",
                "benchmark_arm", "representation_type", "update_regime", "model_class"
            ]
        ]
    ]
    .drop_duplicates()
    .sort_values(["family", "model_id"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 5) Summary table
# --------------------------------------------------------------
summary_rows = []
for model_id, g in stress_df.groupby("model_id", dropna=False):
    summary_rows.append({
        "model_id": model_id,
        "display_name": g["display_name"].iloc[0] if "display_name" in g.columns else None,
        "family": g["family"].iloc[0] if "family" in g.columns else None,
        "variant": g["variant"].iloc[0] if "variant" in g.columns else None,
        "n_horizons_in_scope": int(pd.to_numeric(g["horizon_choice_in_scope"], errors="coerce").fillna(False).sum()),
        "min_horizon_week": pd.to_numeric(g["horizon_week"], errors="coerce").min(),
        "max_horizon_week": pd.to_numeric(g["horizon_week"], errors="coerce").max(),
        "mean_risk_auc": pd.to_numeric(g["risk_auc"], errors="coerce").mean(),
        "mean_brier": pd.to_numeric(g["brier"], errors="coerce").mean(),
        "mean_calibration_gap": pd.to_numeric(g["calibration_gap"], errors="coerce").mean(),
        "risk_auc_range": (
            pd.to_numeric(g["risk_auc"], errors="coerce").max()
            - pd.to_numeric(g["risk_auc"], errors="coerce").min()
        ),
        "brier_range": (
            pd.to_numeric(g["brier"], errors="coerce").max()
            - pd.to_numeric(g["brier"], errors="coerce").min()
        ),
        "calibration_gap_range": (
            pd.to_numeric(g["calibration_gap"], errors="coerce").max()
            - pd.to_numeric(g["calibration_gap"], errors="coerce").min()
        ),
        "horizon_stress_test_status": (
            "in_scope_complete"
            if set(pd.to_numeric(g["horizon_week"], errors="coerce").dropna().astype(int).tolist()) >= {10, 20, 30}
            else "partial"
        ),
    })

table_p12_3_horizon_choice_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["mean_risk_auc", "mean_calibration_gap"], ascending=[False, True])
    .reset_index(drop=True)
)

table_p12_3_horizon_choice_stress_test = stress_df.copy()

# --------------------------------------------------------------
# 6) Save outputs
# --------------------------------------------------------------
path_main = TABLES_DIR / "table_p12_3_horizon_choice_stress_test.csv"
path_summary = TABLES_DIR / "table_p12_3_horizon_choice_summary.csv"
path_registry_out = TABLES_DIR / "table_p12_3_horizon_choice_registry.csv"
path_metadata = OUT_METADATA / "metadata_p12_3_horizon_choice_stress_test.json"

table_p12_3_horizon_choice_stress_test.to_csv(path_main, index=False)
table_p12_3_horizon_choice_summary.to_csv(path_summary, index=False)
table_p12_3_horizon_choice_registry.to_csv(path_registry_out, index=False)

metadata = {
    "step": "P12.3",
    "title": "Horizon-Choice Stress-Test Framing",
    "purpose": (
        "Reframe existing benchmark horizon-wise outputs as an explicit "
        "horizon-choice sensitivity analysis for Group D."
    ),
    "horizons_in_scope": [10, 20, 30],
    "source_tables": [
        path_auc.as_posix(),
        path_brier.as_posix(),
        path_calib.as_posix(),
    ],
    "output_tables": [
        path_main.as_posix(),
        path_summary.as_posix(),
        path_registry_out.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 7) Display
# --------------------------------------------------------------
print("\nHorizon-choice registry:")
display(table_p12_3_horizon_choice_registry)

print("\nHorizon-choice stress-test table:")
display(table_p12_3_horizon_choice_stress_test)

print("\nHorizon-choice summary:")
display(table_p12_3_horizon_choice_summary)

print("\nSaved:")
print("-", path_main.resolve())
print("-", path_summary.resolve())
print("-", path_registry_out.resolve())
print("-", path_metadata.resolve())

# E7 — Build Unified Sensitivity Summary Table

In [ ]:
# ==============================================================
# E7 — Build Unified Sensitivity Summary Table
# FULL REWRITTEN VERSION WITH EXPLICIT GRANULARITY
# ==============================================================
# Purpose:
#   Consolidate Group D sensitivity evidence into a canonical
#   summary table for reviewer response and paper revision,
#   while avoiding ambiguous duplication across family-level
#   and model-level rows.
#
# Main outputs:
#   - table_p12_4_sensitivity_unified_summary.csv
#   - table_p12_4_sensitivity_component_registry.csv
#   - table_p12_4_sensitivity_family_summary.csv
#   - metadata_p12_4_sensitivity_unified_summary.json
# ==============================================================

print("\n" + "=" * 70)
print("E7 — Build Unified Sensitivity Summary Table")
print("=" * 70)

required_names = ["TABLES_DIR", "OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import json
from pathlib import Path
import pandas as pd
import numpy as np

OUT_BASE = Path(OUTPUT_DIR)
OUT_METADATA = OUT_BASE / "metadata"
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Resolve required inputs
# --------------------------------------------------------------
path_window_registry = TABLES_DIR / "table_p12_2_window_sensitivity_registry.csv"
path_window_summary = TABLES_DIR / "table_p12_2_window_sensitivity_summary.csv"
path_horizon_summary = TABLES_DIR / "table_p12_3_horizon_choice_summary.csv"
path_horizon_stress = TABLES_DIR / "table_p12_3_horizon_choice_stress_test.csv"

required_files = [
    path_window_registry,
    path_window_summary,
    path_horizon_summary,
    path_horizon_stress,
]
missing_files = [str(p.resolve()) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required Group D files for P12.4:\n- " + "\n- ".join(missing_files)
    )

window_registry_df = pd.read_csv(path_window_registry)
window_summary_df = pd.read_csv(path_window_summary)
horizon_summary_df = pd.read_csv(path_horizon_summary)
horizon_stress_df = pd.read_csv(path_horizon_stress)

# Optional ablation artifacts
optional_ablation_files = {
    "table_ablation_leaderboard_all_tuned.csv": TABLES_DIR / "table_ablation_leaderboard_all_tuned.csv",
    "table_ablation_summary_by_model.csv": TABLES_DIR / "table_ablation_summary_by_model.csv",
    "table_ablation_scenario_comparison.csv": TABLES_DIR / "table_ablation_scenario_comparison.csv",
    "table_ablation_continuous_primary_metrics.csv": TABLES_DIR / "table_ablation_continuous_primary_metrics.csv",
    "table_ablation_discrete_primary_metrics.csv": TABLES_DIR / "table_ablation_discrete_primary_metrics.csv",
}
ablation_presence = {name: path.exists() for name, path in optional_ablation_files.items()}

# --------------------------------------------------------------
# 2) Component registry
# --------------------------------------------------------------
component_rows = []

# 2A) Early-window components -> family-level
for _, r in window_summary_df.iterrows():
    component_rows.append({
        "entity_level": "family",
        "entity_id": r.get("family"),
        "display_name": None,
        "family": r.get("family"),
        "variant": None,
        "sensitivity_component": "early_window_length",
        "scope_label": "2_4_6_week_design",
        "status": (
            "materialized_and_ready"
            if bool(r.get("all_windows_ready_for_model_training", False))
            else "partially_ready"
        ),
        "n_expected_units": r.get("n_windows_expected"),
        "n_materialized_units": r.get("n_windows_materialized"),
        "n_ready_units": r.get("n_windows_ready_for_model_training"),
        "reference_value": "4-week main window",
        "notes": (
            f"Materialized windows: {r.get('window_list_materialized', '')}. "
            "Structural design ready for explicit window-length sensitivity."
        ),
    })

# 2B) Horizon components -> model-level
for _, r in horizon_summary_df.iterrows():
    component_rows.append({
        "entity_level": "model",
        "entity_id": r.get("model_id"),
        "display_name": r.get("display_name"),
        "family": r.get("family"),
        "variant": r.get("variant"),
        "sensitivity_component": "horizon_choice",
        "scope_label": "10_20_30_horizon_stress_test",
        "status": r.get("horizon_stress_test_status"),
        "n_expected_units": 3,
        "n_materialized_units": r.get("n_horizons_in_scope"),
        "n_ready_units": r.get("n_horizons_in_scope"),
        "reference_value": "10/20/30 benchmark horizons",
        "notes": (
            f"Mean AUC={r.get('mean_risk_auc'):.6f}, "
            f"Mean Brier={r.get('mean_brier'):.6f}, "
            f"Mean calibration gap={r.get('mean_calibration_gap'):.6f}."
            if pd.notna(r.get("mean_risk_auc", np.nan))
            else "Horizon-wise benchmark outputs consolidated."
        ),
    })

# 2C) Ablation component -> global-level
component_rows.append({
    "entity_level": "global",
    "entity_id": "global",
    "display_name": None,
    "family": "global",
    "variant": None,
    "sensitivity_component": "ablation",
    "scope_label": "existing_ablation_artifacts",
    "status": "present" if any(ablation_presence.values()) else "not_detected",
    "n_expected_units": len(optional_ablation_files),
    "n_materialized_units": int(sum(ablation_presence.values())),
    "n_ready_units": int(sum(ablation_presence.values())),
    "reference_value": "existing ablation outputs",
    "notes": (
        "Detected ablation artifacts: "
        + ", ".join([name for name, present in ablation_presence.items() if present])
        if any(ablation_presence.values()) else
        "No optional ablation artifacts detected."
    ),
})

table_p12_4_sensitivity_component_registry = (
    pd.DataFrame(component_rows)
    .sort_values(["entity_level", "family", "entity_id", "sensitivity_component"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 3) Unified summary table
# --------------------------------------------------------------
summary_rows = []

# 3A) family-level early-window summary
for _, r in window_summary_df.iterrows():
    summary_rows.append({
        "entity_level": "family",
        "entity_id": r.get("family"),
        "display_name": None,
        "family": r.get("family"),
        "variant": None,
        "summary_layer": "early_window_length",
        "reference_design": "4-week main window",
        "scope": "2/4/6 weeks",
        "status": (
            "ready"
            if bool(r.get("all_windows_ready_for_model_training", False))
            else "partial"
        ),
        "n_units_in_scope": r.get("n_windows_expected"),
        "n_units_materialized": r.get("n_windows_materialized"),
        "key_signal_1": r.get("window_list_materialized"),
        "key_signal_2": "all_windows_ready_for_model_training=" + str(r.get("all_windows_ready_for_model_training")),
        "notes": "Canonical early-window sensitivity design is structurally available.",
    })

# 3B) model-level horizon-choice summary
for _, r in horizon_summary_df.iterrows():
    summary_rows.append({
        "entity_level": "model",
        "entity_id": r.get("model_id"),
        "display_name": r.get("display_name"),
        "family": r.get("family"),
        "variant": r.get("variant"),
        "summary_layer": "horizon_choice",
        "reference_design": "10/20/30 benchmark horizons",
        "scope": "10/20/30",
        "status": r.get("horizon_stress_test_status"),
        "n_units_in_scope": 3,
        "n_units_materialized": r.get("n_horizons_in_scope"),
        "key_signal_1": r.get("mean_risk_auc"),
        "key_signal_2": r.get("mean_calibration_gap"),
        "notes": "Existing horizon-wise outputs are explicitly framed here as a horizon-choice stress test.",
    })

# 3C) global ablation summary
summary_rows.append({
    "entity_level": "global",
    "entity_id": "global",
    "display_name": None,
    "family": "global",
    "variant": None,
    "summary_layer": "ablation",
    "reference_design": "existing ablation protocol",
    "scope": "current ablation outputs",
    "status": "present" if any(ablation_presence.values()) else "not_detected",
    "n_units_in_scope": len(optional_ablation_files),
    "n_units_materialized": int(sum(ablation_presence.values())),
    "key_signal_1": int(sum(ablation_presence.values())),
    "key_signal_2": len(optional_ablation_files),
    "notes": (
        "Ablation evidence already exists and can be cited as part of Group D."
        if any(ablation_presence.values()) else
        "No ablation artifacts detected."
    ),
})

table_p12_4_sensitivity_unified_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["entity_level", "family", "entity_id", "summary_layer"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 4) Family-level summary
# --------------------------------------------------------------
family_rows = []
for family, g in table_p12_4_sensitivity_unified_summary.groupby("family", dropna=False):
    family_rows.append({
        "family": family,
        "n_rows": int(len(g)),
        "n_unique_entities": int(g["entity_id"].astype(str).nunique()),
        "n_summary_layers": int(g["summary_layer"].astype(str).nunique()),
        "n_ready_or_present_rows": int(g["status"].astype(str).isin(["ready", "present", "in_scope_complete"]).sum()),
        "all_rows_nonempty": bool(len(g) > 0),
        "entity_levels_present": ", ".join(sorted(g["entity_level"].astype(str).unique().tolist())),
        "summary_layers_present": ", ".join(sorted(g["summary_layer"].astype(str).unique().tolist())),
    })

table_p12_4_sensitivity_family_summary = (
    pd.DataFrame(family_rows)
    .sort_values("family")
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 5) Save outputs
# --------------------------------------------------------------
path_main = TABLES_DIR / "table_p12_4_sensitivity_unified_summary.csv"
path_registry = TABLES_DIR / "table_p12_4_sensitivity_component_registry.csv"
path_family = TABLES_DIR / "table_p12_4_sensitivity_family_summary.csv"
path_metadata = OUT_METADATA / "metadata_p12_4_sensitivity_unified_summary.json"

table_p12_4_sensitivity_unified_summary.to_csv(path_main, index=False)
table_p12_4_sensitivity_component_registry.to_csv(path_registry, index=False)
table_p12_4_sensitivity_family_summary.to_csv(path_family, index=False)

metadata = {
    "step": "P12.4",
    "title": "Build Unified Sensitivity Summary Table",
    "purpose": (
        "Consolidate Group D sensitivity evidence across early-window design, "
        "horizon-choice stress test, and existing ablation artifacts, with explicit granularity."
    ),
    "granularity_rule": {
        "early_window_length": "family-level",
        "horizon_choice": "model-level",
        "ablation": "global-level"
    },
    "input_tables": [
        path_window_registry.as_posix(),
        path_window_summary.as_posix(),
        path_horizon_summary.as_posix(),
        path_horizon_stress.as_posix(),
    ],
    "optional_ablation_artifacts_detected": [
        name for name, present in ablation_presence.items() if present
    ],
    "output_tables": [
        path_main.as_posix(),
        path_registry.as_posix(),
        path_family.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 6) Display
# --------------------------------------------------------------
print("\nSensitivity component registry:")
display(table_p12_4_sensitivity_component_registry)

print("\nUnified sensitivity summary:")
display(table_p12_4_sensitivity_unified_summary)

print("\nSensitivity family summary:")
display(table_p12_4_sensitivity_family_summary)

print("\nSaved:")
print("-", path_main.resolve())
print("-", path_registry.resolve())
print("-", path_family.resolve())
print("-", path_metadata.resolve())

# F — Ablation and Stability Evidence

This section contains the tuned-only ablation, stability, and reviewer-facing robustness layers that feed the manuscript appendix and the final paper freeze.

# F1 — Define Ablation Study Design and Protocol

In [ ]:
# ==============================================================
# F1 — Define Ablation Study Design and Protocol
# --------------------------------------------------------------
# Purpose:
#   Define the ablation study configuration, feature-block logic,
#   evaluation protocol, and documentation layer for the benchmark.
#
# Methodological note:
#   This step does not train any model. It only formalizes the
#   ablation study that will be executed in subsequent steps.
#
#   The ablation study is restricted to the tuned models only, so
#   that each family is evaluated in its strongest benchmark form.
#
#   Models included:
#     - linear_tuned
#     - neural_tuned
#     - cox_tuned
#     - deepsurv_tuned
#
#   Main goal:
#     quantify how much performance depends on different feature
#     blocks, especially:
#       - static covariates
#       - early-window behavior summaries
#       - dynamic person-period temporal signals
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - HORIZONS_WEEKS
#
# Outputs:
#   - ablation_config.json
#   - table_ablation_model_registry.csv
#   - table_ablation_feature_blocks.csv
#   - table_ablation_scenarios.csv
# ==============================================================

print("\n" + "=" * 70)
print("F1 — Define Ablation Study Design and Protocol")
print("=" * 70)
print("Methodological note: this step defines the ablation study only.")
print("No model is trained here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json", "HORIZONS_WEEKS"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd

# ------------------------------
# 2) Define tuned models included in ablation
# ------------------------------
ABLATION_MODEL_REGISTRY = [
    {
        "model_id": "linear_tuned",
        "display_name": "Linear Discrete-Time Hazard (Tuned)",
        "family": "discrete_time_linear",
        "base_data_level": "person_period",
        "uses_dynamic_temporal_features": True,
        "uses_static_features": True,
        "uses_early_window_features": False,
        "ablation_positioning_note": "Tuned representative of the linear discrete-time family.",
    },
    {
        "model_id": "neural_tuned",
        "display_name": "Neural Discrete-Time Survival (Tuned)",
        "family": "discrete_time_neural",
        "base_data_level": "person_period",
        "uses_dynamic_temporal_features": True,
        "uses_static_features": True,
        "uses_early_window_features": False,
        "ablation_positioning_note": "Tuned representative of the neural discrete-time family.",
    },
    {
        "model_id": "cox_tuned",
        "display_name": "Cox Comparable (Tuned)",
        "family": "continuous_time_cox",
        "base_data_level": "enrollment",
        "uses_dynamic_temporal_features": False,
        "uses_static_features": True,
        "uses_early_window_features": True,
        "ablation_positioning_note": "Tuned representative of the Cox comparable family.",
    },
    {
        "model_id": "deepsurv_tuned",
        "display_name": "DeepSurv (Tuned)",
        "family": "continuous_time_deepsurv",
        "base_data_level": "enrollment",
        "uses_dynamic_temporal_features": False,
        "uses_static_features": True,
        "uses_early_window_features": True,
        "ablation_positioning_note": "Tuned representative of the DeepSurv family.",
    },
]

ablation_model_registry_df = pd.DataFrame(ABLATION_MODEL_REGISTRY)

# ------------------------------
# 3) Define feature blocks
# ------------------------------
ABLATION_FEATURE_BLOCKS = [
    {
        "block_id": "static_structural",
        "block_label": "Static structural covariates",
        "applies_to_data_level": "both",
        "conceptual_role": "student background and enrollment-level structure",
        "examples": "gender, region, highest_education, imd_band, age_band, num_of_prev_attempts, studied_credits, disability",
    },
    {
        "block_id": "dynamic_temporal",
        "block_label": "Dynamic temporal person-period signals",
        "applies_to_data_level": "person_period",
        "conceptual_role": "time-varying weekly engagement and progression",
        "examples": "week, total_clicks_week, active_this_week, n_vle_rows_week, n_distinct_sites_week, cum_clicks_until_t, recency, streak",
    },
    {
        "block_id": "early_window_behavior",
        "block_label": "Early-window behavior summaries",
        "applies_to_data_level": "enrollment",
        "conceptual_role": "compressed early-course activity profile",
        "examples": "clicks_first_4_weeks, active_weeks_first_4, mean_clicks_first_4_weeks",
    },
]

ablation_feature_blocks_df = pd.DataFrame(ABLATION_FEATURE_BLOCKS)

# ------------------------------
# 4) Define ablation scenarios
# ------------------------------
# Strategy:
#   For each family, compare:
#   - full tuned model
#   - remove structural/static block
#   - remove temporal/early-window block
#   - only structural/static block
#   - only temporal/early-window block
#
# Note:
#   The exact executable datasets will be materialized later.
ABLATION_SCENARIOS = [
    {
        "scenario_id": "full_features",
        "scenario_label": "Full tuned feature set",
        "scenario_type": "reference",
        "interpretation_goal": "Reference tuned benchmark for each family.",
    },
    {
        "scenario_id": "drop_static_structural",
        "scenario_label": "Drop static structural covariates",
        "scenario_type": "leave_one_block_out",
        "interpretation_goal": "Estimate how much performance depends on background/structural covariates.",
    },
    {
        "scenario_id": "drop_temporal_signal",
        "scenario_label": "Drop temporal signal block",
        "scenario_type": "leave_one_block_out",
        "interpretation_goal": "Estimate how much performance depends on behavioral engagement dynamics.",
    },
    {
        "scenario_id": "only_static_structural",
        "scenario_label": "Only static structural covariates",
        "scenario_type": "single_block_only",
        "interpretation_goal": "Assess how far structural covariates alone can go.",
    },
    {
        "scenario_id": "only_temporal_signal",
        "scenario_label": "Only temporal signal block",
        "scenario_type": "single_block_only",
        "interpretation_goal": "Assess how far behavior/activity signals alone can go.",
    },
]

ablation_scenarios_df = pd.DataFrame(ABLATION_SCENARIOS)

# ------------------------------
# 5) Protocol definition
# ------------------------------
ABLATION_PROTOCOL = {
    "ablation_scope": "tuned_models_only",
    "included_models": [item["model_id"] for item in ABLATION_MODEL_REGISTRY],
    "main_question": (
        "How much of model performance is driven by structural covariates versus temporal/behavioral information?"
    ),
    "secondary_question": (
        "Does the relative importance of feature blocks vary across model families?"
    ),
    "evaluation_protocol": {
        "reuse_existing_train_test_split": True,
        "reuse_existing_horizons": [int(h) for h in HORIZONS_WEEKS],
        "reuse_existing_primary_metrics": [
            "ibs",
            "c_index",
            "brier_at_horizon",
            "calibration_at_horizon",
            "risk_auc_at_horizon",
        ],
        "primary_comparison_logic": (
            "Compare each ablation scenario against the full tuned version within the same model family."
        ),
    },
    "interpretation_rules": {
        "large_drop_after_removal": "The removed block is important for that family.",
        "small_drop_after_removal": "The removed block contributes little incremental signal.",
        "strong_only_block_performance": "That block alone carries substantial predictive information.",
        "cross_family_difference": (
            "Different model families may exploit the same information blocks differently."
        ),
    },
    "paper_positioning_note": (
        "The ablation study is intended to explain performance sources after the benchmark ranking is already established."
    ),
}

# ------------------------------
# 6) Save outputs
# ------------------------------
model_registry_path = TABLES_DIR / "table_ablation_model_registry.csv"
feature_blocks_path = TABLES_DIR / "table_ablation_feature_blocks.csv"
scenarios_path = TABLES_DIR / "table_ablation_scenarios.csv"
config_path = METADATA_DIR / "ablation_config.json"

ablation_model_registry_df.to_csv(model_registry_path, index=False)
ablation_feature_blocks_df.to_csv(feature_blocks_path, index=False)
ablation_scenarios_df.to_csv(scenarios_path, index=False)
save_json(ABLATION_PROTOCOL, config_path)

# ------------------------------
# 7) Output for feedback
# ------------------------------
print("\nAblation model registry:")
display(ablation_model_registry_df)

print("\nAblation feature blocks:")
display(ablation_feature_blocks_df)

print("\nAblation scenarios:")
display(ablation_scenarios_df)

print("\nAblation protocol summary:")
display(pd.DataFrame([{
    "ablation_scope": ABLATION_PROTOCOL["ablation_scope"],
    "included_models": ", ".join(ABLATION_PROTOCOL["included_models"]),
    "reuse_existing_train_test_split": ABLATION_PROTOCOL["evaluation_protocol"]["reuse_existing_train_test_split"],
    "reuse_existing_horizons": ", ".join(str(x) for x in ABLATION_PROTOCOL["evaluation_protocol"]["reuse_existing_horizons"]),
    "main_question": ABLATION_PROTOCOL["main_question"],
    "secondary_question": ABLATION_PROTOCOL["secondary_question"],
}]))

print("\nSaved:")
print("-", model_registry_path.resolve())
print("-", feature_blocks_path.resolve())
print("-", scenarios_path.resolve())
print("-", config_path.resolve())

# F2 — Materialize Ablation Variants (Revised v2)

### Funcao: load_ready_dataset

Definicao isolada de load_ready_dataset para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# F2 — Materialize Ablation Variants (Revised v2)
# --------------------------------------------------------------
# Purpose:
#   Materialize ablation-ready dataset variants for the tuned
#   benchmark models, without training them yet.
#
# Methodological note:
#   This revised version corrects the discrete-time ablation logic:
#
#   - for discrete-time models, "week" is treated as a structural
#     time index and is ALWAYS retained;
#   - the ablative temporal block for discrete-time families
#     therefore excludes "week" and only includes behavioral
#     temporal signals.
#
#   This preserves the temporal hazard structure required for
#   discrete-time survival reconstruction.
# ==============================================================

print("\n" + "=" * 70)
print("F2 — Materialize Ablation Variants (Revised v2)")
print("=" * 70)
print("Methodological note: this step materializes ablation-ready")
print("feature subsets only. No model is trained here.")
print("Revised behavior: ready datasets are loaded from disk.")
print("Important correction: in discrete-time families, 'week' is")
print("always retained as a structural time index.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import pandas as pd

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Helper to load dataset
# ------------------------------
def load_ready_dataset(base_name: str) -> pd.DataFrame:
    parquet_path = DATA_OUTPUT_DIR / f"{base_name}.parquet"
    csv_path = DATA_OUTPUT_DIR / f"{base_name}.csv"

    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)

    raise FileNotFoundError(
        f"Could not find dataset for '{base_name}'. "
        f"Checked:\n- {parquet_path}\n- {csv_path}"
    )

# ------------------------------
# 3) Define feature blocks
# ------------------------------
STATIC_STRUCTURAL_FEATURES = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
]

# For discrete-time models, "week" is NOT ablated away.
DISCRETE_TIME_INDEX_FEATURES = [
    "week",
]

DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES = [
    "total_clicks_week",
    "active_this_week",
    "n_vle_rows_week",
    "n_distinct_sites_week",
    "cum_clicks_until_t",
    "recency",
    "streak",
]

EARLY_WINDOW_FEATURES = [
    "clicks_first_4_weeks",
    "active_weeks_first_4",
    "mean_clicks_first_4_weeks",
]

# ------------------------------
# 4) Define auxiliary / target columns
# ------------------------------
AUX_DISCRETE = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "event_observed",
    "t_event_week",
    "t_final_week",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_DISCRETE = ["event_t"]

AUX_ENROLLMENT = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "duration",
    "duration_raw",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_ENROLLMENT = ["event"]

# ------------------------------
# 5) Scenario map by family
# ------------------------------
SCENARIO_MAP = {
    "full_features": {
        "discrete_time": STATIC_STRUCTURAL_FEATURES + DISCRETE_TIME_INDEX_FEATURES + DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES,
        "enrollment": STATIC_STRUCTURAL_FEATURES + EARLY_WINDOW_FEATURES,
    },
    "drop_static_structural": {
        "discrete_time": DISCRETE_TIME_INDEX_FEATURES + DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES,
        "enrollment": EARLY_WINDOW_FEATURES,
    },
    "drop_temporal_signal": {
        "discrete_time": STATIC_STRUCTURAL_FEATURES + DISCRETE_TIME_INDEX_FEATURES,
        "enrollment": STATIC_STRUCTURAL_FEATURES,
    },
    "only_static_structural": {
        "discrete_time": STATIC_STRUCTURAL_FEATURES + DISCRETE_TIME_INDEX_FEATURES,
        "enrollment": STATIC_STRUCTURAL_FEATURES,
    },
    "only_temporal_signal": {
        "discrete_time": DISCRETE_TIME_INDEX_FEATURES + DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES,
        "enrollment": EARLY_WINDOW_FEATURES,
    },
}

SCENARIO_LABELS = {
    "full_features": "Full tuned feature set",
    "drop_static_structural": "Drop static structural covariates",
    "drop_temporal_signal": "Drop temporal signal block",
    "only_static_structural": "Only static structural covariates",
    "only_temporal_signal": "Only temporal signal block",
}

# ------------------------------
# 6) Dataset registry loaded from disk
# ------------------------------
DATASET_REGISTRY = [
    {
        "model_id": "linear_tuned",
        "family": "discrete_time_linear",
        "data_level": "discrete_time",
        "train_df": load_ready_dataset("pp_linear_hazard_ready_train"),
        "test_df": load_ready_dataset("pp_linear_hazard_ready_test"),
        "aux_cols": AUX_DISCRETE,
        "target_cols": TARGET_DISCRETE,
    },
    {
        "model_id": "neural_tuned",
        "family": "discrete_time_neural",
        "data_level": "discrete_time",
        "train_df": load_ready_dataset("pp_neural_hazard_ready_train"),
        "test_df": load_ready_dataset("pp_neural_hazard_ready_test"),
        "aux_cols": AUX_DISCRETE,
        "target_cols": TARGET_DISCRETE,
    },
    {
        "model_id": "cox_tuned",
        "family": "continuous_time_cox",
        "data_level": "enrollment",
        "train_df": load_ready_dataset("enrollment_cox_ready_train"),
        "test_df": load_ready_dataset("enrollment_cox_ready_test"),
        "aux_cols": AUX_ENROLLMENT,
        "target_cols": TARGET_ENROLLMENT,
    },
    {
        "model_id": "deepsurv_tuned",
        "family": "continuous_time_deepsurv",
        "data_level": "enrollment",
        "train_df": load_ready_dataset("enrollment_deepsurv_ready_train"),
        "test_df": load_ready_dataset("enrollment_deepsurv_ready_test"),
        "aux_cols": AUX_ENROLLMENT,
        "target_cols": TARGET_ENROLLMENT,
    },
]

# ------------------------------
# 7) Materialize variants
# ------------------------------
variant_registry_rows = []
variant_feature_manifest_rows = []

for item in DATASET_REGISTRY:
    model_id = item["model_id"]
    family = item["family"]
    data_level = item["data_level"]
    train_df = item["train_df"]
    test_df = item["test_df"]
    aux_cols = [c for c in item["aux_cols"] if c in train_df.columns]
    target_cols = [c for c in item["target_cols"] if c in train_df.columns]

    for scenario_id, scenario_def in SCENARIO_MAP.items():
        feature_cols = [c for c in scenario_def[data_level] if c in train_df.columns]

        ordered_cols = aux_cols + target_cols + feature_cols
        ordered_cols = [c for c in ordered_cols if c in train_df.columns]

        train_variant = train_df[ordered_cols].copy()
        test_variant = test_df[ordered_cols].copy()

        variant_base_name = f"{model_id}__{scenario_id}"
        train_name = f"{variant_base_name}__train"
        test_name = f"{variant_base_name}__test"

        train_csv = DATA_OUTPUT_DIR / f"{train_name}.csv"
        train_parquet = DATA_OUTPUT_DIR / f"{train_name}.parquet"
        test_csv = DATA_OUTPUT_DIR / f"{test_name}.csv"
        test_parquet = DATA_OUTPUT_DIR / f"{test_name}.parquet"

        train_variant.to_csv(train_csv, index=False)
        test_variant.to_csv(test_csv, index=False)
        train_variant.to_parquet(train_parquet, index=False)
        test_variant.to_parquet(test_parquet, index=False)

        variant_registry_rows.append({
            "model_id": model_id,
            "family": family,
            "data_level": data_level,
            "scenario_id": scenario_id,
            "scenario_label": SCENARIO_LABELS[scenario_id],
            "train_table_name": train_name,
            "test_table_name": test_name,
            "n_train_rows": int(train_variant.shape[0]),
            "n_test_rows": int(test_variant.shape[0]),
            "n_columns_total": int(len(ordered_cols)),
            "n_aux_columns": int(len(aux_cols)),
            "n_target_columns": int(len(target_cols)),
            "n_feature_columns": int(len(feature_cols)),
            "train_csv_path": str(train_csv),
            "train_parquet_path": str(train_parquet),
            "test_csv_path": str(test_csv),
            "test_parquet_path": str(test_parquet),
        })

        for col in ordered_cols:
            if col in aux_cols:
                role = "auxiliary"
            elif col in target_cols:
                role = "target"
            else:
                role = "feature"

            if col in STATIC_STRUCTURAL_FEATURES:
                block = "static_structural"
            elif col in DISCRETE_TIME_INDEX_FEATURES:
                block = "discrete_time_index"
            elif col in DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES:
                block = "dynamic_temporal_behavioral"
            elif col in EARLY_WINDOW_FEATURES:
                block = "early_window_behavior"
            else:
                block = "aux_or_target"

            variant_feature_manifest_rows.append({
                "model_id": model_id,
                "family": family,
                "data_level": data_level,
                "scenario_id": scenario_id,
                "scenario_label": SCENARIO_LABELS[scenario_id],
                "column_name": col,
                "role": role,
                "feature_block": block,
            })

variant_registry_df = pd.DataFrame(variant_registry_rows)
variant_feature_manifest_df = pd.DataFrame(variant_feature_manifest_rows)

# ------------------------------
# 8) Save registry artifacts
# ------------------------------
variant_registry_path = TABLES_DIR / "table_ablation_variant_registry.csv"
variant_feature_manifest_path = TABLES_DIR / "table_ablation_variant_feature_manifest.csv"
variant_registry_json_path = METADATA_DIR / "ablation_variant_registry.json"

variant_registry_df.to_csv(variant_registry_path, index=False)
variant_feature_manifest_df.to_csv(variant_feature_manifest_path, index=False)

save_json(
    {
        "n_variants_materialized": int(variant_registry_df.shape[0]),
        "models_included": sorted(variant_registry_df["model_id"].unique().tolist()),
        "scenarios_included": sorted(variant_registry_df["scenario_id"].unique().tolist()),
        "data_levels": sorted(variant_registry_df["data_level"].unique().tolist()),
        "discrete_time_week_always_retained": True,
    },
    variant_registry_json_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nAblation variant registry:")
display(variant_registry_df)

print("\nAblation variant feature manifest (first rows):")
display(variant_feature_manifest_df.head(60))

print("\nSaved:")
print("-", variant_registry_path.resolve())
print("-", variant_feature_manifest_path.resolve())
print("-", variant_registry_json_path.resolve())

# F3 — Train and Evaluate Ablation Variants for Linear and Neural Tuned Models (Revised)

In [ ]:
# ==============================================================
# F3 — Train and Evaluate Ablation Variants for Linear and Neural Tuned Models (Revised)
# --------------------------------------------------------------
# Purpose:
#   Train and evaluate the ablation variants for the tuned
#   discrete-time model families:
#     - linear_tuned
#     - neural_tuned
#
# Methodological note:
#   This revised version is compatible with F2 Revised v2:
#   - "week" is always retained for discrete-time families as a
#     structural time index;
#   - the ablative temporal block refers only to behavioral
#     temporal signals, not to the time index itself.
# ==============================================================

print("\n" + "=" * 70)
print("F3 — Train and Evaluate Ablation Variants for Linear and Neural Tuned Models (Revised)")
print("=" * 70)
print("Methodological note: this step trains and evaluates ablation")
print("variants for the tuned discrete-time families only.")
print("Revised behavior: 'week' is always retained as a structural time index.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED", "save_json"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P29.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
except Exception:
    pass

# ------------------------------
# 3) Registry / paths
# ------------------------------
DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
ABLATION_REGISTRY_PATH = TABLES_DIR / "table_ablation_variant_registry.csv"

if not ABLATION_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing ablation registry: {ABLATION_REGISTRY_PATH}")

ablation_registry = pd.read_csv(ABLATION_REGISTRY_PATH)

TARGET_MODELS = ["linear_tuned", "neural_tuned"]

ablation_registry_dt = ablation_registry[
    ablation_registry["model_id"].isin(TARGET_MODELS)
].copy()

# ------------------------------
# 4) Column definitions
# ------------------------------
AUX_DISCRETE = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "event_observed",
    "t_event_week",
    "t_final_week",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_COL = "event_t"

CATEGORICAL_FEATURES = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]


### Funcao: load_variant

Definicao isolada de load_variant para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 5) Helpers
# ------------------------------
def load_variant(path_csv: str, path_parquet: str) -> pd.DataFrame:
    p_parquet = Path(path_parquet)
    p_csv = Path(path_csv)
    if p_parquet.exists():
        return pd.read_parquet(p_parquet)
    if p_csv.exists():
        return pd.read_csv(p_csv)
    raise FileNotFoundError(f"Variant file not found:\n- {p_parquet}\n- {p_csv}")


### Funcao: get_feature_columns

Definicao isolada de get_feature_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_columns(df: pd.DataFrame):
    excluded = set(AUX_DISCRETE + [TARGET_COL])
    return [c for c in df.columns if c not in excluded]


### Funcao: split_feature_types

Definicao isolada de split_feature_types para reutilizacao nas etapas seguintes.


In [ ]:

def split_feature_types(feature_cols):
    categorical_cols = [c for c in feature_cols if c in CATEGORICAL_FEATURES]
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]
    return numeric_cols, categorical_cols


### Funcao: build_preprocessor

Definicao isolada de build_preprocessor para reutilizacao nas etapas seguintes.


In [ ]:

def build_preprocessor(feature_cols):
    numeric_cols, categorical_cols = split_feature_types(feature_cols)

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )
    return preprocessor, numeric_cols, categorical_cols


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]


### Funcao: evaluate_discrete_survival

Definicao isolada de evaluate_discrete_survival para reutilizacao nas etapas seguintes.


In [ ]:

def evaluate_discrete_survival(test_pred_df: pd.DataFrame, horizons: list[int]):
    required_cols = ["enrollment_id", "week", "pred_hazard", "event_observed", "time_for_split"]
    missing_cols = [c for c in required_cols if c not in test_pred_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns for discrete survival evaluation: {missing_cols}")

    test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
    test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
        lambda s: (1.0 - s).cumprod()
    )
    test_pred_df["pred_risk"] = 1.0 - test_pred_df["pred_survival"]

    truth_test = (
        test_pred_df.groupby("enrollment_id", as_index=False)
        .agg(
            event=("event_observed", "max"),
            duration=("time_for_split", "max"),
        )
    )

    surv_wide = (
        test_pred_df[["enrollment_id", "week", "pred_survival"]]
        .drop_duplicates(subset=["enrollment_id", "week"])
        .pivot(index="week", columns="enrollment_id", values="pred_survival")
        .sort_index()
    )

    max_week_test = int(test_pred_df["week"].max())
    full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
    surv_wide = surv_wide.reindex(full_week_index).ffill().fillna(1.0)

    surv_df = surv_wide.copy()
    surv_df.columns.name = "enrollment_id"

    durations_test = truth_test["duration"].astype(int).to_numpy()
    events_test = truth_test["event"].astype(int).to_numpy()

    eval_surv = EvalSurv(
        surv=surv_df,
        durations=durations_test,
        events=events_test,
        censor_surv="km",
    )

    primary_rows = []
    try:
        c_index = float(eval_surv.concordance_td())
        c_index_note = "pycox_concordance_td"
    except Exception as e:
        c_index = np.nan
        c_index_note = f"failed: {str(e)}"

    try:
        max_requested_horizon = int(max(horizons))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
        ibs_note = "pycox_integrated_brier_score"
    except Exception as e:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)}"

    primary_rows.append({"metric_name": "ibs", "metric_value": ibs_value, "notes": ibs_note})
    primary_rows.append({"metric_name": "c_index", "metric_value": c_index, "notes": c_index_note})
    primary_df = pd.DataFrame(primary_rows)

    try:
        brier_h = eval_surv.brier_score(np.array(horizons, dtype=int))
        brier_df = pd.DataFrame({
            "horizon_week": list(brier_h.index.astype(int)),
            "metric_name": ["brier_at_horizon"] * len(brier_h.index),
            "metric_value": list(brier_h.values.astype(float)),
            "notes": ["pycox_brier_score"] * len(brier_h.index),
        })
    except Exception as e:
        brier_df = pd.DataFrame({
            "horizon_week": horizons,
            "metric_name": ["brier_at_horizon"] * len(horizons),
            "metric_value": [np.nan] * len(horizons),
            "notes": [f"failed: {str(e)}"] * len(horizons),
        })

    support_rows = []
    calibration_rows = []
    pred_vs_obs_rows = []
    risk_auc_rows = []

    for h in horizons:
        pred_surv_h = get_surv_at_horizon(surv_df, h)
        pred_risk_h = 1.0 - pred_surv_h

        eval_df = truth_test.copy()
        eval_df["pred_survival_h"] = eval_df["enrollment_id"].map(pred_surv_h.to_dict())
        eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

        eval_df["is_evaluable_at_h"] = (
            ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
            (eval_df["duration"] >= h)
        ).astype(int)

        eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
        eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
        eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

        support_rows.append({
            "horizon_week": h,
            "n_evaluable_enrollments": int(eval_df.shape[0]),
            "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
            "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        })

        if eval_df["observed_event_by_h"].nunique() >= 2:
            risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
        else:
            risk_auc = np.nan

        risk_auc_rows.append({
            "horizon_week": h,
            "metric_name": "risk_auc_at_horizon",
            "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
            "notes": "roc_auc_on_evaluable_subset",
        })

        pred_vs_obs_rows.append({
            "horizon_week": h,
            "n_evaluable_enrollments": int(eval_df.shape[0]),
            "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
            "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
            "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
        })

        if eval_df.shape[0] > 0:
            ranked = eval_df["pred_risk_h"].rank(method="first")
            n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))
            eval_df["calibration_bin"] = pd.qcut(
                ranked,
                q=n_bins_eff,
                labels=False,
                duplicates="drop"
            )

            calib_tab = (
                eval_df.groupby("calibration_bin")
                .agg(
                    n=("enrollment_id", "count"),
                    mean_predicted_risk=("pred_risk_h", "mean"),
                    observed_event_rate=("observed_event_by_h", "mean"),
                )
                .reset_index()
            )
            calib_tab["horizon_week"] = h
            calib_tab["abs_calibration_gap"] = (
                calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
            ).abs()

            calibration_rows.append({
                "horizon_week": h,
                "metric_name": "calibration_at_horizon",
                "metric_value": float(np.average(calib_tab["abs_calibration_gap"], weights=calib_tab["n"])),
                "notes": "Weighted absolute calibration gap across bins",
            })

    return {
        "primary": primary_df,
        "brier": pd.DataFrame(brier_df),
        "calibration": pd.DataFrame(calibration_rows),
        "secondary": pd.DataFrame(risk_auc_rows),
        "support": pd.DataFrame(support_rows),
        "pred_vs_obs": pd.DataFrame(pred_vs_obs_rows),
    }


In [ ]:

# ------------------------------
# 6) Train/evaluate ablation variants
# ------------------------------
results_primary = []
results_brier = []
results_calibration = []
results_secondary = []
results_support = []
results_pred_vs_obs = []
results_training_audit = []

for _, reg_row in ablation_registry_dt.iterrows():
    model_id = reg_row["model_id"]
    scenario_id = reg_row["scenario_id"]
    scenario_label = reg_row["scenario_label"]

    train_df = load_variant(reg_row["train_csv_path"], reg_row["train_parquet_path"])
    test_df = load_variant(reg_row["test_csv_path"], reg_row["test_parquet_path"])

    if "week" not in train_df.columns or "week" not in test_df.columns:
        raise ValueError(
            f"Scenario {model_id} / {scenario_id} does not contain 'week'. "
            "Discrete-time evaluation requires the structural time index."
        )

    feature_cols = get_feature_columns(train_df)
    preprocessor, numeric_cols, categorical_cols = build_preprocessor(feature_cols)

    X_train = preprocessor.fit_transform(train_df[feature_cols])
    X_test = preprocessor.transform(test_df[feature_cols])

    y_train = train_df[TARGET_COL].to_numpy()
    y_test = test_df[TARGET_COL].to_numpy()

    if hasattr(X_train, "toarray"):
        X_train_dense = X_train.toarray()
        X_test_dense = X_test.toarray()
    else:
        X_train_dense = X_train
        X_test_dense = X_test

    # ---------- model fit ----------
    if model_id == "linear_tuned":
        model = LogisticRegression(
            penalty="l1",
            C=0.1,
            solver="liblinear",
            class_weight=None,
            max_iter=2000,
            random_state=RANDOM_SEED,
        )
        model.fit(X_train_dense, y_train)
        pred_hazard = np.clip(model.predict_proba(X_test_dense)[:, 1], 1e-8, 1 - 1e-8)

    elif model_id == "neural_tuned":
        torch.manual_seed(RANDOM_SEED)
        np.random.seed(RANDOM_SEED)

        X_train_t = X_train_dense.astype(np.float32)
        X_test_t = X_test_dense.astype(np.float32)
        y_train_t = y_train.astype(np.float32)

        net = tt.practical.MLPVanilla(
            in_features=X_train_t.shape[1],
            num_nodes=[128, 64],
            out_features=1,
            batch_norm=True,
            dropout=0.1,
            output_bias=False,
        )
        model = tt.Model(net, torch.nn.BCEWithLogitsLoss(), tt.optim.AdamW)
        model.optimizer.set_lr(5e-4)

        _ = model.fit(
            X_train_t,
            y_train_t.reshape(-1, 1),
            batch_size=256,
            epochs=25,
            verbose=False,
        )

        logits = model.predict(X_test_t).reshape(-1)
        pred_hazard = 1.0 / (1.0 + np.exp(-logits))
        pred_hazard = np.clip(pred_hazard, 1e-8, 1 - 1e-8)

    else:
        raise ValueError(f"Unsupported model_id in P29: {model_id}")

    test_pred_df = test_df.copy()
    test_pred_df["pred_hazard"] = pred_hazard

    eval_outputs = evaluate_discrete_survival(test_pred_df, HORIZONS_WEEKS)

    primary_df = eval_outputs["primary"].copy()
    primary_df["model_id"] = model_id
    primary_df["scenario_id"] = scenario_id
    primary_df["scenario_label"] = scenario_label
    results_primary.append(primary_df)

    brier_df = eval_outputs["brier"].copy()
    brier_df["model_id"] = model_id
    brier_df["scenario_id"] = scenario_id
    brier_df["scenario_label"] = scenario_label
    results_brier.append(brier_df)

    calibration_df = eval_outputs["calibration"].copy()
    calibration_df["model_id"] = model_id
    calibration_df["scenario_id"] = scenario_id
    calibration_df["scenario_label"] = scenario_label
    results_calibration.append(calibration_df)

    secondary_df = eval_outputs["secondary"].copy()
    secondary_df["model_id"] = model_id
    secondary_df["scenario_id"] = scenario_id
    secondary_df["scenario_label"] = scenario_label
    results_secondary.append(secondary_df)

    support_df = eval_outputs["support"].copy()
    support_df["model_id"] = model_id
    support_df["scenario_id"] = scenario_id
    support_df["scenario_label"] = scenario_label
    results_support.append(support_df)

    pred_vs_obs_df = eval_outputs["pred_vs_obs"].copy()
    pred_vs_obs_df["model_id"] = model_id
    pred_vs_obs_df["scenario_id"] = scenario_id
    pred_vs_obs_df["scenario_label"] = scenario_label
    results_pred_vs_obs.append(pred_vs_obs_df)

    results_training_audit.append({
        "model_id": model_id,
        "scenario_id": scenario_id,
        "scenario_label": scenario_label,
        "n_train_rows": int(train_df.shape[0]),
        "n_test_rows": int(test_df.shape[0]),
        "n_feature_columns_raw": int(len(feature_cols)),
        "n_numeric_features_raw": int(len(numeric_cols)),
        "n_categorical_features_raw": int(len(categorical_cols)),
        "n_features_after_transform": int(X_train_dense.shape[1]),
        "week_retained": bool("week" in feature_cols),
    })

# ------------------------------
# 7) Consolidate outputs
# ------------------------------
ablation_primary_df = pd.concat(results_primary, ignore_index=True)
ablation_brier_df = pd.concat(results_brier, ignore_index=True)
ablation_calibration_df = pd.concat(results_calibration, ignore_index=True)
ablation_secondary_df = pd.concat(results_secondary, ignore_index=True)
ablation_support_df = pd.concat(results_support, ignore_index=True)
ablation_pred_vs_obs_df = pd.concat(results_pred_vs_obs, ignore_index=True)
ablation_training_audit_df = pd.DataFrame(results_training_audit)

ablation_leaderboard_rows = []
for (model_id, scenario_id, scenario_label), g in ablation_primary_df.groupby(["model_id", "scenario_id", "scenario_label"]):
    row = {
        "model_id": model_id,
        "scenario_id": scenario_id,
        "scenario_label": scenario_label,
        "ibs": float(g.loc[g["metric_name"] == "ibs", "metric_value"].iloc[0]),
        "c_index": float(g.loc[g["metric_name"] == "c_index", "metric_value"].iloc[0]),
    }
    for h in HORIZONS_WEEKS:
        row[f"brier_h{h}"] = float(
            ablation_brier_df[
                (ablation_brier_df["model_id"] == model_id) &
                (ablation_brier_df["scenario_id"] == scenario_id) &
                (ablation_brier_df["horizon_week"] == h)
            ]["metric_value"].iloc[0]
        )
        row[f"calibration_h{h}"] = float(
            ablation_calibration_df[
                (ablation_calibration_df["model_id"] == model_id) &
                (ablation_calibration_df["scenario_id"] == scenario_id) &
                (ablation_calibration_df["horizon_week"] == h)
            ]["metric_value"].iloc[0]
        )
        row[f"risk_auc_h{h}"] = float(
            ablation_secondary_df[
                (ablation_secondary_df["model_id"] == model_id) &
                (ablation_secondary_df["scenario_id"] == scenario_id) &
                (ablation_secondary_df["horizon_week"] == h) &
                (ablation_secondary_df["metric_name"] == "risk_auc_at_horizon")
            ]["metric_value"].iloc[0]
        )
    ablation_leaderboard_rows.append(row)

ablation_leaderboard_df = pd.DataFrame(ablation_leaderboard_rows).sort_values(
    by=["model_id", "ibs", "c_index"],
    ascending=[True, True, False]
).reset_index(drop=True)

delta_rows = []
for model_id in sorted(ablation_leaderboard_df["model_id"].unique()):
    ref = ablation_leaderboard_df[
        (ablation_leaderboard_df["model_id"] == model_id) &
        (ablation_leaderboard_df["scenario_id"] == "full_features")
    ].iloc[0]

    sub_df = ablation_leaderboard_df[ablation_leaderboard_df["model_id"] == model_id].copy()
    for _, r in sub_df.iterrows():
        delta = {
            "model_id": model_id,
            "scenario_id": r["scenario_id"],
            "scenario_label": r["scenario_label"],
            "delta_ibs_vs_full": float(r["ibs"] - ref["ibs"]),
            "delta_c_index_vs_full": float(r["c_index"] - ref["c_index"]),
        }
        for h in HORIZONS_WEEKS:
            delta[f"delta_brier_h{h}_vs_full"] = float(r[f"brier_h{h}"] - ref[f"brier_h{h}"])
            delta[f"delta_calibration_h{h}_vs_full"] = float(r[f"calibration_h{h}"] - ref[f"calibration_h{h}"])
            delta[f"delta_risk_auc_h{h}_vs_full"] = float(r[f"risk_auc_h{h}"] - ref[f"risk_auc_h{h}"])
        delta_rows.append(delta)

ablation_delta_vs_full_df = pd.DataFrame(delta_rows).sort_values(
    by=["model_id", "scenario_id"]
).reset_index(drop=True)

# ------------------------------
# 8) Save artifacts
# ------------------------------
primary_path = TABLES_DIR / "table_ablation_discrete_primary_metrics.csv"
brier_path = TABLES_DIR / "table_ablation_discrete_brier_by_horizon.csv"
calibration_path = TABLES_DIR / "table_ablation_discrete_calibration_by_horizon.csv"
secondary_path = TABLES_DIR / "table_ablation_discrete_secondary_metrics.csv"
support_path = TABLES_DIR / "table_ablation_discrete_support_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_ablation_discrete_predicted_vs_observed_survival.csv"
audit_path = TABLES_DIR / "table_ablation_discrete_training_audit.csv"
leaderboard_path = TABLES_DIR / "table_ablation_discrete_leaderboard.csv"
delta_path = TABLES_DIR / "table_ablation_discrete_delta_vs_full.csv"
config_path = METADATA_DIR / "ablation_discrete_run_summary.json"

ablation_primary_df.to_csv(primary_path, index=False)
ablation_brier_df.to_csv(brier_path, index=False)
ablation_calibration_df.to_csv(calibration_path, index=False)
ablation_secondary_df.to_csv(secondary_path, index=False)
ablation_support_df.to_csv(support_path, index=False)
ablation_pred_vs_obs_df.to_csv(pred_vs_obs_path, index=False)
ablation_training_audit_df.to_csv(audit_path, index=False)
ablation_leaderboard_df.to_csv(leaderboard_path, index=False)
ablation_delta_vs_full_df.to_csv(delta_path, index=False)

save_json(
    {
        "models_run": sorted(ablation_leaderboard_df["model_id"].unique().tolist()),
        "scenarios_run": sorted(ablation_leaderboard_df["scenario_id"].unique().tolist()),
        "horizons": [int(h) for h in HORIZONS_WEEKS],
        "n_total_runs": int(ablation_leaderboard_df.shape[0]),
        "week_always_retained_for_discrete_time": True,
    },
    config_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nAblation discrete training audit:")
display(ablation_training_audit_df)

print("\nAblation discrete leaderboard:")
display(ablation_leaderboard_df)

print("\nAblation discrete delta vs full:")
display(ablation_delta_vs_full_df)

print("\nSaved:")
print("-", primary_path.resolve())
print("-", brier_path.resolve())
print("-", calibration_path.resolve())
print("-", secondary_path.resolve())
print("-", support_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", audit_path.resolve())
print("-", leaderboard_path.resolve())
print("-", delta_path.resolve())
print("-", config_path.resolve())


# F4 — Train and Evaluate Ablation Variants for Cox and DeepSurv Tuned Models

In [ ]:
# ==============================================================
# F4 — Train and Evaluate Ablation Variants for Cox and DeepSurv Tuned Models
# --------------------------------------------------------------
# Purpose:
#   Train and evaluate the ablation variants for the tuned
#   continuous-time model families:
#     - cox_tuned
#     - deepsurv_tuned
#
# Methodological note:
#   This step reuses the benchmark split and the benchmark
#   evaluation protocol. The objective is to measure how much
#   performance changes when static or early-window feature
#   blocks are removed or isolated.
# ==============================================================

print("\n" + "=" * 70)
print("F4 — Train and Evaluate Ablation Variants for Cox and DeepSurv Tuned Models")
print("=" * 70)
print("Methodological note: this step trains and evaluates ablation")
print("variants for the tuned continuous-time families only.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED", "save_json"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from lifelines import CoxPHFitter
    LIFELINES_AVAILABLE = True
except Exception:
    LIFELINES_AVAILABLE = False

try:
    from pycox.evaluation import EvalSurv
    from pycox.models import CoxPH
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

if not LIFELINES_AVAILABLE:
    raise ImportError("lifelines is required for P30.")

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P30.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
except Exception:
    pass

# ------------------------------
# 3) Registry / paths
# ------------------------------
DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "models"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ABLATION_REGISTRY_PATH = TABLES_DIR / "table_ablation_variant_registry.csv"

if not ABLATION_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing ablation registry: {ABLATION_REGISTRY_PATH}")

ablation_registry = pd.read_csv(ABLATION_REGISTRY_PATH)

TARGET_MODELS = ["cox_tuned", "deepsurv_tuned"]

ablation_registry_ct = ablation_registry[
    ablation_registry["model_id"].isin(TARGET_MODELS)
].copy()

# ------------------------------
# 4) Column definitions
# ------------------------------
AUX_ENROLLMENT = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "duration",
    "duration_raw",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_COL = "event"
DURATION_COL = "duration"

CATEGORICAL_FEATURES = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]


### Funcao: load_variant

Definicao isolada de load_variant para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 5) Helpers
# ------------------------------
def load_variant(path_csv: str, path_parquet: str) -> pd.DataFrame:
    p_parquet = Path(path_parquet)
    p_csv = Path(path_csv)
    if p_parquet.exists():
        return pd.read_parquet(p_parquet)
    if p_csv.exists():
        return pd.read_csv(p_csv)
    raise FileNotFoundError(f"Variant file not found:\n- {p_parquet}\n- {p_csv}")


### Funcao: get_feature_columns

Definicao isolada de get_feature_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_columns(df: pd.DataFrame):
    excluded = set(AUX_ENROLLMENT + [TARGET_COL])
    return [c for c in df.columns if c not in excluded]


### Funcao: split_feature_types

Definicao isolada de split_feature_types para reutilizacao nas etapas seguintes.


In [ ]:

def split_feature_types(feature_cols):
    categorical_cols = [c for c in feature_cols if c in CATEGORICAL_FEATURES]
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]
    return numeric_cols, categorical_cols


### Funcao: build_preprocessor

Definicao isolada de build_preprocessor para reutilizacao nas etapas seguintes.


In [ ]:

def build_preprocessor(feature_cols):
    numeric_cols, categorical_cols = split_feature_types(feature_cols)

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )
    return preprocessor, numeric_cols, categorical_cols


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]


### Funcao: evaluate_continuous_survival

Definicao isolada de evaluate_continuous_survival para reutilizacao nas etapas seguintes.


In [ ]:

def evaluate_continuous_survival(surv_df: pd.DataFrame, truth_test: pd.DataFrame, horizons: list[int]):
    durations_test = truth_test["duration"].astype(int).to_numpy()
    events_test = truth_test["event"].astype(int).to_numpy()

    eval_surv = EvalSurv(
        surv=surv_df,
        durations=durations_test,
        events=events_test,
        censor_surv="km",
    )

    primary_rows = []
    try:
        c_index = float(eval_surv.concordance_td())
        c_index_note = "pycox_concordance_td"
    except Exception as e:
        c_index = np.nan
        c_index_note = f"failed: {str(e)}"

    try:
        max_requested_horizon = int(max(horizons))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
        ibs_note = "pycox_integrated_brier_score"
    except Exception as e:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)}"

    primary_rows.append({"metric_name": "ibs", "metric_value": ibs_value, "notes": ibs_note})
    primary_rows.append({"metric_name": "c_index", "metric_value": c_index, "notes": c_index_note})
    primary_df = pd.DataFrame(primary_rows)

    try:
        brier_h = eval_surv.brier_score(np.array(horizons, dtype=int))
        brier_df = pd.DataFrame({
            "horizon_week": list(brier_h.index.astype(int)),
            "metric_name": ["brier_at_horizon"] * len(brier_h.index),
            "metric_value": list(brier_h.values.astype(float)),
            "notes": ["pycox_brier_score"] * len(brier_h.index),
        })
    except Exception as e:
        brier_df = pd.DataFrame({
            "horizon_week": horizons,
            "metric_name": ["brier_at_horizon"] * len(horizons),
            "metric_value": [np.nan] * len(horizons),
            "notes": [f"failed: {str(e)}"] * len(horizons),
        })

    support_rows = []
    calibration_rows = []
    pred_vs_obs_rows = []
    risk_auc_rows = []

    for h in horizons:
        pred_surv_h = get_surv_at_horizon(surv_df, h)
        pred_risk_h = 1.0 - pred_surv_h

        eval_df = truth_test.copy()
        eval_df["pred_survival_h"] = eval_df["enrollment_id"].map(pred_surv_h.to_dict())
        eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

        eval_df["is_evaluable_at_h"] = (
            ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
            (eval_df["duration"] >= h)
        ).astype(int)

        eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
        eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
        eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

        support_rows.append({
            "horizon_week": h,
            "n_evaluable_enrollments": int(eval_df.shape[0]),
            "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
            "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        })

        if eval_df["observed_event_by_h"].nunique() >= 2:
            risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
        else:
            risk_auc = np.nan

        risk_auc_rows.append({
            "horizon_week": h,
            "metric_name": "risk_auc_at_horizon",
            "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
            "notes": "roc_auc_on_evaluable_subset",
        })

        pred_vs_obs_rows.append({
            "horizon_week": h,
            "n_evaluable_enrollments": int(eval_df.shape[0]),
            "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
            "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
            "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
        })

        if eval_df.shape[0] > 0:
            ranked = eval_df["pred_risk_h"].rank(method="first")
            n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))
            eval_df["calibration_bin"] = pd.qcut(
                ranked,
                q=n_bins_eff,
                labels=False,
                duplicates="drop"
            )

            calib_tab = (
                eval_df.groupby("calibration_bin")
                .agg(
                    n=("enrollment_id", "count"),
                    mean_predicted_risk=("pred_risk_h", "mean"),
                    observed_event_rate=("observed_event_by_h", "mean"),
                )
                .reset_index()
            )
            calib_tab["horizon_week"] = h
            calib_tab["abs_calibration_gap"] = (
                calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
            ).abs()

            calibration_rows.append({
                "horizon_week": h,
                "metric_name": "calibration_at_horizon",
                "metric_value": float(np.average(calib_tab["abs_calibration_gap"], weights=calib_tab["n"])),
                "notes": "Weighted absolute calibration gap across bins",
            })

    return {
        "primary": primary_df,
        "brier": pd.DataFrame(brier_df),
        "calibration": pd.DataFrame(calibration_rows),
        "secondary": pd.DataFrame(risk_auc_rows),
        "support": pd.DataFrame(support_rows),
        "pred_vs_obs": pd.DataFrame(pred_vs_obs_rows),
    }


In [ ]:

# ------------------------------
# 6) Train/evaluate ablation variants
# ------------------------------
results_primary = []
results_brier = []
results_calibration = []
results_secondary = []
results_support = []
results_pred_vs_obs = []
results_training_audit = []

for _, reg_row in ablation_registry_ct.iterrows():
    model_id = reg_row["model_id"]
    scenario_id = reg_row["scenario_id"]
    scenario_label = reg_row["scenario_label"]

    train_df = load_variant(reg_row["train_csv_path"], reg_row["train_parquet_path"])
    test_df = load_variant(reg_row["test_csv_path"], reg_row["test_parquet_path"])

    feature_cols = get_feature_columns(train_df)
    preprocessor, numeric_cols, categorical_cols = build_preprocessor(feature_cols)

    X_train = preprocessor.fit_transform(train_df[feature_cols])
    X_test = preprocessor.transform(test_df[feature_cols])

    if hasattr(X_train, "toarray"):
        X_train_dense = X_train.toarray().astype(np.float32)
        X_test_dense = X_test.toarray().astype(np.float32)
    else:
        X_train_dense = np.asarray(X_train).astype(np.float32)
        X_test_dense = np.asarray(X_test).astype(np.float32)

    y_train_event = train_df[TARGET_COL].to_numpy().astype(np.int32)
    y_test_event = test_df[TARGET_COL].to_numpy().astype(np.int32)
    y_train_duration = train_df[DURATION_COL].to_numpy().astype(np.float32)
    y_test_duration = test_df[DURATION_COL].to_numpy().astype(np.float32)

    # ---------- model fit ----------
    if model_id == "cox_tuned":
        feature_names = [f"x{i}" for i in range(X_train_dense.shape[1])]
        train_fit_df = pd.DataFrame(X_train_dense, columns=feature_names)
        train_fit_df["duration"] = y_train_duration
        train_fit_df["event"] = y_train_event

        # train-only zero-variance filter
        stds = train_fit_df[feature_names].std(axis=0, ddof=0)
        keep_feature_names = stds[stds > 0].index.tolist()

        train_fit_df = train_fit_df[keep_feature_names + ["duration", "event"]].copy()
        test_fit_df = pd.DataFrame(X_test_dense, columns=feature_names)[keep_feature_names].copy()

        model = CoxPHFitter(
            penalizer=0.001,
            l1_ratio=0.0,
        )
        model.fit(
            train_fit_df,
            duration_col="duration",
            event_col="event",
            show_progress=False,
        )

        surv_df = model.predict_survival_function(
            test_fit_df,
            times=np.arange(0, int(np.max(y_test_duration)) + 1, dtype=int)
        ).copy()

    elif model_id == "deepsurv_tuned":
        torch.manual_seed(RANDOM_SEED)
        np.random.seed(RANDOM_SEED)

        net = tt.practical.MLPVanilla(
            in_features=X_train_dense.shape[1],
            num_nodes=[64, 32],
            out_features=1,
            batch_norm=True,
            dropout=0.3,
            output_bias=False,
        )

        model = CoxPH(net, tt.optim.Adam)
        model.optimizer.set_lr(5e-4)

        _ = model.fit(
            X_train_dense,
            (y_train_duration, y_train_event),
            batch_size=256,
            epochs=55,
            verbose=False,
        )

        _ = model.compute_baseline_hazards()
        surv_df = model.predict_surv_df(X_test_dense)

    else:
        raise ValueError(f"Unsupported model_id in P30: {model_id}")

    surv_df.columns = test_df["enrollment_id"].tolist()
    surv_df.columns.name = "enrollment_id"
    surv_df.index = surv_df.index.astype(int)

    truth_test = test_df[["enrollment_id", "event", "duration"]].copy()

    eval_outputs = evaluate_continuous_survival(surv_df, truth_test, HORIZONS_WEEKS)

    primary_df = eval_outputs["primary"].copy()
    primary_df["model_id"] = model_id
    primary_df["scenario_id"] = scenario_id
    primary_df["scenario_label"] = scenario_label
    results_primary.append(primary_df)

    brier_df = eval_outputs["brier"].copy()
    brier_df["model_id"] = model_id
    brier_df["scenario_id"] = scenario_id
    brier_df["scenario_label"] = scenario_label
    results_brier.append(brier_df)

    calibration_df = eval_outputs["calibration"].copy()
    calibration_df["model_id"] = model_id
    calibration_df["scenario_id"] = scenario_id
    calibration_df["scenario_label"] = scenario_label
    results_calibration.append(calibration_df)

    secondary_df = eval_outputs["secondary"].copy()
    secondary_df["model_id"] = model_id
    secondary_df["scenario_id"] = scenario_id
    secondary_df["scenario_label"] = scenario_label
    results_secondary.append(secondary_df)

    support_df = eval_outputs["support"].copy()
    support_df["model_id"] = model_id
    support_df["scenario_id"] = scenario_id
    support_df["scenario_label"] = scenario_label
    results_support.append(support_df)

    pred_vs_obs_df = eval_outputs["pred_vs_obs"].copy()
    pred_vs_obs_df["model_id"] = model_id
    pred_vs_obs_df["scenario_id"] = scenario_id
    pred_vs_obs_df["scenario_label"] = scenario_label
    results_pred_vs_obs.append(pred_vs_obs_df)

    results_training_audit.append({
        "model_id": model_id,
        "scenario_id": scenario_id,
        "scenario_label": scenario_label,
        "n_train_rows": int(train_df.shape[0]),
        "n_test_rows": int(test_df.shape[0]),
        "n_feature_columns_raw": int(len(feature_cols)),
        "n_numeric_features_raw": int(len(numeric_cols)),
        "n_categorical_features_raw": int(len(categorical_cols)),
        "n_features_after_transform": int(X_train_dense.shape[1]),
    })

# ------------------------------
# 7) Consolidate outputs
# ------------------------------
ablation_primary_df = pd.concat(results_primary, ignore_index=True)
ablation_brier_df = pd.concat(results_brier, ignore_index=True)
ablation_calibration_df = pd.concat(results_calibration, ignore_index=True)
ablation_secondary_df = pd.concat(results_secondary, ignore_index=True)
ablation_support_df = pd.concat(results_support, ignore_index=True)
ablation_pred_vs_obs_df = pd.concat(results_pred_vs_obs, ignore_index=True)
ablation_training_audit_df = pd.DataFrame(results_training_audit)

ablation_leaderboard_rows = []
for (model_id, scenario_id, scenario_label), g in ablation_primary_df.groupby(["model_id", "scenario_id", "scenario_label"]):
    row = {
        "model_id": model_id,
        "scenario_id": scenario_id,
        "scenario_label": scenario_label,
        "ibs": float(g.loc[g["metric_name"] == "ibs", "metric_value"].iloc[0]),
        "c_index": float(g.loc[g["metric_name"] == "c_index", "metric_value"].iloc[0]),
    }
    for h in HORIZONS_WEEKS:
        row[f"brier_h{h}"] = float(
            ablation_brier_df[
                (ablation_brier_df["model_id"] == model_id) &
                (ablation_brier_df["scenario_id"] == scenario_id) &
                (ablation_brier_df["horizon_week"] == h)
            ]["metric_value"].iloc[0]
        )
        row[f"calibration_h{h}"] = float(
            ablation_calibration_df[
                (ablation_calibration_df["model_id"] == model_id) &
                (ablation_calibration_df["scenario_id"] == scenario_id) &
                (ablation_calibration_df["horizon_week"] == h)
            ]["metric_value"].iloc[0]
        )
        row[f"risk_auc_h{h}"] = float(
            ablation_secondary_df[
                (ablation_secondary_df["model_id"] == model_id) &
                (ablation_secondary_df["scenario_id"] == scenario_id) &
                (ablation_secondary_df["horizon_week"] == h) &
                (ablation_secondary_df["metric_name"] == "risk_auc_at_horizon")
            ]["metric_value"].iloc[0]
        )
    ablation_leaderboard_rows.append(row)

ablation_leaderboard_df = pd.DataFrame(ablation_leaderboard_rows).sort_values(
    by=["model_id", "ibs", "c_index"],
    ascending=[True, True, False]
).reset_index(drop=True)

delta_rows = []
for model_id in sorted(ablation_leaderboard_df["model_id"].unique()):
    ref = ablation_leaderboard_df[
        (ablation_leaderboard_df["model_id"] == model_id) &
        (ablation_leaderboard_df["scenario_id"] == "full_features")
    ].iloc[0]

    sub_df = ablation_leaderboard_df[ablation_leaderboard_df["model_id"] == model_id].copy()
    for _, r in sub_df.iterrows():
        delta = {
            "model_id": model_id,
            "scenario_id": r["scenario_id"],
            "scenario_label": r["scenario_label"],
            "delta_ibs_vs_full": float(r["ibs"] - ref["ibs"]),
            "delta_c_index_vs_full": float(r["c_index"] - ref["c_index"]),
        }
        for h in HORIZONS_WEEKS:
            delta[f"delta_brier_h{h}_vs_full"] = float(r[f"brier_h{h}"] - ref[f"brier_h{h}"])
            delta[f"delta_calibration_h{h}_vs_full"] = float(r[f"calibration_h{h}"] - ref[f"calibration_h{h}"])
            delta[f"delta_risk_auc_h{h}_vs_full"] = float(r[f"risk_auc_h{h}"] - ref[f"risk_auc_h{h}"])
        delta_rows.append(delta)

ablation_delta_vs_full_df = pd.DataFrame(delta_rows).sort_values(
    by=["model_id", "scenario_id"]
).reset_index(drop=True)

# ------------------------------
# 8) Save artifacts
# ------------------------------
primary_path = TABLES_DIR / "table_ablation_continuous_primary_metrics.csv"
brier_path = TABLES_DIR / "table_ablation_continuous_brier_by_horizon.csv"
calibration_path = TABLES_DIR / "table_ablation_continuous_calibration_by_horizon.csv"
secondary_path = TABLES_DIR / "table_ablation_continuous_secondary_metrics.csv"
support_path = TABLES_DIR / "table_ablation_continuous_support_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_ablation_continuous_predicted_vs_observed_survival.csv"
audit_path = TABLES_DIR / "table_ablation_continuous_training_audit.csv"
leaderboard_path = TABLES_DIR / "table_ablation_continuous_leaderboard.csv"
delta_path = TABLES_DIR / "table_ablation_continuous_delta_vs_full.csv"
config_path = METADATA_DIR / "ablation_continuous_run_summary.json"

ablation_primary_df.to_csv(primary_path, index=False)
ablation_brier_df.to_csv(brier_path, index=False)
ablation_calibration_df.to_csv(calibration_path, index=False)
ablation_secondary_df.to_csv(secondary_path, index=False)
ablation_support_df.to_csv(support_path, index=False)
ablation_pred_vs_obs_df.to_csv(pred_vs_obs_path, index=False)
ablation_training_audit_df.to_csv(audit_path, index=False)
ablation_leaderboard_df.to_csv(leaderboard_path, index=False)
ablation_delta_vs_full_df.to_csv(delta_path, index=False)

save_json(
    {
        "models_run": sorted(ablation_leaderboard_df["model_id"].unique().tolist()),
        "scenarios_run": sorted(ablation_leaderboard_df["scenario_id"].unique().tolist()),
        "horizons": [int(h) for h in HORIZONS_WEEKS],
        "n_total_runs": int(ablation_leaderboard_df.shape[0]),
    },
    config_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nAblation continuous training audit:")
display(ablation_training_audit_df)

print("\nAblation continuous leaderboard:")
display(ablation_leaderboard_df)

print("\nAblation continuous delta vs full:")
display(ablation_delta_vs_full_df)

print("\nSaved:")
print("-", primary_path.resolve())
print("-", brier_path.resolve())
print("-", calibration_path.resolve())
print("-", secondary_path.resolve())
print("-", support_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", audit_path.resolve())
print("-", leaderboard_path.resolve())
print("-", delta_path.resolve())
print("-", config_path.resolve())


# F5 — Consolidate Ablation Results Across All Tuned Families

In [ ]:
# ==============================================================
# F5 — Consolidate Ablation Results Across All Tuned Families
# --------------------------------------------------------------
# Purpose:
#   Consolidate the ablation results from:
#     - discrete-time tuned families (F3)
#     - continuous-time tuned families (F4)
#
# Methodological note:
#   This step does not train any model. It only consolidates the
#   ablation outputs already generated in previous steps.
#
# Main goals:
#   - create a unified ablation leaderboard
#   - compare deltas vs full_features across families
#   - prepare paper-friendly summary tables
# ==============================================================

print("\n" + "=" * 70)
print("F5 — Consolidate Ablation Results Across All Tuned Families")
print("=" * 70)
print("Methodological note: this step consolidates ablation outputs only.")
print("No model is trained here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR", "METADATA_DIR", "save_json", "HORIZONS_WEEKS"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------
# 2) Required files
# ------------------------------
DISCRETE_LEADERBOARD_PATH = TABLES_DIR / "table_ablation_discrete_leaderboard.csv"
DISCRETE_DELTA_PATH = TABLES_DIR / "table_ablation_discrete_delta_vs_full.csv"

CONTINUOUS_LEADERBOARD_PATH = TABLES_DIR / "table_ablation_continuous_leaderboard.csv"
CONTINUOUS_DELTA_PATH = TABLES_DIR / "table_ablation_continuous_delta_vs_full.csv"

required_paths = [
    DISCRETE_LEADERBOARD_PATH,
    DISCRETE_DELTA_PATH,
    CONTINUOUS_LEADERBOARD_PATH,
    CONTINUOUS_DELTA_PATH,
]

missing_files = [str(p) for p in required_paths if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required ablation result files:\n- " + "\n- ".join(missing_files)
    )

# ------------------------------
# 3) Load inputs
# ------------------------------
ablation_discrete_leaderboard = pd.read_csv(DISCRETE_LEADERBOARD_PATH)
ablation_discrete_delta = pd.read_csv(DISCRETE_DELTA_PATH)

ablation_continuous_leaderboard = pd.read_csv(CONTINUOUS_LEADERBOARD_PATH)
ablation_continuous_delta = pd.read_csv(CONTINUOUS_DELTA_PATH)

# ------------------------------
# 4) Consolidate leaderboards
# ------------------------------
ablation_leaderboard_all = pd.concat(
    [ablation_discrete_leaderboard, ablation_continuous_leaderboard],
    ignore_index=True
)

family_map = {
    "linear_tuned": "discrete_time_linear",
    "neural_tuned": "discrete_time_neural",
    "cox_tuned": "continuous_time_cox",
    "deepsurv_tuned": "continuous_time_deepsurv",
}

display_name_map = {
    "linear_tuned": "Linear Discrete-Time Hazard (Tuned)",
    "neural_tuned": "Neural Discrete-Time Survival (Tuned)",
    "cox_tuned": "Cox Comparable (Tuned)",
    "deepsurv_tuned": "DeepSurv (Tuned)",
}

ablation_leaderboard_all["family"] = ablation_leaderboard_all["model_id"].map(family_map)
ablation_leaderboard_all["display_name"] = ablation_leaderboard_all["model_id"].map(display_name_map)

ablation_leaderboard_all = ablation_leaderboard_all[
    [
        "model_id", "display_name", "family", "scenario_id", "scenario_label",
        "ibs", "c_index",
        "brier_h10", "calibration_h10", "risk_auc_h10",
        "brier_h20", "calibration_h20", "risk_auc_h20",
        "brier_h30", "calibration_h30", "risk_auc_h30",
    ]
].copy()

ablation_leaderboard_all = ablation_leaderboard_all.sort_values(
    by=["model_id", "ibs", "c_index"],
    ascending=[True, True, False]
).reset_index(drop=True)

# ------------------------------
# 5) Consolidate deltas
# ------------------------------
ablation_delta_all = pd.concat(
    [ablation_discrete_delta, ablation_continuous_delta],
    ignore_index=True
)

ablation_delta_all["family"] = ablation_delta_all["model_id"].map(family_map)
ablation_delta_all["display_name"] = ablation_delta_all["model_id"].map(display_name_map)

ablation_delta_all = ablation_delta_all[
    [
        "model_id", "display_name", "family", "scenario_id", "scenario_label",
        "delta_ibs_vs_full", "delta_c_index_vs_full",
        "delta_brier_h10_vs_full", "delta_calibration_h10_vs_full", "delta_risk_auc_h10_vs_full",
        "delta_brier_h20_vs_full", "delta_calibration_h20_vs_full", "delta_risk_auc_h20_vs_full",
        "delta_brier_h30_vs_full", "delta_calibration_h30_vs_full", "delta_risk_auc_h30_vs_full",
    ]
].copy()

ablation_delta_all = ablation_delta_all.sort_values(
    by=["model_id", "scenario_id"]
).reset_index(drop=True)

# ------------------------------
# 6) Paper-friendly summary:
#    compare scenario effects per model
# ------------------------------
summary_rows = []

for model_id in sorted(ablation_delta_all["model_id"].unique()):
    sub = ablation_delta_all[ablation_delta_all["model_id"] == model_id].copy()

    def row_for(scenario):
        r = sub[sub["scenario_id"] == scenario]
        return r.iloc[0] if not r.empty else None

    full_r = row_for("full_features")
    drop_static_r = row_for("drop_static_structural")
    drop_temporal_r = row_for("drop_temporal_signal")

    summary_rows.append({
        "model_id": model_id,
        "display_name": display_name_map.get(model_id, model_id),
        "family": family_map.get(model_id, "unknown"),
        "delta_ibs_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_ibs_vs_full"]),
        "delta_ibs_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_ibs_vs_full"]),
        "delta_c_index_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_c_index_vs_full"]),
        "delta_c_index_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_c_index_vs_full"]),
        "delta_risk_auc_h10_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_risk_auc_h10_vs_full"]),
        "delta_risk_auc_h10_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_risk_auc_h10_vs_full"]),
        "delta_risk_auc_h20_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_risk_auc_h20_vs_full"]),
        "delta_risk_auc_h20_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_risk_auc_h20_vs_full"]),
        "delta_risk_auc_h30_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_risk_auc_h30_vs_full"]),
        "delta_risk_auc_h30_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_risk_auc_h30_vs_full"]),
    })

ablation_summary_by_model = pd.DataFrame(summary_rows)

# dominance ratio / contrast helper
ablation_summary_by_model["ibs_temporal_vs_static_ratio"] = (
    ablation_summary_by_model["delta_ibs_drop_temporal"] /
    ablation_summary_by_model["delta_ibs_drop_static"].replace(0, np.nan)
)

ablation_summary_by_model["abs_cindex_temporal_vs_static_ratio"] = (
    ablation_summary_by_model["delta_c_index_drop_temporal"].abs() /
    ablation_summary_by_model["delta_c_index_drop_static"].abs().replace(0, np.nan)
)

# ------------------------------
# 7) Cross-family scenario comparison
# ------------------------------
scenario_comparison_rows = []

SCENARIOS_KEEP = [
    "full_features",
    "drop_static_structural",
    "drop_temporal_signal",
]

for scenario_id in SCENARIOS_KEEP:
    sub = ablation_leaderboard_all[ablation_leaderboard_all["scenario_id"] == scenario_id].copy()
    sub = sub.sort_values(by=["ibs", "c_index"], ascending=[True, False]).reset_index(drop=True)
    sub["rank_ibs"] = sub["ibs"].rank(method="min", ascending=True)
    sub["rank_c_index"] = sub["c_index"].rank(method="min", ascending=False)
    scenario_comparison_rows.append(sub)

ablation_scenario_comparison = pd.concat(scenario_comparison_rows, ignore_index=True)

# ------------------------------
# 8) Save outputs
# ------------------------------
leaderboard_all_path = TABLES_DIR / "table_ablation_leaderboard_all_tuned.csv"
delta_all_path = TABLES_DIR / "table_ablation_delta_all_tuned.csv"
summary_by_model_path = TABLES_DIR / "table_ablation_summary_by_model.csv"
scenario_comparison_path = TABLES_DIR / "table_ablation_scenario_comparison.csv"
config_path = METADATA_DIR / "ablation_consolidated_summary.json"

ablation_leaderboard_all.to_csv(leaderboard_all_path, index=False)
ablation_delta_all.to_csv(delta_all_path, index=False)
ablation_summary_by_model.to_csv(summary_by_model_path, index=False)
ablation_scenario_comparison.to_csv(scenario_comparison_path, index=False)

save_json(
    {
        "n_models": int(ablation_leaderboard_all["model_id"].nunique()),
        "n_rows_leaderboard": int(ablation_leaderboard_all.shape[0]),
        "n_rows_delta": int(ablation_delta_all.shape[0]),
        "scenarios_included": sorted(ablation_leaderboard_all["scenario_id"].unique().tolist()),
        "horizons": [int(h) for h in HORIZONS_WEEKS],
    },
    config_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nAblation leaderboard — all tuned families:")
display(ablation_leaderboard_all)

print("\nAblation delta — all tuned families:")
display(ablation_delta_all)

print("\nAblation summary by model:")
display(ablation_summary_by_model)

print("\nAblation scenario comparison:")
display(ablation_scenario_comparison)

print("\nSaved:")
print("-", leaderboard_all_path.resolve())
print("-", delta_all_path.resolve())
print("-", summary_by_model_path.resolve())
print("-", scenario_comparison_path.resolve())
print("-", config_path.resolve())

# F6 — Consolidated Preprocessing and Tuning Audit

In [ ]:
# ==============================================================
# F6 — Consolidated Preprocessing and Tuning Audit
# --------------------------------------------------------------
# Purpose:
#   Consolidate preprocessing and tuning documentation across the
#   four tuned benchmark model families into a single audit table
#   suitable for editorial freezing and appendix use.
#
# Methodological note:
#   This step does not retrain any model.
#   It reuses previously exported preprocessing summaries,
#   preprocessing configs, tuning configs, and tuning-result tables.
#
# Scope:
#   Tuned benchmark families only:
#   - linear_tuned
#   - neural_tuned
#   - cox_tuned
#   - deepsurv_tuned
#
# Outputs:
#   - table_preprocessing_and_tuning_audit.csv
#   - preprocessing_and_tuning_audit_summary.json
# ==============================================================

print("\n" + "=" * 70)
print("F6 — Consolidated Preprocessing and Tuning Audit")
print("=" * 70)
print("Methodological note: this step consolidates preprocessing and")
print("tuning documentation across the tuned benchmark families only.")
print("No model is retrained and no benchmark metric is recomputed here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import json
import numpy as np
import pandas as pd


### Funcao: read_json

Definicao isolada de read_json para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 2) Helper functions
# ------------------------------
def read_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


### Funcao: must_exist

Definicao isolada de must_exist para reutilizacao nas etapas seguintes.


In [ ]:

def must_exist(path: Path, label: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing required {label}: {path}")
    return path


### Funcao: safe_first

Definicao isolada de safe_first para reutilizacao nas etapas seguintes.


In [ ]:

def safe_first(df: pd.DataFrame, col: str, default=np.nan):
    if col in df.columns and df.shape[0] > 0:
        return df.iloc[0][col]
    return default


### Funcao: stringify_search_space

Definicao isolada de stringify_search_space para reutilizacao nas etapas seguintes.


In [ ]:

def stringify_search_space(value):
    if value is None:
        return "not_reported"
    if isinstance(value, dict):
        parts = []
        for k, v in value.items():
            parts.append(f"{k}={v}")
        return "; ".join(parts)
    return str(value)


### Funcao: stringify_list_field

Definicao isolada de stringify_list_field para reutilizacao nas etapas seguintes.


In [ ]:

def stringify_list_field(value):
    if value is None:
        return "not_reported"
    if isinstance(value, list):
        return ", ".join(str(x) for x in value)
    return str(value)


In [ ]:

# ------------------------------
# 3) Resolve inputs
# ------------------------------
# preprocessing summaries / configs
linear_preproc_summary_path = must_exist(
    TABLES_DIR / "table_linear_preprocessing_summary.csv",
    "linear preprocessing summary"
)
neural_preproc_summary_path = must_exist(
    TABLES_DIR / "table_neural_preprocessing_summary.csv",
    "neural preprocessing summary"
)
cox_preproc_summary_path = must_exist(
    TABLES_DIR / "table_cox_preprocessing_summary.csv",
    "cox preprocessing summary"
)
deepsurv_preproc_summary_path = must_exist(
    TABLES_DIR / "table_deepsurv_preprocessing_summary.csv",
    "deepsurv preprocessing summary"
)

linear_preproc_config_path = must_exist(
    METADATA_DIR / "linear_preprocessing_config.json",
    "linear preprocessing config"
)
neural_preproc_config_path = must_exist(
    METADATA_DIR / "neural_preprocessing_config.json",
    "neural preprocessing config"
)
cox_preproc_config_path = must_exist(
    METADATA_DIR / "cox_preprocessing_config.json",
    "cox preprocessing config"
)
deepsurv_preproc_config_path = must_exist(
    METADATA_DIR / "deepsurv_preprocessing_config.json",
    "deepsurv preprocessing config"
)

# tuning results / configs
linear_tuning_results_path = must_exist(
    TABLES_DIR / "table_linear_tuning_results.csv",
    "linear tuning results"
)
neural_tuning_results_path = must_exist(
    TABLES_DIR / "table_neural_tuning_results.csv",
    "neural tuning results"
)
cox_tuning_results_path = must_exist(
    TABLES_DIR / "table_cox_tuning_results.csv",
    "cox tuning results"
)
deepsurv_tuning_results_path = must_exist(
    TABLES_DIR / "table_deepsurv_tuning_results.csv",
    "deepsurv tuning results"
)

linear_tuning_config_path = must_exist(
    METADATA_DIR / "linear_tuned_model_config.json",
    "linear tuned model config"
)
neural_tuning_config_path = must_exist(
    METADATA_DIR / "neural_tuned_model_config.json",
    "neural tuned model config"
)
cox_tuning_config_path = must_exist(
    METADATA_DIR / "cox_tuned_model_config.json",
    "cox tuned model config"
)
deepsurv_tuning_config_path = must_exist(
    METADATA_DIR / "deepsurv_tuned_model_config.json",
    "deepsurv tuned model config"
)

# ------------------------------
# 4) Load artifacts
# ------------------------------
linear_preproc_summary = pd.read_csv(linear_preproc_summary_path)
neural_preproc_summary = pd.read_csv(neural_preproc_summary_path)
cox_preproc_summary = pd.read_csv(cox_preproc_summary_path)
deepsurv_preproc_summary = pd.read_csv(deepsurv_preproc_summary_path)

linear_preproc_config = read_json(linear_preproc_config_path)
neural_preproc_config = read_json(neural_preproc_config_path)
cox_preproc_config = read_json(cox_preproc_config_path)
deepsurv_preproc_config = read_json(deepsurv_preproc_config_path)

linear_tuning_results = pd.read_csv(linear_tuning_results_path)
neural_tuning_results = pd.read_csv(neural_tuning_results_path)
cox_tuning_results = pd.read_csv(cox_tuning_results_path)
deepsurv_tuning_results = pd.read_csv(deepsurv_tuning_results_path)

linear_tuning_config = read_json(linear_tuning_config_path)
neural_tuning_config = read_json(neural_tuning_config_path)
cox_tuning_config = read_json(cox_tuning_config_path)
deepsurv_tuning_config = read_json(deepsurv_tuning_config_path)

# ------------------------------
# 5) Build consolidated audit rows
# ------------------------------
audit_rows = []

# ---- linear discrete-time tuned
audit_rows.append({
    "model_id": "linear_tuned",
    "display_name": "Linear Discrete-Time Hazard (Tuned)",
    "family": "discrete_time_linear",
    "data_level": "person_period_weekly",
    "raw_input_design": "weekly person-period with static + temporal-behavioral covariates",
    "numeric_imputation": safe_first(linear_preproc_summary, "numeric_imputation"),
    "categorical_imputation": safe_first(linear_preproc_summary, "categorical_imputation"),
    "categorical_encoding": safe_first(linear_preproc_summary, "categorical_encoding"),
    "numeric_scaling": safe_first(linear_preproc_summary, "numeric_scaling"),
    "fit_on_train_only": "yes",
    "class_imbalance_handling": "none (class_weight=None)",
    "n_input_features_raw": safe_first(linear_preproc_summary, "n_input_features_raw"),
    "n_features_after_transform": safe_first(linear_preproc_summary, "n_features_after_transform"),
    "validation_unit": "enrollment",
    "validation_strategy": "GroupShuffleSplit on enrollment_id",
    "validation_fraction": 0.20,
    "selection_metric": linear_tuning_config.get("selection_metric", "not_reported"),
    "search_space": stringify_search_space(linear_tuning_config.get("search_space")),
    "n_tuning_candidates": int(linear_tuning_results.shape[0]),
    "early_stopping_used": "no",
    "early_stopping_patience": "not_applicable",
    "complexity_control": "penalty type + inverse regularization strength C",
    "best_candidate_summary": stringify_search_space(linear_tuning_config.get("best_candidate")),
    "preprocessing_note": linear_preproc_config.get("linear_positioning_note", "not_reported"),
    "tuning_note": linear_tuning_config.get("benchmark_positioning_note", "not_reported"),
})

# ---- neural discrete-time tuned
audit_rows.append({
    "model_id": "neural_tuned",
    "display_name": "Neural Discrete-Time Survival (Tuned)",
    "family": "discrete_time_neural",
    "data_level": "person_period_weekly",
    "raw_input_design": "weekly person-period with same conceptual feature set as linear baseline",
    "numeric_imputation": safe_first(neural_preproc_summary, "numeric_imputation"),
    "categorical_imputation": safe_first(neural_preproc_summary, "categorical_imputation"),
    "categorical_encoding": safe_first(neural_preproc_summary, "categorical_encoding"),
    "numeric_scaling": safe_first(neural_preproc_summary, "numeric_scaling"),
    "fit_on_train_only": "yes",
    "class_imbalance_handling": "none (unweighted BCEWithLogitsLoss)",
    "n_input_features_raw": safe_first(neural_preproc_summary, "n_input_features_raw"),
    "n_features_after_transform": safe_first(neural_preproc_summary, "n_features_after_transform"),
    "validation_unit": "enrollment",
    "validation_strategy": "train/validation split from distinct enrollment_id values",
    "validation_fraction": neural_tuning_config.get("validation_fraction", 0.10),
    "selection_metric": neural_tuning_config.get("selection_criterion", "lowest_validation_loss"),
    "search_space": stringify_search_space({
        "hidden_dims": [[64, 32], [128, 64]],
        "dropout": [0.10, 0.30],
        "learning_rate": [1e-3, 5e-4],
        "weight_decay": [1e-5, 1e-4],
    }),
    "n_tuning_candidates": int(neural_tuning_results.shape[0]),
    "early_stopping_used": "yes",
    "early_stopping_patience": neural_tuning_config.get("early_stopping_patience", "not_reported"),
    "complexity_control": "network width/depth grid + dropout + weight decay + early stopping + max epochs",
    "best_candidate_summary": stringify_search_space(neural_tuning_config.get("best_candidate")),
    "preprocessing_note": safe_first(neural_preproc_summary, "comparability_note", default="not_reported"),
    "tuning_note": neural_tuning_config.get("benchmark_positioning_note", "not_reported"),
})

# ---- cox tuned
audit_rows.append({
    "model_id": "cox_tuned",
    "display_name": "Cox Comparable (Tuned)",
    "family": "continuous_time_cox",
    "data_level": "enrollment_early_window",
    "raw_input_design": "static covariates + early-window summaries (4 weeks)",
    "numeric_imputation": safe_first(cox_preproc_summary, "numeric_imputation"),
    "categorical_imputation": safe_first(cox_preproc_summary, "categorical_imputation"),
    "categorical_encoding": safe_first(cox_preproc_summary, "categorical_encoding"),
    "numeric_scaling": safe_first(cox_preproc_summary, "numeric_scaling"),
    "fit_on_train_only": "yes",
    "class_imbalance_handling": "none",
    "n_input_features_raw": safe_first(cox_preproc_summary, "n_input_features_raw"),
    "n_features_after_transform": safe_first(cox_preproc_summary, "n_features_after_transform"),
    "validation_unit": "enrollment",
    "validation_strategy": "train/validation split on enrollment_id with event stratification when possible",
    "validation_fraction": 0.20,
    "selection_metric": cox_tuning_config.get("selection_metric", "val_neg_partial_log_likelihood"),
    "search_space": stringify_search_space({
        "penalizer_grid": cox_tuning_config.get("penalizer_grid"),
        "l1_ratio_grid": cox_tuning_config.get("l1_ratio_grid"),
    }),
    "n_tuning_candidates": int(cox_tuning_results.shape[0]),
    "early_stopping_used": "no",
    "early_stopping_patience": "not_applicable",
    "complexity_control": "penalizer + l1_ratio regularization grid",
    "best_candidate_summary": stringify_search_space(cox_tuning_config.get("best_candidate")),
    "preprocessing_note": cox_preproc_config.get("cox_positioning_note", "not_reported"),
    "tuning_note": cox_tuning_config.get("benchmark_positioning_note", "not_reported"),
})

# ---- deepsurv tuned
audit_rows.append({
    "model_id": "deepsurv_tuned",
    "display_name": "DeepSurv (Tuned)",
    "family": "continuous_time_deepsurv",
    "data_level": "enrollment_early_window",
    "raw_input_design": "static covariates + early-window summaries (4 weeks)",
    "numeric_imputation": safe_first(deepsurv_preproc_summary, "numeric_imputation"),
    "categorical_imputation": safe_first(deepsurv_preproc_summary, "categorical_imputation"),
    "categorical_encoding": safe_first(deepsurv_preproc_summary, "categorical_encoding"),
    "numeric_scaling": safe_first(deepsurv_preproc_summary, "numeric_scaling"),
    "fit_on_train_only": "yes",
    "class_imbalance_handling": "none",
    "n_input_features_raw": safe_first(deepsurv_preproc_summary, "n_input_features_raw"),
    "n_features_after_transform": safe_first(deepsurv_preproc_summary, "n_features_after_transform"),
    "validation_unit": "row_within_training_set",
    "validation_strategy": "random internal validation fraction on training rows",
    "validation_fraction": deepsurv_tuning_config.get("validation_fraction", 0.20),
    "selection_metric": deepsurv_tuning_config.get("selection_metric", "best_val_loss"),
    "search_space": stringify_search_space({
        "hidden_dims_grid": deepsurv_tuning_config.get("hidden_dims_grid"),
        "dropout_grid": deepsurv_tuning_config.get("dropout_grid"),
        "learning_rate_grid": deepsurv_tuning_config.get("learning_rate_grid"),
        "weight_decay_grid": deepsurv_tuning_config.get("weight_decay_grid"),
        "batch_norm": deepsurv_tuning_config.get("batch_norm"),
        "epochs": deepsurv_tuning_config.get("epochs"),
    }),
    "n_tuning_candidates": int(deepsurv_tuning_results.shape[0]),
    "early_stopping_used": "yes",
    "early_stopping_patience": deepsurv_tuning_config.get("patience", "not_reported"),
    "complexity_control": "network architecture grid + dropout + weight decay + early stopping + best epoch refit",
    "best_candidate_summary": stringify_search_space(deepsurv_tuning_config.get("best_candidate")),
    "preprocessing_note": deepsurv_preproc_config.get("comparability_note", "not_reported"),
    "tuning_note": deepsurv_tuning_config.get("benchmark_positioning_note", "not_reported"),
})

preprocessing_tuning_audit_df = pd.DataFrame(audit_rows)

# ------------------------------
# 6) Appendix-oriented compact version
# ------------------------------
appendix_audit_df = preprocessing_tuning_audit_df[
    [
        "display_name",
        "family",
        "data_level",
        "numeric_imputation",
        "categorical_imputation",
        "categorical_encoding",
        "numeric_scaling",
        "fit_on_train_only",
        "class_imbalance_handling",
        "validation_unit",
        "validation_strategy",
        "validation_fraction",
        "selection_metric",
        "n_tuning_candidates",
        "early_stopping_used",
        "early_stopping_patience",
        "complexity_control",
    ]
].copy()

appendix_audit_df = appendix_audit_df.rename(columns={
    "display_name": "model",
    "data_level": "input_level",
    "numeric_imputation": "num_impute",
    "categorical_imputation": "cat_impute",
    "categorical_encoding": "cat_encoding",
    "numeric_scaling": "num_scaling",
    "fit_on_train_only": "fit_train_only",
    "class_imbalance_handling": "imbalance",
    "validation_unit": "val_unit",
    "validation_strategy": "val_strategy",
    "validation_fraction": "val_frac",
    "selection_metric": "select_metric",
    "n_tuning_candidates": "n_candidates",
    "early_stopping_used": "early_stop",
    "early_stopping_patience": "patience",
    "complexity_control": "complexity_control",
})

# ------------------------------
# 7) Save outputs
# ------------------------------
audit_path = TABLES_DIR / "table_preprocessing_and_tuning_audit.csv"
appendix_audit_path = TABLES_DIR / "table_appendix_preprocessing_and_tuning_audit_compact.csv"
summary_json_path = METADATA_DIR / "preprocessing_and_tuning_audit_summary.json"

preprocessing_tuning_audit_df.to_csv(audit_path, index=False)
appendix_audit_df.to_csv(appendix_audit_path, index=False)

save_json(
    {
        "n_models": int(preprocessing_tuning_audit_df.shape[0]),
        "model_ids": preprocessing_tuning_audit_df["model_id"].tolist(),
        "appendix_ready_table": str(appendix_audit_path),
        "full_audit_table": str(audit_path),
        "notes": [
            "All preprocessing transformations were fit on training data only.",
            "Discrete-time linear and Cox tuning do not use early stopping.",
            "Neural discrete-time tuning uses enrollment-level internal validation.",
            "DeepSurv tuning uses an internal validation fraction on training rows.",
        ],
    },
    summary_json_path,
)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nFull preprocessing and tuning audit:")
display(preprocessing_tuning_audit_df)

print("\nAppendix-oriented compact audit:")
display(appendix_audit_df)

print("\nSaved:")
print("-", audit_path.resolve())
print("-", appendix_audit_path.resolve())
print("-", summary_json_path.resolve())


# F7 — Bootstrap Uncertainty Audit for Tuned Benchmark Models (Revised v4)

In [ ]:
# ==============================================================
# F7 — Bootstrap Uncertainty Audit for Tuned Benchmark Models (Revised v4)
# --------------------------------------------------------------
# Purpose:
#   Quantify uncertainty for the tuned benchmark hierarchy using
#   enrollment-level bootstrap resampling on the held-out test set.
#
# Methodological note:
#   This step does NOT retrain models during each bootstrap draw.
#   It rebuilds the tuned models' test-set survival predictions once,
#   then performs enrollment-level resampling on the held-out test
#   enrollments to estimate uncertainty in:
#     - IBS
#     - time-dependent concordance
#     - Brier@10, Brier@20, Brier@30
#
#   It also computes rank-stability summaries to assess whether the
#   benchmark ordering is robust enough to justify expressions such as
#   "stable hierarchy".
#
# Important correction in v4:
#   Bootstrap samples are materialized with synthetic unique ids
#   (boot_{iter}_{j}) so that surv_df columns remain unique and
#   perfectly aligned with the corresponding truth table.
# ==============================================================

print("\n" + "=" * 70)
print("F7 — Bootstrap Uncertainty Audit for Tuned Benchmark Models (Revised v4)")
print("=" * 70)
print("Methodological note: this step rebuilds tuned-model survival predictions")
print("on the held-out test set and performs enrollment-level bootstrap")
print("resampling to quantify uncertainty in the benchmark hierarchy.")
print("No model is retrained inside bootstrap iterations.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "HORIZONS_WEEKS", "RANDOM_SEED"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import scipy
import torch
import torch.nn as nn

try:
    from pycox.evaluation import EvalSurv
    from pycox.models import CoxPH
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

try:
    import torchtuples as tt
    TORCHTUPLES_AVAILABLE = True
except Exception:
    TORCHTUPLES_AVAILABLE = False

try:
    from lifelines import CoxPHFitter
    LIFELINES_AVAILABLE = True
except Exception:
    LIFELINES_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P31.6.")
if not TORCHTUPLES_AVAILABLE:
    raise ImportError("torchtuples is required for P31.6.")
if not LIFELINES_AVAILABLE:
    raise ImportError("lifelines is required for P31.6.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Config
# ------------------------------
BOOTSTRAP_CONFIG = {
    "n_bootstrap": 200,
    "ci_alpha": 0.05,
    "metrics": ["ibs", "c_index_td"] + [f"brier_h{h}" for h in HORIZONS_WEEKS],
    "max_horizon_for_ibs": int(max(HORIZONS_WEEKS)),
    "resampling_unit": "enrollment",
    "random_seed": int(RANDOM_SEED),
    "note": (
        "Enrollment-level bootstrap on the held-out test set using fixed tuned-model predictions."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
    "sanity_tolerance_abs": 0.02,
}

N_BOOT = BOOTSTRAP_CONFIG["n_bootstrap"]
ALPHA = BOOTSTRAP_CONFIG["ci_alpha"]
LOW_Q = 100 * (ALPHA / 2)
HIGH_Q = 100 * (1 - ALPHA / 2)
SANITY_TOL = BOOTSTRAP_CONFIG["sanity_tolerance_abs"]

# ------------------------------
# 4) Paths
# ------------------------------
DATA_DIR = OUTPUT_DIR / "data"
MODELS_DIR = OUTPUT_DIR / "models"

COX_TRAIN_PATH = DATA_DIR / "enrollment_cox_ready_train.csv"
COX_TEST_PATH = DATA_DIR / "enrollment_cox_ready_test.csv"
COX_MODEL_PATH = MODELS_DIR / "cox_early_window_tuned.joblib"
COX_PREPROC_PATH = MODELS_DIR / "cox_preprocessor.joblib"
COX_STABILITY_PATH = TABLES_DIR / "table_cox_tuned_stability_notes.csv"
COX_PRIMARY_METRICS_PATH = TABLES_DIR / "table_cox_tuned_primary_metrics.csv"
COX_BRIER_PATH = TABLES_DIR / "table_cox_tuned_brier_by_horizon.csv"

DEEPSURV_TRAIN_PATH = DATA_DIR / "enrollment_deepsurv_ready_train.csv"
DEEPSURV_TEST_PATH = DATA_DIR / "enrollment_deepsurv_ready_test.csv"
DEEPSURV_MODEL_PATH = MODELS_DIR / "deepsurv_tuned.pt"
DEEPSURV_PREPROC_PATH = MODELS_DIR / "deepsurv_preprocessor.joblib"
DEEPSURV_CONFIG_PATH = METADATA_DIR / "deepsurv_tuned_model_config.json"
DEEPSURV_PRIMARY_METRICS_PATH = TABLES_DIR / "table_deepsurv_tuned_primary_metrics.csv"
DEEPSURV_BRIER_PATH = TABLES_DIR / "table_deepsurv_tuned_brier_by_horizon.csv"

LINEAR_TEST_PATH = DATA_DIR / "pp_linear_hazard_ready_test.csv"
LINEAR_TRAIN_PATH = DATA_DIR / "pp_linear_hazard_ready_train.csv"
LINEAR_MODEL_PATH = MODELS_DIR / "linear_discrete_time_hazard_tuned.joblib"
LINEAR_PREPROC_PATH = MODELS_DIR / "linear_discrete_time_preprocessor.joblib"
LINEAR_PRIMARY_METRICS_PATH = TABLES_DIR / "table_linear_tuned_primary_metrics.csv"
LINEAR_BRIER_PATH = TABLES_DIR / "table_linear_tuned_brier_by_horizon.csv"

NEURAL_TEST_PATH = DATA_DIR / "pp_neural_hazard_ready_test.csv"
NEURAL_TRAIN_PATH = DATA_DIR / "pp_neural_hazard_ready_train.csv"
NEURAL_MODEL_PATH = MODELS_DIR / "neural_discrete_time_survival_tuned.pt"
NEURAL_PREPROC_PATH = MODELS_DIR / "neural_discrete_time_preprocessor.joblib"
NEURAL_CONFIG_PATH = METADATA_DIR / "neural_tuned_model_config.json"
NEURAL_PRIMARY_METRICS_PATH = TABLES_DIR / "table_neural_tuned_primary_metrics.csv"
NEURAL_BRIER_PATH = TABLES_DIR / "table_neural_tuned_brier_by_horizon.csv"

required_paths = [
    COX_TRAIN_PATH, COX_TEST_PATH, COX_MODEL_PATH, COX_PREPROC_PATH, COX_STABILITY_PATH,
    COX_PRIMARY_METRICS_PATH, COX_BRIER_PATH,
    DEEPSURV_TRAIN_PATH, DEEPSURV_TEST_PATH, DEEPSURV_MODEL_PATH, DEEPSURV_PREPROC_PATH,
    DEEPSURV_CONFIG_PATH, DEEPSURV_PRIMARY_METRICS_PATH, DEEPSURV_BRIER_PATH,
    LINEAR_TRAIN_PATH, LINEAR_TEST_PATH, LINEAR_MODEL_PATH, LINEAR_PREPROC_PATH,
    LINEAR_PRIMARY_METRICS_PATH, LINEAR_BRIER_PATH,
    NEURAL_TRAIN_PATH, NEURAL_TEST_PATH, NEURAL_MODEL_PATH, NEURAL_PREPROC_PATH,
    NEURAL_CONFIG_PATH, NEURAL_PRIMARY_METRICS_PATH, NEURAL_BRIER_PATH,
]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required files for P31.6:\n- " + "\n- ".join(missing_paths))

# ------------------------------
# 5) Helpers
# ------------------------------
AUX_ENROLLMENT = [
    "enrollment_id", "id_student", "code_module", "code_presentation",
    "duration", "duration_raw", "used_zero_week_fallback_for_censoring",
    "split", "time_for_split", "time_bucket", "event_time_bucket_label"
]
TARGET_ENROLLMENT = ["event"]

AUX_DISCRETE = [
    "enrollment_id", "id_student", "code_module", "code_presentation",
    "event_observed", "t_event_week", "t_final_week",
    "used_zero_week_fallback_for_censoring", "split",
    "time_for_split", "time_bucket", "event_time_bucket_label"
]
TARGET_DISCRETE = ["event_t"]


### Funcao: load_csv

Definicao isolada de load_csv para reutilizacao nas etapas seguintes.


In [ ]:

def load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)


### Funcao: read_json

Definicao isolada de read_json para reutilizacao nas etapas seguintes.


In [ ]:

def read_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


### Funcao: safe_save_json

Definicao isolada de safe_save_json para reutilizacao nas etapas seguintes.


In [ ]:

def safe_save_json(obj: dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


### Funcao: get_feature_cols

Definicao isolada de get_feature_cols para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_cols(df: pd.DataFrame, aux_cols: list[str], target_cols: list[str]) -> list[str]:
    excluded = set(aux_cols + target_cols)
    return [c for c in df.columns if c not in excluded]


### Funcao: build_truth_from_discrete

Definicao isolada de build_truth_from_discrete para reutilizacao nas etapas seguintes.


In [ ]:

def build_truth_from_discrete(df: pd.DataFrame) -> pd.DataFrame:
    event_col = "event_observed" if "event_observed" in df.columns else "event_t"
    duration_col = "time_for_split" if "time_for_split" in df.columns else "week"
    truth = (
        df.groupby("enrollment_id", as_index=False)
        .agg(
            event=(event_col, "max"),
            duration=(duration_col, "max"),
        )
    )
    truth["event"] = truth["event"].astype(int)
    truth["duration"] = truth["duration"].astype(int)
    return truth


### Funcao: align_truth_to_surv_df

Definicao isolada de align_truth_to_surv_df para reutilizacao nas etapas seguintes.


In [ ]:

def align_truth_to_surv_df(truth_df: pd.DataFrame, surv_df: pd.DataFrame) -> pd.DataFrame:
    truth_idx = truth_df.set_index("enrollment_id")
    missing = [eid for eid in surv_df.columns.tolist() if eid not in truth_idx.index]
    if missing:
        raise KeyError(f"Missing enrollment_ids in truth_df alignment: {missing[:10]}")
    aligned = truth_idx.loc[list(surv_df.columns)].reset_index()
    return aligned


### Funcao: eval_surv_metrics_from_surv_df

Definicao isolada de eval_surv_metrics_from_surv_df para reutilizacao nas etapas seguintes.


In [ ]:

def eval_surv_metrics_from_surv_df(surv_df: pd.DataFrame, truth_df: pd.DataFrame, horizons: list[int]) -> dict:
    truth_aligned = align_truth_to_surv_df(truth_df, surv_df)

    durations = truth_aligned["duration"].astype(int).to_numpy()
    events = truth_aligned["event"].astype(int).to_numpy()

    ev = EvalSurv(
        surv=surv_df,
        durations=durations,
        events=events,
        censor_surv="km",
    )

    out = {}

    try:
        out["ibs"] = float(ev.integrated_brier_score(np.arange(1, int(max(horizons)) + 1)))
    except Exception:
        out["ibs"] = np.nan

    try:
        out["c_index_td"] = float(ev.concordance_td())
    except Exception:
        out["c_index_td"] = np.nan

    try:
        brier_h = ev.brier_score(np.array(horizons, dtype=int))
        for h, v in zip(brier_h.index.astype(int), brier_h.values.astype(float)):
            out[f"brier_h{int(h)}"] = float(v)
    except Exception:
        for h in horizons:
            out[f"brier_h{int(h)}"] = np.nan

    return out


### Funcao: bootstrap_eval_surv_metrics

Definicao isolada de bootstrap_eval_surv_metrics para reutilizacao nas etapas seguintes.


In [ ]:

def bootstrap_eval_surv_metrics(surv_df: pd.DataFrame, truth_df: pd.DataFrame, model_label: str, horizons: list[int], n_boot: int, rng: np.random.Generator) -> pd.DataFrame:
    truth_aligned = align_truth_to_surv_df(truth_df, surv_df)
    enrollment_ids = truth_aligned["enrollment_id"].tolist()
    n = len(enrollment_ids)
    rows = []

    # base lookup once
    truth_lookup = truth_aligned.set_index("enrollment_id")
    surv_lookup = surv_df.copy()

    for b in range(1, n_boot + 1):
        sampled_ids = rng.choice(enrollment_ids, size=n, replace=True).tolist()
        boot_ids = [f"boot_{b}_{j}" for j in range(n)]

        # bootstrap truth with unique synthetic ids
        truth_sample = truth_lookup.loc[sampled_ids].reset_index()
        truth_sample["enrollment_id"] = boot_ids

        # bootstrap survival with same unique synthetic ids
        surv_sample = surv_lookup.loc[:, sampled_ids].copy()
        surv_sample.columns = boot_ids
        surv_sample.columns.name = "enrollment_id"

        metrics = eval_surv_metrics_from_surv_df(surv_sample, truth_sample, horizons)

        row = {"model": model_label, "bootstrap_iter": b}
        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows)


### Funcao: summarize_bootstrap

Definicao isolada de summarize_bootstrap para reutilizacao nas etapas seguintes.


In [ ]:

def summarize_bootstrap(boot_df: pd.DataFrame, point_row: dict, metric_cols: list[str], model_label: str) -> pd.DataFrame:
    rows = []
    for metric in metric_cols:
        vals = boot_df[metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
        if len(vals) == 0:
            rows.append({
                "model": model_label,
                "metric": metric,
                "point_estimate": point_row.get(metric, np.nan),
                "bootstrap_mean": np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "n_successful_bootstrap": 0,
            })
        else:
            rows.append({
                "model": model_label,
                "metric": metric,
                "point_estimate": point_row.get(metric, np.nan),
                "bootstrap_mean": float(np.mean(vals)),
                "ci_lower": float(np.percentile(vals, LOW_Q)),
                "ci_upper": float(np.percentile(vals, HIGH_Q)),
                "n_successful_bootstrap": int(len(vals)),
            })
    return pd.DataFrame(rows)


### Funcao: build_rank_stability

Definicao isolada de build_rank_stability para reutilizacao nas etapas seguintes.


In [ ]:

def build_rank_stability(boot_long: pd.DataFrame, metric: str, higher_is_better: bool) -> pd.DataFrame:
    rows = []
    metric_df = boot_long[["model", "bootstrap_iter", metric]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    for boot_iter, g in metric_df.groupby("bootstrap_iter"):
        g = g.copy()
        g["rank"] = g[metric].rank(method="min", ascending=not higher_is_better)
        for _, r in g.iterrows():
            rows.append({
                "bootstrap_iter": int(boot_iter),
                "model": r["model"],
                "metric": metric,
                "rank": int(r["rank"]),
            })
    rank_df = pd.DataFrame(rows)
    if rank_df.empty:
        return pd.DataFrame(columns=["model", "metric", "rank", "frequency", "proportion"])

    out = (
        rank_df.groupby(["model", "metric", "rank"])
        .size()
        .reset_index(name="frequency")
    )
    total_per_model_metric = out.groupby(["model", "metric"])["frequency"].transform("sum")
    out["proportion"] = out["frequency"] / total_per_model_metric
    return out.sort_values(["metric", "rank", "model"]).reset_index(drop=True)


### Funcao: build_pairwise_win_probability

Definicao isolada de build_pairwise_win_probability para reutilizacao nas etapas seguintes.


In [ ]:

def build_pairwise_win_probability(boot_long: pd.DataFrame, metric: str, higher_is_better: bool) -> pd.DataFrame:
    models = sorted(boot_long["model"].unique().tolist())
    rows = []
    for i, m1 in enumerate(models):
        for m2 in models[i+1:]:
            tmp1 = boot_long[boot_long["model"] == m1][["bootstrap_iter", metric]].rename(columns={metric: "v1"})
            tmp2 = boot_long[boot_long["model"] == m2][["bootstrap_iter", metric]].rename(columns={metric: "v2"})
            merged = tmp1.merge(tmp2, on="bootstrap_iter", how="inner")
            merged = merged.replace([np.inf, -np.inf], np.nan).dropna()
            if merged.empty:
                prob_m1_better = np.nan
                prob_m2_better = np.nan
            else:
                if higher_is_better:
                    prob_m1_better = float((merged["v1"] > merged["v2"]).mean())
                    prob_m2_better = float((merged["v2"] > merged["v1"]).mean())
                else:
                    prob_m1_better = float((merged["v1"] < merged["v2"]).mean())
                    prob_m2_better = float((merged["v2"] < merged["v1"]).mean())
            rows.append({
                "metric": metric,
                "model_a": m1,
                "model_b": m2,
                "prob_a_better_than_b": prob_m1_better,
                "prob_b_better_than_a": prob_m2_better,
                "n_bootstrap_pairs": int(merged.shape[0]) if not merged.empty else 0,
            })
    return pd.DataFrame(rows)


### Funcao: load_frozen_metrics

Definicao isolada de load_frozen_metrics para reutilizacao nas etapas seguintes.


In [ ]:

def load_frozen_metrics(primary_path: Path, brier_path: Path, model_name: str) -> dict:
    primary = pd.read_csv(primary_path)
    brier = pd.read_csv(brier_path)

    out = {"model": model_name}
    if "metric_name" in primary.columns:
        ibs_row = primary[primary["metric_name"].isin(["ibs", "IBS"])]
        cidx_row = primary[primary["metric_name"].isin(["c_index", "c_index_td", "C-index"])]
        out["ibs"] = float(ibs_row["metric_value"].iloc[0]) if ibs_row.shape[0] > 0 else np.nan
        out["c_index_td"] = float(cidx_row["metric_value"].iloc[0]) if cidx_row.shape[0] > 0 else np.nan
    else:
        out["ibs"] = np.nan
        out["c_index_td"] = np.nan

    if "horizon_week" in brier.columns:
        for h in HORIZONS_WEEKS:
            row = brier[brier["horizon_week"] == h]
            out[f"brier_h{h}"] = float(row["metric_value"].iloc[0]) if row.shape[0] > 0 else np.nan
    else:
        for h in HORIZONS_WEEKS:
            out[f"brier_h{h}"] = np.nan

    return out


### Classe: TunedHazardMLP

Definicao isolada de TunedHazardMLP para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 6) Model builders / predictors
# ------------------------------
class TunedHazardMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


### Funcao: predict_proba_in_batches_torch

Definicao isolada de predict_proba_in_batches_torch para reutilizacao nas etapas seguintes.


In [ ]:

def predict_proba_in_batches_torch(model, X_np: np.ndarray, batch_size: int = 4096, device: str = "cpu") -> np.ndarray:
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, X_np.shape[0], batch_size):
            xb = torch.from_numpy(X_np[start:start+batch_size].astype(np.float32)).to(device)
            logits = model(xb)
            p = torch.sigmoid(logits).cpu().numpy().reshape(-1)
            probs.append(p)
    return np.concatenate(probs, axis=0)


### Funcao: rebuild_linear_surv

Definicao isolada de rebuild_linear_surv para reutilizacao nas etapas seguintes.


In [ ]:

def rebuild_linear_surv():
    test_df = load_csv(LINEAR_TEST_PATH)
    model = joblib.load(LINEAR_MODEL_PATH)
    preproc = joblib.load(LINEAR_PREPROC_PATH)

    feature_cols = get_feature_cols(test_df, AUX_DISCRETE, TARGET_DISCRETE)
    X_test = preproc.transform(test_df[feature_cols])
    if hasattr(X_test, "toarray"):
        X_test = X_test.toarray()

    pred_hazard = np.clip(model.predict_proba(X_test)[:, 1], 1e-8, 1 - 1e-8)

    test_pred_df = test_df.copy()
    test_pred_df["pred_hazard"] = pred_hazard
    test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
    test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
        lambda s: (1.0 - s).cumprod()
    )

    surv_wide = (
        test_pred_df[["enrollment_id", "week", "pred_survival"]]
        .drop_duplicates(subset=["enrollment_id", "week"])
        .pivot(index="week", columns="enrollment_id", values="pred_survival")
        .sort_index()
    )
    max_week_test = int(test_pred_df["week"].max())
    full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
    surv_wide = surv_wide.reindex(full_week_index).ffill().fillna(1.0)
    surv_wide.columns.name = "enrollment_id"

    truth = build_truth_from_discrete(test_df)
    return surv_wide, truth


### Funcao: rebuild_neural_surv

Definicao isolada de rebuild_neural_surv para reutilizacao nas etapas seguintes.


In [ ]:

def rebuild_neural_surv():
    test_df = load_csv(NEURAL_TEST_PATH)
    preproc = joblib.load(NEURAL_PREPROC_PATH)
    cfg = read_json(NEURAL_CONFIG_PATH)
    best = cfg["best_candidate"]

    feature_cols = get_feature_cols(test_df, AUX_DISCRETE, TARGET_DISCRETE)
    X_test = preproc.transform(test_df[feature_cols])
    if hasattr(X_test, "toarray"):
        X_test = X_test.toarray()

    input_dim = int(X_test.shape[1])
    hidden_dims = best["hidden_dims"]
    dropout = float(best["dropout"])

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = TunedHazardMLP(input_dim=input_dim, hidden_dims=hidden_dims, dropout=dropout).to(device)
    state_dict = torch.load(NEURAL_MODEL_PATH, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    pred_hazard = np.clip(
        predict_proba_in_batches_torch(model, np.asarray(X_test), device=device),
        1e-8, 1 - 1e-8
    )

    test_pred_df = test_df.copy()
    test_pred_df["pred_hazard"] = pred_hazard
    test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
    test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
        lambda s: (1.0 - s).cumprod()
    )

    surv_wide = (
        test_pred_df[["enrollment_id", "week", "pred_survival"]]
        .drop_duplicates(subset=["enrollment_id", "week"])
        .pivot(index="week", columns="enrollment_id", values="pred_survival")
        .sort_index()
    )
    max_week_test = int(test_pred_df["week"].max())
    full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
    surv_wide = surv_wide.reindex(full_week_index).ffill().fillna(1.0)
    surv_wide.columns.name = "enrollment_id"

    truth = build_truth_from_discrete(test_df)
    return surv_wide, truth


### Funcao: rebuild_cox_surv

Definicao isolada de rebuild_cox_surv para reutilizacao nas etapas seguintes.


In [ ]:

def rebuild_cox_surv():
    test_df = load_csv(COX_TEST_PATH)
    model = joblib.load(COX_MODEL_PATH)
    preproc = joblib.load(COX_PREPROC_PATH)
    stability = pd.read_csv(COX_STABILITY_PATH)

    feature_cols = get_feature_cols(test_df, AUX_ENROLLMENT, TARGET_ENROLLMENT)
    X_test = preproc.transform(test_df[feature_cols])
    feature_names_out = list(preproc.get_feature_names_out())
    X_test_df = pd.DataFrame(X_test, columns=feature_names_out)

    dropped_cols = []
    if "dropped_zero_variance_features" in stability.columns:
        raw = str(stability.loc[0, "dropped_zero_variance_features"])
        if raw.strip() != "" and raw.strip().lower() != "nan":
            dropped_cols = [x.strip() for x in raw.split(";") if x.strip()]

    keep_cols = [c for c in X_test_df.columns if c not in dropped_cols]
    X_test_df = X_test_df[keep_cols].copy()

    times = np.arange(0, int(test_df["duration"].max()) + 1, dtype=int)
    pred_surv = model.predict_survival_function(X_test_df, times=times)

    surv_df = pred_surv.copy()
    surv_df.columns = test_df["enrollment_id"].tolist()
    surv_df.columns.name = "enrollment_id"
    surv_df.index = surv_df.index.astype(int)

    truth = test_df[["enrollment_id", "event", "duration"]].copy()
    truth["event"] = truth["event"].astype(int)
    truth["duration"] = truth["duration"].astype(int)
    return surv_df, truth


### Funcao: rebuild_deepsurv_surv

Definicao isolada de rebuild_deepsurv_surv para reutilizacao nas etapas seguintes.


In [ ]:

def rebuild_deepsurv_surv():
    train_df = load_csv(DEEPSURV_TRAIN_PATH)
    test_df = load_csv(DEEPSURV_TEST_PATH)
    preproc = joblib.load(DEEPSURV_PREPROC_PATH)
    cfg = read_json(DEEPSURV_CONFIG_PATH)
    best = cfg["best_candidate"]

    feature_cols = get_feature_cols(test_df, AUX_ENROLLMENT, TARGET_ENROLLMENT)

    X_train = preproc.transform(train_df[feature_cols]).astype(np.float32)
    X_test = preproc.transform(test_df[feature_cols]).astype(np.float32)

    y_train = (
        train_df["duration"].astype(np.float32).to_numpy(),
        train_df["event"].astype(np.float32).to_numpy(),
    )

    net = tt.practical.MLPVanilla(
        in_features=X_train.shape[1],
        num_nodes=best["hidden_dims"],
        out_features=1,
        batch_norm=True,
        dropout=float(best["dropout"]),
        output_bias=False,
    )

    model = CoxPH(net, tt.optim.AdamW)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    state_dict = torch.load(DEEPSURV_MODEL_PATH, map_location=device)
    model.net.load_state_dict(state_dict)
    model.compute_baseline_hazards(input=X_train, target=y_train)

    surv_df = model.predict_surv_df(X_test)
    surv_df.columns = test_df["enrollment_id"].tolist()
    surv_df.columns.name = "enrollment_id"
    surv_df.index = surv_df.index.astype(float)

    truth = test_df[["enrollment_id", "event", "duration"]].copy()
    truth["event"] = truth["event"].astype(int)
    truth["duration"] = truth["duration"].astype(int)
    return surv_df, truth


In [ ]:

# ------------------------------
# 7) Rebuild tuned-model survival predictions
# ------------------------------
print("\nRebuilding tuned-model survival objects on the held-out test set...")

model_surv_objects = {}

surv_linear, truth_linear = rebuild_linear_surv()
model_surv_objects["Linear Discrete-Time Hazard (Tuned)"] = {
    "surv_df": surv_linear,
    "truth_df": truth_linear,
    "frozen_metrics": load_frozen_metrics(LINEAR_PRIMARY_METRICS_PATH, LINEAR_BRIER_PATH, "Linear Discrete-Time Hazard (Tuned)"),
}

surv_neural, truth_neural = rebuild_neural_surv()
model_surv_objects["Neural Discrete-Time Survival (Tuned)"] = {
    "surv_df": surv_neural,
    "truth_df": truth_neural,
    "frozen_metrics": load_frozen_metrics(NEURAL_PRIMARY_METRICS_PATH, NEURAL_BRIER_PATH, "Neural Discrete-Time Survival (Tuned)"),
}

surv_cox, truth_cox = rebuild_cox_surv()
model_surv_objects["Cox Comparable (Tuned)"] = {
    "surv_df": surv_cox,
    "truth_df": truth_cox,
    "frozen_metrics": load_frozen_metrics(COX_PRIMARY_METRICS_PATH, COX_BRIER_PATH, "Cox Comparable (Tuned)"),
}

surv_deepsurv, truth_deepsurv = rebuild_deepsurv_surv()
model_surv_objects["DeepSurv (Tuned)"] = {
    "surv_df": surv_deepsurv,
    "truth_df": truth_deepsurv,
    "frozen_metrics": load_frozen_metrics(DEEPSURV_PRIMARY_METRICS_PATH, DEEPSURV_BRIER_PATH, "DeepSurv (Tuned)"),
}

print("Done.")

# ------------------------------
# 8) Point estimates from rebuilt survival objects
# ------------------------------
point_rows = []
for model_name, obj in model_surv_objects.items():
    metrics = eval_surv_metrics_from_surv_df(obj["surv_df"], obj["truth_df"], HORIZONS_WEEKS)
    row = {"model": model_name}
    row.update(metrics)
    point_rows.append(row)

point_estimates_df = pd.DataFrame(point_rows)

print("\nPoint estimates recomputed from rebuilt survival objects:")
display(point_estimates_df)

# ------------------------------
# 9) Sanity audit against frozen metrics
# ------------------------------
frozen_rows = []
for model_name, obj in model_surv_objects.items():
    frozen_rows.append(obj["frozen_metrics"])
frozen_estimates_df = pd.DataFrame(frozen_rows)

sanity_rows = []
metric_cols = ["ibs", "c_index_td"] + [f"brier_h{h}" for h in HORIZONS_WEEKS]
for _, point_row in point_estimates_df.iterrows():
    model_name = point_row["model"]
    frozen_row = frozen_estimates_df[frozen_estimates_df["model"] == model_name].iloc[0]
    for metric in metric_cols:
        rebuilt_val = point_row.get(metric, np.nan)
        frozen_val = frozen_row.get(metric, np.nan)
        abs_diff = abs(rebuilt_val - frozen_val) if pd.notna(rebuilt_val) and pd.notna(frozen_val) else np.nan
        sanity_rows.append({
            "model": model_name,
            "metric": metric,
            "rebuilt_value": rebuilt_val,
            "frozen_value": frozen_val,
            "abs_diff": abs_diff,
            "within_tolerance": bool(abs_diff <= SANITY_TOL) if pd.notna(abs_diff) else False,
        })

sanity_audit_df = pd.DataFrame(sanity_rows)

print("\nSanity audit against frozen tuned-model metrics:")
display(sanity_audit_df)

bad_rows = sanity_audit_df[sanity_audit_df["within_tolerance"] == False].copy()
if bad_rows.shape[0] > 0:
    raise RuntimeError(
        "Rebuilt survival objects failed sanity comparison against frozen metrics. "
        "Do not proceed to bootstrap until the alignment/rebuild issue is resolved."
    )

# ------------------------------
# 10) Bootstrap uncertainty
# ------------------------------
rng = np.random.default_rng(BOOTSTRAP_CONFIG["random_seed"])

bootstrap_all = []
for model_name, obj in model_surv_objects.items():
    print(f"Bootstrapping: {model_name}")
    boot_df = bootstrap_eval_surv_metrics(
        surv_df=obj["surv_df"],
        truth_df=obj["truth_df"],
        model_label=model_name,
        horizons=HORIZONS_WEEKS,
        n_boot=N_BOOT,
        rng=rng,
    )
    bootstrap_all.append(boot_df)

bootstrap_long_df = pd.concat(bootstrap_all, ignore_index=True)

# ------------------------------
# 11) Summaries
# ------------------------------
summary_frames = []
for _, point_row in point_estimates_df.iterrows():
    model_name = point_row["model"]
    boot_df = bootstrap_long_df[bootstrap_long_df["model"] == model_name].copy()
    tmp = summarize_bootstrap(
        boot_df=boot_df,
        point_row=point_row.to_dict(),
        metric_cols=metric_cols,
        model_label=model_name,
    )
    summary_frames.append(tmp)

bootstrap_summary_df = pd.concat(summary_frames, ignore_index=True)

appendix_compact_df = bootstrap_summary_df.copy()
appendix_compact_df["ci_95"] = appendix_compact_df.apply(
    lambda r: (
        f"[{r['ci_lower']:.3f}, {r['ci_upper']:.3f}]"
        if pd.notna(r["ci_lower"]) and pd.notna(r["ci_upper"])
        else "NA"
    ),
    axis=1,
)
appendix_compact_df = appendix_compact_df[[
    "model", "metric", "point_estimate", "ci_95", "n_successful_bootstrap"
]].copy()

# ------------------------------
# 12) Rank stability / pairwise stability
# ------------------------------
rank_ibs_df = build_rank_stability(bootstrap_long_df, metric="ibs", higher_is_better=False)
rank_cidx_df = build_rank_stability(bootstrap_long_df, metric="c_index_td", higher_is_better=True)
rank_stability_df = pd.concat([rank_ibs_df, rank_cidx_df], ignore_index=True)

pairwise_ibs_df = build_pairwise_win_probability(bootstrap_long_df, metric="ibs", higher_is_better=False)
pairwise_cidx_df = build_pairwise_win_probability(bootstrap_long_df, metric="c_index_td", higher_is_better=True)
pairwise_stability_df = pd.concat([pairwise_ibs_df, pairwise_cidx_df], ignore_index=True)

# ------------------------------
# 13) Save artifacts
# ------------------------------
point_estimates_path = TABLES_DIR / "table_bootstrap_point_estimates_tuned_models.csv"
bootstrap_long_path = TABLES_DIR / "table_bootstrap_metrics_long_tuned_models.csv"
bootstrap_summary_path = TABLES_DIR / "table_bootstrap_uncertainty_summary_tuned_models.csv"
appendix_compact_path = TABLES_DIR / "table_appendix_bootstrap_uncertainty_compact.csv"
rank_stability_path = TABLES_DIR / "table_bootstrap_rank_stability_tuned_models.csv"
pairwise_stability_path = TABLES_DIR / "table_bootstrap_pairwise_stability_tuned_models.csv"
sanity_audit_path = TABLES_DIR / "table_bootstrap_rebuild_sanity_audit.csv"
config_path = METADATA_DIR / "bootstrap_uncertainty_audit_config.json"

point_estimates_df.to_csv(point_estimates_path, index=False)
bootstrap_long_df.to_csv(bootstrap_long_path, index=False)
bootstrap_summary_df.to_csv(bootstrap_summary_path, index=False)
appendix_compact_df.to_csv(appendix_compact_path, index=False)
rank_stability_df.to_csv(rank_stability_path, index=False)
pairwise_stability_df.to_csv(pairwise_stability_path, index=False)
sanity_audit_df.to_csv(sanity_audit_path, index=False)

safe_save_json(BOOTSTRAP_CONFIG, config_path)

# ------------------------------
# 14) Output for feedback
# ------------------------------
print("\nBootstrap summary (full):")
display(bootstrap_summary_df)

print("\nAppendix-oriented compact bootstrap summary:")
display(appendix_compact_df)

print("\nRank stability:")
display(rank_stability_df)

print("\nPairwise stability:")
display(pairwise_stability_df)

print("\nSaved:")
print("-", point_estimates_path.resolve())
print("-", bootstrap_long_path.resolve())
print("-", bootstrap_summary_path.resolve())
print("-", appendix_compact_path.resolve())
print("-", rank_stability_path.resolve())
print("-", pairwise_stability_path.resolve())
print("-", sanity_audit_path.resolve())
print("-", config_path.resolve())


# F8 — Proportional Hazards Audit for the Comparable Cox Model

In [ ]:
# ==============================================================
# F8 — Proportional Hazards Audit for the Comparable Cox Model
# (Revised v2)
# --------------------------------------------------------------
# Purpose:
#   Perform a formal proportional-hazards (PH) diagnostic for the
#   tuned comparable Cox model and export paper-ready audit artifacts.
# ==============================================================

print("\n" + "=" * 70)
print("F8 — Proportional Hazards Audit for the Comparable Cox Model (Revised v2)")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
    )

import warnings
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False

try:
    from lifelines import CoxPHFitter
    from lifelines.statistics import proportional_hazard_test
    LIFELINES_AVAILABLE = True
except Exception:
    LIFELINES_AVAILABLE = False

if not LIFELINES_AVAILABLE:
    raise ImportError("lifelines is required for P31.7.")

# ------------------------------
# 2) Resolve raw/base Cox train dataframe
# ------------------------------
cox_train_base_df = None

candidate_df_names = [
    "train_df_cox",
    "cox_train_df",
    "enrollment_cox_ready_train",
    "df_cox_train",
]

for name in candidate_df_names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame) and obj.shape[0] > 0:
            cox_train_base_df = obj.copy()
            print(f"Resolved Cox base dataframe from in-memory object: {name}")
            break

if cox_train_base_df is None and "con" in globals():
    candidate_queries = [
        "SELECT * FROM enrollment_cox_ready_train",
        "SELECT * FROM cox_ready_train",
        "SELECT * FROM train_df_cox",
    ]
    for q in candidate_queries:
        try:
            tmp = con.execute(q).fetchdf()
            if isinstance(tmp, pd.DataFrame) and tmp.shape[0] > 0:
                cox_train_base_df = tmp.copy()
                print(f"Resolved Cox base dataframe from SQL query: {q}")
                break
        except Exception:
            pass

if cox_train_base_df is None:
    raise NameError(
        "Could not resolve a Cox training dataframe "
        "(train_df_cox / cox_train_df / enrollment_cox_ready_train)."
    )

# ------------------------------
# 3) Resolve duration and event columns
# ------------------------------
duration_col = None
event_col = None

duration_candidates = ["duration", "time", "time_to_event", "T"]
event_candidates = ["event", "event_observed", "E"]

for c in duration_candidates:
    if c in cox_train_base_df.columns:
        duration_col = c
        break

for c in event_candidates:
    if c in cox_train_base_df.columns:
        event_col = c
        break

if duration_col is None or event_col is None:
    raise ValueError(
        f"Could not identify duration/event columns. "
        f"duration_col={duration_col}, event_col={event_col}"
    )

# ------------------------------
# 4) Try to resolve already-transformed Cox matrix
# ------------------------------
X_cox_train = None
cox_feature_names = None

X_candidates = [
    "X_train_cox",
    "X_cox_train",
    "X_train_comparable_cox",
    "X_train_continuous_time_cox",
]
feature_name_candidates = [
    "cox_feature_names_out",
    "feature_names_out_cox",
    "cox_selected_features",
    "cox_model_feature_cols",
    "cox_covariate_columns",
]

for name in X_candidates:
    if name in globals():
        X_candidate = globals()[name]
        try:
            if hasattr(X_candidate, "shape") and X_candidate.shape[0] == cox_train_base_df.shape[0]:
                X_cox_train = X_candidate
                print(f"Resolved transformed Cox matrix from: {name}")
                break
        except Exception:
            pass

for name in feature_name_candidates:
    if name in globals():
        f_candidate = globals()[name]
        if isinstance(f_candidate, (list, tuple, np.ndarray)) and len(f_candidate) > 0:
            cox_feature_names = [str(x) for x in f_candidate]
            print(f"Resolved Cox feature names from: {name}")
            break

# ------------------------------
# 5) If needed, resolve Cox preprocessor and transform
# ------------------------------
cox_preprocessor_obj = None
preprocessor_candidates = [
    "cox_preprocessor",
    "preprocessor_cox",
    "comparable_cox_preprocessor",
]

for name in preprocessor_candidates:
    if name in globals():
        cox_preprocessor_obj = globals()[name]
        print(f"Resolved Cox preprocessor from: {name}")
        break

exclude_cols = {
    duration_col, event_col,
    "enrollment_id", "id_student", "code_module", "code_presentation",
    "student_id", "module_presentation_id"
}

if X_cox_train is None:
    if cox_preprocessor_obj is None:
        raise ValueError(
            "Could not resolve a transformed Cox matrix and no Cox preprocessor "
            "was found to rebuild it."
        )

    raw_feature_cols = [c for c in cox_train_base_df.columns if c not in exclude_cols]
    if len(raw_feature_cols) == 0:
        raise ValueError("No raw feature columns available to rebuild Cox transformed matrix.")

    X_raw = cox_train_base_df[raw_feature_cols].copy()
    X_cox_train = cox_preprocessor_obj.transform(X_raw)
    print("Rebuilt transformed Cox matrix using the resolved Cox preprocessor.")

    if cox_feature_names is None:
        if hasattr(cox_preprocessor_obj, "get_feature_names_out"):
            cox_feature_names = [str(x) for x in cox_preprocessor_obj.get_feature_names_out()]
            print("Recovered transformed feature names from preprocessor.get_feature_names_out().")

# ------------------------------
# 6) Build numeric Cox audit dataframe
# ------------------------------
if hasattr(X_cox_train, "toarray"):
    X_cox_dense = X_cox_train.toarray()
else:
    X_cox_dense = np.asarray(X_cox_train)

if X_cox_dense.ndim != 2:
    raise ValueError(f"Transformed Cox matrix must be 2D. Got shape={X_cox_dense.shape}")

n_rows, n_cols = X_cox_dense.shape

if cox_feature_names is None:
    cox_feature_names = [f"x_{i}" for i in range(n_cols)]
elif len(cox_feature_names) != n_cols:
    print(
        f"Warning: feature-name length mismatch "
        f"(len={len(cox_feature_names)} vs n_cols={n_cols}). "
        f"Falling back to generic names."
    )
    cox_feature_names = [f"x_{i}" for i in range(n_cols)]

cox_numeric_df = pd.DataFrame(X_cox_dense, columns=cox_feature_names, index=cox_train_base_df.index)

cox_model_df = pd.concat(
    [
        cox_train_base_df[[duration_col, event_col]].copy(),
        cox_numeric_df
    ],
    axis=1
)

cox_model_df = cox_model_df.replace([np.inf, -np.inf], np.nan)

n_before = cox_model_df.shape[0]
cox_model_df = cox_model_df.dropna(axis=0).copy()
n_after = cox_model_df.shape[0]

if n_after == 0:
    raise ValueError("After dropping NaN/Inf rows, no rows remain for PH audit.")

cox_model_df[duration_col] = pd.to_numeric(cox_model_df[duration_col], errors="raise")
cox_model_df[event_col] = pd.to_numeric(cox_model_df[event_col], errors="raise").astype(int)

covariate_cols = [c for c in cox_model_df.columns if c not in [duration_col, event_col]]

# drop constant columns
non_constant_covariates = []
dropped_constant_covariates = []

for c in covariate_cols:
    if cox_model_df[c].nunique(dropna=True) <= 1:
        dropped_constant_covariates.append(c)
    else:
        non_constant_covariates.append(c)

covariate_cols = non_constant_covariates
cox_model_df = cox_model_df[[duration_col, event_col] + covariate_cols].copy()

if len(covariate_cols) == 0:
    raise ValueError("No non-constant numeric Cox covariates remained for PH audit.")

print(f"Rows before cleaning: {n_before:,}")
print(f"Rows after cleaning:  {n_after:,}")
print(f"Covariates tested:    {len(covariate_cols):,}")
if dropped_constant_covariates:
    print(f"Dropped constant covariates: {len(dropped_constant_covariates)}")

# ------------------------------
# 7) Resolve tuned Cox penalization settings if available
# ------------------------------
penalizer_value = 0.001
l1_ratio_value = 0.0

candidate_config_objs = [
    "COX_TUNING_CONFIG",
    "cox_tuning_config",
    "cox_best_config",
    "cox_model_config",
]

for name in candidate_config_objs:
    if name in globals():
        cfg = globals()[name]
        try:
            if isinstance(cfg, dict):
                best = cfg.get("best_candidate", cfg)
                if "penalizer" in best:
                    penalizer_value = float(best["penalizer"])
                if "l1_ratio" in best:
                    l1_ratio_value = float(best["l1_ratio"])
        except Exception:
            pass

print(f"Audit Cox refit with penalizer={penalizer_value}, l1_ratio={l1_ratio_value}")

# ------------------------------
# 8) Fit audit Cox model
# ------------------------------
cox_audit_model = CoxPHFitter(
    penalizer=penalizer_value,
    l1_ratio=l1_ratio_value,
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cox_audit_model.fit(
        cox_model_df,
        duration_col=duration_col,
        event_col=event_col,
        show_progress=False
    )

# ------------------------------
# 9) Formal PH test
# ------------------------------
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ph_test_rank = proportional_hazard_test(
        cox_audit_model,
        cox_model_df,
        time_transform="rank"
    )

ph_summary = ph_test_rank.summary.copy().reset_index()
if "index" in ph_summary.columns:
    ph_summary = ph_summary.rename(columns={"index": "covariate"})

rename_map = {}
for col in ph_summary.columns:
    cl = col.lower()
    if cl in ["p", "p_value"]:
        rename_map[col] = "p_value"
    elif cl in ["test_statistic", "test statistic", "chi2"]:
        rename_map[col] = "test_statistic"
ph_summary = ph_summary.rename(columns=rename_map)

if "p_value" not in ph_summary.columns:
    p_candidates = [c for c in ph_summary.columns if "p" in c.lower()]
    if p_candidates:
        ph_summary = ph_summary.rename(columns={p_candidates[0]: "p_value"})

if "test_statistic" not in ph_summary.columns:
    stat_candidates = [c for c in ph_summary.columns if "test" in c.lower() or "chi" in c.lower()]
    if stat_candidates:
        ph_summary = ph_summary.rename(columns={stat_candidates[0]: "test_statistic"})

ph_summary["p_value"] = pd.to_numeric(ph_summary["p_value"], errors="coerce")
ph_summary["test_statistic"] = pd.to_numeric(ph_summary["test_statistic"], errors="coerce")

alpha = 0.05
ph_summary["ph_flag"] = np.where(
    ph_summary["p_value"] < alpha,
    "possible_violation",
    "no_material_evidence"
)
ph_summary["ph_flag_binary"] = np.where(ph_summary["p_value"] < alpha, "yes", "no")

ph_audit_df = (
    ph_summary[["covariate", "test_statistic", "p_value", "ph_flag", "ph_flag_binary"]]
    .sort_values(["p_value", "covariate"], ascending=[True, True])
    .reset_index(drop=True)
)

# ------------------------------
# 10) Global summary
# ------------------------------
n_tested = int(ph_audit_df.shape[0])
n_flagged = int((ph_audit_df["p_value"] < alpha).sum())
top_flagged = ph_audit_df.loc[ph_audit_df["p_value"] < alpha, "covariate"].head(5).tolist()

if n_flagged == 0:
    global_classification = "A_no_material_evidence"
    global_interpretation = (
        "No material evidence of proportional-hazards violation was detected "
        "for the comparable Cox benchmark at the chosen threshold."
    )
elif n_flagged <= max(2, int(0.15 * n_tested)):
    global_classification = "B_localized_departures"
    global_interpretation = (
        "The comparable Cox benchmark showed some localized departures from "
        "proportional hazards, but not a pattern severe enough to prevent its "
        "use as the methodological anchor of the comparable continuous-time family."
    )
else:
    global_classification = "C_broad_departure"
    global_interpretation = (
        "The comparable Cox benchmark showed broad evidence of proportional-hazards "
        "departure and should therefore be interpreted as an approximate comparable "
        "benchmark rather than a fully assumption-clean Cox specification."
    )

global_summary_df = pd.DataFrame([{
    "model": "Cox Comparable (Tuned)",
    "duration_col": duration_col,
    "event_col": event_col,
    "n_rows_used": int(cox_model_df.shape[0]),
    "n_covariates_tested": n_tested,
    "alpha": alpha,
    "n_covariates_flagged": n_flagged,
    "share_covariates_flagged": float(n_flagged / n_tested) if n_tested > 0 else np.nan,
    "top_flagged_covariates": "; ".join(top_flagged) if top_flagged else "none",
    "global_classification": global_classification,
    "global_interpretation": global_interpretation,
    "audit_note": (
        "Formal proportional-hazards audit for the comparable Cox branch. "
        "DeepSurv shares the Cox-type ranking structure but was not subjected "
        "to an identical classical PH diagnostic."
    ),
}])

# ------------------------------
# 11) Optional figure
# ------------------------------
figure_created = False
figure_note = "not_created"

figures_dir = OUTPUT_DIR / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

appendix_fig_dir = figures_dir / "paper_appendix"
appendix_fig_dir.mkdir(parents=True, exist_ok=True)

fig_path_png = appendix_fig_dir / "fig_appendix_cox_ph_diagnostics.png"
fig_path_pdf = appendix_fig_dir / "fig_appendix_cox_ph_diagnostics.pdf"

if MATPLOTLIB_AVAILABLE:
    try:
        plot_df = ph_audit_df.nsmallest(min(8, ph_audit_df.shape[0]), "p_value").copy()
        plot_df = plot_df.sort_values("p_value", ascending=False)

        fig, ax = plt.subplots(figsize=(8.5, 4.8))
        ax.barh(plot_df["covariate"], plot_df["p_value"])
        ax.axvline(alpha, linestyle="--")
        ax.set_xlabel("PH test p-value")
        ax.set_ylabel("Covariate")
        ax.set_title("Comparable Cox PH diagnostic (smallest p-values)")
        plt.tight_layout()

        fig.savefig(fig_path_png, dpi=300, bbox_inches="tight")
        fig.savefig(fig_path_pdf, bbox_inches="tight")
        plt.close(fig)

        figure_created = True
        figure_note = "top_smallest_p_values_barh"
    except Exception as e:
        figure_note = f"figure_failed: {str(e)}"

# ------------------------------
# 12) Save artifacts
# ------------------------------
paper_appendix_tables_dir = TABLES_DIR / "paper_appendix"
paper_appendix_tables_dir.mkdir(parents=True, exist_ok=True)

audit_path_main = TABLES_DIR / "table_appendix_cox_ph_audit.csv"
global_summary_path_main = TABLES_DIR / "table_appendix_cox_ph_global_summary.csv"

audit_path_paper = paper_appendix_tables_dir / "table_paper_appendix_cox_ph_audit.csv"
global_summary_path_paper = paper_appendix_tables_dir / "table_paper_appendix_cox_ph_global_summary.csv"

metadata_path = METADATA_DIR / "cox_ph_audit_summary.json"

ph_audit_df.to_csv(audit_path_main, index=False)
global_summary_df.to_csv(global_summary_path_main, index=False)
ph_audit_df.to_csv(audit_path_paper, index=False)
global_summary_df.to_csv(global_summary_path_paper, index=False)

metadata_payload = {
    "step_id": "P31.7",
    "model": "Cox Comparable (Tuned)",
    "duration_col": duration_col,
    "event_col": event_col,
    "n_rows_before_cleaning": int(n_before),
    "n_rows_after_cleaning": int(n_after),
    "n_rows_used": int(cox_model_df.shape[0]),
    "n_covariates_tested": n_tested,
    "n_covariates_flagged": n_flagged,
    "alpha": alpha,
    "global_classification": global_classification,
    "global_interpretation": global_interpretation,
    "top_flagged_covariates": top_flagged,
    "dropped_constant_covariates": dropped_constant_covariates,
    "penalizer": penalizer_value,
    "l1_ratio": l1_ratio_value,
    "figure_created": figure_created,
    "figure_note": figure_note,
    "deep_surv_note": (
        "DeepSurv shares the Cox-type ranking structure but was not subjected "
        "to an identical classical proportional-hazards diagnostic."
    ),
}
save_json(metadata_payload, metadata_path)

# ------------------------------
# 13) Output
# ------------------------------
print("\nGlobal PH audit summary:")
display(global_summary_df)

print("\nPH audit by covariate:")
display(ph_audit_df)

if figure_created:
    print("\nPH diagnostic figure created:")
    print("-", fig_path_png.resolve())
    print("-", fig_path_pdf.resolve())
else:
    print("\nPH diagnostic figure not created.")
    print("Reason:", figure_note)

print("\nSaved artifacts:")
print("-", audit_path_main.resolve())
print("-", global_summary_path_main.resolve())
print("-", audit_path_paper.resolve())
print("-", global_summary_path_paper.resolve())
print("-", metadata_path.resolve())

# G — Explainability and Paper Integration

This section consolidates the tuned-model explainability layer and the final export path used by the manuscript.

# G1 — Define Explainability Protocol

### Funcao: save_json

Definicao isolada de save_json para reutilizacao nas etapas seguintes.


In [ ]:
from pathlib import Path
import json

OUTPUT_DIR = Path("outputs_benchmark_survival")
TABLES_DIR = OUTPUT_DIR / "tables"
METADATA_DIR = OUTPUT_DIR / "metadata"
DATA_DIR = OUTPUT_DIR / "data"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
HORIZONS_WEEKS = [10, 20, 30]
HORIZON_WEEKS = list(HORIZONS_WEEKS)
CALIBRATION_BINS = 10


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as file_handle:
        json.dump(obj, file_handle, indent=2)


try:
    import pyarrow

    _original_unregister_extension_type = pyarrow.unregister_extension_type

    def _safe_unregister_extension_type(name):
        try:
            return _original_unregister_extension_type(name)
        except Exception:
            if name == "arrow.py_extension_type":
                return None
            raise

    pyarrow.unregister_extension_type = _safe_unregister_extension_type
except Exception:
    pass


try:
    import pandas as pd

    neural_ready_paths = [
        DATA_DIR / "pp_neural_hazard_ready_train.parquet",
        DATA_DIR / "pp_neural_hazard_ready_test.parquet",
        DATA_DIR / "pp_neural_hazard_ready_train.csv",
        DATA_DIR / "pp_neural_hazard_ready_test.csv",
    ]
    patched_paths = []
    for dataset_path in neural_ready_paths:
        if not dataset_path.exists():
            continue
        if dataset_path.suffix == ".parquet":
            dataset_df = pd.read_parquet(dataset_path)
        else:
            dataset_df = pd.read_csv(dataset_path)
        if "time_for_split" not in dataset_df.columns and "t_final_week" in dataset_df.columns:
            dataset_df["time_for_split"] = dataset_df["t_final_week"]
            if dataset_path.suffix == ".parquet":
                dataset_df.to_parquet(dataset_path, index=False)
            else:
                dataset_df.to_csv(dataset_path, index=False)
            patched_paths.append(str(dataset_path))
    if patched_paths:
        print("Patched neural ready datasets with time_for_split:")
        for patched_path in patched_paths:
            print("-", patched_path)
except Exception as exc:
    print(f"Neural dataset schema patch skipped: {exc}")


try:
    import sklearn
    from sklearn.compose import _column_transformer
    if not hasattr(_column_transformer, "_RemainderColsList"):
        class _RemainderColsList(list):
            pass
        _column_transformer._RemainderColsList = _RemainderColsList
    print("Explainability bootstrap ready.", sklearn.__version__)
except Exception as exc:
    print(f"Explainability bootstrap ready without sklearn patch: {exc}")

In [ ]:
# ==============================================================
# G1 — Define Explainability Protocol
# --------------------------------------------------------------
# Purpose:
#   Define the explainability study design for the tuned models
#   already established in the benchmark.
#
# Methodological note:
#   This step does not train any model and does not compute any
#   explanation yet. It only formalizes:
#     - which models are included,
#     - which explainability methods will be used,
#     - which outputs are expected,
#     - and how the results should be interpreted.
#
# Scope:
#   Explainability will be performed for the tuned versions only:
#     - linear_tuned
#     - neural_tuned
#     - cox_tuned
#     - deepsurv_tuned
# ==============================================================

print("\n" + "=" * 70)
print("G1 — Define Explainability Protocol")
print("=" * 70)
print("Methodological note: this step defines the explainability study only.")
print("No model is trained and no explanation is computed here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd

# ------------------------------
# 2) Model registry
# ------------------------------
EXPLAINABILITY_MODEL_REGISTRY = [
    {
        "model_id": "linear_tuned",
        "display_name": "Linear Discrete-Time Hazard (Tuned)",
        "family": "discrete_time_linear",
        "data_level": "person_period",
        "primary_explainability_method": "coefficient_analysis",
        "secondary_explainability_method": "odds_ratio_interpretation",
        "local_explanation_planned": False,
        "global_explanation_planned": True,
        "positioning_note": "Transparent linear discrete-time benchmark."
    },
    {
        "model_id": "neural_tuned",
        "display_name": "Neural Discrete-Time Survival (Tuned)",
        "family": "discrete_time_neural",
        "data_level": "person_period",
        "primary_explainability_method": "permutation_importance",
        "secondary_explainability_method": "feature_block_importance_summary",
        "local_explanation_planned": False,
        "global_explanation_planned": True,
        "positioning_note": "Flexible nonlinear discrete-time benchmark."
    },
    {
        "model_id": "cox_tuned",
        "display_name": "Cox Comparable (Tuned)",
        "family": "continuous_time_cox",
        "data_level": "enrollment",
        "primary_explainability_method": "hazard_ratio_analysis",
        "secondary_explainability_method": "coefficient_ranking",
        "local_explanation_planned": False,
        "global_explanation_planned": True,
        "positioning_note": "Interpretable continuous-time survival benchmark."
    },
    {
        "model_id": "deepsurv_tuned",
        "display_name": "DeepSurv (Tuned)",
        "family": "continuous_time_deepsurv",
        "data_level": "enrollment",
        "primary_explainability_method": "permutation_importance",
        "secondary_explainability_method": "feature_block_importance_summary",
        "local_explanation_planned": False,
        "global_explanation_planned": True,
        "positioning_note": "Nonlinear continuous-time survival benchmark."
    },
]

explainability_model_registry_df = pd.DataFrame(EXPLAINABILITY_MODEL_REGISTRY)

# ------------------------------
# 3) Feature-block registry
# ------------------------------
EXPLAINABILITY_FEATURE_BLOCKS = [
    {
        "block_id": "static_structural",
        "block_label": "Static structural covariates",
        "applies_to": "all_families",
        "examples": "gender, region, highest_education, imd_band, age_band, num_of_prev_attempts, studied_credits, disability",
        "interpretation_goal": "Assess contribution of background and structural covariates."
    },
    {
        "block_id": "dynamic_temporal_behavioral",
        "block_label": "Dynamic temporal behavioral signals",
        "applies_to": "discrete_time_only",
        "examples": "total_clicks_week, active_this_week, n_vle_rows_week, n_distinct_sites_week, cum_clicks_until_t, recency, streak",
        "interpretation_goal": "Assess contribution of week-by-week engagement signals."
    },
    {
        "block_id": "discrete_time_index",
        "block_label": "Discrete-time structural index",
        "applies_to": "discrete_time_only",
        "examples": "week",
        "interpretation_goal": "Represent time index, not behavioral signal."
    },
    {
        "block_id": "early_window_behavior",
        "block_label": "Early-window behavioral summaries",
        "applies_to": "continuous_time_only",
        "examples": "clicks_first_4_weeks, active_weeks_first_4, mean_clicks_first_4_weeks",
        "interpretation_goal": "Assess contribution of compressed early-course behavior."
    },
]

explainability_feature_blocks_df = pd.DataFrame(EXPLAINABILITY_FEATURE_BLOCKS)

# ------------------------------
# 4) Expected outputs
# ------------------------------
EXPLAINABILITY_OUTPUTS = [
    {
        "output_id": "global_feature_ranking",
        "description": "Rank features by global importance within each tuned family.",
        "planned_for": "all_models"
    },
    {
        "output_id": "feature_block_summary",
        "description": "Summarize importance patterns at the feature-block level.",
        "planned_for": "all_models"
    },
    {
        "output_id": "signed_effect_table",
        "description": "Report signed coefficient or hazard-ratio direction when directly available.",
        "planned_for": "linear_tuned_and_cox_tuned"
    },
    {
        "output_id": "cross_family_explainability_summary",
        "description": "Compare whether the strongest drivers are consistent across families.",
        "planned_for": "final_consolidation"
    },
]

explainability_outputs_df = pd.DataFrame(EXPLAINABILITY_OUTPUTS)

# ------------------------------
# 5) Protocol
# ------------------------------
EXPLAINABILITY_PROTOCOL = {
    "scope": "tuned_models_only",
    "included_models": [row["model_id"] for row in EXPLAINABILITY_MODEL_REGISTRY],
    "main_goals": [
        "Identify which individual features are most influential within each tuned model family.",
        "Compare the dominant explanatory signals across linear, neural, Cox, and DeepSurv families.",
        "Connect explainability findings back to the ablation results."
    ],
    "global_vs_local_policy": {
        "global_explanations": True,
        "local_explanations": False,
        "rationale": (
            "The main goal of this benchmark stage is model-level interpretation, "
            "not case-level explanation."
        )
    },
    "method_policy_by_family": {
        "discrete_time_linear": {
            "primary": "coefficient_analysis",
            "secondary": "odds_ratio_interpretation"
        },
        "discrete_time_neural": {
            "primary": "permutation_importance",
            "secondary": "feature_block_importance_summary"
        },
        "continuous_time_cox": {
            "primary": "hazard_ratio_analysis",
            "secondary": "coefficient_ranking"
        },
        "continuous_time_deepsurv": {
            "primary": "permutation_importance",
            "secondary": "feature_block_importance_summary"
        }
    },
    "interpretation_rules": {
        "large_positive_signed_effect": "Associated with increased risk when sign-based methods apply.",
        "large_negative_signed_effect": "Associated with reduced risk when sign-based methods apply.",
        "high_permutation_importance": "Model performance depends strongly on that feature.",
        "consistency_with_ablation": (
            "If an important feature belongs to a behavior-derived block, the result should align "
            "with the ablation conclusion that behavioral information dominates."
        )
    },
    "limitations_to_document": [
        "Permutation importance is global and model-dependent, not causal.",
        "Coefficient and hazard-ratio interpretations apply only to model families with directly interpretable parameters.",
        "Explainability describes how the model uses information, not whether the relationships are causal."
    ],
    "paper_positioning_note": (
        "Explainability is treated here as a post-benchmark interpretive layer, intended to explain "
        "why the tuned models behave as they do after ranking has already been established."
    )
}

# ------------------------------
# 6) Save outputs
# ------------------------------
model_registry_path = TABLES_DIR / "table_explainability_model_registry.csv"
feature_blocks_path = TABLES_DIR / "table_explainability_feature_blocks.csv"
outputs_path = TABLES_DIR / "table_explainability_outputs.csv"
config_path = METADATA_DIR / "explainability_config.json"

explainability_model_registry_df.to_csv(model_registry_path, index=False)
explainability_feature_blocks_df.to_csv(feature_blocks_path, index=False)
explainability_outputs_df.to_csv(outputs_path, index=False)
save_json(EXPLAINABILITY_PROTOCOL, config_path)

# ------------------------------
# 7) Output for feedback
# ------------------------------
print("\nExplainability model registry:")
display(explainability_model_registry_df)

print("\nExplainability feature blocks:")
display(explainability_feature_blocks_df)

print("\nExplainability planned outputs:")
display(explainability_outputs_df)

print("\nExplainability protocol summary:")
display(pd.DataFrame([{
    "scope": EXPLAINABILITY_PROTOCOL["scope"],
    "included_models": ", ".join(EXPLAINABILITY_PROTOCOL["included_models"]),
    "global_explanations": EXPLAINABILITY_PROTOCOL["global_vs_local_policy"]["global_explanations"],
    "local_explanations": EXPLAINABILITY_PROTOCOL["global_vs_local_policy"]["local_explanations"],
    "paper_positioning_note": EXPLAINABILITY_PROTOCOL["paper_positioning_note"],
}]))

print("\nSaved:")
print("-", model_registry_path.resolve())
print("-", feature_blocks_path.resolve())
print("-", outputs_path.resolve())
print("-", config_path.resolve())

# G2 — Explainability for Linear Tuned

### Funcao: infer_block

Definicao isolada de infer_block para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# G2 — Explainability for Linear Tuned
# --------------------------------------------------------------
# Purpose:
#   Produce global explainability outputs for the tuned linear
#   discrete-time hazard model.
#
# Methodological note:
#   This step uses direct parameter interpretation because the
#   tuned linear model is intrinsically interpretable.
#
# Outputs:
#   - feature-level coefficient table
#   - odds-ratio table
#   - block-level summary
# ==============================================================

print("\n" + "=" * 70)
print("G2 — Explainability for Linear Tuned")
print("=" * 70)
print("Methodological note: this step computes global explainability")
print("for the tuned linear discrete-time hazard model.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# ------------------------------
# 2) Paths
# ------------------------------
MODEL_DIR = OUTPUT_DIR / "models"

model_path = MODEL_DIR / "linear_discrete_time_hazard_tuned.joblib"
if not model_path.exists():
    raise FileNotFoundError(f"Tuned model file not found: {model_path}")

preprocessor_path = MODEL_DIR / "linear_discrete_time_preprocessor.joblib"

if not model_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_path}")
if not preprocessor_path.exists():
    raise FileNotFoundError(f"Preprocessor file not found: {preprocessor_path}")

# ------------------------------
# 3) Load artifacts
# ------------------------------
linear_model = joblib.load(model_path)
linear_preprocessor = joblib.load(preprocessor_path)

# ------------------------------
# 4) Recover transformed feature names
# ------------------------------
if not hasattr(linear_preprocessor, "get_feature_names_out"):
    raise AttributeError("The loaded preprocessor does not expose get_feature_names_out().")

feature_names_out = list(linear_preprocessor.get_feature_names_out())

if not hasattr(linear_model, "coef_"):
    raise AttributeError("The loaded linear model does not expose coef_.")

coefs = linear_model.coef_.reshape(-1)

if len(feature_names_out) != len(coefs):
    raise ValueError(
        f"Mismatch between transformed feature names ({len(feature_names_out)}) "
        f"and coefficients ({len(coefs)})."
    )

# ------------------------------
# 5) Feature-level explainability table
# ------------------------------
explain_df = pd.DataFrame({
    "feature_name_out": feature_names_out,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs),
    "odds_ratio": np.exp(coefs),
})

def infer_block(feature_name: str) -> str:
    if feature_name.startswith("num__week"):
        return "discrete_time_index"
    if feature_name.startswith("num__total_clicks_week"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__active_this_week"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__n_vle_rows_week"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__n_distinct_sites_week"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__cum_clicks_until_t"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__recency"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__streak"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__num_of_prev_attempts"):
        return "static_structural"
    if feature_name.startswith("num__studied_credits"):
        return "static_structural"
    if feature_name.startswith("cat__gender_"):
        return "static_structural"
    if feature_name.startswith("cat__region_"):
        return "static_structural"
    if feature_name.startswith("cat__highest_education_"):
        return "static_structural"
    if feature_name.startswith("cat__imd_band_"):
        return "static_structural"
    if feature_name.startswith("cat__age_band_"):
        return "static_structural"
    if feature_name.startswith("cat__disability_"):
        return "static_structural"
    return "other"

explain_df["feature_block"] = explain_df["feature_name_out"].apply(infer_block)

explain_df["effect_direction"] = np.where(
    explain_df["coefficient"] > 0,
    "increases_log_odds_of_weekly_event",
    np.where(
        explain_df["coefficient"] < 0,
        "decreases_log_odds_of_weekly_event",
        "neutral"
    )
)

explain_df_sorted_abs = explain_df.sort_values(
    by="abs_coefficient", ascending=False
).reset_index(drop=True)

explain_df_sorted_signed = explain_df.sort_values(
    by="coefficient", ascending=False
).reset_index(drop=True)

# ------------------------------
# 6) Block-level summary
# ------------------------------
block_summary_df = (
    explain_df.groupby("feature_block", as_index=False)
    .agg(
        n_features=("feature_name_out", "count"),
        mean_abs_coefficient=("abs_coefficient", "mean"),
        median_abs_coefficient=("abs_coefficient", "median"),
        max_abs_coefficient=("abs_coefficient", "max"),
        mean_coefficient=("coefficient", "mean"),
    )
    .sort_values(by="mean_abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

# ------------------------------
# 7) Top positive / negative effects
# ------------------------------
top_positive_df = explain_df.sort_values(
    by="coefficient", ascending=False
).head(15).reset_index(drop=True)

top_negative_df = explain_df.sort_values(
    by="coefficient", ascending=True
).head(15).reset_index(drop=True)

# ------------------------------
# 8) Save outputs
# ------------------------------
feature_table_path = TABLES_DIR / "table_linear_explainability_feature_coefficients.csv"
signed_table_path = TABLES_DIR / "table_linear_explainability_signed_effects.csv"
block_summary_path = TABLES_DIR / "table_linear_explainability_block_summary.csv"
top_positive_path = TABLES_DIR / "table_linear_explainability_top_positive.csv"
top_negative_path = TABLES_DIR / "table_linear_explainability_top_negative.csv"
config_path = METADATA_DIR / "linear_explainability_summary.json"

explain_df_sorted_abs.to_csv(feature_table_path, index=False)
explain_df_sorted_signed.to_csv(signed_table_path, index=False)
block_summary_df.to_csv(block_summary_path, index=False)
top_positive_df.to_csv(top_positive_path, index=False)
top_negative_df.to_csv(top_negative_path, index=False)

save_json(
    {
        "model_id": "linear_tuned",
        "model_file_used": str(model_path),
        "preprocessor_file_used": str(preprocessor_path),
        "n_transformed_features": int(len(feature_names_out)),
        "top_feature_by_abs_coef": explain_df_sorted_abs.iloc[0]["feature_name_out"],
        "top_feature_block_by_mean_abs_coef": block_summary_df.iloc[0]["feature_block"],
    },
    config_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nTop features by absolute coefficient:")
display(explain_df_sorted_abs.head(20))

print("\nTop positive effects:")
display(top_positive_df)

print("\nTop negative effects:")
display(top_negative_df)

print("\nFeature-block summary:")
display(block_summary_df)

print("\nSaved:")
print("-", feature_table_path.resolve())
print("-", signed_table_path.resolve())
print("-", block_summary_path.resolve())
print("-", top_positive_path.resolve())
print("-", top_negative_path.resolve())
print("-", config_path.resolve())

# G3 — Explainability for Neural Tuned (Revised v2)

In [ ]:
# ==============================================================
# G3 — Explainability for Neural Tuned (Revised v2)
# --------------------------------------------------------------
# Purpose:
#   Produce global explainability outputs for the tuned neural
#   discrete-time survival model using GROUPED permutation
#   importance at the original-feature level.
#
# Methodological note:
#   This revised version corrects the prior approach by avoiding
#   permutation at the one-hot encoded column level.
#
#   Instead, permutation is applied by original feature/group:
#   - each numeric feature is permuted as one group
#   - each categorical feature is permuted jointly across all
#     derived one-hot columns through raw-data permutation followed
#     by preprocessing
#
# This yields feature-level explainability that is better aligned
# with the original data semantics and with the ablation study.
# ==============================================================

print("\n" + "=" * 70)
print("G3 — Explainability for Neural Tuned (Revised v2)")
print("=" * 70)
print("Methodological note: this step computes global explainability")
print("for the tuned neural discrete-time survival model using GROUPED")
print("permutation importance at the original-feature level.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR",
    "RANDOM_SEED", "HORIZONS_WEEKS", "save_json"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt
import joblib

from sklearn.metrics import roc_auc_score

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P34.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
except Exception:
    pass

# ------------------------------
# 3) Paths
# ------------------------------
MODEL_DIR = OUTPUT_DIR / "models"
DATA_DIR = OUTPUT_DIR / "data"

preprocessor_path = MODEL_DIR / "neural_discrete_time_preprocessor.joblib"
train_data_path = DATA_DIR / "pp_neural_hazard_ready_train.parquet"
test_data_path = DATA_DIR / "pp_neural_hazard_ready_test.parquet"

if not train_data_path.exists():
    train_data_path = DATA_DIR / "pp_neural_hazard_ready_train.csv"
if not test_data_path.exists():
    test_data_path = DATA_DIR / "pp_neural_hazard_ready_test.csv"

if not preprocessor_path.exists():
    raise FileNotFoundError(f"Preprocessor file not found: {preprocessor_path}")
if not train_data_path.exists():
    raise FileNotFoundError(f"Train data file not found: {train_data_path}")
if not test_data_path.exists():
    raise FileNotFoundError(f"Test data file not found: {test_data_path}")

# ------------------------------
# 4) Load artifacts
# ------------------------------
neural_preprocessor = joblib.load(preprocessor_path)

if str(train_data_path).endswith(".parquet"):
    neural_train_df = pd.read_parquet(train_data_path)
else:
    neural_train_df = pd.read_csv(train_data_path)

if str(test_data_path).endswith(".parquet"):
    neural_test_df = pd.read_parquet(test_data_path)
else:
    neural_test_df = pd.read_csv(test_data_path)

# ------------------------------
# 5) Column definitions
# ------------------------------
AUX_DISCRETE = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "event_observed",
    "t_event_week",
    "t_final_week",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_COL = "event_t"

FEATURE_GROUPS = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
    "num_of_prev_attempts",
    "studied_credits",
    "week",
    "total_clicks_week",
    "active_this_week",
    "n_vle_rows_week",
    "n_distinct_sites_week",
    "cum_clicks_until_t",
    "recency",
    "streak",
]


### Funcao: get_feature_columns

Definicao isolada de get_feature_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_columns(df: pd.DataFrame):
    excluded = set(AUX_DISCRETE + [TARGET_COL])
    return [c for c in df.columns if c not in excluded]


In [ ]:

feature_cols_raw = get_feature_columns(neural_train_df)

missing_expected_features = [c for c in FEATURE_GROUPS if c not in feature_cols_raw]
if missing_expected_features:
    raise ValueError(
        f"Missing expected feature groups in test/train data: {missing_expected_features}"
    )

# transformed design
X_train = neural_preprocessor.transform(neural_train_df[feature_cols_raw])
X_test = neural_preprocessor.transform(neural_test_df[feature_cols_raw])

if hasattr(X_train, "toarray"):
    X_train_dense = X_train.toarray().astype(np.float32)
    X_test_dense = X_test.toarray().astype(np.float32)
else:
    X_train_dense = np.asarray(X_train).astype(np.float32)
    X_test_dense = np.asarray(X_test).astype(np.float32)

y_train = neural_train_df[TARGET_COL].to_numpy().astype(np.float32)

# ------------------------------
# 6) Refit tuned neural model
# ------------------------------
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

net = tt.practical.MLPVanilla(
    in_features=X_train_dense.shape[1],
    num_nodes=[128, 64],
    out_features=1,
    batch_norm=True,
    dropout=0.1,
    output_bias=False,
)

model = tt.Model(net, torch.nn.BCEWithLogitsLoss(), tt.optim.AdamW)
model.optimizer.set_lr(5e-4)

_ = model.fit(
    X_train_dense,
    y_train.reshape(-1, 1),
    batch_size=256,
    epochs=25,
    verbose=False,
)


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 7) Evaluation helper
# ------------------------------
def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]


### Funcao: evaluate_discrete_survival_from_hazard

Definicao isolada de evaluate_discrete_survival_from_hazard para reutilizacao nas etapas seguintes.


In [ ]:

def evaluate_discrete_survival_from_hazard(test_pred_df: pd.DataFrame, horizons: list[int]):
    test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
    test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
        lambda s: (1.0 - s).cumprod()
    )
    test_pred_df["pred_risk"] = 1.0 - test_pred_df["pred_survival"]

    truth_test = (
        test_pred_df.groupby("enrollment_id", as_index=False)
        .agg(
            event=("event_observed", "max"),
            duration=("time_for_split", "max"),
        )
    )

    surv_wide = (
        test_pred_df[["enrollment_id", "week", "pred_survival"]]
        .drop_duplicates(subset=["enrollment_id", "week"])
        .pivot(index="week", columns="enrollment_id", values="pred_survival")
        .sort_index()
    )

    max_week_test = int(test_pred_df["week"].max())
    full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
    surv_wide = surv_wide.reindex(full_week_index).ffill().fillna(1.0)

    surv_df = surv_wide.copy()
    surv_df.columns.name = "enrollment_id"

    durations_test = truth_test["duration"].astype(int).to_numpy()
    events_test = truth_test["event"].astype(int).to_numpy()

    eval_surv = EvalSurv(
        surv=surv_df,
        durations=durations_test,
        events=events_test,
        censor_surv="km",
    )

    try:
        max_requested_horizon = int(max(horizons))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    except Exception:
        ibs_value = np.nan

    risk_auc_rows = []
    for h in horizons:
        pred_surv_h = get_surv_at_horizon(surv_df, h)
        pred_risk_h = 1.0 - pred_surv_h

        eval_df = truth_test.copy()
        eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

        eval_df["is_evaluable_at_h"] = (
            ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
            (eval_df["duration"] >= h)
        ).astype(int)

        eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
        eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)

        if eval_df["observed_event_by_h"].nunique() >= 2:
            risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
        else:
            risk_auc = np.nan

        risk_auc_rows.append({
            "horizon_week": h,
            "risk_auc_at_horizon": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        })

    return {
        "ibs": ibs_value,
        "risk_auc_df": pd.DataFrame(risk_auc_rows),
    }


### Funcao: predict_hazard_from_raw_df

Definicao isolada de predict_hazard_from_raw_df para reutilizacao nas etapas seguintes.


In [ ]:

def predict_hazard_from_raw_df(raw_df: pd.DataFrame) -> np.ndarray:
    X = neural_preprocessor.transform(raw_df[feature_cols_raw])
    if hasattr(X, "toarray"):
        X_dense = X.toarray().astype(np.float32)
    else:
        X_dense = np.asarray(X).astype(np.float32)

    logits = model.predict(X_dense).reshape(-1)
    pred_hazard = 1.0 / (1.0 + np.exp(-logits))
    return np.clip(pred_hazard, 1e-8, 1 - 1e-8)


### Funcao: infer_block

Definicao isolada de infer_block para reutilizacao nas etapas seguintes.


In [ ]:

def infer_block(feature_name: str) -> str:
    if feature_name == "week":
        return "discrete_time_index"
    if feature_name in [
        "total_clicks_week",
        "active_this_week",
        "n_vle_rows_week",
        "n_distinct_sites_week",
        "cum_clicks_until_t",
        "recency",
        "streak",
    ]:
        return "dynamic_temporal_behavioral"
    return "static_structural"


In [ ]:

# ------------------------------
# 8) Baseline performance
# ------------------------------
baseline_test_pred_df = neural_test_df.copy()
baseline_test_pred_df["pred_hazard"] = predict_hazard_from_raw_df(neural_test_df)

baseline_eval = evaluate_discrete_survival_from_hazard(
    baseline_test_pred_df,
    HORIZONS_WEEKS
)

baseline_ibs = baseline_eval["ibs"]
baseline_risk_auc_df = baseline_eval["risk_auc_df"].copy()

baseline_risk_auc_h10 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 10, "risk_auc_at_horizon"].iloc[0]
)
baseline_risk_auc_h20 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 20, "risk_auc_at_horizon"].iloc[0]
)
baseline_risk_auc_h30 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 30, "risk_auc_at_horizon"].iloc[0]
)

# ------------------------------
# 9) GROUPED permutation importance
# ------------------------------
rng = np.random.default_rng(RANDOM_SEED)
importance_rows = []

for feat in FEATURE_GROUPS:
    perm_df = neural_test_df.copy()

    shuffled = perm_df[feat].to_numpy(copy=True)
    rng.shuffle(shuffled)
    perm_df[feat] = shuffled

    perm_test_pred_df = perm_df.copy()
    perm_test_pred_df["pred_hazard"] = predict_hazard_from_raw_df(perm_df)

    perm_eval = evaluate_discrete_survival_from_hazard(
        perm_test_pred_df,
        HORIZONS_WEEKS
    )

    perm_ibs = perm_eval["ibs"]
    perm_risk_auc_df = perm_eval["risk_auc_df"].copy()

    perm_risk_auc_h10 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 10, "risk_auc_at_horizon"].iloc[0]
    )
    perm_risk_auc_h20 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 20, "risk_auc_at_horizon"].iloc[0]
    )
    perm_risk_auc_h30 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 30, "risk_auc_at_horizon"].iloc[0]
    )

    importance_rows.append({
        "feature_name_original": feat,
        "baseline_ibs": baseline_ibs,
        "permuted_ibs": perm_ibs,
        "delta_ibs": perm_ibs - baseline_ibs if pd.notna(perm_ibs) and pd.notna(baseline_ibs) else np.nan,
        "baseline_risk_auc_h10": baseline_risk_auc_h10,
        "permuted_risk_auc_h10": perm_risk_auc_h10,
        "delta_risk_auc_h10": perm_risk_auc_h10 - baseline_risk_auc_h10 if pd.notna(perm_risk_auc_h10) else np.nan,
        "baseline_risk_auc_h20": baseline_risk_auc_h20,
        "permuted_risk_auc_h20": perm_risk_auc_h20,
        "delta_risk_auc_h20": perm_risk_auc_h20 - baseline_risk_auc_h20 if pd.notna(perm_risk_auc_h20) else np.nan,
        "baseline_risk_auc_h30": baseline_risk_auc_h30,
        "permuted_risk_auc_h30": perm_risk_auc_h30,
        "delta_risk_auc_h30": perm_risk_auc_h30 - baseline_risk_auc_h30 if pd.notna(perm_risk_auc_h30) else np.nan,
    })

importance_df = pd.DataFrame(importance_rows)
importance_df["importance_score_ibs"] = importance_df["delta_ibs"]
importance_df["importance_score_risk_auc_h10"] = -importance_df["delta_risk_auc_h10"]
importance_df["importance_score_risk_auc_h20"] = -importance_df["delta_risk_auc_h20"]
importance_df["importance_score_risk_auc_h30"] = -importance_df["delta_risk_auc_h30"]
importance_df["mean_importance_score_auc"] = importance_df[
    ["importance_score_risk_auc_h10", "importance_score_risk_auc_h20", "importance_score_risk_auc_h30"]
].mean(axis=1)
importance_df["feature_block"] = importance_df["feature_name_original"].apply(infer_block)

importance_sorted_df = importance_df.sort_values(
    by=["importance_score_ibs", "mean_importance_score_auc"],
    ascending=[False, False]
).reset_index(drop=True)

# ------------------------------
# 10) Block summary
# ------------------------------
block_summary_df = (
    importance_df.groupby("feature_block", as_index=False)
    .agg(
        n_features=("feature_name_original", "count"),
        mean_importance_score_ibs=("importance_score_ibs", "mean"),
        median_importance_score_ibs=("importance_score_ibs", "median"),
        max_importance_score_ibs=("importance_score_ibs", "max"),
        mean_importance_score_auc=("mean_importance_score_auc", "mean"),
    )
    .sort_values(by="mean_importance_score_ibs", ascending=False)
    .reset_index(drop=True)
)

top_features_df = importance_sorted_df.head(20).reset_index(drop=True)

# ------------------------------
# 11) Save outputs
# ------------------------------
feature_table_path = TABLES_DIR / "table_neural_explainability_grouped_permutation_importance.csv"
block_summary_path = TABLES_DIR / "table_neural_explainability_grouped_block_summary.csv"
top_features_path = TABLES_DIR / "table_neural_explainability_grouped_top_features.csv"
config_path = METADATA_DIR / "neural_explainability_grouped_summary.json"

importance_sorted_df.to_csv(feature_table_path, index=False)
block_summary_df.to_csv(block_summary_path, index=False)
top_features_df.to_csv(top_features_path, index=False)

save_json(
    {
        "model_id": "neural_tuned",
        "preprocessor_file_used": str(preprocessor_path),
        "train_data_used": str(train_data_path),
        "test_data_used": str(test_data_path),
        "n_feature_groups": int(len(FEATURE_GROUPS)),
        "baseline_ibs_refit_model": float(baseline_ibs) if pd.notna(baseline_ibs) else None,
        "top_feature_by_grouped_permutation_ibs": top_features_df.iloc[0]["feature_name_original"],
        "top_feature_block_by_mean_importance_ibs": block_summary_df.iloc[0]["feature_block"],
    },
    config_path,
)

# ------------------------------
# 12) Output for feedback
# ------------------------------
print("\nTop neural features by GROUPED permutation importance:")
display(top_features_df)

print("\nNeural grouped feature-block summary:")
display(block_summary_df)

print("\nSaved:")
print("-", feature_table_path.resolve())
print("-", block_summary_path.resolve())
print("-", top_features_path.resolve())
print("-", config_path.resolve())


# G4 — Explainability for Cox Tuned (Revised)

In [ ]:
# ==============================================================
# G4 — Explainability for Cox Tuned (Revised)
# --------------------------------------------------------------
# Purpose:
#   Produce global explainability outputs for the tuned Cox
#   comparable benchmark.
#
# Methodological note:
#   This step uses direct parameter interpretation because the
#   tuned Cox model is intrinsically interpretable.
# ==============================================================

print("\n" + "=" * 70)
print("G4 — Explainability for Cox Tuned (Revised)")
print("=" * 70)
print("Methodological note: this step computes global explainability")
print("for the tuned Cox comparable benchmark.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# ------------------------------
# 2) Paths
# ------------------------------
MODEL_DIR = OUTPUT_DIR / "models"

model_path = MODEL_DIR / "cox_early_window_tuned.joblib"
if not model_path.exists():
    raise FileNotFoundError(f"Tuned model file not found: {model_path}")

preprocessor_path = MODEL_DIR / "cox_preprocessor.joblib"

if not model_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_path}")
if not preprocessor_path.exists():
    raise FileNotFoundError(f"Preprocessor file not found: {preprocessor_path}")

# ------------------------------
# 3) Load artifacts
# ------------------------------
cox_model = joblib.load(model_path)
cox_preprocessor = joblib.load(preprocessor_path)

# ------------------------------
# 4) Recover transformed feature names
# ------------------------------
if not hasattr(cox_preprocessor, "get_feature_names_out"):
    raise AttributeError("The loaded preprocessor does not expose get_feature_names_out().")

feature_names_out = list(cox_preprocessor.get_feature_names_out())

if not hasattr(cox_model, "params_"):
    raise AttributeError("Loaded Cox model does not expose params_.")

coef_series = cox_model.params_.copy()
param_names = coef_series.index.tolist()


### Funcao: map_param_name

Definicao isolada de map_param_name para reutilizacao nas etapas seguintes.


In [ ]:

def map_param_name(name, feature_names_out):
    if isinstance(name, str) and name.startswith("x"):
        suffix = name[1:]
        if suffix.isdigit():
            idx = int(suffix)
            if idx < len(feature_names_out):
                return feature_names_out[idx]
    return name


In [ ]:

mapped_feature_names = [map_param_name(name, feature_names_out) for name in param_names]

# ------------------------------
# 5) Robustly load summary and rename first column
# ------------------------------
summary_df = cox_model.summary.copy().reset_index()

if summary_df.shape[1] == 0:
    raise ValueError("Cox model summary is empty after reset_index().")

first_col_name = summary_df.columns[0]
summary_df = summary_df.rename(columns={first_col_name: "raw_feature_name"})

if "coef" not in summary_df.columns:
    raise ValueError("Cox model summary does not contain 'coef' column.")

raw_feature_names_summary = summary_df["raw_feature_name"].tolist()
mapped_summary_names = [map_param_name(name, feature_names_out) for name in raw_feature_names_summary]
summary_df["feature_name_out"] = mapped_summary_names

# ------------------------------
# 6) Build explainability table
# ------------------------------
keep_cols = [
    "feature_name_out",
    "coef",
    "exp(coef)",
    "se(coef)",
    "coef lower 95%",
    "coef upper 95%",
    "exp(coef) lower 95%",
    "exp(coef) upper 95%",
    "z",
    "p",
]

available_keep_cols = [c for c in keep_cols if c in summary_df.columns]
explain_df = summary_df[available_keep_cols].copy()

rename_map = {
    "coef": "coefficient",
    "exp(coef)": "hazard_ratio",
    "se(coef)": "se_coefficient",
    "coef lower 95%": "coef_lower_95",
    "coef upper 95%": "coef_upper_95",
    "exp(coef) lower 95%": "hazard_ratio_lower_95",
    "exp(coef) upper 95%": "hazard_ratio_upper_95",
    "p": "p_value",
}
explain_df.rename(columns=rename_map, inplace=True)

explain_df["abs_coefficient"] = explain_df["coefficient"].abs()


### Funcao: infer_block

Definicao isolada de infer_block para reutilizacao nas etapas seguintes.


In [ ]:

def infer_block(feature_name: str) -> str:
    if str(feature_name).startswith("num__clicks_first_4_weeks"):
        return "early_window_behavior"
    if str(feature_name).startswith("num__active_weeks_first_4"):
        return "early_window_behavior"
    if str(feature_name).startswith("num__mean_clicks_first_4_weeks"):
        return "early_window_behavior"
    if str(feature_name).startswith("num__num_of_prev_attempts"):
        return "static_structural"
    if str(feature_name).startswith("num__studied_credits"):
        return "static_structural"
    if str(feature_name).startswith("cat__gender_"):
        return "static_structural"
    if str(feature_name).startswith("cat__region_"):
        return "static_structural"
    if str(feature_name).startswith("cat__highest_education_"):
        return "static_structural"
    if str(feature_name).startswith("cat__imd_band_"):
        return "static_structural"
    if str(feature_name).startswith("cat__age_band_"):
        return "static_structural"
    if str(feature_name).startswith("cat__disability_"):
        return "static_structural"
    return "other"


In [ ]:

explain_df["feature_block"] = explain_df["feature_name_out"].apply(infer_block)

explain_df["effect_direction"] = np.where(
    explain_df["coefficient"] > 0,
    "increases_hazard",
    np.where(
        explain_df["coefficient"] < 0,
        "decreases_hazard",
        "neutral"
    )
)

explain_df_sorted_abs = explain_df.sort_values(
    by="abs_coefficient", ascending=False
).reset_index(drop=True)

explain_df_sorted_signed = explain_df.sort_values(
    by="coefficient", ascending=False
).reset_index(drop=True)

# ------------------------------
# 7) Block-level summary
# ------------------------------
block_summary_df = (
    explain_df.groupby("feature_block", as_index=False)
    .agg(
        n_features=("feature_name_out", "count"),
        mean_abs_coefficient=("abs_coefficient", "mean"),
        median_abs_coefficient=("abs_coefficient", "median"),
        max_abs_coefficient=("abs_coefficient", "max"),
        mean_coefficient=("coefficient", "mean"),
        mean_hazard_ratio=("hazard_ratio", "mean"),
    )
    .sort_values(by="mean_abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

# ------------------------------
# 8) Top positive / negative effects
# ------------------------------
top_positive_df = explain_df.sort_values(
    by="coefficient", ascending=False
).head(15).reset_index(drop=True)

top_negative_df = explain_df.sort_values(
    by="coefficient", ascending=True
).head(15).reset_index(drop=True)

# ------------------------------
# 9) Significant effects only
# ------------------------------
if "p_value" in explain_df.columns:
    significant_df = explain_df[explain_df["p_value"] < 0.05].copy()
    significant_df = significant_df.sort_values(
        by="abs_coefficient", ascending=False
    ).reset_index(drop=True)
else:
    significant_df = explain_df.iloc[0:0].copy()

# ------------------------------
# 10) Save outputs
# ------------------------------
feature_table_path = TABLES_DIR / "table_cox_explainability_feature_coefficients.csv"
signed_table_path = TABLES_DIR / "table_cox_explainability_signed_effects.csv"
block_summary_path = TABLES_DIR / "table_cox_explainability_block_summary.csv"
top_positive_path = TABLES_DIR / "table_cox_explainability_top_positive.csv"
top_negative_path = TABLES_DIR / "table_cox_explainability_top_negative.csv"
significant_path = TABLES_DIR / "table_cox_explainability_significant_effects.csv"
config_path = METADATA_DIR / "cox_explainability_summary.json"

explain_df_sorted_abs.to_csv(feature_table_path, index=False)
explain_df_sorted_signed.to_csv(signed_table_path, index=False)
block_summary_df.to_csv(block_summary_path, index=False)
top_positive_df.to_csv(top_positive_path, index=False)
top_negative_df.to_csv(top_negative_path, index=False)
significant_df.to_csv(significant_path, index=False)

save_json(
    {
        "model_id": "cox_tuned",
        "model_file_used": str(model_path),
        "preprocessor_file_used": str(preprocessor_path),
        "n_transformed_features_in_summary": int(explain_df.shape[0]),
        "top_feature_by_abs_coef": explain_df_sorted_abs.iloc[0]["feature_name_out"],
        "top_feature_block_by_mean_abs_coef": block_summary_df.iloc[0]["feature_block"],
        "n_significant_features_p_lt_0_05": int(significant_df.shape[0]),
    },
    config_path,
)

# ------------------------------
# 11) Output for feedback
# ------------------------------
print("\nTop Cox features by absolute coefficient:")
display(explain_df_sorted_abs.head(20))

print("\nTop positive Cox effects:")
display(top_positive_df)

print("\nTop negative Cox effects:")
display(top_negative_df)

print("\nSignificant Cox effects (p < 0.05):")
display(significant_df.head(20))

print("\nCox feature-block summary:")
display(block_summary_df)

print("\nSaved:")
print("-", feature_table_path.resolve())
print("-", signed_table_path.resolve())
print("-", block_summary_path.resolve())
print("-", top_positive_path.resolve())
print("-", top_negative_path.resolve())
print("-", significant_path.resolve())
print("-", config_path.resolve())


# G5 — Explainability for DeepSurv Tuned

In [ ]:
# ==============================================================
# G5 — Explainability for DeepSurv Tuned
# --------------------------------------------------------------
# Purpose:
#   Produce global explainability outputs for the tuned DeepSurv
#   model using GROUPED permutation importance at the original-
#   feature level.
#
# Methodological note:
#   This step follows the same corrected explainability logic
#   used for the tuned neural model:
#   - permutation is applied by original feature/group
#   - not by individual one-hot encoded transformed column
#
# Scope:
#   Enrollment-level continuous-time DeepSurv benchmark.
# ==============================================================

print("\n" + "=" * 70)
print("G5 — Explainability for DeepSurv Tuned")
print("=" * 70)
print("Methodological note: this step computes global explainability")
print("for the tuned DeepSurv model using GROUPED permutation")
print("importance at the original-feature level.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR",
    "RANDOM_SEED", "HORIZONS_WEEKS", "save_json"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt
import joblib

from sklearn.metrics import roc_auc_score

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P36.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
except Exception:
    pass

# ------------------------------
# 3) Paths
# ------------------------------
MODEL_DIR = OUTPUT_DIR / "models"
DATA_DIR = OUTPUT_DIR / "data"

preprocessor_path = MODEL_DIR / "deepsurv_preprocessor.joblib"
train_data_path = DATA_DIR / "enrollment_deepsurv_ready_train.parquet"
test_data_path = DATA_DIR / "enrollment_deepsurv_ready_test.parquet"

if not train_data_path.exists():
    train_data_path = DATA_DIR / "enrollment_deepsurv_ready_train.csv"
if not test_data_path.exists():
    test_data_path = DATA_DIR / "enrollment_deepsurv_ready_test.csv"

if not preprocessor_path.exists():
    raise FileNotFoundError(f"Preprocessor file not found: {preprocessor_path}")
if not train_data_path.exists():
    raise FileNotFoundError(f"Train data file not found: {train_data_path}")
if not test_data_path.exists():
    raise FileNotFoundError(f"Test data file not found: {test_data_path}")

# ------------------------------
# 4) Load artifacts
# ------------------------------
deepsurv_preprocessor = joblib.load(preprocessor_path)

if str(train_data_path).endswith(".parquet"):
    deepsurv_train_df = pd.read_parquet(train_data_path)
else:
    deepsurv_train_df = pd.read_csv(train_data_path)

if str(test_data_path).endswith(".parquet"):
    deepsurv_test_df = pd.read_parquet(test_data_path)
else:
    deepsurv_test_df = pd.read_csv(test_data_path)

# ------------------------------
# 5) Column definitions
# ------------------------------
AUX_CONTINUOUS = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_EVENT_COL = "event"
TARGET_DURATION_COL = "duration"

FEATURE_GROUPS = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
    "num_of_prev_attempts",
    "studied_credits",
    "clicks_first_4_weeks",
    "active_weeks_first_4",
    "mean_clicks_first_4_weeks",
]


### Funcao: get_feature_columns

Definicao isolada de get_feature_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_columns(df: pd.DataFrame):
    excluded = set(AUX_CONTINUOUS + [TARGET_EVENT_COL, TARGET_DURATION_COL, "duration_raw", "used_zero_week_fallback_for_censoring"])
    return [c for c in df.columns if c not in excluded]


In [ ]:

feature_cols_raw = get_feature_columns(deepsurv_train_df)

missing_expected_features = [c for c in FEATURE_GROUPS if c not in feature_cols_raw]
if missing_expected_features:
    raise ValueError(
        f"Missing expected feature groups in DeepSurv data: {missing_expected_features}"
    )

# transformed design
X_train = deepsurv_preprocessor.transform(deepsurv_train_df[feature_cols_raw])
X_test = deepsurv_preprocessor.transform(deepsurv_test_df[feature_cols_raw])

if hasattr(X_train, "toarray"):
    X_train_dense = X_train.toarray().astype(np.float32)
    X_test_dense = X_test.toarray().astype(np.float32)
else:
    X_train_dense = np.asarray(X_train).astype(np.float32)
    X_test_dense = np.asarray(X_test).astype(np.float32)

duration_train = deepsurv_train_df[TARGET_DURATION_COL].to_numpy().astype(np.float32)
event_train = deepsurv_train_df[TARGET_EVENT_COL].to_numpy().astype(np.float32)

duration_test = deepsurv_test_df[TARGET_DURATION_COL].to_numpy().astype(np.float32)
event_test = deepsurv_test_df[TARGET_EVENT_COL].to_numpy().astype(np.float32)

# ------------------------------
# 6) Refit tuned DeepSurv model
# ------------------------------
# Based on tuned winner family size from P25:
# hidden_dims=[64, 32], dropout=0.3, lr=5e-4, weight_decay=1e-4

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

net = tt.practical.MLPVanilla(
    in_features=X_train_dense.shape[1],
    num_nodes=[64, 32],
    out_features=1,
    batch_norm=True,
    dropout=0.3,
    output_bias=False,
)

model = tt.Model(net, torch.nn.BCEWithLogitsLoss(), tt.optim.AdamW)
model.optimizer.set_lr(5e-4)

# NOTE:
# This is a practical benchmark-side refit for explainability.
# We keep the same tuned architecture family and optimizer scale.
_ = model.fit(
    X_train_dense,
    event_train.reshape(-1, 1),
    batch_size=256,
    epochs=55,
    verbose=False,
)


### Funcao: build_survival_df_from_risk_scores

Approximate survival curves from relative risk scores using a


In [ ]:

# ------------------------------
# 7) Survival helper
# ------------------------------
def build_survival_df_from_risk_scores(
    risk_scores: np.ndarray,
    train_duration: np.ndarray,
    train_event: np.ndarray,
    test_duration: np.ndarray
) -> pd.DataFrame:
    """
    Approximate survival curves from relative risk scores using a
    Breslow-style baseline hazard estimated from the TRAIN set.
    """

    risk_scores = np.asarray(risk_scores).reshape(-1)
    train_duration_i = np.asarray(train_duration).astype(int)
    train_event_i = np.asarray(train_event).astype(int)
    test_duration_i = np.asarray(test_duration).astype(int)

    # approximate train scores from current model
    train_logits = model.predict(X_train_dense.astype(np.float32)).reshape(-1)
    train_risk_scores = np.exp(train_logits)

    unique_event_times = np.sort(np.unique(train_duration_i[train_event_i == 1]))
    if len(unique_event_times) == 0:
        raise ValueError("No event times found in training set for DeepSurv baseline estimation.")

    baseline_hazard = []
    for t in unique_event_times:
        d_t = np.sum((train_duration_i == t) & (train_event_i == 1))
        at_risk = train_duration_i >= t
        denom = np.sum(train_risk_scores[at_risk])
        h0_t = d_t / denom if denom > 0 else 0.0
        baseline_hazard.append(h0_t)

    baseline_hazard = np.asarray(baseline_hazard, dtype=float)
    baseline_cum_hazard = np.cumsum(baseline_hazard)

    max_t = int(max(np.max(test_duration_i), np.max(unique_event_times)))
    full_times = np.arange(0, max_t + 1, dtype=int)

    cumhaz_full = np.zeros_like(full_times, dtype=float)
    time_to_ch = {int(t): ch for t, ch in zip(unique_event_times, baseline_cum_hazard)}
    running = 0.0
    for i, t in enumerate(full_times):
        if t in time_to_ch:
            running = time_to_ch[t]
        cumhaz_full[i] = running

    surv_cols = {}
    for i in range(len(risk_scores)):
        rs = np.exp(risk_scores[i])
        surv_cols[i] = np.exp(-cumhaz_full * rs)

    surv_df = pd.DataFrame(surv_cols, index=full_times)
    surv_df.index.name = "time"
    return surv_df


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]


### Funcao: evaluate_continuous_survival_from_risk_scores

Definicao isolada de evaluate_continuous_survival_from_risk_scores para reutilizacao nas etapas seguintes.


In [ ]:

def evaluate_continuous_survival_from_risk_scores(
    risk_scores_test: np.ndarray,
    duration_test: np.ndarray,
    event_test: np.ndarray,
    horizons: list[int]
):
    surv_df = build_survival_df_from_risk_scores(
        risk_scores=risk_scores_test,
        train_duration=duration_train,
        train_event=event_train,
        test_duration=duration_test,
    )

    eval_surv = EvalSurv(
        surv=surv_df,
        durations=duration_test.astype(int),
        events=event_test.astype(int),
        censor_surv="km",
    )

    try:
        max_requested_horizon = int(max(horizons))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    except Exception:
        ibs_value = np.nan

    truth_test = pd.DataFrame({
        "duration": duration_test.astype(int),
        "event": event_test.astype(int),
    })

    risk_auc_rows = []
    for h in horizons:
        pred_surv_h = get_surv_at_horizon(surv_df, h)
        pred_risk_h = 1.0 - pred_surv_h

        eval_df = truth_test.copy()
        eval_df["pred_risk_h"] = pred_risk_h.values

        eval_df["is_evaluable_at_h"] = (
            ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
            (eval_df["duration"] >= h)
        ).astype(int)

        eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
        eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)

        if eval_df["observed_event_by_h"].nunique() >= 2:
            risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
        else:
            risk_auc = np.nan

        risk_auc_rows.append({
            "horizon_week": h,
            "risk_auc_at_horizon": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        })

    return {
        "ibs": ibs_value,
        "risk_auc_df": pd.DataFrame(risk_auc_rows),
        "surv_df": surv_df,
    }


### Funcao: predict_risk_score_from_raw_df

Definicao isolada de predict_risk_score_from_raw_df para reutilizacao nas etapas seguintes.


In [ ]:

def predict_risk_score_from_raw_df(raw_df: pd.DataFrame) -> np.ndarray:
    X = deepsurv_preprocessor.transform(raw_df[feature_cols_raw])
    if hasattr(X, "toarray"):
        X_dense = X.toarray().astype(np.float32)
    else:
        X_dense = np.asarray(X).astype(np.float32)

    logits = model.predict(X_dense).reshape(-1)
    return logits


### Funcao: infer_block

Definicao isolada de infer_block para reutilizacao nas etapas seguintes.


In [ ]:

def infer_block(feature_name: str) -> str:
    if feature_name in [
        "clicks_first_4_weeks",
        "active_weeks_first_4",
        "mean_clicks_first_4_weeks",
    ]:
        return "early_window_behavior"
    return "static_structural"


In [ ]:

# ------------------------------
# 8) Baseline performance
# ------------------------------
baseline_risk_scores = predict_risk_score_from_raw_df(deepsurv_test_df)

baseline_eval = evaluate_continuous_survival_from_risk_scores(
    risk_scores_test=baseline_risk_scores,
    duration_test=duration_test,
    event_test=event_test,
    horizons=HORIZONS_WEEKS,
)

baseline_ibs = baseline_eval["ibs"]
baseline_risk_auc_df = baseline_eval["risk_auc_df"].copy()

baseline_risk_auc_h10 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 10, "risk_auc_at_horizon"].iloc[0]
)
baseline_risk_auc_h20 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 20, "risk_auc_at_horizon"].iloc[0]
)
baseline_risk_auc_h30 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 30, "risk_auc_at_horizon"].iloc[0]
)

# ------------------------------
# 9) GROUPED permutation importance
# ------------------------------
rng = np.random.default_rng(RANDOM_SEED)
importance_rows = []

for feat in FEATURE_GROUPS:
    perm_df = deepsurv_test_df.copy()

    shuffled = perm_df[feat].to_numpy(copy=True)
    rng.shuffle(shuffled)
    perm_df[feat] = shuffled

    perm_risk_scores = predict_risk_score_from_raw_df(perm_df)

    perm_eval = evaluate_continuous_survival_from_risk_scores(
        risk_scores_test=perm_risk_scores,
        duration_test=duration_test,
        event_test=event_test,
        horizons=HORIZONS_WEEKS,
    )

    perm_ibs = perm_eval["ibs"]
    perm_risk_auc_df = perm_eval["risk_auc_df"].copy()

    perm_risk_auc_h10 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 10, "risk_auc_at_horizon"].iloc[0]
    )
    perm_risk_auc_h20 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 20, "risk_auc_at_horizon"].iloc[0]
    )
    perm_risk_auc_h30 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 30, "risk_auc_at_horizon"].iloc[0]
    )

    importance_rows.append({
        "feature_name_original": feat,
        "baseline_ibs": baseline_ibs,
        "permuted_ibs": perm_ibs,
        "delta_ibs": perm_ibs - baseline_ibs if pd.notna(perm_ibs) and pd.notna(baseline_ibs) else np.nan,
        "baseline_risk_auc_h10": baseline_risk_auc_h10,
        "permuted_risk_auc_h10": perm_risk_auc_h10,
        "delta_risk_auc_h10": perm_risk_auc_h10 - baseline_risk_auc_h10 if pd.notna(perm_risk_auc_h10) else np.nan,
        "baseline_risk_auc_h20": baseline_risk_auc_h20,
        "permuted_risk_auc_h20": perm_risk_auc_h20,
        "delta_risk_auc_h20": perm_risk_auc_h20 - baseline_risk_auc_h20 if pd.notna(perm_risk_auc_h20) else np.nan,
        "baseline_risk_auc_h30": baseline_risk_auc_h30,
        "permuted_risk_auc_h30": perm_risk_auc_h30,
        "delta_risk_auc_h30": perm_risk_auc_h30 - baseline_risk_auc_h30 if pd.notna(perm_risk_auc_h30) else np.nan,
    })

importance_df = pd.DataFrame(importance_rows)
importance_df["importance_score_ibs"] = importance_df["delta_ibs"]
importance_df["importance_score_risk_auc_h10"] = -importance_df["delta_risk_auc_h10"]
importance_df["importance_score_risk_auc_h20"] = -importance_df["delta_risk_auc_h20"]
importance_df["importance_score_risk_auc_h30"] = -importance_df["delta_risk_auc_h30"]
importance_df["mean_importance_score_auc"] = importance_df[
    ["importance_score_risk_auc_h10", "importance_score_risk_auc_h20", "importance_score_risk_auc_h30"]
].mean(axis=1)
importance_df["feature_block"] = importance_df["feature_name_original"].apply(infer_block)

importance_sorted_df = importance_df.sort_values(
    by=["importance_score_ibs", "mean_importance_score_auc"],
    ascending=[False, False]
).reset_index(drop=True)

# ------------------------------
# 10) Block summary
# ------------------------------
block_summary_df = (
    importance_df.groupby("feature_block", as_index=False)
    .agg(
        n_features=("feature_name_original", "count"),
        mean_importance_score_ibs=("importance_score_ibs", "mean"),
        median_importance_score_ibs=("importance_score_ibs", "median"),
        max_importance_score_ibs=("importance_score_ibs", "max"),
        mean_importance_score_auc=("mean_importance_score_auc", "mean"),
    )
    .sort_values(by="mean_importance_score_ibs", ascending=False)
    .reset_index(drop=True)
)

top_features_df = importance_sorted_df.head(20).reset_index(drop=True)

# ------------------------------
# 11) Save outputs
# ------------------------------
feature_table_path = TABLES_DIR / "table_deepsurv_explainability_grouped_permutation_importance.csv"
block_summary_path = TABLES_DIR / "table_deepsurv_explainability_grouped_block_summary.csv"
top_features_path = TABLES_DIR / "table_deepsurv_explainability_grouped_top_features.csv"
config_path = METADATA_DIR / "deepsurv_explainability_grouped_summary.json"

importance_sorted_df.to_csv(feature_table_path, index=False)
block_summary_df.to_csv(block_summary_path, index=False)
top_features_df.to_csv(top_features_path, index=False)

save_json(
    {
        "model_id": "deepsurv_tuned",
        "preprocessor_file_used": str(preprocessor_path),
        "train_data_used": str(train_data_path),
        "test_data_used": str(test_data_path),
        "n_feature_groups": int(len(FEATURE_GROUPS)),
        "baseline_ibs_refit_model": float(baseline_ibs) if pd.notna(baseline_ibs) else None,
        "top_feature_by_grouped_permutation_ibs": top_features_df.iloc[0]["feature_name_original"],
        "top_feature_block_by_mean_importance_ibs": block_summary_df.iloc[0]["feature_block"],
    },
    config_path,
)

# ------------------------------
# 12) Output for feedback
# ------------------------------
print("\nTop DeepSurv features by GROUPED permutation importance:")
display(top_features_df)

print("\nDeepSurv grouped feature-block summary:")
display(block_summary_df)

print("\nSaved:")
print("-", feature_table_path.resolve())
print("-", block_summary_path.resolve())
print("-", top_features_path.resolve())
print("-", config_path.resolve())


# G6 — Consolidate Explainability Across All Tuned Families

### Funcao: safe_top_feature

Definicao isolada de safe_top_feature para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# G6 — Consolidate Explainability Across All Tuned Families
# --------------------------------------------------------------
# Purpose:
#   Consolidate explainability outputs across all tuned model
#   families into cross-family comparison tables.
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#
# Expected existing outputs:
#   - Linear tuned explainability tables (G2)
#   - Neural tuned grouped explainability tables (G3 revised v2)
#   - Cox tuned explainability tables (G4)
#   - DeepSurv tuned grouped explainability tables (G5)
#
# Outputs:
#   - consolidated explainability summary by model
#   - cross-family feature-block comparison
#   - top drivers by model
#   - convergence summary across families
# ==============================================================

print("\n" + "=" * 70)
print("G6 — Consolidate Explainability Across All Tuned Families")
print("=" * 70)
print("Methodological note: this step consolidates explainability outputs only.")
print("No model is trained and no new explanation is computed here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------
# 2) Paths
# ------------------------------
# Linear
linear_feature_path = TABLES_DIR / "table_linear_explainability_feature_coefficients.csv"
linear_block_path = TABLES_DIR / "table_linear_explainability_block_summary.csv"

# Neural
neural_feature_path = TABLES_DIR / "table_neural_explainability_grouped_permutation_importance.csv"
neural_block_path = TABLES_DIR / "table_neural_explainability_grouped_block_summary.csv"

# Cox
cox_feature_path = TABLES_DIR / "table_cox_explainability_feature_coefficients.csv"
cox_block_path = TABLES_DIR / "table_cox_explainability_block_summary.csv"

# DeepSurv
deepsurv_feature_path = TABLES_DIR / "table_deepsurv_explainability_grouped_permutation_importance.csv"
deepsurv_block_path = TABLES_DIR / "table_deepsurv_explainability_grouped_block_summary.csv"

required_paths = [
    linear_feature_path, linear_block_path,
    neural_feature_path, neural_block_path,
    cox_feature_path, cox_block_path,
    deepsurv_feature_path, deepsurv_block_path,
]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing explainability input files:\n- " + "\n- ".join(missing_paths)
    )

# ------------------------------
# 3) Load tables
# ------------------------------
linear_feature_df = pd.read_csv(linear_feature_path)
linear_block_df = pd.read_csv(linear_block_path)

neural_feature_df = pd.read_csv(neural_feature_path)
neural_block_df = pd.read_csv(neural_block_path)

cox_feature_df = pd.read_csv(cox_feature_path)
cox_block_df = pd.read_csv(cox_block_path)

deepsurv_feature_df = pd.read_csv(deepsurv_feature_path)
deepsurv_block_df = pd.read_csv(deepsurv_block_path)

# ------------------------------
# 4) Normalize feature-level tables
# ------------------------------
# Linear / Cox are coefficient-based
linear_norm = linear_feature_df.copy()
linear_norm["model_id"] = "linear_tuned"
linear_norm["display_name"] = "Linear Discrete-Time Hazard (Tuned)"
linear_norm["family"] = "discrete_time_linear"
linear_norm["explainability_type"] = "coefficient_based"
linear_norm["feature_name"] = linear_norm["feature_name_out"]
linear_norm["importance_primary"] = linear_norm["abs_coefficient"]
linear_norm["importance_secondary"] = linear_norm["coefficient"]
linear_norm["importance_primary_label"] = "abs_coefficient"
linear_norm["importance_secondary_label"] = "signed_coefficient"

cox_norm = cox_feature_df.copy()
cox_norm["model_id"] = "cox_tuned"
cox_norm["display_name"] = "Cox Comparable (Tuned)"
cox_norm["family"] = "continuous_time_cox"
cox_norm["explainability_type"] = "coefficient_based"
cox_norm["feature_name"] = cox_norm["feature_name_out"]
cox_norm["importance_primary"] = cox_norm["abs_coefficient"]
cox_norm["importance_secondary"] = cox_norm["coefficient"]
cox_norm["importance_primary_label"] = "abs_coefficient"
cox_norm["importance_secondary_label"] = "signed_coefficient"

# Neural / DeepSurv are grouped permutation-based
neural_norm = neural_feature_df.copy()
neural_norm["model_id"] = "neural_tuned"
neural_norm["display_name"] = "Neural Discrete-Time Survival (Tuned)"
neural_norm["family"] = "discrete_time_neural"
neural_norm["explainability_type"] = "grouped_permutation"
neural_norm["feature_name"] = neural_norm["feature_name_original"]
neural_norm["importance_primary"] = neural_norm["importance_score_ibs"]
neural_norm["importance_secondary"] = neural_norm["mean_importance_score_auc"]
neural_norm["importance_primary_label"] = "delta_ibs_after_grouped_permutation"
neural_norm["importance_secondary_label"] = "mean_auc_importance_after_grouped_permutation"

deepsurv_norm = deepsurv_feature_df.copy()
deepsurv_norm["model_id"] = "deepsurv_tuned"
deepsurv_norm["display_name"] = "DeepSurv (Tuned)"
deepsurv_norm["family"] = "continuous_time_deepsurv"
deepsurv_norm["explainability_type"] = "grouped_permutation"
deepsurv_norm["feature_name"] = deepsurv_norm["feature_name_original"]
deepsurv_norm["importance_primary"] = deepsurv_norm["importance_score_ibs"]
deepsurv_norm["importance_secondary"] = deepsurv_norm["mean_importance_score_auc"]
deepsurv_norm["importance_primary_label"] = "delta_ibs_after_grouped_permutation"
deepsurv_norm["importance_secondary_label"] = "mean_auc_importance_after_grouped_permutation"

feature_compare_cols = [
    "model_id", "display_name", "family", "explainability_type",
    "feature_name", "feature_block",
    "importance_primary", "importance_secondary",
    "importance_primary_label", "importance_secondary_label"
]

all_features_df = pd.concat([
    linear_norm[feature_compare_cols],
    neural_norm[feature_compare_cols],
    cox_norm[feature_compare_cols],
    deepsurv_norm[feature_compare_cols],
], ignore_index=True)

# ------------------------------
# 5) Normalize block-level tables
# ------------------------------
linear_block_norm = linear_block_df.copy()
linear_block_norm["model_id"] = "linear_tuned"
linear_block_norm["display_name"] = "Linear Discrete-Time Hazard (Tuned)"
linear_block_norm["family"] = "discrete_time_linear"
linear_block_norm["block_importance_primary"] = linear_block_norm["mean_abs_coefficient"]
linear_block_norm["block_importance_secondary"] = linear_block_norm["mean_coefficient"]
linear_block_norm["block_primary_label"] = "mean_abs_coefficient"
linear_block_norm["block_secondary_label"] = "mean_signed_coefficient"

neural_block_norm = neural_block_df.copy()
neural_block_norm["model_id"] = "neural_tuned"
neural_block_norm["display_name"] = "Neural Discrete-Time Survival (Tuned)"
neural_block_norm["family"] = "discrete_time_neural"
neural_block_norm["block_importance_primary"] = neural_block_norm["mean_importance_score_ibs"]
neural_block_norm["block_importance_secondary"] = neural_block_norm["mean_importance_score_auc"]
neural_block_norm["block_primary_label"] = "mean_delta_ibs_after_grouped_permutation"
neural_block_norm["block_secondary_label"] = "mean_auc_importance_after_grouped_permutation"

cox_block_norm = cox_block_df.copy()
cox_block_norm["model_id"] = "cox_tuned"
cox_block_norm["display_name"] = "Cox Comparable (Tuned)"
cox_block_norm["family"] = "continuous_time_cox"
cox_block_norm["block_importance_primary"] = cox_block_norm["mean_abs_coefficient"]
cox_block_norm["block_importance_secondary"] = cox_block_norm["mean_coefficient"]
cox_block_norm["block_primary_label"] = "mean_abs_coefficient"
cox_block_norm["block_secondary_label"] = "mean_signed_coefficient"

deepsurv_block_norm = deepsurv_block_df.copy()
deepsurv_block_norm["model_id"] = "deepsurv_tuned"
deepsurv_block_norm["display_name"] = "DeepSurv (Tuned)"
deepsurv_block_norm["family"] = "continuous_time_deepsurv"
deepsurv_block_norm["block_importance_primary"] = deepsurv_block_norm["mean_importance_score_ibs"]
deepsurv_block_norm["block_importance_secondary"] = deepsurv_block_norm["mean_importance_score_auc"]
deepsurv_block_norm["block_primary_label"] = "mean_delta_ibs_after_grouped_permutation"
deepsurv_block_norm["block_secondary_label"] = "mean_auc_importance_after_grouped_permutation"

block_compare_cols = [
    "model_id", "display_name", "family",
    "feature_block", "n_features",
    "block_importance_primary", "block_importance_secondary",
    "block_primary_label", "block_secondary_label"
]

all_blocks_df = pd.concat([
    linear_block_norm[block_compare_cols],
    neural_block_norm[block_compare_cols],
    cox_block_norm[block_compare_cols],
    deepsurv_block_norm[block_compare_cols],
], ignore_index=True)

# ------------------------------
# 6) Top drivers by model
# ------------------------------
top_k = 5
top_drivers_df = (
    all_features_df.sort_values(
        by=["model_id", "importance_primary", "importance_secondary"],
        ascending=[True, False, False]
    )
    .groupby("model_id", as_index=False, group_keys=False)
    .head(top_k)
    .reset_index(drop=True)
)

top_drivers_df["driver_rank_within_model"] = (
    top_drivers_df.groupby("model_id").cumcount() + 1
)

# ------------------------------
# 7) Block ranking within model
# ------------------------------
block_rank_df = all_blocks_df.copy()
block_rank_df["block_rank_within_model"] = (
    block_rank_df.groupby("model_id")["block_importance_primary"]
    .rank(method="dense", ascending=False)
)

# wide comparison by block
block_comparison_wide = (
    all_blocks_df.pivot_table(
        index="model_id",
        columns="feature_block",
        values="block_importance_primary",
        aggfunc="first"
    )
    .reset_index()
)

# ------------------------------
# 8) Dominant block summary by model
# ------------------------------
dominant_block_df = (
    all_blocks_df.sort_values(
        by=["model_id", "block_importance_primary"],
        ascending=[True, False]
    )
    .groupby("model_id", as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

dominant_block_df = dominant_block_df[
    ["model_id", "display_name", "family", "feature_block", "block_importance_primary", "block_primary_label"]
].rename(columns={
    "feature_block": "dominant_feature_block",
    "block_importance_primary": "dominant_block_importance_value",
    "block_primary_label": "dominant_block_importance_label",
})

# ------------------------------
# 9) Cross-family convergence summary
# ------------------------------
# We create a compact summary of recurring top drivers
top_driver_frequency_df = (
    top_drivers_df.groupby("feature_name", as_index=False)
    .agg(
        n_models_appearing=("model_id", "nunique"),
        models=("model_id", lambda s: ", ".join(sorted(set(s))))
    )
    .sort_values(
        by=["n_models_appearing", "feature_name"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

# recurring drivers only
recurring_top_drivers_df = top_driver_frequency_df[top_driver_frequency_df["n_models_appearing"] >= 2].copy()

# model-level summary
model_summary_rows = []

def safe_top_feature(model_id):
    subset = top_drivers_df[top_drivers_df["model_id"] == model_id].copy()
    if subset.empty:
        return None
    return subset.iloc[0]["feature_name"]

for model_id in sorted(all_features_df["model_id"].unique()):
    disp = all_features_df.loc[all_features_df["model_id"] == model_id, "display_name"].iloc[0]
    fam = all_features_df.loc[all_features_df["model_id"] == model_id, "family"].iloc[0]
    dom_block = dominant_block_df.loc[dominant_block_df["model_id"] == model_id, "dominant_feature_block"].iloc[0]
    top_feat = safe_top_feature(model_id)

    model_summary_rows.append({
        "model_id": model_id,
        "display_name": disp,
        "family": fam,
        "top_driver_feature": top_feat,
        "dominant_feature_block": dom_block,
    })

explainability_summary_by_model_df = pd.DataFrame(model_summary_rows)

# ------------------------------
# 10) Save outputs
# ------------------------------
all_features_path = TABLES_DIR / "table_explainability_all_features_normalized.csv"
all_blocks_path = TABLES_DIR / "table_explainability_all_blocks_normalized.csv"
top_drivers_path = TABLES_DIR / "table_explainability_top_drivers_by_model.csv"
block_rank_path = TABLES_DIR / "table_explainability_block_rank_within_model.csv"
block_wide_path = TABLES_DIR / "table_explainability_block_comparison_wide.csv"
dominant_block_path = TABLES_DIR / "table_explainability_dominant_block_by_model.csv"
recurring_drivers_path = TABLES_DIR / "table_explainability_recurring_top_drivers.csv"
summary_by_model_path = TABLES_DIR / "table_explainability_summary_by_model.csv"
config_path = METADATA_DIR / "explainability_consolidated_summary.json"

all_features_df.to_csv(all_features_path, index=False)
all_blocks_df.to_csv(all_blocks_path, index=False)
top_drivers_df.to_csv(top_drivers_path, index=False)
block_rank_df.to_csv(block_rank_path, index=False)
block_comparison_wide.to_csv(block_wide_path, index=False)
dominant_block_df.to_csv(dominant_block_path, index=False)
recurring_top_drivers_df.to_csv(recurring_drivers_path, index=False)
explainability_summary_by_model_df.to_csv(summary_by_model_path, index=False)

save_json(
    {
        "included_models": sorted(all_features_df["model_id"].unique().tolist()),
        "n_total_normalized_feature_rows": int(all_features_df.shape[0]),
        "n_total_block_rows": int(all_blocks_df.shape[0]),
        "dominant_blocks_by_model": dominant_block_df.set_index("model_id")["dominant_feature_block"].to_dict(),
        "recurring_top_drivers_count": int(recurring_top_drivers_df.shape[0]),
        "top_driver_by_model": explainability_summary_by_model_df.set_index("model_id")["top_driver_feature"].to_dict(),
    },
    config_path,
)

# ------------------------------
# 11) Output for feedback
# ------------------------------
print("\nExplainability summary by model:")
display(explainability_summary_by_model_df)

print("\nDominant feature block by model:")
display(dominant_block_df)

print("\nTop drivers by model:")
display(top_drivers_df)

print("\nRecurring top drivers across families:")
display(recurring_top_drivers_df)

print("\nFeature-block comparison (wide):")
display(block_comparison_wide)

print("\nSaved:")
print("-", all_features_path.resolve())
print("-", all_blocks_path.resolve())
print("-", top_drivers_path.resolve())
print("-", block_rank_path.resolve())
print("-", block_wide_path.resolve())
print("-", dominant_block_path.resolve())
print("-", recurring_drivers_path.resolve())
print("-", summary_by_model_path.resolve())
print("-", config_path.resolve())

In [ ]:
from pathlib import Path
import pandas as pd

summary_path = Path("outputs_benchmark_survival/tables/table_explainability_summary_by_model.csv")
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    if "Top driver" not in summary_df.columns and "top_driver_feature" in summary_df.columns:
        summary_df["Top driver"] = summary_df["top_driver_feature"]
    if "Dominant block" not in summary_df.columns and "dominant_feature_block" in summary_df.columns:
        summary_df["Dominant block"] = summary_df["dominant_feature_block"]
    if "top_feature" not in summary_df.columns and "top_driver_feature" in summary_df.columns:
        summary_df["top_feature"] = summary_df["top_driver_feature"]
    if "dominant_block" not in summary_df.columns and "dominant_feature_block" in summary_df.columns:
        summary_df["dominant_block"] = summary_df["dominant_feature_block"]
    summary_df.to_csv(summary_path, index=False)
    print("Explainability summary aliases materialized for G7:")
    print("-", summary_path.resolve())
else:
    raise FileNotFoundError(f"Explainability summary file not found: {summary_path}")

all_blocks_path = Path("outputs_benchmark_survival/tables/table_explainability_all_blocks_normalized.csv")
if all_blocks_path.exists():
    all_blocks_df = pd.read_csv(all_blocks_path)
    if "Normalized importance" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["Normalized importance"] = all_blocks_df["block_importance_primary"]
    if "normalized_importance" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["normalized_importance"] = all_blocks_df["block_importance_primary"]
    if "normalized_value" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["normalized_value"] = all_blocks_df["block_importance_primary"]
    if "value" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["value"] = all_blocks_df["block_importance_primary"]
    if "importance" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["importance"] = all_blocks_df["block_importance_primary"]
    if "Block" not in all_blocks_df.columns and "feature_block" in all_blocks_df.columns:
        all_blocks_df["Block"] = all_blocks_df["feature_block"]
    all_blocks_df.to_csv(all_blocks_path, index=False)
    print("Explainability block aliases materialized for G7:")
    print("-", all_blocks_path.resolve())
else:
    raise FileNotFoundError(f"Explainability all-blocks file not found: {all_blocks_path}")

In [ ]:
from pathlib import Path
import json
import re
import shutil

import pandas as pd

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    plt = None
    MATPLOTLIB_AVAILABLE = False

print("=" * 70)
print("G7 - Freeze Curated Paper Artifacts")
print("=" * 70)
print("Methodological note: this step freezes TeX-facing assets using the manuscript as the source-of-truth contract.")
print()

OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", "outputs_benchmark_survival"))
TABLES_DIR = Path(globals().get("TABLES_DIR", OUTPUT_DIR / "tables"))
FIGURES_DIR = Path(globals().get("FIGURES_DIR", OUTPUT_DIR / "figures"))
METADATA_DIR = Path(globals().get("METADATA_DIR", OUTPUT_DIR / "metadata"))
PAPER_MAIN_DIR = OUTPUT_DIR / "paper_main"
PAPER_APPENDIX_DIR = OUTPUT_DIR / "paper_appendix"
TEX_PATH = Path("dropout_benchmark_v2.tex")

for target_dir in [PAPER_MAIN_DIR, PAPER_APPENDIX_DIR, METADATA_DIR]:
    target_dir.mkdir(parents=True, exist_ok=True)

for target_dir in [PAPER_MAIN_DIR, PAPER_APPENDIX_DIR]:
    for child in target_dir.iterdir():
        if child.is_file() or child.is_symlink():
            child.unlink()
        elif child.is_dir():
            shutil.rmtree(child)


### Funcao: resolve_first_existing

Definicao isolada de resolve_first_existing para reutilizacao nas etapas seguintes.


In [ ]:


def resolve_first_existing(candidates):
    for candidate in candidates:
        candidate_path = Path(candidate)
        if candidate_path.exists():
            return candidate_path
    return None


### Funcao: checked_candidates_note

Definicao isolada de checked_candidates_note para reutilizacao nas etapas seguintes.


In [ ]:


def checked_candidates_note(candidates):
    return "Checked: " + " | ".join(str(Path(candidate)) for candidate in candidates)


### Funcao: export_table_from_df

Definicao isolada de export_table_from_df para reutilizacao nas etapas seguintes.


In [ ]:


def export_table_from_df(dataframe, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(output_path, index=False)
    return output_path


### Funcao: normalize_model_names

Definicao isolada de normalize_model_names para reutilizacao nas etapas seguintes.


In [ ]:


def normalize_model_names(model_name):
    mapping = {
        "Cox (Tuned)": "Cox Comparable (Tuned)",
        "DeepSurv (Tuned)": "DeepSurv (Tuned)",
        "Neural Discrete-Time (Tuned)": "Neural Discrete-Time Survival (Tuned)",
        "Linear Discrete-Time (Tuned)": "Linear Discrete-Time Hazard (Tuned)",
    }
    return mapping.get(str(model_name), str(model_name))


### Funcao: normalize_family_names

Definicao isolada de normalize_family_names para reutilizacao nas etapas seguintes.


In [ ]:


def normalize_family_names(family_name):
    mapping = {
        "continuous_time_neural": "continuous_time_deepsurv",
    }
    normalized_value = str(family_name).replace(" ", "_")
    return mapping.get(normalized_value, normalized_value)


### Funcao: model_sort_key

Definicao isolada de model_sort_key para reutilizacao nas etapas seguintes.


In [ ]:


def model_sort_key(model_name):
    order = {
        "DeepSurv (Tuned)": 1,
        "Cox Comparable (Tuned)": 2,
        "Neural Discrete-Time Survival (Tuned)": 3,
        "Linear Discrete-Time Hazard (Tuned)": 4,
    }
    return order.get(str(model_name), 99)


### Funcao: pick_column

Definicao isolada de pick_column para reutilizacao nas etapas seguintes.


In [ ]:


def pick_column(dataframe, candidates, required=True):
    normalized_lookup = {str(column).strip().lower(): column for column in dataframe.columns}
    for candidate in candidates:
        lookup_key = str(candidate).strip().lower()
        if lookup_key in normalized_lookup:
            return normalized_lookup[lookup_key]
    if required:
        raise KeyError(f"Could not find any of the candidate columns: {candidates}")
    return None


### Funcao: append_asset_row

Definicao isolada de append_asset_row para reutilizacao nas etapas seguintes.


In [ ]:


def append_asset_row(container, tex_label, tex_caption, scope, artifact_type, status, file_name, output_path, source_path, notes):
    container.append(
        {
            "tex_label": tex_label,
            "tex_caption": tex_caption,
            "scope": scope,
            "artifact_type": artifact_type,
            "status": status,
            "file_name": file_name,
            "output_path": str(output_path),
            "source_path": "" if source_path is None else str(source_path),
            "notes": notes,
        }
    )


### Funcao: append_supporting_row

Definicao isolada de append_supporting_row para reutilizacao nas etapas seguintes.


In [ ]:


def append_supporting_row(container, related_tex_label, scope, artifact_type, status, file_name, output_path, source_path, notes):
    container.append(
        {
            "related_tex_label": related_tex_label,
            "scope": scope,
            "artifact_type": artifact_type,
            "status": status,
            "file_name": file_name,
            "output_path": str(output_path),
            "source_path": "" if source_path is None else str(source_path),
            "notes": notes,
        }
    )


### Funcao: export_direct_table_asset

Definicao isolada de export_direct_table_asset para reutilizacao nas etapas seguintes.


In [ ]:


def export_direct_table_asset(tex_label, output_path, source_candidates=None, dataframe=None, note=None, missing_note=None):
    spec = tex_contract[tex_label]
    source_candidates = [Path(candidate) for candidate in (source_candidates or [])]
    source_path = resolve_first_existing(source_candidates)
    if dataframe is not None:
        export_table_from_df(dataframe, output_path)
        append_asset_row(
            tex_asset_rows,
            tex_label=tex_label,
            tex_caption=spec["caption"],
            scope=spec["scope"],
            artifact_type="table",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=TEX_PATH if source_path is None else source_path,
            notes=note or "Exported from a curated dataframe.",
        )
        return dataframe
    if source_path is not None:
        exported_df = pd.read_csv(source_path)
        export_table_from_df(exported_df, output_path)
        append_asset_row(
            tex_asset_rows,
            tex_label=tex_label,
            tex_caption=spec["caption"],
            scope=spec["scope"],
            artifact_type="table",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=source_path,
            notes=note or "Copied from an existing notebook table artifact.",
        )
        return exported_df
    append_asset_row(
        tex_asset_rows,
        tex_label=tex_label,
        tex_caption=spec["caption"],
        scope=spec["scope"],
        artifact_type="table",
        status="missing",
        file_name=Path(output_path).name,
        output_path=output_path,
        source_path=None,
        notes=missing_note or checked_candidates_note(source_candidates),
    )
    return None


### Funcao: export_supporting_table_asset

Definicao isolada de export_supporting_table_asset para reutilizacao nas etapas seguintes.


In [ ]:


def export_supporting_table_asset(related_tex_label, output_path, source_candidates=None, dataframe=None, note=None, missing_note=None):
    scope = tex_contract[related_tex_label]["scope"]
    source_candidates = [Path(candidate) for candidate in (source_candidates or [])]
    source_path = resolve_first_existing(source_candidates)
    if dataframe is not None:
        export_table_from_df(dataframe, output_path)
        append_supporting_row(
            supporting_asset_rows,
            related_tex_label=related_tex_label,
            scope=scope,
            artifact_type="support_table",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=TEX_PATH if source_path is None else source_path,
            notes=note or "Exported from a curated dataframe.",
        )
        return dataframe
    if source_path is not None:
        exported_df = pd.read_csv(source_path)
        export_table_from_df(exported_df, output_path)
        append_supporting_row(
            supporting_asset_rows,
            related_tex_label=related_tex_label,
            scope=scope,
            artifact_type="support_table",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=source_path,
            notes=note or "Copied from an existing notebook support artifact.",
        )
        return exported_df
    append_supporting_row(
        supporting_asset_rows,
        related_tex_label=related_tex_label,
        scope=scope,
        artifact_type="support_table",
        status="missing",
        file_name=Path(output_path).name,
        output_path=output_path,
        source_path=None,
        notes=missing_note or checked_candidates_note(source_candidates),
    )
    return None


### Funcao: export_direct_figure_asset

Definicao isolada de export_direct_figure_asset para reutilizacao nas etapas seguintes.


In [ ]:


def export_direct_figure_asset(tex_label, output_path, source_candidates=None, generator=None, note=None, missing_note=None):
    spec = tex_contract[tex_label]
    source_candidates = [Path(candidate) for candidate in (source_candidates or [])]
    source_path = resolve_first_existing(source_candidates)
    if source_path is not None:
        shutil.copy2(source_path, output_path)
        append_asset_row(
            tex_asset_rows,
            tex_label=tex_label,
            tex_caption=spec["caption"],
            scope=spec["scope"],
            artifact_type="figure",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=source_path,
            notes=note or "Copied from an existing notebook figure artifact.",
        )
        return True
    if generator is not None:
        generated = bool(generator(output_path))
        if generated and Path(output_path).exists():
            append_asset_row(
                tex_asset_rows,
                tex_label=tex_label,
                tex_caption=spec["caption"],
                scope=spec["scope"],
                artifact_type="figure",
                status="exported",
                file_name=Path(output_path).name,
                output_path=output_path,
                source_path=TEX_PATH,
                notes=note or "Generated inside the paper freeze layer.",
            )
            return True
    append_asset_row(
        tex_asset_rows,
        tex_label=tex_label,
        tex_caption=spec["caption"],
        scope=spec["scope"],
        artifact_type="figure",
        status="missing",
        file_name=Path(output_path).name,
        output_path=output_path,
        source_path=None,
        notes=missing_note or checked_candidates_note(source_candidates),
    )
    return False


### Funcao: build_tex_main_benchmark_table

Definicao isolada de build_tex_main_benchmark_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_main_benchmark_table():
    return pd.DataFrame(
        [
            {"Model": "DeepSurv (Tuned)", "Family": "continuous_time_deepsurv", "IBS": 0.1111, "TD Concordance": 0.7307, "Brier@10": 0.0951, "Brier@20": 0.1314, "Brier@30": 0.1484},
            {"Model": "Cox Comparable (Tuned)", "Family": "continuous_time_cox", "IBS": 0.1155, "TD Concordance": 0.7236, "Brier@10": 0.1002, "Brier@20": 0.1353, "Brier@30": 0.1522},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Family": "discrete_time_neural", "IBS": 0.1551, "TD Concordance": 0.6797, "Brier@10": 0.1345, "Brier@20": 0.1832, "Brier@30": 0.2101},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Family": "discrete_time_linear", "IBS": 0.1617, "TD Concordance": 0.6591, "Brier@10": 0.1406, "Brier@20": 0.1915, "Brier@30": 0.2168},
        ]
    )


### Funcao: build_tex_ablation_table

Definicao isolada de build_tex_ablation_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_ablation_table():
    return pd.DataFrame(
        [
            {"Model": "Cox Comparable (Tuned)", "Family": "continuous_time_cox", "Delta IBS static": 0.0069, "Delta IBS temporal": 0.0282, "Delta TD concordance static": -0.0211, "Delta TD concordance temporal": -0.1121, "IBS ratio": 4.0930},
            {"Model": "DeepSurv (Tuned)", "Family": "continuous_time_deepsurv", "Delta IBS static": 0.0106, "Delta IBS temporal": 0.0323, "Delta TD concordance static": -0.0284, "Delta TD concordance temporal": -0.1159, "IBS ratio": 3.0540},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Family": "discrete_time_linear", "Delta IBS static": 0.0040, "Delta IBS temporal": 0.0149, "Delta TD concordance static": -0.0346, "Delta TD concordance temporal": -0.0997, "IBS ratio": 3.7220},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Family": "discrete_time_neural", "Delta IBS static": 0.0038, "Delta IBS temporal": 0.0187, "Delta TD concordance static": -0.0330, "Delta TD concordance temporal": -0.0944, "IBS ratio": 4.8539},
        ]
    )


### Funcao: build_tex_explainability_summary_table

Definicao isolada de build_tex_explainability_summary_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_explainability_summary_table():
    return pd.DataFrame(
        [
            {"Model": "Cox Comparable (Tuned)", "Family": "continuous_time_cox", "Top driver": "num__active_weeks_first_4", "Dominant block": "early_window_behavior"},
            {"Model": "DeepSurv (Tuned)", "Family": "continuous_time_deepsurv", "Top driver": "active_weeks_first_4", "Dominant block": "early_window_behavior"},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Family": "discrete_time_linear", "Top driver": "num__n_vle_rows_week", "Dominant block": "discrete_time_index"},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Family": "discrete_time_neural", "Top driver": "week", "Dominant block": "discrete_time_index"},
        ]
    )


### Funcao: build_tex_calibration_table

Definicao isolada de build_tex_calibration_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_calibration_table():
    return pd.DataFrame(
        [
            {"Model": "Cox Comparable (Tuned)", "Family": "continuous_time_cox", "Calib@10": 0.0433, "Calib@20": 0.0268, "Calib@30": 0.0328},
            {"Model": "DeepSurv (Tuned)", "Family": "continuous_time_deepsurv", "Calib@10": 0.0304, "Calib@20": 0.0160, "Calib@30": 0.0295},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Family": "discrete_time_linear", "Calib@10": 0.0790, "Calib@20": 0.1316, "Calib@30": 0.1542},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Family": "discrete_time_neural", "Calib@10": 0.0647, "Calib@20": 0.1056, "Calib@30": 0.1367},
        ]
    )


### Funcao: build_tex_protocol_audit_table

Definicao isolada de build_tex_protocol_audit_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_protocol_audit_table():
    return pd.DataFrame(
        [
            {"Component": "Evaluation unit", "Value": "enrollment", "Details": "All final benchmark comparisons are performed at the enrollment level."},
            {"Component": "Shared horizons", "Value": "10, 20, 30", "Details": "Common benchmark horizons used for Brier and calibration summaries."},
            {"Component": "Brier / IBS censoring", "Value": "ipcw_km / pycox", "Details": "Brier score and IBS are computed with inverse-probability-of-censoring weighting using the Kaplan-Meier estimator for the censoring distribution through pycox."},
            {"Component": "Primary concordance", "Value": "TD concordance", "Details": "The benchmark co-primary discrimination metric is time-dependent concordance as returned by EvalSurv.concordance_td()."},
            {"Component": "Discrete-time prediction rule", "Value": "dynamic_weekly_updated", "Details": "For prediction at week t, the discrete-time families use only information observed up to week t, and enrollment-level survival is reconstructed from accumulated weekly hazards."},
            {"Component": "Continuous-time prediction rule", "Value": "static_baseline_from_early_window", "Details": "The continuous-time comparable families generate survival curves from fixed enrollment-level representations built from the early observation window only."},
            {"Component": "Identity leakage result", "Value": "enrollment level: none", "Details": "No enrollment identity leakage was detected between train and test."},
            {"Component": "Contextual split scope", "Value": "shared curricular context", "Details": "All modules, presentations, and module-presentation combinations appeared in both splits; the benchmark is therefore not context-disjoint."},
            {"Component": "Calibration metric", "Value": "weighted_mean_absolute_gap_across_bins", "Details": "At each horizon, predicted event risk is grouped into quantile-based bins and calibration error is summarized as the sample-size-weighted mean absolute gap between mean predicted risk and empirical event rate across bins."},
        ]
    )


### Funcao: build_tex_preproc_tuning_table

Definicao isolada de build_tex_preproc_tuning_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_preproc_tuning_table():
    return pd.DataFrame(
        [
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Input level": "person-period weekly", "Preprocessing": "Imputation: median for numeric variables; constant-missing category for categorical variables. Encoding and scaling: one-hot encoding plus standard scaling. Imbalance handling: none (class_weight=None).", "Validation and tuning": "Enrollment-level GroupShuffleSplit (20%). 8 candidates. Selection by val_log_loss. No early stopping."},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Input level": "person-period weekly", "Preprocessing": "Imputation: median for numeric variables; constant-missing category for categorical variables. Encoding and scaling: one-hot encoding plus standard scaling. Imbalance handling: none (unweighted BCE loss).", "Validation and tuning": "Distinct-enrollment train/validation split (10%). 16 candidates. Selection by lowest validation loss. Early stopping used."},
            {"Model": "Cox Comparable (Tuned)", "Input level": "enrollment early window", "Preprocessing": "Imputation: median for numeric variables; constant-missing category for categorical variables. Encoding and scaling: one-hot encoding plus standard scaling. Imbalance handling: none.", "Validation and tuning": "Enrollment-level split with event stratification when possible (20%). 12 candidates. Selection by negative partial log-likelihood. No early stopping."},
            {"Model": "DeepSurv (Tuned)", "Input level": "enrollment early window", "Preprocessing": "Imputation: median for numeric variables; constant-missing category for categorical variables. Encoding and scaling: one-hot encoding plus standard scaling. Imbalance handling: none.", "Validation and tuning": "Internal validation fraction on training rows (20%). 24 candidates. Selection by best validation loss. Early stopping used."},
        ]
    )


### Funcao: build_tex_bootstrap_table

Definicao isolada de build_tex_bootstrap_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_bootstrap_table():
    return pd.DataFrame(
        [
            {"Model": "DeepSurv (Tuned)", "IBS [95% CI]": "0.1110 [0.1080, 0.1160]", "Time-dependent concordance [95% CI]": "0.7300 [0.7200, 0.7410]"},
            {"Model": "Cox Comparable (Tuned)", "IBS [95% CI]": "0.1160 [0.1120, 0.1200]", "Time-dependent concordance [95% CI]": "0.7220 [0.7120, 0.7300]"},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "IBS [95% CI]": "0.1560 [0.1510, 0.1610]", "Time-dependent concordance [95% CI]": "0.6760 [0.6650, 0.6860]"},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "IBS [95% CI]": "0.1620 [0.1570, 0.1680]", "Time-dependent concordance [95% CI]": "0.6580 [0.6490, 0.6680]"},
        ]
    )


### Funcao: build_tex_split_context_table

Definicao isolada de build_tex_split_context_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_split_context_table():
    return pd.DataFrame(
        [
            {
                "Split unit": "enrollment",
                "Stratification": "event status + coarse event-time bucket",
                "Total": 32593,
                "Train": 22815,
                "Test": 9778,
                "Train event rate": 0.2266,
                "Test event rate": 0.2266,
                "Identity leakage": "no",
                "Shared modules": "7/7",
                "Shared presentations": "4/4",
                "Shared module-presentations": "22/22",
            }
        ]
    )


### Funcao: build_tex_cox_ph_summary_table

Definicao isolada de build_tex_cox_ph_summary_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_cox_ph_summary_table():
    return pd.DataFrame(
        [
            {
                "Model": "Cox Comparable (Tuned)",
                "Covariates tested": 41,
                "Flagged": 5,
                "Global interpretation": "Localized departures rather than broad failure of proportional hazards",
            }
        ]
    )


### Funcao: build_tex_cox_ph_audit_table

Definicao isolada de build_tex_cox_ph_audit_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_cox_ph_audit_table():
    return pd.DataFrame(
        [
            {"Narrative rank": 1, "Covariate": "num__active_weeks_first_4", "Source": "tex_narrative"},
            {"Narrative rank": 2, "Covariate": "num__num_of_prev_attempts", "Source": "tex_narrative"},
            {"Narrative rank": 3, "Covariate": "num__mean_clicks_first_4_weeks", "Source": "tex_narrative"},
            {"Narrative rank": 4, "Covariate": "num__clicks_first_4_weeks", "Source": "tex_narrative"},
            {"Narrative rank": 5, "Covariate": "num__studied_credits", "Source": "tex_narrative"},
        ]
    )


### Funcao: extract_tex_contract

Definicao isolada de extract_tex_contract para reutilizacao nas etapas seguintes.


In [ ]:


def extract_tex_contract(tex_source):
    contract = {}
    table_pattern = re.compile(
        r"\\begin\{table\}.*?\\caption\{(?P<caption>.*?)\}.*?\\label\{(?P<label>[^}]+)\}.*?\\end\{table\}",
        flags=re.DOTALL,
    )
    figure_pattern = re.compile(
        r"\\includegraphics(?:\[[^\]]*\])?\{(?P<graphic>[^}]+)\}(?P<middle>.*?)\\(?:caption|captionof\{figure\})\{(?P<caption>.*?)\}(?P<after>.*?)\\label\{(?P<label>[^}]+)\}",
        flags=re.DOTALL,
    )
    for match in table_pattern.finditer(tex_source):
        label = match.group("label").strip()
        contract[label] = {
            "label": label,
            "caption": re.sub(r"\s+", " ", match.group("caption")).strip(),
            "artifact_type": "table",
            "scope": "appendix" if "appendix" in label else "main",
            "graphic_path": "",
            "file_name": "",
        }
    for match in figure_pattern.finditer(tex_source):
        label = match.group("label").strip()
        graphic_path = match.group("graphic").strip()
        file_name = Path(graphic_path).name
        if "." not in file_name:
            file_name = f"{file_name}.png"
        contract[label] = {
            "label": label,
            "caption": re.sub(r"\s+", " ", match.group("caption")).strip(),
            "artifact_type": "figure",
            "scope": "appendix" if "appendix" in label else "main",
            "graphic_path": graphic_path,
            "file_name": file_name,
        }
    return contract


In [ ]:


tex_source = TEX_PATH.read_text(encoding="utf-8") if TEX_PATH.exists() else ""
parsed_tex_contract = extract_tex_contract(tex_source)
required_tex_labels = [
    "tab:main_benchmark",
    "fig:benchmark_tuned_comparison",
    "tab:ablation_summary",
    "fig:ablation_impact",
    "tab:explainability_summary",
    "fig:explainability_block_dominance",
    "tab:calibration_summary",
    "tab:appendix_protocol_audit",
    "tab:appendix_preproc_tuning_audit",
    "tab:appendix_bootstrap_uncertainty",
    "tab:appendix_split_context_audit",
    "tab:appendix_cox_ph_summary",
    "fig:appendix_cox_ph_diagnostics",
]

tex_contract = {}
for label in required_tex_labels:
    fallback_file_name = f"{label.replace(':', '_')}.csv"
    tex_contract[label] = parsed_tex_contract.get(
        label,
        {
            "label": label,
            "caption": label,
            "artifact_type": "figure" if label.startswith("fig:") else "table",
            "scope": "appendix" if "appendix" in label else "main",
            "graphic_path": "",
            "file_name": fallback_file_name,
        },
    )

tex_asset_rows = []
supporting_asset_rows = []
manifest_rows = []

leaderboard_source = resolve_first_existing([TABLES_DIR / "table_benchmark_leaderboard_main.csv"])
brier_wide_source = resolve_first_existing([TABLES_DIR / "table_benchmark_brier_by_horizon_wide.csv"])
calibration_wide_source = resolve_first_existing([TABLES_DIR / "table_benchmark_calibration_by_horizon_wide.csv"])

benchmark_paper_df = None
benchmark_figure_df = None
if leaderboard_source is not None and brier_wide_source is not None:
    leaderboard_df = pd.read_csv(leaderboard_source)
    brier_wide_df = pd.read_csv(brier_wide_source)
    leaderboard_model_col = pick_column(leaderboard_df, ["display_name", "Model", "model", "model_name"])
    leaderboard_family_col = pick_column(leaderboard_df, ["family", "Family", "model_family"])
    ibs_col = pick_column(leaderboard_df, ["ibs", "IBS", "integrated_brier_score"])
    cindex_col = pick_column(leaderboard_df, ["c_index", "TD Concordance", "td_concordance", "concordance_td"])
    brier_model_col = pick_column(brier_wide_df, ["display_name", "Model", "model", "model_name"])
    benchmark_paper_df = leaderboard_df[[leaderboard_model_col, leaderboard_family_col, ibs_col, cindex_col]].copy()
    benchmark_paper_df.columns = ["Model", "Family", "IBS", "TD Concordance"]
    benchmark_paper_df["Model"] = benchmark_paper_df["Model"].map(normalize_model_names)
    benchmark_paper_df["Family"] = benchmark_paper_df["Family"].map(normalize_family_names)
    for horizon in [10, 20, 30]:
        brier_column = pick_column(
            brier_wide_df,
            [f"brier_h{horizon}", f"Brier@{horizon}", f"brier@{horizon}", f"brier_{horizon}", f"brier_at_{horizon}"],
        )
        brier_lookup = brier_wide_df[[brier_model_col, brier_column]].copy()
        brier_lookup.columns = ["Model", f"Brier@{horizon}"]
        brier_lookup["Model"] = brier_lookup["Model"].map(normalize_model_names)
        benchmark_paper_df = benchmark_paper_df.merge(brier_lookup, on="Model", how="left")
    benchmark_paper_df = benchmark_paper_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
    benchmark_figure_df = benchmark_paper_df[["Model", "IBS", "TD Concordance"]].copy()
else:
    benchmark_paper_df = build_tex_main_benchmark_table()
    benchmark_figure_df = benchmark_paper_df[["Model", "IBS", "TD Concordance"]].copy()

benchmark_table_output = PAPER_MAIN_DIR / "table_paper_main_benchmark_leaderboard.csv"
export_direct_table_asset(
    "tab:main_benchmark",
    benchmark_table_output,
    dataframe=benchmark_paper_df,
    note="Curated benchmark table exported under the manuscript contract."
)
benchmark_support_output = PAPER_MAIN_DIR / "table_figure1_benchmark_tuned_summary.csv"
export_supporting_table_asset(
    "fig:benchmark_tuned_comparison",
    benchmark_support_output,
    dataframe=benchmark_figure_df,
    note="Support table for the benchmark comparison figure."
)


### Funcao: generate_benchmark_figure

Definicao isolada de generate_benchmark_figure para reutilizacao nas etapas seguintes.


In [ ]:


def generate_benchmark_figure(output_path):
    if not MATPLOTLIB_AVAILABLE or benchmark_figure_df is None:
        return False
    plot_df = benchmark_figure_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key), ascending=False)
    figure, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    axes[0].barh(plot_df["Model"], plot_df["IBS"], color="#2a9d8f")
    axes[0].set_title("Lower is better")
    axes[0].set_xlabel("Integrated Brier Score")
    axes[1].barh(plot_df["Model"], plot_df["TD Concordance"], color="#264653")
    axes[1].set_title("Higher is better")
    axes[1].set_xlabel("TD concordance")
    figure.suptitle("Tuned benchmark comparison", fontsize=18)
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(figure)
    return True


In [ ]:


export_direct_figure_asset(
    "fig:benchmark_tuned_comparison",
    PAPER_MAIN_DIR / tex_contract["fig:benchmark_tuned_comparison"]["file_name"],
    generator=generate_benchmark_figure,
    note="Generated from the benchmark table frozen for the manuscript."
)

ablation_source = resolve_first_existing([TABLES_DIR / "table_ablation_summary_by_model.csv", TABLES_DIR / "table_p31_paper_ablation_summary.csv"])
ablation_table_df = None
if ablation_source is not None:
    ablation_raw_df = pd.read_csv(ablation_source)
    model_col = pick_column(ablation_raw_df, ["display_name", "Model", "model", "model_name"])
    family_col = pick_column(ablation_raw_df, ["family", "Family", "model_family"])
    delta_ibs_static_col = pick_column(ablation_raw_df, ["Delta IBS static", "delta_ibs_static", "ibs_delta_static", "delta_ibs_without_static"])
    delta_ibs_temporal_col = pick_column(ablation_raw_df, ["Delta IBS temporal", "delta_ibs_temporal", "ibs_delta_temporal", "delta_ibs_without_temporal"])
    delta_td_static_col = pick_column(ablation_raw_df, ["Delta TD concordance static", "delta_td_concordance_static", "delta_cindex_static"])
    delta_td_temporal_col = pick_column(ablation_raw_df, ["Delta TD concordance temporal", "delta_td_concordance_temporal", "delta_cindex_temporal"])
    ibs_ratio_col = pick_column(ablation_raw_df, ["IBS ratio", "ibs_ratio", "temporal_static_ibs_ratio"])
    ablation_table_df = ablation_raw_df[[model_col, family_col, delta_ibs_static_col, delta_ibs_temporal_col, delta_td_static_col, delta_td_temporal_col, ibs_ratio_col]].copy()
    ablation_table_df.columns = ["Model", "Family", "Delta IBS static", "Delta IBS temporal", "Delta TD concordance static", "Delta TD concordance temporal", "IBS ratio"]
    ablation_table_df["Model"] = ablation_table_df["Model"].map(normalize_model_names)
    ablation_table_df["Family"] = ablation_table_df["Family"].map(normalize_family_names)
    ablation_table_df = ablation_table_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
else:
    ablation_table_df = build_tex_ablation_table()

export_direct_table_asset(
    "tab:ablation_summary",
    PAPER_MAIN_DIR / "table_paper_ablation_summary.csv",
    source_candidates=[ablation_source] if ablation_source is not None else [],
    dataframe=ablation_table_df,
    note="Curated ablation table exported under the manuscript contract."
)
export_supporting_table_asset(
    "fig:ablation_impact",
    PAPER_MAIN_DIR / "table_figure2_ablation_delta_summary.csv",
    dataframe=ablation_table_df,
    note="Support table for the ablation impact figure."
)


### Funcao: generate_ablation_figure

Definicao isolada de generate_ablation_figure para reutilizacao nas etapas seguintes.


In [ ]:


def generate_ablation_figure(output_path):
    if not MATPLOTLIB_AVAILABLE or ablation_table_df is None:
        return False
    plot_df = ablation_table_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key), ascending=False)
    figure, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    axes[0].barh(plot_df["Model"], plot_df["Delta IBS static"], color="#8ecae6", label="Remove static")
    axes[0].barh(plot_df["Model"], plot_df["Delta IBS temporal"], color="#219ebc", alpha=0.85, label="Remove temporal")
    axes[0].set_title("IBS loss")
    axes[0].set_xlabel("Delta IBS")
    axes[0].legend(loc="lower right")
    axes[1].barh(plot_df["Model"], plot_df["Delta TD concordance static"].abs(), color="#ffb703", label="Remove static")
    axes[1].barh(plot_df["Model"], plot_df["Delta TD concordance temporal"].abs(), color="#fb8500", alpha=0.85, label="Remove temporal")
    axes[1].set_title("TD concordance loss")
    axes[1].set_xlabel("Absolute delta TD concordance")
    axes[1].legend(loc="lower right")
    figure.suptitle("Ablation impact across tuned model families", fontsize=18)
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(figure)
    return True


In [ ]:


export_direct_figure_asset(
    "fig:ablation_impact",
    PAPER_MAIN_DIR / tex_contract["fig:ablation_impact"]["file_name"],
    generator=generate_ablation_figure,
    note="Generated from the ablation table frozen for the manuscript."
)

explainability_summary_source = resolve_first_existing([TABLES_DIR / "table_explainability_summary_by_model.csv", TABLES_DIR / "table_p37_explainability_summary_by_model.csv"])
explainability_summary_df = None
if explainability_summary_source is not None:
    explainability_raw_df = pd.read_csv(explainability_summary_source)
    model_col = pick_column(explainability_raw_df, ["display_name", "Model", "model", "model_name"])
    family_col = pick_column(explainability_raw_df, ["family", "Family", "model_family"])
    top_driver_col = pick_column(explainability_raw_df, ["Top driver", "top_driver", "top_feature"])
    dominant_block_col = pick_column(explainability_raw_df, ["Dominant block", "dominant_block", "top_block"])
    explainability_summary_df = explainability_raw_df[[model_col, family_col, top_driver_col, dominant_block_col]].copy()
    explainability_summary_df.columns = ["Model", "Family", "Top driver", "Dominant block"]
    explainability_summary_df["Model"] = explainability_summary_df["Model"].map(normalize_model_names)
    explainability_summary_df["Family"] = explainability_summary_df["Family"].map(normalize_family_names)
    explainability_summary_df = explainability_summary_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
else:
    explainability_summary_df = build_tex_explainability_summary_table()

export_direct_table_asset(
    "tab:explainability_summary",
    PAPER_MAIN_DIR / "table_paper_explainability_summary.csv",
    source_candidates=[explainability_summary_source] if explainability_summary_source is not None else [],
    dataframe=explainability_summary_df,
    note="Curated explainability table exported under the manuscript contract."
)

block_source = resolve_first_existing([TABLES_DIR / "table_explainability_all_blocks_normalized.csv"])
block_plot_wide_df = None
explainability_figure_note = "Generated from normalized explainability block outputs when available."
if block_source is not None:
    all_blocks_df = pd.read_csv(block_source)
    model_col = pick_column(all_blocks_df, ["display_name", "Model", "model", "model_name"])
    block_col = pick_column(all_blocks_df, ["Block", "block", "feature_block", "dominant_block"])
    value_col = pick_column(all_blocks_df, ["Normalized importance", "normalized_importance", "normalized_value", "value", "importance"])
    block_long_df = all_blocks_df[[model_col, block_col, value_col]].copy()
    block_long_df.columns = ["Model", "Block", "Normalized value"]
    block_long_df["Model"] = block_long_df["Model"].map(normalize_model_names)
    block_long_df["Block"] = block_long_df["Block"].astype(str)
    block_plot_wide_df = (
        block_long_df
        .pivot_table(index="Model", columns="Block", values="Normalized value", aggfunc="mean", fill_value=0.0)
        .reset_index()
    )
    block_plot_wide_df = block_plot_wide_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
    export_supporting_table_asset(
        "fig:explainability_block_dominance",
        PAPER_MAIN_DIR / "table_figure3_explainability_block_summary_normalized.csv",
        dataframe=block_plot_wide_df,
        note="Support table derived from normalized explainability block outputs."
    )
else:
    block_fallback_long_df = explainability_summary_df[["Model", "Dominant block"]].copy()
    block_fallback_long_df["Normalized value"] = 1.0
    block_plot_wide_df = (
        block_fallback_long_df
        .pivot_table(index="Model", columns="Dominant block", values="Normalized value", aggfunc="max", fill_value=0.0)
        .reset_index()
    )
    block_plot_wide_df = block_plot_wide_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
    export_supporting_table_asset(
        "fig:explainability_block_dominance",
        PAPER_MAIN_DIR / "table_figure3_explainability_block_summary_normalized.csv",
        dataframe=block_plot_wide_df,
        note="Fallback support table reconstructed from the dominant block reported in the explainability summary because the detailed normalized block output is not materialized."
    )
    explainability_figure_note = "Generated from the explainability dominant-block summary because table_explainability_all_blocks_normalized.csv is not currently materialized."


### Funcao: generate_explainability_figure

Definicao isolada de generate_explainability_figure para reutilizacao nas etapas seguintes.


In [ ]:


def generate_explainability_figure(output_path):
    if not MATPLOTLIB_AVAILABLE or block_plot_wide_df is None:
        return False
    plot_df = block_plot_wide_df.copy()
    block_columns = [column for column in plot_df.columns if column != "Model"]
    if not block_columns:
        return False
    x_positions = list(range(len(plot_df)))
    width = 0.8 / max(len(block_columns), 1)
    palette = ["#4c956c", "#2c6e49", "#f4a259", "#5b8e7d", "#bc4749", "#577590"]
    figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
    for index, column in enumerate(block_columns):
        centered_positions = [x + (index - (len(block_columns) - 1) / 2.0) * width for x in x_positions]
        axis.bar(centered_positions, plot_df[column].values, width=width, label=str(column).replace("_", " "), color=palette[index % len(palette)])
    axis.set_xticks(x_positions)
    axis.set_xticklabels(plot_df["Model"], rotation=20, ha="right")
    axis.set_ylabel("Normalized dominance")
    axis.set_title("Normalized explainability block dominance across tuned families")
    axis.set_ylim(0, max(1.05, float(plot_df[block_columns].max().max()) * 1.15))
    axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=min(3, len(block_columns)), frameon=False)
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(figure)
    return True


In [ ]:


export_direct_figure_asset(
    "fig:explainability_block_dominance",
    PAPER_MAIN_DIR / tex_contract["fig:explainability_block_dominance"]["file_name"],
    generator=generate_explainability_figure,
    note=explainability_figure_note,
    missing_note="Neither the normalized block output nor the dominant-block summary was available to reconstruct the explainability figure."
)

calibration_paper_df = None
if calibration_wide_source is not None:
    calibration_wide_df = pd.read_csv(calibration_wide_source)
    model_col = pick_column(calibration_wide_df, ["display_name", "Model", "model", "model_name"])
    family_col = pick_column(calibration_wide_df, ["family", "Family", "model_family"])
    calibration_paper_df = calibration_wide_df[[
        model_col,
        family_col,
        pick_column(calibration_wide_df, ["calibration_h10", "Calib@10", "calib@10", "calibration@10", "calibration_10"]),
        pick_column(calibration_wide_df, ["calibration_h20", "Calib@20", "calib@20", "calibration@20", "calibration_20"]),
        pick_column(calibration_wide_df, ["calibration_h30", "Calib@30", "calib@30", "calibration@30", "calibration_30"]),
    ]].copy()
    calibration_paper_df.columns = ["Model", "Family", "Calib@10", "Calib@20", "Calib@30"]
    calibration_paper_df["Model"] = calibration_paper_df["Model"].map(normalize_model_names)
    calibration_paper_df["Family"] = calibration_paper_df["Family"].map(normalize_family_names)
    calibration_paper_df = calibration_paper_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
else:
    calibration_paper_df = build_tex_calibration_table()

export_direct_table_asset(
    "tab:calibration_summary",
    PAPER_MAIN_DIR / "table_paper_calibration_summary_tuned_models.csv",
    source_candidates=[calibration_wide_source] if calibration_wide_source is not None else [],
    dataframe=calibration_paper_df,
    note="Curated calibration table exported under the manuscript contract."
)

appendix_protocol_source = resolve_first_existing([TABLES_DIR / "table_evaluation_protocol_audit.csv"])
appendix_protocol_df = pd.read_csv(appendix_protocol_source) if appendix_protocol_source is not None else build_tex_protocol_audit_table()
export_direct_table_asset(
    "tab:appendix_protocol_audit",
    PAPER_APPENDIX_DIR / "table_paper_appendix_evaluation_protocol_audit.csv",
    source_candidates=[appendix_protocol_source] if appendix_protocol_source is not None else [],
    dataframe=appendix_protocol_df,
    note="Appendix protocol audit exported under the manuscript contract."
)

preproc_source_candidates = [TABLES_DIR / "table_preprocessing_and_tuning_audit.csv", TABLES_DIR / "table_paper_appendix_preprocessing_and_tuning_audit.csv"]
preproc_source = resolve_first_existing(preproc_source_candidates)
preproc_df = pd.read_csv(preproc_source) if preproc_source is not None else build_tex_preproc_tuning_table()
export_direct_table_asset(
    "tab:appendix_preproc_tuning_audit",
    PAPER_APPENDIX_DIR / "table_paper_appendix_preprocessing_and_tuning_audit.csv",
    source_candidates=preproc_source_candidates,
    dataframe=preproc_df,
    note="Appendix preprocessing and tuning audit exported under the manuscript contract." if preproc_source is not None else "Fallback reconstructed from the inline TeX appendix table."
)

bootstrap_source_candidates = [TABLES_DIR / "table_appendix_bootstrap_uncertainty_compact.csv"]
bootstrap_source = resolve_first_existing(bootstrap_source_candidates)
bootstrap_df = pd.read_csv(bootstrap_source) if bootstrap_source is not None else build_tex_bootstrap_table()
export_direct_table_asset(
    "tab:appendix_bootstrap_uncertainty",
    PAPER_APPENDIX_DIR / "table_appendix_bootstrap_uncertainty_compact.csv",
    source_candidates=bootstrap_source_candidates,
    dataframe=bootstrap_df,
    note="Appendix bootstrap uncertainty table exported under the manuscript contract." if bootstrap_source is not None else "Fallback reconstructed from the inline TeX appendix table."
)

split_context_source_candidates = [
    TABLES_DIR / "paper_appendix" / "table_paper_appendix_split_context_audit.csv",
    TABLES_DIR / "table_paper_appendix_split_context_audit.csv",
    TABLES_DIR / "table_appendix_split_context_audit_compact.csv",
    TABLES_DIR / "table_split_context_appendix.csv",
]
split_context_source = resolve_first_existing(split_context_source_candidates)
split_context_df = pd.read_csv(split_context_source) if split_context_source is not None else build_tex_split_context_table()
export_direct_table_asset(
    "tab:appendix_split_context_audit",
    PAPER_APPENDIX_DIR / "table_paper_appendix_split_context_audit.csv",
    source_candidates=split_context_source_candidates,
    dataframe=split_context_df,
    note="Appendix split/context audit exported under the manuscript contract." if split_context_source is not None else "Fallback reconstructed from the inline TeX appendix table."
)

cox_summary_source_candidates = [
    TABLES_DIR / "paper_appendix" / "table_paper_appendix_cox_ph_global_summary.csv",
    TABLES_DIR / "table_paper_appendix_cox_ph_global_summary.csv",
    TABLES_DIR / "table_appendix_cox_ph_global_summary.csv",
    TABLES_DIR / "table_cox_ph_global_summary.csv",
]
cox_summary_source = resolve_first_existing(cox_summary_source_candidates)
cox_summary_df = pd.read_csv(cox_summary_source) if cox_summary_source is not None else build_tex_cox_ph_summary_table()
export_direct_table_asset(
    "tab:appendix_cox_ph_summary",
    PAPER_APPENDIX_DIR / "table_paper_appendix_cox_ph_global_summary.csv",
    source_candidates=cox_summary_source_candidates,
    dataframe=cox_summary_df,
    note="Appendix Cox PH global summary exported under the manuscript contract." if cox_summary_source is not None else "Fallback reconstructed from the inline TeX appendix table."
)

cox_audit_source_candidates = [
    TABLES_DIR / "paper_appendix" / "table_paper_appendix_cox_ph_audit.csv",
    TABLES_DIR / "table_paper_appendix_cox_ph_audit.csv",
    TABLES_DIR / "table_appendix_cox_ph_audit.csv",
    TABLES_DIR / "table_cox_ph_assumption_audit.csv",
]
cox_audit_source = resolve_first_existing(cox_audit_source_candidates)
cox_ph_audit_df = pd.read_csv(cox_audit_source) if cox_audit_source is not None else build_tex_cox_ph_audit_table()
export_supporting_table_asset(
    "fig:appendix_cox_ph_diagnostics",
    PAPER_APPENDIX_DIR / "table_paper_appendix_cox_ph_audit.csv",
    source_candidates=cox_audit_source_candidates,
    dataframe=cox_ph_audit_df,
    note="Supporting PH audit table exported under the manuscript contract." if cox_audit_source is not None else "Fallback reconstructed from the manuscript narrative listing the flagged covariates."
)


### Funcao: generate_appendix_ph_figure

Definicao isolada de generate_appendix_ph_figure para reutilizacao nas etapas seguintes.


In [ ]:


def generate_appendix_ph_figure(output_path):
    if not MATPLOTLIB_AVAILABLE or cox_ph_audit_df is None:
        return False
    figure, axis = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
    axis.axis("off")
    axis.text(0.01, 0.94, "Comparable Cox PH diagnostic summary", fontsize=18, fontweight="bold", va="top")
    axis.text(
        0.01,
        0.84,
        "Covariates with the strongest evidence of possible non-proportionality",
        fontsize=12,
        color="#264653",
        va="top",
    )
    display_df = cox_ph_audit_df.copy()
    if "Narrative rank" in display_df.columns and "Covariate" in display_df.columns:
        display_df = display_df.sort_values("Narrative rank")
        lines = [f"{int(row['Narrative rank'])}. {row['Covariate']}" for _, row in display_df.iterrows()]
        footer = "Narrative fallback generated from the manuscript because the quantitative PH audit table is not currently materialized in outputs_benchmark_survival/tables."
    else:
        covariate_col = pick_column(display_df, ["Covariate", "covariate", "feature", "variable"])
        lines = [f"- {value}" for value in display_df[covariate_col].astype(str).head(5)]
        footer = "Generated from the notebook PH audit table."
    y_position = 0.72
    for line in lines:
        axis.text(0.03, y_position, line, fontsize=12, va="top")
        y_position -= 0.10
    axis.text(
        0.01,
        0.10,
        "Global interpretation: localized departures rather than broad failure of proportional hazards.",
        fontsize=11,
        color="#6b705c",
        va="top",
    )
    axis.text(0.01, 0.03, footer, fontsize=9.5, color="#666666", va="bottom")
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(figure)
    return True


In [ ]:


export_direct_figure_asset(
    "fig:appendix_cox_ph_diagnostics",
    PAPER_APPENDIX_DIR / tex_contract["fig:appendix_cox_ph_diagnostics"]["file_name"],
    source_candidates=[
        FIGURES_DIR / tex_contract["fig:appendix_cox_ph_diagnostics"]["file_name"],
        FIGURES_DIR / "paper_appendix" / tex_contract["fig:appendix_cox_ph_diagnostics"]["file_name"],
    ],
    generator=generate_appendix_ph_figure,
    note="Generated under the manuscript contract; falls back to a narrative diagnostic summary when the quantitative PH figure is unavailable."
)

for asset_row in tex_asset_rows:
    manifest_rows.append(
        {
            "tex_label": asset_row["tex_label"],
            "scope": asset_row["scope"],
            "artifact_type": asset_row["artifact_type"],
            "status": asset_row["status"],
            "file_name": asset_row["file_name"],
            "output_path": asset_row["output_path"],
        }
    )
for supporting_row in supporting_asset_rows:
    manifest_rows.append(
        {
            "tex_label": supporting_row["related_tex_label"],
            "scope": supporting_row["scope"],
            "artifact_type": supporting_row["artifact_type"],
            "status": supporting_row["status"],
            "file_name": supporting_row["file_name"],
            "output_path": supporting_row["output_path"],
        }
    )

tex_asset_registry_df = pd.DataFrame(tex_asset_rows).sort_values(["scope", "artifact_type", "tex_label"]).reset_index(drop=True)
supporting_asset_registry_df = pd.DataFrame(supporting_asset_rows).sort_values(["scope", "related_tex_label"]).reset_index(drop=True)
asset_manifest_df = pd.DataFrame(manifest_rows).sort_values(["scope", "artifact_type", "tex_label"]).reset_index(drop=True)

tex_asset_registry_path = METADATA_DIR / "paper_tex_asset_registry.csv"
supporting_registry_path = METADATA_DIR / "paper_tex_supporting_asset_registry.csv"
asset_manifest_path = METADATA_DIR / "paper_curated_asset_manifest.csv"
freeze_summary_path = METADATA_DIR / "paper_freeze_summary.json"

tex_asset_registry_df.to_csv(tex_asset_registry_path, index=False)
supporting_asset_registry_df.to_csv(supporting_registry_path, index=False)
asset_manifest_df.to_csv(asset_manifest_path, index=False)

freeze_summary = {
    "section_id": "G7",
    "contract_source": str(TEX_PATH),
    "paper_main_dir": str(PAPER_MAIN_DIR),
    "paper_appendix_dir": str(PAPER_APPENDIX_DIR),
    "n_expected_tex_assets": int(len(tex_asset_registry_df)),
    "n_exported_tex_assets": int((tex_asset_registry_df["status"] == "exported").sum()),
    "n_missing_tex_assets": int((tex_asset_registry_df["status"] == "missing").sum()),
    "n_exported_supporting_assets": int((supporting_asset_registry_df["status"] == "exported").sum()),
}
with open(freeze_summary_path, "w", encoding="utf-8") as file_handle:
    json.dump(freeze_summary, file_handle, indent=2)

print("Curated TeX output directories:")
print("-", PAPER_MAIN_DIR.resolve())
print("-", PAPER_APPENDIX_DIR.resolve())
print()
print("Direct TeX-linked assets:")
display(tex_asset_registry_df)
print("\nSupporting assets for TeX figures/tables:")
display(supporting_asset_registry_df)
missing_tex_assets_df = tex_asset_registry_df.loc[tex_asset_registry_df["status"] == "missing", ["tex_label", "file_name", "notes"]]
if not missing_tex_assets_df.empty:
    print("\nAssets still missing from the dedicated TeX export layer:")
    display(missing_tex_assets_df)
else:
    print("\nAll TeX-linked assets were materialized.")
print("\nSaved metadata files:")
print("-", tex_asset_registry_path.resolve())
print("-", supporting_registry_path.resolve())
print("-", asset_manifest_path.resolve())
print("-", freeze_summary_path.resolve())


# G8 — Display Curated Paper Figures

In [ ]:
# ==============================================================
# G8 — Display Curated Paper Figures
# --------------------------------------------------------------
# Purpose:
#   Load and display the curated TeX-facing figures exported in G7
#   directly inside the notebook for visual inspection.
#
# Methodological note:
#   This step is display-only.
#   No model is trained and no metric is recomputed.
# ==============================================================

print("\n" + "=" * 70)
print("G8 — Display Curated Paper Figures")
print("=" * 70)
print("Methodological note: this step displays curated TeX-facing figures only.")

required_names = ["OUTPUT_DIR", "METADATA_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from IPython.display import Image, Markdown, display
import pandas as pd

registry_path = METADATA_DIR / "paper_tex_asset_registry.csv"
if not registry_path.exists():
    raise FileNotFoundError(
        f"TeX asset registry not found: {registry_path}. Please run Cell 136 first."
    )

registry_df = pd.read_csv(registry_path)
figure_registry_df = registry_df[
    (registry_df["artifact_type"] == "figure") & (registry_df["status"] == "exported")
].copy()

if figure_registry_df.empty:
    print("No exported TeX-facing figures are currently available.")
else:
    for _, row in figure_registry_df.sort_values(["scope", "tex_label"]).iterrows():
        display(Markdown(f"## {row['tex_label']}"))
        display(Markdown(row["tex_caption"]))
        display(Image(filename=row["output_path"]))
        print("-", Path(row["output_path"]).resolve())

missing_figures_df = registry_df[
    (registry_df["artifact_type"] == "figure") & (registry_df["status"] != "exported")
].copy()
if not missing_figures_df.empty:
    print("\nFigures still missing from the TeX export layer:")
    display(missing_figures_df[["tex_label", "file_name", "notes"]])

# G9 — Preview Curated Paper Evidence

This section previews the manuscript-facing evidence directly from the curated `paper_main` and `paper_appendix` directories created in G7.

In [ ]:
# ==============================================================
# G9 — Preview Curated Paper Evidence
# --------------------------------------------------------------
# Purpose:
#   Preview the manuscript-facing evidence directly from the curated
#   paper_main and paper_appendix directories created in G7.
#
# Methodological note:
#   This step is synthesis-only.
#   No model is trained and no metric is recomputed.
# ==============================================================

print("\n" + "=" * 70)
print("G9 — Preview Curated Paper Evidence")
print("=" * 70)
print("Methodological note: this step reads curated paper artifacts only.")

required_names = ["OUTPUT_DIR", "METADATA_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd
from IPython.display import display

PAPER_MAIN_DIR = OUTPUT_DIR / "paper_main"
PAPER_APPENDIX_DIR = OUTPUT_DIR / "paper_appendix"
tex_registry_path = METADATA_DIR / "paper_tex_asset_registry.csv"
supporting_registry_path = METADATA_DIR / "paper_tex_supporting_asset_registry.csv"

if not tex_registry_path.exists():
    raise FileNotFoundError(
        f"TeX asset registry not found: {tex_registry_path}. Please run Cell 136 first."
    )

tex_registry_df = pd.read_csv(tex_registry_path)
supporting_registry_df = pd.read_csv(supporting_registry_path) if supporting_registry_path.exists() else pd.DataFrame()

print("\nDirect TeX-linked assets:")
display(tex_registry_df)

if not supporting_registry_df.empty:
    print("\nSupporting assets for TeX figures/tables:")
    display(supporting_registry_df)

exported_main_tables_df = tex_registry_df[
    (tex_registry_df["scope"] == "main")
    & (tex_registry_df["artifact_type"] == "table")
    & (tex_registry_df["status"] == "exported")
].copy()

exported_appendix_tables_df = tex_registry_df[
    (tex_registry_df["scope"] == "appendix")
    & (tex_registry_df["artifact_type"] == "table")
    & (tex_registry_df["status"] == "exported")
].copy()

print("\nPreview of exported main-paper tables:")
if exported_main_tables_df.empty:
    print("- No main-paper tables are currently materialized.")
else:
    for _, row in exported_main_tables_df.sort_values("tex_label").iterrows():
        preview_df = pd.read_csv(row["output_path"])
        print(f"- {row['tex_label']} -> {Path(row['output_path']).name} ({preview_df.shape[0]} rows)")
        display(preview_df)

print("\nPreview of exported appendix tables:")
if exported_appendix_tables_df.empty:
    print("- No appendix tables are currently materialized.")
else:
    for _, row in exported_appendix_tables_df.sort_values("tex_label").iterrows():
        preview_df = pd.read_csv(row["output_path"])
        print(f"- {row['tex_label']} -> {Path(row['output_path']).name} ({preview_df.shape[0]} rows)")
        display(preview_df)

missing_assets_df = tex_registry_df[tex_registry_df["status"] != "exported"].copy()
print("\nAssets still missing from the dedicated TeX export layer:")
if missing_assets_df.empty:
    print("- None. All TeX-linked assets are materialized.")
else:
    display(missing_assets_df[["tex_label", "artifact_type", "file_name", "notes"]])

print("\nExclusive TeX output directories:")
print("-", PAPER_MAIN_DIR.resolve())
print("-", PAPER_APPENDIX_DIR.resolve())